# Recourse under competition — consolidated experiments

One numerical implementation, the five paper methods, and all experiments/follow-ups from the seven supplied notebooks. Start at the top and run the setup once. Each study below has an independent dataset/model configuration and a guarded execution cell.

**Methods (fixed order/colors):** No action; Original threshold; Common target; Personalized target without selection; Personalized target with selection (ours). Common target uses **30 equally spaced scores**, with no cost-based recipient search. The separate grid-resolution diagnostic deliberately tests larger common-target grids. All learned methods choose their checkpoint and, where applicable, recommendation cutoff using validation only. The all-eligible method fixes recommendation to every feasible eligible applicant.

**Paper settings:** five seeds 42–46; affine and concave-quadratic logits; $B=0.75$, $\alpha=0.4$, $\tau=0.01$, $\eta=0.2$, $T=500$. Main budget grid: `[0.25, 0.75, 1.5]`; admission grid: `[0.2, 0.4, 0.6]`; temperature grid: `[0.005, 0.01, 0.02, 0.05]` at $\lambda=30$, $\eta=0.05$. The trade-off sweep has 14 values from 0.3 to 30. The target/group follow-up retains its own $\lambda\in\{1,3,10\}$ protocol. Sharper-temperature stress tests are explicitly separate follow-ups.

| ID | Study | Dependencies |
|---|---|---|
| 1–8 | Convergence; trade-offs; budget; admission; temperature; ablations; numerical/model audit; partial adoption | None |
| 2a | Validation-cohort transfer, cost decomposition, pooled candidates | Saved experiment 2 and caches |
| 2b | Joint target/recommendation-cutoff calibration | Saved experiment 2 and caches |
| 9–10 | Focused temperature/step-size diagnostics and validation-selected trade-offs | None |
| 11 | Common-target grid resolution (30 / 100 / 300) | Saved experiment 10 |
| 12 | Fresh synthetic cohort transfer | Saved 10 and 11; synthetic datasets only |
| 13 | Shared-validation policy-family selection | Saved 10–12; synthetic datasets only |
| 14–15 | Budget/admission follow-ups with repeated fresh synthetic cohorts | None; real data retain one held-out cohort |
| 16 | Full / all-eligible / detached-threshold multi-cohort comparison | None |
| 17 | Targets, recipient selection, paired target comparison, group audit | None |
| 18 | Personalized candidates at the common target's validation cost | Completed 17 and its candidate/scorer caches |
| diagnostics | Inspect existing saved main/focused trials without retraining | Saved trial context manifests |

**Execution:** `run` resumes compatible saved results; `preflight` performs a small/check-only diagnostic; `load` reads a completed matching study and regenerates reports; `skip` does nothing. Each source-only follow-up reports missing inputs instead of starting hidden training. Dependencies are never run automatically. Source settings in studies 11–13 are independent snapshots; if you edit a source study after this setup, also update the corresponding `source_config`/`source_refinement`/`source_transfer` entry or point to its saved ID.

**Outputs:** results go under `./Results`, figures under `./figures`. Scientific settings and implementation hashes distinguish changed parameters. Earlier files are not overwritten or relabeled. Tables-only historical results can be inspected with `inventory_saved_studies`; replay requires the original `Results/cache` and policy files. Old results using retired baselines remain historical records, not outputs of the new comparison.

**Interpretation:** the scorer and immutable coordinates remain fixed. Validation/test pools are separate. Repeated real-data cohorts are resamples of held-out pools, not independent new datasets. Seed-level summaries are the uncertainty units. Distribution plots in study 17 use the predeclared first seed. Mutable categorical/integer features retain the existing continuous relaxation.

Dependencies: Python 3.10+, NumPy, pandas, SciPy, scikit-learn >=1.2, PyTorch >=2.0, matplotlib, IPython, threadpoolctl; `folktables` for ACSIncome. Supply `GiveMeSomeCredit.csv` locally. MLP scorer support is retained for exploratory use, but no paper configuration selects it.


In [ ]:
# One control panel. Modes remain "run" unless overridden for a particular study.
RUN_MODE = "run"
experiments_to_run = [1]  # e.g. [2, "2a", "2b", 17]; dependencies are explicit.
default_selected_pairs = [("German Credit", "affine")]
# To run every dataset/scorer pair, set default_selected_pairs = ALL_PAIRS below.
RESULTS_DIR = "./Results"
FIGURES_DIR = "./figures"
GMSC_PATH = "GiveMeSomeCredit.csv"
SHOW_FIGURES = True
DATASETS = ("German Credit", "Adult", "ACSIncome", "Give Me Some Credit",
            "Synthetic nonlinear", "Synthetic curved")
MODELS = ("affine", "quadratic")
ALL_PAIRS = [(dataset, model) for dataset in DATASETS for model in MODELS]
PAPER_SEEDS = [42, 43, 44, 45, 46]
PAPER_LAMBDAS = [0.3,0.5,0.75,1.0,1.5,2.0,3.0,4.0,5.0,7.5,10.0,15.0,20.0,30.0]
RHO_CUTOFFS = [0.05,0.1,0.25,0.5,0.75,0.9,0.95]


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import copy
import math
import time
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from scipy.optimize import minimize, root
from collections import OrderedDict
from sklearn.metrics import roc_auc_score, log_loss
from scipy.special import expit
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt
from IPython.display import display

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def as_tensor(X):
    return torch.as_tensor(X, dtype=torch.float64, device="cpu")


def generate_population_nd(n_samples, w_star, noise_std=0.0, seed=42):
    rng = np.random.default_rng(seed)
    w_star = np.asarray(w_star, dtype=float)
    X = rng.normal(size=(n_samples, len(w_star)))
    X += rng.normal(scale=noise_std, size=X.shape)
    return X, expit(X @ w_star)


def generate_correlated_nonlinear_population(
    n_samples, correlation=0.8, d=3, noise_std=0.0, seed=42
):
    if d < 2 or not (-1 / (d - 1) < correlation < 1):
        raise ValueError("The correlation matrix must be positive definite; d >= 2.")
    rng = np.random.default_rng(seed)
    covariance = (1 - correlation) * np.eye(d) + correlation * np.ones((d, d))
    X = rng.multivariate_normal(np.zeros(d), covariance, n_samples)
    X += rng.normal(scale=noise_std, size=X.shape)
    w_star = np.array([1.0, -0.5] + [0.0] * (d - 2))
    w_star /= np.linalg.norm(w_star)
    f_star_scores = expit(X @ w_star + 0.8 * np.sin(np.pi * X[:, 0]) * X[:, 1])
    return X, f_star_scores, w_star


def split_indices(X, seed=42):
    """Split before preprocessing/sampling; identical feature rows stay together."""
    frame = pd.DataFrame(X).reset_index(drop=True)
    groups = pd.util.hash_pandas_object(frame, index=False).to_numpy()
    indices = np.arange(len(frame))

    def split(indices, fraction, random_state):
        splitter = GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=random_state)
        left, right = next(splitter.split(indices, groups=groups[indices]))
        return indices[left], indices[right]

    development, test = split(indices, 0.20, seed)
    development, validation = split(development, 0.25, seed + 1)
    fitting, reference = split(development, 0.50, seed + 2)
    result = dict(fit=fitting, train=reference, val=validation, test=test)
    names = list(result)
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            assert set(groups[result[left]]).isdisjoint(groups[result[right]])
    return result


def prepare_population(X, f_star_scores, seed=42, immutable_indices=()):
    """Synthetic population: no sample-dependent rescaling or bootstrap duplicates."""
    X = np.asarray(X, dtype=float)
    f_star_scores = np.asarray(f_star_scores, dtype=float).reshape(-1)
    indices = split_indices(X, seed)
    population = {f"X_{name}": X[idx] for name, idx in indices.items()}
    population["f_star_fit"] = f_star_scores[indices["fit"]]
    population["indices"] = indices
    population["actionable_mask"] = np.ones(X.shape[1], dtype=bool)
    population["actionable_mask"][list(immutable_indices)] = False
    population["feature_names"] = [f"x_{j}" for j in range(X.shape[1])]
    return population


def prepare_real_population(X, y, n_samples=None, seed=42, immutable_features=None, proxy_options=None):
    """Group-disjoint records; preprocessing and qualification proxy fit on fit only.

    n_samples is a cap per partition, sampled WITHOUT replacement.
    Recourse later treats all resulting encoded coordinates as continuous.
    """
    X = pd.DataFrame(X).reset_index(drop=True)
    y = np.asarray(y, dtype=int).reshape(-1)
    indices = split_indices(X, seed)
    if n_samples is not None:
        if n_samples < 2:
            raise ValueError("n_samples must be at least two.")
        rng = np.random.default_rng(seed)
        indices = {name: rng.choice(idx, min(len(idx), n_samples), replace=False)
                   for name, idx in indices.items()}
    numeric = X.select_dtypes(include="number").columns.tolist()
    categorical = [column for column in X if column not in numeric]
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])
    X_fit = preprocessor.fit_transform(X.iloc[indices["fit"]])
    if len(np.unique(y[indices["fit"]])) != 2:
        raise ValueError("The fitting partition must contain both outcome classes.")
    proxy_options = dict(proxy_options or {})
    qualification_proxy = MLPClassifier(
        hidden_layer_sizes=tuple(proxy_options.get("hidden", [32,16])), activation="tanh",
        max_iter=proxy_options.get("max_iter",1500), early_stopping=True,
        random_state=seed,
    ).fit(X_fit, y[indices["fit"]])
    population = {f"X_{name}": np.asarray(preprocessor.transform(X.iloc[idx]), dtype=float)
                  for name, idx in indices.items()}
    population["f_star_fit"] = qualification_proxy.predict_proba(X_fit)[:, 1]

    immutable_features = set(immutable_features or [])
    missing = immutable_features - set(X.columns)
    if missing:
        raise ValueError(f"Immutable columns were not found: {sorted(missing)}")
    # Exact source-column mapping, not string-prefix matching on one-hot names.
    source_features = list(numeric)
    if categorical:
        encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
        source_features += [column for column, categories in zip(categorical, encoder.categories_)
                            for _ in categories]
    feature_names = preprocessor.get_feature_names_out()
    assert len(source_features) == len(feature_names) == X_fit.shape[1]
    population["actionable_mask"] = np.array([name not in immutable_features for name in source_features])
    population["feature_names"] = feature_names.tolist()
    population["encoded_source_features"] = source_features
    population["immutable_features"] = sorted(immutable_features)
    population["y_test"] = y[indices["test"]]
    population.update(indices=indices, preprocessor=preprocessor,
                      qualification_proxy=qualification_proxy)
    return population


def prepare_adult_data(n_samples=None, seed=42, proxy_options=None):
    data = fetch_openml(name="adult", version=2, as_frame=True)
    y = data.target.astype(str).str.rstrip(".").eq(">50K").to_numpy(dtype=int)
    immutable_features = ['age', 'sex', 'race']
    return prepare_real_population(data.data, y, n_samples, seed, immutable_features=immutable_features, proxy_options=proxy_options)


def prepare_credit_data(n_samples=None, seed=42, proxy_options=None):
    data = fetch_openml(data_id=31, as_frame=True)
    y = data.target.astype(str).eq("good").to_numpy(dtype=int)
    immutable_features = ['age', 'personal_status', 'foreign_worker', 'num_dependents']
    return prepare_real_population(data.data, y, n_samples, seed, immutable_features=immutable_features, proxy_options=proxy_options)




def prepare_gmsc_data(path="GiveMeSomeCredit.csv", n_samples=None, seed=42, proxy_options=None):
    frame = pd.read_csv(path)
    y = (frame["SeriousDlqin2yrs"] == 0).to_numpy(dtype=int)
    X = frame.drop(columns=["SeriousDlqin2yrs", "Unnamed: 0"], errors="ignore")
    immutable_features = ['age', 'NumberOfDependents']
    return prepare_real_population(X, y, n_samples, seed, immutable_features=immutable_features, proxy_options=proxy_options)


def prepare_folktables_data(n_samples=None, seed=42, proxy_options=None):
    from folktables import ACSDataSource, ACSIncome  # Optional dependency; install explicitly.
    source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    frame = source.get_data(states=["CA"], download=True)
    features, labels, _ = ACSIncome.df_to_numpy(frame)
    X = pd.DataFrame(features, columns=ACSIncome.features)
    for column in ["COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "SEX", "RAC1P"]:
        if column in X:
            X[column] = X[column].astype("category")

    immutable_features = ['AGEP', 'SEX', 'RAC1P', 'MAR', 'RELP', 'POBP']
    return prepare_real_population(X, labels.astype(int), n_samples, seed, immutable_features=immutable_features, proxy_options=proxy_options)


## Data, frozen scorers, and immutable coordinates


In [ ]:
class FixedScorer(nn.Module):
    """Affine logit; immutable coordinates still contribute to the decision."""
    model_type = "affine"
    def __init__(self, w_0):
        super().__init__()
        self.register_buffer("w_0", as_tensor(w_0).reshape(-1).detach().clone())
        self.register_buffer("h", torch.zeros(len(self.w_0) - 1))

    def logits(self, X):
        return self.w_0[0] + X @ self.w_0[1:]

    def forward(self, X):
        return torch.sigmoid(self.logits(X))


class QuadraticScorer(FixedScorer):
    """Nonlinear score; concave logit gives convex target-attainment sets."""
    model_type = "quadratic"
    def __init__(self, w_0, h):
        super().__init__(w_0)
        self.h.copy_(as_tensor(h))
        if not (self.h > 0).all():
            raise ValueError("Quadratic curvatures must be strictly positive.")

    def logits(self, X):
        return super().logits(X) - 0.5 * (X.square() * self.h).sum(-1)


class SmoothMLPScorer(nn.Module):
    """Smooth nonlinear residual network; no global recourse-optimality guarantee."""
    model_type = "mlp"
    def __init__(self, w_0, hidden=12, seed=42):
        super().__init__()
        with torch.random.fork_rng():
            torch.manual_seed(seed)
            self.w_0 = nn.Parameter(as_tensor(w_0).detach().clone())
            self.network = nn.Sequential(nn.Linear(len(w_0)-1, hidden), nn.Tanh(),
                                         nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))
            nn.init.zeros_(self.network[-1].weight)
            nn.init.zeros_(self.network[-1].bias)

    def logits(self, X):
        return self.w_0[0] + X @ self.w_0[1:] + self.network(X).squeeze(-1)

    def forward(self, X):
        return torch.sigmoid(self.logits(X))


def fit_f_w0(X_fit, f_star_scores, model_type="quadratic", seed=42, l2_reg=1e-2, epochs=1000, lr=0.01, hidden=12):
    """Fit once on the fitting split; all returned scoring parameters are frozen."""
    X = np.asarray(X_fit, dtype=float)
    y = np.asarray(f_star_scores, dtype=float).reshape(-1)
    if X.ndim != 2 or len(X) != len(y) or not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Invalid fitting data.")
    if not ((0 <= y) & (y <= 1)).all() or l2_reg < 0:
        raise ValueError("Require qualification scores in [0,1] and nonnegative regularization.")
    d = X.shape[1]
    X_aug = np.column_stack([np.ones(len(X)), X])
    design = np.column_stack([X_aug, -0.5 * X**2]) if model_type == "quadratic" else X_aug
    bounds = [(-5, 5)] * (d + 1)
    initial = np.zeros(d + 1)
    if model_type == "quadratic":
        bounds += [(1e-4, 3.0)] * d
        initial = np.r_[initial, np.full(d, 0.05)]

    def objective(parameters):
        logits = design @ parameters
        penalty = parameters.copy(); penalty[0] = 0
        loss = np.mean(np.logaddexp(0.0, logits) - y * logits) + l2_reg * np.dot(penalty, penalty) / 2
        grad = design.T @ (expit(logits) - y) / len(y) + l2_reg * penalty
        return loss, grad

    fitted = minimize(objective, initial, jac=True, method="L-BFGS-B", bounds=bounds,
                      options={"maxiter": 10000, "ftol": 1e-12, "gtol": 1e-8})
    if not fitted.success:
        raise RuntimeError(f"Initial scoring fit failed: {fitted.message}")
    if model_type == "affine":
        f_w0 = FixedScorer(fitted.x)
    elif model_type == "quadratic":
        f_w0 = QuadraticScorer(fitted.x[:d+1], fitted.x[d+1:])
    elif model_type == "mlp":
        f_w0 = SmoothMLPScorer(fitted.x, hidden=hidden, seed=seed)
        optimizer = torch.optim.Adam(f_w0.parameters(), lr=lr)
        X_tensor, y_tensor = as_tensor(X), as_tensor(y)
        for _ in range(epochs):
            optimizer.zero_grad()
            loss = F.binary_cross_entropy_with_logits(f_w0.logits(X_tensor), y_tensor)
            loss = loss + 0.5 * l2_reg * sum(p.square().sum() for p in f_w0.parameters())
            loss.backward(); optimizer.step()
        f_w0.fit_loss = float(loss.detach())
    else:
        raise ValueError("model_type must be affine, quadratic, or mlp.")
    f_w0.eval().requires_grad_(False)
    return f_w0


def cost(X, X_prime):
    return torch.linalg.vector_norm(X_prime - X, dim=-1)


def project_actions(delta, B, actionable_mask):
    delta = delta * actionable_mask
    norms = torch.linalg.vector_norm(delta, dim=-1, keepdim=True)
    return delta * torch.clamp(B / norms.clamp_min(torch.finfo(delta.dtype).tiny), max=1.0)


@torch.no_grad()
def exact_reachable(X, f_w0, B, actionable_mask):
    """Global maximizer within the mutable-coordinate Euclidean ball."""
    a = (f_w0.w_0[1:] - X * f_w0.h) * actionable_mask
    if B == 0 or not actionable_mask.any():
        return X.clone(), f_w0(X)
    if f_w0.model_type == "affine":
        delta = B * a / torch.linalg.vector_norm(a, dim=1, keepdim=True).clamp_min(1e-300)
    else:
        h = f_w0.h
        unconstrained = a / h
        low = torch.zeros(len(X))
        high = torch.linalg.vector_norm(a, dim=1) / B
        for _ in range(75):
            multiplier = (low + high) / 2
            delta = a / (h + multiplier[:, None])
            too_far = torch.linalg.vector_norm(delta, dim=1) > B
            low = torch.where(too_far, multiplier, low)
            high = torch.where(too_far, high, multiplier)
        delta = a / (h + high[:, None])
        delta = torch.where((torch.linalg.vector_norm(unconstrained, dim=1) <= B)[:, None],
                            unconstrained, delta)
    witness = X + delta
    return witness, f_w0(witness)


def numerical_reachable(X, f_w0, B, actionable_mask, seed=42, starts=3, steps=60):
    """A feasible witness/lower bound on the reachable maximum, never a global certificate."""
    if B == 0 or not actionable_mask.any():
        return X.clone(), f_w0(X).detach()
    generator = torch.Generator().manual_seed(seed)
    best_X, best_score = X.clone(), f_w0(X).detach()
    for start in range(starts):
        delta = torch.zeros_like(X) if start == 0 else project_actions(
            torch.randn(X.shape, generator=generator), B, actionable_mask)
        for step in range(steps):
            with torch.enable_grad():
                delta = delta.detach().requires_grad_(True)
                gradient, = torch.autograd.grad(f_w0.logits(X + delta).sum(), delta)
            gradient *= actionable_mask
            gradient /= torch.linalg.vector_norm(gradient, dim=1, keepdim=True).clamp_min(1e-14)
            with torch.no_grad():
                delta = project_actions(delta + (0.25 * B / math.sqrt(1 + step/10)) * gradient,
                                        B, actionable_mask)
                score = f_w0(X + delta)
                better = score > best_score
                best_X[better], best_score[better] = (X + delta)[better], score[better]
    return best_X.detach(), best_score.detach()


@dataclass
class Cohort:
    X: torch.Tensor
    X_aug: torch.Tensor
    f_w0: nn.Module
    t_0: float
    B: float
    alpha: float
    s_0: torch.Tensor
    q_bar: object  # Exact maximum only for affine/quadratic; None for MLP.
    q_cap: torch.Tensor  # Known-feasible target cap used by every comparison method.
    witness: torch.Tensor
    X_minus: torch.Tensor
    eligible: torch.Tensor
    N: int
    N_minus: int
    k: int
    actionable_mask: torch.Tensor
    response_cache: object
    solver_audit: list
    options: dict


def make_cohort(X, f_w0, t_0, B, alpha, actionable_mask=None, seed=42, solver_options=None):
    options = dict(solver_options or {})
    X = as_tensor(X).detach().clone()
    if X.ndim != 2 or not len(X) or not torch.isfinite(X).all():
        raise ValueError("X must be a nonempty finite matrix.")
    if not (0 < t_0 < 1 and 0 < alpha < 1 and np.isfinite(B) and B >= 0):
        raise ValueError("Invalid t_0, alpha, or B.")
    mask = torch.ones(X.shape[1], dtype=torch.bool) if actionable_mask is None else torch.as_tensor(actionable_mask, dtype=torch.bool)
    if mask.shape != (X.shape[1],):
        raise ValueError("actionable_mask must match the encoded feature dimension.")
    s_0 = f_w0(X).detach()
    if not ((0 < s_0) & (s_0 < 1)).all():
        raise FloatingPointError("Scoring saturation: rescale features or strengthen scorer regularization.")
    if getattr(f_w0, "model_type", None) in {"affine", "quadratic"}:
        witness, q_reached = exact_reachable(X, f_w0, B, mask)
        q_bar = q_reached.clone()
        slack = options.get("exact_margin", 1e-4)
    else:
        witness, q_reached = numerical_reachable(X, f_w0, B, mask, seed=seed,
                                                   starts=options.get("reachability_starts",3),
                                                   steps=options.get("reachability_steps",60))
        q_bar, slack = None, options.get("neural_margin",0.02)
    q_cap = torch.minimum(s_0 + (1-slack) * (q_reached-s_0), torch.full_like(s_0, 1-1e-8))
    q_cap = torch.maximum(q_cap, s_0)
    X_minus = s_0 < t_0
    eligible = X_minus & (q_cap >= t_0)
    assert torch.equal(witness[:, ~mask], X[:, ~mask])
    assert (cost(X, witness) <= B + 1e-9).all()
    return Cohort(X, torch.cat([torch.ones((len(X), 1)), X], 1), f_w0, float(t_0), float(B),
                  float(alpha), s_0, q_bar, q_cap, witness, X_minus, eligible, len(X),
                  int(X_minus.sum()), math.floor(alpha*len(X)), mask, OrderedDict(), [], options)


def target_feasible(cohort, q):
    q = as_tensor(q).expand(cohort.N)
    return cohort.X_minus & (q >= cohort.t_0) & (q <= cohort.q_cap) & (q < 1)


## Exact-response layers and numerical fallbacks


In [ ]:
class QuadraticBR(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, cohort):
        active = target_feasible(cohort, q)
        response, derivative = cohort.X.clone(), torch.zeros_like(cohort.X)
        if active.any():
            X, target = cohort.X[active], q[active]
            h = cohort.f_w0.h
            a = (cohort.f_w0.w_0[1:] - X*h) * cohort.actionable_mask
            gain = torch.logit(target) - cohort.f_w0.logits(X)
            if cohort.f_w0.model_type == "affine":
                squared_norm = a.square().sum(1)
                if (squared_norm <= 0).any():
                    raise ArithmeticError("No mutable score-increasing direction.")
                delta = gain[:, None] * a / squared_norm[:, None]
                jacobian = a / squared_norm[:, None] / (target*(1-target))[:, None]
            else:
                def at(multiplier):
                    delta = multiplier[:, None]*a / (1+multiplier[:, None]*h)
                    achieved = (a*delta - 0.5*h*delta.square()).sum(1)
                    return delta, achieved
                low, high = torch.zeros_like(target), torch.ones_like(target)
                for _ in range(70):
                    insufficient = at(high)[1] < gain
                    if not insufficient.any():
                        break
                    high = torch.where(insufficient, high*2, high)
                else:
                    raise ArithmeticError("Unable to bracket quadratic response multiplier.")
                for _ in range(75):
                    middle = (low+high)/2
                    insufficient = at(middle)[1] < gain
                    low = torch.where(insufficient, middle, low)
                    high = torch.where(insufficient, high, middle)
                delta, _ = at(high)
                slope = a/(1+high[:, None]*h).square()
                denominator = ((a-h*delta)*slope).sum(1)
                if (denominator <= 1e-18).any():
                    raise FloatingPointError("Degenerate best response; increase the target safety margin.")
                jacobian = slope/denominator[:, None]/(target*(1-target))[:, None]
            response[active], derivative[active] = X + delta, jacobian
        ctx.save_for_backward(derivative)
        return response

    @staticmethod
    def backward(ctx, grad_output):
        derivative, = ctx.saved_tensors
        return (grad_output*derivative).sum(1), None


def solve_local_br(cohort, index, q):
    """Multistart constrained solve + KKT polishing; returns a certified local branch.

    No fallback to an unchecked direction, interpolation, or fabricated gradient.
    The witness proves this target is feasible even if numerical optimization fails.
    """
    key = (int(index), float(q))
    if key in cohort.response_cache:
        return cohort.response_cache[key]
    x = cohort.X[index].detach()
    mask = cohort.actionable_mask
    mutable = torch.where(mask)[0]
    d = len(mutable)
    logit_q = math.log(q) - math.log1p(-q)
    witness_delta = (cohort.witness[index]-x)[mask].numpy()
    cache = {}

    def derivatives(delta, hessian=False):
        signature = (np.asarray(delta, dtype=float).tobytes(), hessian)
        if signature in cache:
            return cache[signature]
        with torch.enable_grad():
            u = as_tensor(delta).detach().clone().requires_grad_(True)
            def logit(u):
                z = x.clone(); z[mutable] = x[mutable]+u
                return cohort.f_w0.logits(z[None])[0]
            value = logit(u)
            gradient, = torch.autograd.grad(value, u, create_graph=hessian)
            H = torch.autograd.functional.hessian(logit, u, vectorize=True).detach().numpy() if hessian else None
        answer = (float(value.detach()), gradient.detach().numpy(), H)
        if len(cache) > 100:
            cache.clear()
        cache[signature] = answer
        return answer

    # Deterministic feasible ray start; continuity suffices, no monotonicity is assumed.
    low, high = 0.0, 1.0
    for _ in range(55):
        middle = (low+high)/2
        if derivatives(middle*witness_delta)[0] >= logit_q:
            high = middle
        else:
            low = middle
    ray = high*witness_delta
    constraints = [
        {"type": "ineq", "fun": lambda u: derivatives(u)[0]-logit_q,
         "jac": lambda u: derivatives(u)[1]},
        {"type": "ineq", "fun": lambda u: cohort.B**2-np.dot(u,u),
         "jac": lambda u: -2*u},
    ]
    candidates = []
    # Fixed starts make the branch-selection rule reproducible and independent of training history.
    for initial in [ray, np.zeros(d), witness_delta]:
        result = minimize(lambda u: 0.5*np.dot(u,u), initial, jac=lambda u: u,
                          constraints=constraints, method="SLSQP",
                          options={"ftol": cohort.options.get("response_ftol",1e-11), "maxiter": cohort.options.get("response_maxiter",150)})
        delta = result.x
        value, g, H = derivatives(delta, True)
        if np.dot(g,g) < 1e-16:
            continue
        multiplier = np.dot(delta,g)/np.dot(g,g)

        def equations(z):
            value, gradient, H = derivatives(z[:-1], True)
            return np.r_[z[:-1]-z[-1]*gradient, value-logit_q]

        def jacobian(z):
            _, gradient, H = derivatives(z[:-1], True)
            return np.block([[np.eye(d)-z[-1]*H, -gradient[:,None]],
                             [gradient[None,:], np.zeros((1,1))]])

        polished = root(equations, np.r_[delta,multiplier], jac=jacobian,
                        method="hybr", options={"xtol": 1e-10})
        z = polished.x
        residual = float(np.max(np.abs(equations(z))))
        delta, multiplier = z[:-1], z[-1]
        value, g, H = derivatives(delta, True)
        K = jacobian(z)
        norm = np.linalg.norm(delta)
        if (residual > 2e-7 or multiplier <= 0 or norm > np.linalg.norm(ray)+1e-7 or norm > cohort.B+1e-9
                or norm >= cohort.B-1e-9 or abs(value-logit_q) > 1e-9):
            continue
        tangent = np.linalg.svd(g[None,:], full_matrices=True)[2][1:].T
        reduced = tangent.T @ (np.eye(d)-multiplier*H) @ tangent
        minimum_eigenvalue = np.linalg.eigvalsh(reduced).min() if reduced.size else math.inf
        condition = np.linalg.cond(K)
        if minimum_eigenvalue <= 1e-8 or not np.isfinite(condition) or condition > 1e12:
            continue
        rhs = np.r_[np.zeros(d), 1/(q*(1-q))]
        sensitivity = np.linalg.solve(K, rhs)[:-1]
        candidates.append((norm, delta.copy(), sensitivity, residual, condition))
    if not candidates:
        raise RuntimeError(f"Nonlinear br failed local-optimality/gradient checks at applicant {index}; "
                           "the target has a feasible witness. Increase solver effort or use the quadratic model.")
    _, delta, sensitivity, residual, condition = min(candidates, key=lambda item:item[0])
    response, dq = x.clone(), torch.zeros_like(x)
    response[mask] += as_tensor(delta); dq[mask] = as_tensor(sensitivity)
    audit = dict(target_error=abs(float(cohort.f_w0(response[None])[0])-q),
                 kkt_residual=residual, kkt_condition=condition,
                 local_cost_spread=max(item[0] for item in candidates)-min(item[0] for item in candidates),
                 starts_passed=len(candidates), global_certificate=False)
    answer = (response, dq, audit)
    cohort.solver_audit.append(audit)
    cohort.response_cache[key] = answer
    if len(cohort.response_cache) > 4096:
        cohort.response_cache.popitem(last=False)
    return answer


class NonlinearBR(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, cohort):
        response, derivative = cohort.X.clone(), torch.zeros_like(cohort.X)
        for index in torch.where(target_feasible(cohort,q))[0].tolist():
            response[index], derivative[index], _ = solve_local_br(cohort,index,float(q[index]))
        ctx.save_for_backward(derivative)
        return response

    @staticmethod
    def backward(ctx, grad_output):
        derivative, = ctx.saved_tensors
        return (grad_output*derivative).sum(1), None


def br(cohort, q):
    q = as_tensor(q).expand(cohort.N)
    layer = QuadraticBR if cohort.f_w0.model_type in {"affine","quadratic"} else NonlinearBR
    response = layer.apply(q,cohort)
    if not torch.equal(response[:,~cohort.actionable_mask],cohort.X[:,~cohort.actionable_mask]):
        raise AssertionError("Immutable features changed.")
    if (cost(cohort.X,response).detach() > cohort.B+1e-8).any():
        raise ArithmeticError("Response exceeded the budget.")
    return response


def binary_vector(values, N, name="rho"):
    values = torch.as_tensor(values).expand(N)
    if not ((values==0)|(values==1)).all():
        raise ValueError(f"{name} must be binary; convert surrogates with DefinitePolicy.")
    return values.bool()


def r(cohort, q, rho):
    rho = binary_vector(rho,cohort.N) & target_feasible(cohort,q)
    # Unselected targets are set below eligibility only for the internal solver call.
    solver_q = torch.where(rho,as_tensor(q).expand(cohort.N),torch.zeros_like(cohort.s_0))
    return br(cohort,solver_q)


## Capacity and the paper baseline policies


In [ ]:
def initial_threshold(scores, alpha):
    scores = as_tensor(scores).reshape(-1)
    N = len(scores)
    k = math.floor(alpha * N)
    if not (0 < alpha < 1 and 1 <= k < N):
        raise ValueError("Reference capacity requires 1 <= floor(alpha*N) < N.")
    ordered = torch.sort(scores).values
    lower, upper = ordered[N - k - 1], ordered[N - k]
    if not lower < upper:
        raise ValueError("Initial boundary ties violate this experiment's continuous-score assumption.")
    t_0 = float((lower + upper) / 2)
    if not float(lower) < t_0 <= float(upper):
        t_0 = float(upper)  # Adjacent representable values: still exactly k accepted.
    return t_0


def exact_allocation(scores, k, mass=None):
    """Empirical CDF cutoff and uniform tie probabilities; also supports mixture mass."""
    scores = as_tensor(scores).detach().reshape(-1)
    mass = torch.ones_like(scores) if mass is None else as_tensor(mass).detach().reshape(-1)
    if mass.shape != scores.shape or len(scores) == 0:
        raise ValueError("Nonempty scores and matching masses are required.")
    if not torch.isfinite(scores).all() or not torch.isfinite(mass).all() or (mass < 0).any():
        raise ValueError("Invalid scores or masses.")
    total_mass = float(mass.sum())
    if not 0 <= k <= total_mass or total_mass == 0:
        raise ValueError("Capacity must lie between zero and total mass.")
    positive = mass > 0
    if k == 0:
        return torch.zeros_like(scores), float(scores[positive].max()), 0.0
    if k == total_mass:
        return torch.ones_like(scores), float(scores[positive].min()), 1.0
    ordered, indices = torch.sort(scores[positive])
    cumulative = torch.cumsum(mass[positive][indices], dim=0)
    # inf{t: F(t) >= 1-k/total_mass}; no rounding of distinct scores into ties.
    index = int(torch.searchsorted(cumulative, cumulative.new_tensor(total_mass - k)))
    t_1 = ordered[index]
    above, tied = scores > t_1, scores == t_1
    beta_r = (k - float(mass[above].sum())) / float(mass[tied].sum())
    if not -1e-12 <= beta_r <= 1 + 1e-12:
        raise ArithmeticError("Cutoff tie probability is outside [0,1].")
    beta_r = min(1.0, max(0.0, beta_r))
    y_hat_1 = above.to(scores.dtype) + beta_r * tied.to(scores.dtype)
    assert abs(float((mass * y_hat_1).sum()) - k) <= 1e-10 * max(1.0, total_mass)
    return y_hat_1, float(t_1), beta_r


@torch.no_grad()
def evaluate_policy(cohort, policy, lambda_, adoption_mask=None):
    """Deterministic recommendations; expected acceptance under uniform cutoff ties.

    The optional adoption_mask is only for the separately labelled stress test.
    t_0 is mandatory inside cohort and is never recalibrated here.
    """
    if not (np.isfinite(lambda_) and lambda_ > 0) or cohort.N_minus == 0:
        raise ValueError("Require lambda_ > 0 and at least one initially rejected applicant.")
    q, rho = policy(cohort)
    q = as_tensor(q).expand(cohort.N)
    rho = binary_vector(rho, cohort.N)
    rho = rho & target_feasible(cohort, q)
    if not torch.isfinite(q).all() or ((q[cohort.X_minus] < cohort.t_0) | (q[cohort.X_minus] > 1)).any():
        raise ValueError("Policy targets for rejected applicants must lie in [t_0,1].")
    implements = rho.clone()
    if adoption_mask is not None:
        adoption_mask = torch.as_tensor(adoption_mask)
        if adoption_mask.shape != (cohort.N,):
            raise ValueError("adoption_mask must have one entry per applicant.")
        implements &= binary_vector(adoption_mask, cohort.N, "adoption_mask")
    r_X = r(cohort, q, implements)
    costs = cost(cohort.X, r_X)
    actual_scores = cohort.f_w0(r_X)
    # Target equality is verified below before canonicalizing solver roundoff.
    # This explicit 2e-9 score tolerance preserves target ties; report its residual.
    scores = torch.where(implements, q, cohort.s_0)
    if not torch.allclose(actual_scores, scores, atol=2e-9, rtol=2e-9):
        raise ArithmeticError("The response failed its deployed-score attainment check.")
    if (costs > cohort.B + 1e-10 * max(1.0, cohort.B)).any():
        raise ArithmeticError("A response exceeded the budget.")
    assert torch.equal(r_X[~cohort.X_minus], cohort.X[~cohort.X_minus])
    assert torch.equal(r_X[:, ~cohort.actionable_mask], cohort.X[:, ~cohort.actionable_mask])
    y_hat_1, t_1, beta_r = exact_allocation(scores, cohort.k)
    C = float(costs[cohort.X_minus].mean())
    V = float(y_hat_1[cohort.X_minus].mean())
    assert V <= min(1.0,cohort.k/cohort.N_minus)+1e-12
    return dict(C=C, V=V, J=C - lambda_ * V,
                capacity_validity_bound=min(1.0,cohort.k/cohort.N_minus),
                max_target_error=float((actual_scores-scores).abs().max()),
                wasted_cost=float((costs*(1-y_hat_1))[cohort.X_minus].mean()),
                eligible_fraction=float(cohort.eligible[cohort.X_minus].double().mean()),
                globally_optimal_br=cohort.f_w0.model_type in {"affine", "quadratic"}, t_0=cohort.t_0, t_1=t_1,
                t_shift=t_1 - cohort.t_0, beta_r=beta_r,
                recommendation_rate=float(rho[cohort.X_minus].double().mean()),
                implementation_rate=float(implements[cohort.X_minus].double().mean()),
                accepted_mass=float(y_hat_1.sum()), N_minus=cohort.N_minus, k=cohort.k,
                q=q, rho=rho, r=r_X, scores=scores, y_hat_1=y_hat_1)


def scalar_metrics(result):
    return {key: value for key, value in result.items() if np.isscalar(value)}


class NoRecourse:
    def __call__(self, cohort):
        return torch.full_like(cohort.s_0, cohort.t_0), torch.zeros(cohort.N, dtype=torch.bool)


@dataclass(frozen=True)
class CommonTarget:
    q_c: float
    cost_limit: float = math.inf

    def __call__(self, cohort):
        if not cohort.t_0 <= self.q_c <= 1 or self.cost_limit < 0:
            raise ValueError("Require t_0 <= q_c <= 1 and a nonnegative cost limit.")
        q = torch.full_like(cohort.s_0, self.q_c)
        rho = target_feasible(cohort, q)
        if np.isfinite(self.cost_limit):
            rho &= cost(cohort.X, br(cohort, q)) <= self.cost_limit
        return q, rho


def select_on_validation(candidates, validation, lambda_):
    """Ties in J keep no action/earlier candidate; never use test data to select."""
    selected = NoRecourse()
    best = evaluate_policy(validation, selected, lambda_)
    for candidate in candidates:
        result = evaluate_policy(validation, candidate, lambda_)
        if result["J"] < best["J"]:
            selected, best = copy.deepcopy(candidate), result
    return selected, scalar_metrics(best)


## Smooth targets, recommendation weights, and threshold derivatives


In [ ]:
class PolicySurrogates(nn.Module):
    """Eq. (policy-parameterization): independent affine sigmoid surrogates."""
    def __init__(self, d, Theta_1_radius=3.0, Theta_2_radius=3.0, seed=42, select_recipients=True):
        super().__init__()
        if not all(np.isfinite(value) and value > 0 for value in [Theta_1_radius, Theta_2_radius]):
            raise ValueError("Parameter-set radii must be positive.")
        generator = torch.Generator().manual_seed(seed)
        self.theta_1 = nn.Parameter(0.02 * torch.randn(d + 1, generator=generator))
        self.theta_2 = nn.Parameter(0.02 * torch.randn(d + 1, generator=generator))
        with torch.no_grad():
            self.theta_1[0] = -1.0
            self.theta_2[0] = 0.0
        self.select_recipients = select_recipients
        self.Theta_1_radius = float(Theta_1_radius)
        self.Theta_2_radius = float(Theta_2_radius)
        self.project_()

    def forward(self, cohort):
        q_hat = cohort.t_0 + (cohort.q_cap - cohort.t_0) * torch.sigmoid(cohort.X_aug @ self.theta_1)
        q_hat = torch.where(cohort.eligible, q_hat, torch.full_like(q_hat, cohort.t_0))
        rho_hat = torch.where(cohort.eligible, torch.sigmoid(cohort.X_aug @ self.theta_2),
                              torch.zeros_like(cohort.s_0))
        if not self.select_recipients:
            rho_hat = cohort.eligible.double() + 0.0*self.theta_2.sum()
        if not torch.equal(target_feasible(cohort, q_hat), cohort.eligible):
            raise FloatingPointError("Target interpolation lost feasibility at numerical precision.")
        return q_hat, rho_hat

    @torch.no_grad()
    def project_(self):
        # Theta_1 and Theta_2 are separate closed Euclidean balls.
        for parameter, radius in [(self.theta_1, self.Theta_1_radius),
                                  (self.theta_2, self.Theta_2_radius)]:
            norm = torch.linalg.vector_norm(parameter)
            if norm > radius:
                parameter.mul_(radius / norm)
        return self


class DefinitePolicy:
    def __init__(self, surrogates, rho_cutoff=0.5):
        if not np.isfinite(rho_cutoff) or not 0 < rho_cutoff < 1:
            raise ValueError("rho_cutoff must lie strictly between zero and one.")
        self.rho_cutoff = float(rho_cutoff)
        # Clone the checkpoint: later training cannot mutate a selected policy.
        self.surrogates = copy.deepcopy(surrogates).eval()
        self.surrogates.requires_grad_(False)

    def __call__(self, cohort):
        q, rho_hat = self.surrogates(cohort)
        rho = cohort.eligible & (rho_hat >= self.rho_cutoff)
        return q, rho


def a_i(t, s_0, s_br, rho_hat, tau):
    """Eq. (allocation-weights), over the entire fixed reference cohort."""
    return ((1 - rho_hat) * torch.sigmoid((s_0 - t) / tau)
            + rho_hat * torch.sigmoid((s_br - t) / tau))


def sigmoid_prime(z):
    e = torch.exp(-torch.abs(z))
    return e / (1 + e).square()


def sigmoid_difference(z_b, z_a):
    lower, upper = torch.minimum(z_a, z_b), torch.maximum(z_a, z_b)
    return (torch.sign(z_b - z_a) * torch.sigmoid(upper) * torch.sigmoid(-lower)
            * (-torch.expm1(lower - upper)))


def threshold_partials(s_0, s_br, rho_hat, t_hat, tau):
    z_0, z_br = (s_0 - t_hat) / tau, (s_br - t_hat) / tau
    derivative_0, derivative_br = sigmoid_prime(z_0), sigmoid_prime(z_br)
    denominator = ((1 - rho_hat) * derivative_0 + rho_hat * derivative_br).sum()
    if not torch.isfinite(denominator) or denominator <= torch.finfo(s_0.dtype).tiny:
        raise FloatingPointError("Threshold derivative is numerically unresolved; increase tau.")
    dt_ds_0 = (1 - rho_hat) * derivative_0 / denominator
    dt_ds_br = rho_hat * derivative_br / denominator
    dt_drho_hat = tau * sigmoid_difference(z_br, z_0) / denominator
    if not all(torch.isfinite(v).all() for v in [dt_ds_0, dt_ds_br, dt_drho_hat]):
        raise FloatingPointError("Nonfinite implicit gradient; increase tau.")
    return dt_ds_0, dt_ds_br, dt_drho_hat


class SmoothCapacity(torch.autograd.Function):
    @staticmethod
    def forward(ctx, s_0, s_br, rho_hat, k, tau):
        N = len(s_0)
        if not (1 <= k < N and isinstance(k, int) and np.isfinite(tau) and tau > 0):
            raise ValueError("Require integer 1 <= k < N and tau > 0.")
        if s_0.shape != s_br.shape or s_0.shape != rho_hat.shape:
            raise ValueError("Scores and weights must have matching shapes.")
        if not all(torch.isfinite(v).all() and ((0 <= v) & (v <= 1)).all()
                   for v in [s_0, s_br, rho_hat]):
            raise ValueError("Scores and mixture weights must lie in [0,1].")
        shift = tau * math.log((N - k) / k)
        low, high = s_0.new_tensor(shift), s_0.new_tensor(1 + shift)
        for _ in range(80):
            midpoint = (low + high) / 2
            if float(a_i(midpoint, s_0, s_br, rho_hat, tau).sum()) > k:
                low = midpoint
            else:
                high = midpoint
        t_hat = (low + high) / 2
        residual = abs(float(a_i(t_hat, s_0, s_br, rho_hat, tau).sum()) - k)
        if residual > 1e-10 * N:
            raise FloatingPointError("Bisection failed the capacity residual check.")
        ctx.save_for_backward(s_0, s_br, rho_hat, t_hat)
        ctx.tau = tau
        return t_hat

    @staticmethod
    def backward(ctx, grad_output):
        s_0, s_br, rho_hat, t_hat = ctx.saved_tensors
        derivatives = threshold_partials(s_0, s_br, rho_hat, t_hat, ctx.tau)
        return *(grad_output * derivative for derivative in derivatives), None, None




def J_hat(cohort, surrogates, tau, lambda_, mode="full"):
    """Eq. (empirical-objective); other modes are explicitly labelled ablations."""
    if (cohort.N_minus == 0 or not 1 <= cohort.k < cohort.N
            or not all(np.isfinite(value) and value > 0 for value in [tau, lambda_])):
        raise ValueError("Training requires N_minus > 0, 1 <= k < N, tau > 0, lambda_ > 0.")
    if mode not in {"full", "direct_only", "fixed_t_0"}:
        raise ValueError("Unknown gradient mode.")
    q_hat, rho_hat = surrogates(cohort)
    br_X = br(cohort, q_hat)
    # Score the actual nonlinear response and differentiate through its optimization layer.
    s_br = cohort.f_w0(br_X)
    if mode == "fixed_t_0":
        t_hat = cohort.s_0.new_tensor(cohort.t_0)
    else:
        t_hat = SmoothCapacity.apply(cohort.s_0, s_br, rho_hat, cohort.k, float(tau))
        if mode == "direct_only":
            t_hat = t_hat.detach()
    acceptance = a_i(t_hat, cohort.s_0, s_br, rho_hat, tau)
    C_hat = (rho_hat * cost(cohort.X, br_X))[cohort.X_minus].mean()
    V_hat = acceptance[cohort.X_minus].mean()
    objective = C_hat - lambda_ * V_hat
    return objective, dict(q_hat=q_hat, rho_hat=rho_hat, br=br_X, s_br=s_br,
                           t_hat=t_hat, a_i=acceptance, C_hat=C_hat, V_hat=V_hat)


def validated_rho_cutoffs(values):
    values = [float(value) for value in values]
    if (not values or not np.isfinite(values).all() or len(set(values)) != len(values)
            or any(not 0 < value < 1 for value in values) or 0.5 not in values):
        raise ValueError("rho_cutoffs must be distinct values in (0,1), including 0.5.")
    # Prefer 0.5 on exact objective ties; other cutoffs follow in increasing order.
    return [0.5] + sorted(value for value in values if value != 0.5)


@torch.no_grad()
def validation_cutoff_metrics(cohort, surrogates, lambda_, rho_cutoffs):
    """Evaluate full minimum-cost responses once, then compare binary rules on validation."""
    q_hat, rho_hat = surrogates(cohort)
    responses = br(cohort, q_hat)
    costs = cost(cohort.X, responses)
    actual = cohort.f_w0(responses)
    scores = torch.where(cohort.eligible, q_hat, cohort.s_0)
    if not torch.allclose(actual, scores, atol=2e-9, rtol=2e-9):
        raise ArithmeticError('Cutoff selection failed target attainment.')
    if (costs > cohort.B + 1e-10 * max(1., cohort.B)).any():
        raise ArithmeticError('Cutoff selection exceeded the recourse budget.')
    rows = []
    for cutoff in rho_cutoffs:
        rho = cohort.eligible & (rho_hat >= cutoff)
        allocation, _, _ = exact_allocation(torch.where(rho, scores, cohort.s_0), cohort.k)
        C = float((rho * costs)[cohort.X_minus].mean())
        V = float(allocation[cohort.X_minus].mean())
        rows.append(dict(rho_cutoff=float(cutoff), C=C, V=V, J=C-lambda_*V))
    return rows


## Configuration and persistent records


In [ ]:
import hashlib
import inspect
import json
import os
import pickle
import platform
import re
import tempfile
import traceback
import types
from datetime import datetime, timezone
import scipy
import sklearn


# One order and palette for every comparison figure. Internal method keys are
# retained for compatibility; display labels below are used in all legends.
MAIN_METHODS = (
    "No recourse", "Original threshold", "Common target",
    "Personalized targets, all eligible", "Personalized targets + selection",
)
ABLATION_METHODS = ("Fixed t_0 (ablation)", "Detached threshold (ablation)")
METHOD_ORDER = MAIN_METHODS + ABLATION_METHODS
METHOD_LABELS = {
    "No recourse": "No action",
    "Original threshold": "Original threshold",
    "Common target": "Common target",
    "Personalized targets, all eligible": "Personalized target without selection",
    "Personalized targets + selection": "Personalized target with selection (ours)",
    "Fixed t_0 (ablation)": "Fixed initial threshold",
    "Detached threshold (ablation)": "Detached threshold",
}
METHOD_COLORS = dict(zip(METHOD_ORDER, (
    "#7f7f7f", "#E69F00", "#0072B2", "#CC79A7", "#009E73", "#D55E00", "#56B4E9",
)))
METHOD_MARKERS = dict(zip(METHOD_ORDER, ("s", "^", "D", "v", "o", "P", "X")))


def ordered_methods(methods):
    present = set(methods)
    unknown = present - set(METHOD_ORDER)
    if unknown:
        raise ValueError(f"No plotting style is defined for: {sorted(unknown)}")
    return [method for method in METHOD_ORDER if method in present]


def experiment_config(experiment, dataset, **overrides):
    """A fresh, fully resolved configuration. No QUICK_RUN or mutable shared defaults."""
    config = dict(
        experiment=experiment, dataset=dataset, model="quadratic", seeds=[42,43,44,45,46],
        comparison_protocol="validation_cutoff_grid_v1",
        alpha=0.4, B=0.75, lambda_=3.0, tau=0.01, T=500, eta=0.2,
        cohort_size=256, fit_cap=5000, synthetic_n=6000, gmsc_path="GiveMeSomeCredit.csv",
        methods=list(MAIN_METHODS),
        lambda_values=[0.3,0.5,0.75,1.0,1.5,2.0,3.0,4.0,5.0,7.5,10.0,15.0,20.0,30.0], B_values=[0.25,0.75,1.5],
        alpha_values=[0.2,0.4,0.6], tau_values=[0.005,0.01,0.02,0.05], eta_values=[0.05,0.2],
        scorer=dict(epochs=1000,lr=0.01,hidden=12,l2_reg=0.01),
        qualification_proxy=dict(hidden=[32,16],max_iter=1500),
        solver=dict(reachability_starts=3,reachability_steps=60,response_maxiter=150,
                    response_ftol=1e-11,exact_margin=1e-4,neural_margin=0.02),
        policy=dict(Theta_1_radius=3.0,Theta_2_radius=3.0,validation_every=5,
                    rho_cutoffs=[0.05,0.1,0.25,0.5,0.75,0.9,0.95]),
        baselines=dict(target_points=30),
        diagnostics=dict(enabled=True,every=25,figures=True,rho_cutoffs=[.05,.1,.25,.5,.75,.9,.95]),
        preflight=dict(cohort_size=32,T=10,finite_difference=True),
        audit_fractions=[0.25,0.5,0.75], audit_limit=32,
        adoption_probabilities=[0,0.25,0.5,0.75,1], adoption_draws=200,
        results_dir="./Results", figures_dir="./figures", checkpoint_every=5, retry_failed=False,
    )
    if experiment in [9,10]:
        config.update(tau_values=[.005,.01,.02,.05], eta_values=[.05,.2],
                      lambda_values=[.3,1.,3.,10.,30.])
    if experiment == 1:
        config.update(T=500,methods=["Personalized targets + selection"])
    elif experiment == 6:
        config["methods"] += ["Fixed t_0 (ablation)","Detached threshold (ablation)"]
    elif experiment == 7:
        config["methods"] = ["Personalized targets + selection"]
    for key,value in overrides.items():
        if key not in config:
            raise ValueError(f"Unknown configuration key {key!r}; check its spelling.")
        if isinstance(config[key],dict):
            if not isinstance(value,dict) or set(value)-set(config[key]):
                raise ValueError(f"Unknown or malformed options for {key}.")
            config[key].update(copy.deepcopy(value))
        else:
            config[key]=copy.deepcopy(value)
    validate_config(config)
    return config


def learned_method(method):
    choices={"Personalized targets + selection":("full",True),
             "Personalized targets, all eligible":("full",False),
             "Fixed t_0 (ablation)":("fixed_t_0",True),
             "Detached threshold (ablation)":("direct_only",True)}
    return choices.get(method)


def validate_config(c):
    datasets={"Synthetic nonlinear","Synthetic curved","German Credit","Adult",
              "ACSIncome","Give Me Some Credit"}
    if c["dataset"] not in datasets or c["model"] not in {"affine","quadratic","mlp"}:
        raise ValueError("Unknown dataset or model.")
    if c["experiment"] not in range(1,11):raise ValueError("Choose experiment 1 through 10.")
    def positive_integer(value,name,minimum=1):
        if isinstance(value,bool) or not isinstance(value,(int,np.integer)) or value<minimum:
            raise ValueError(f"{name} must be an integer >= {minimum}.")
    for name in ["T","checkpoint_every","synthetic_n","audit_limit","adoption_draws"]:
        positive_integer(c[name],name)
    positive_integer(c["cohort_size"],"cohort_size",2)
    if c["fit_cap"] is not None:positive_integer(c["fit_cap"],"fit_cap",2)
    if not c["seeds"] or len(set(c["seeds"]))!=len(c["seeds"]):
        raise ValueError("Provide distinct seeds.")
    for seed in c["seeds"]:positive_integer(seed,"seed",0)
    for key in ["lambda_","tau","eta"]:
        if not np.isfinite(c[key]) or c[key]<=0:raise ValueError(f"{key} must be positive.")
    if not np.isfinite(c["B"]) or c["B"]<0 or not 0<c["alpha"]<1:
        raise ValueError("Require B >= 0 and 0 < alpha < 1.")
    for name in ["lambda_values","B_values","alpha_values","tau_values","eta_values"]:
        values=c[name]
        if not values or len(set(values))!=len(values) or not np.isfinite(values).all():
            raise ValueError(f"{name} must contain distinct finite values.")
        if name=="B_values":valid=all(v>=0 for v in values)
        elif name=="alpha_values":valid=all(0<v<1 for v in values)
        else:valid=all(v>0 for v in values)
        if not valid:raise ValueError(f"Invalid {name}.")
    baseline_names={"No recourse","Original threshold","Common target"}
    if not c["methods"] or len(set(c["methods"]))!=len(c["methods"]):
        raise ValueError("Provide distinct methods.")
    if any(m not in baseline_names and learned_method(m) is None for m in c["methods"]):
        raise ValueError("Unknown method name.")
    if c["experiment"] == 1 and any(learned_method(m) is None for m in c["methods"]):
        raise ValueError("Convergence experiments require learned methods.")
    for name in ["Theta_1_radius","Theta_2_radius"]:
        if not np.isfinite(c["policy"][name]) or c["policy"][name]<=0:raise ValueError(name)
    positive_integer(c["policy"]["validation_every"],"validation_every")
    validated_rho_cutoffs(c["policy"]["rho_cutoffs"])
    positive_integer(c["preflight"]["cohort_size"],"preflight cohort_size",2)
    positive_integer(c["preflight"]["T"],"preflight T")
    for name in ["epochs","hidden"]:positive_integer(c["scorer"][name],name)
    if c["scorer"]["lr"]<=0 or c["scorer"]["l2_reg"]<0:raise ValueError("Invalid scorer settings.")
    positive_integer(c["qualification_proxy"]["max_iter"],"qualification max_iter")
    for width in c["qualification_proxy"]["hidden"]:positive_integer(width,"qualification hidden width")
    for name in ["reachability_starts","reachability_steps","response_maxiter"]:
        positive_integer(c["solver"][name],name)
    for name in ["exact_margin","neural_margin"]:
        if not 0<c["solver"][name]<1:raise ValueError(name)
    if c["solver"]["response_ftol"]<=0:raise ValueError("response_ftol")
    positive_integer(c["baselines"]["target_points"],"target_points",2)
    if not c["audit_fractions"] or any(not 0<v<1 for v in c["audit_fractions"]):raise ValueError("audit_fractions")
    if not c["adoption_probabilities"] or any(not 0<=v<=1 for v in c["adoption_probabilities"]):
        raise ValueError("adoption_probabilities")
    if not str(c["results_dir"]).strip():raise ValueError("Provide results_dir.")
    if not str(c.get("figures_dir", "./figures")).strip():raise ValueError("Provide figures_dir.")


def json_value(value):
    if isinstance(value,dict):return {str(k):json_value(v) for k,v in value.items()}
    if isinstance(value,(tuple,list)):return [json_value(v) for v in value]
    if isinstance(value,Path):return str(value)
    if isinstance(value,np.generic):return json_value(value.item())
    if isinstance(value,float) and not math.isfinite(value):return None
    return value


def stable_id(value):
    return hashlib.sha256(json.dumps(json_value(value),sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()[:24]


def array_id(array):
    if torch.is_tensor(array):array=array.detach().cpu().numpy()
    array=np.ascontiguousarray(array)
    return hashlib.sha256(str((array.shape,array.dtype)).encode()+array.tobytes()).hexdigest()


def slug(value):return re.sub(r"[^a-zA-Z0-9_.-]+","_",str(value)).strip("_") or "unnamed"
def utc_now():return datetime.now(timezone.utc).isoformat()


def _constant_signature(value):
    if isinstance(value,types.CodeType):return _code_signature(value)
    if isinstance(value,frozenset):return ['frozenset',sorted(_constant_signature(x) for x in value)]
    if isinstance(value,tuple):return ['tuple',[_constant_signature(x) for x in value]]
    return repr(value)


def _code_signature(code):
    # Exclude notebook cell names/line numbers so a kernel restart preserves the key.
    return [code.co_code.hex(),code.co_names,code.co_varnames,code.co_freevars,
            [_constant_signature(x) for x in code.co_consts]]


def implementation_id():
    signatures={}
    for name in IMPLEMENTATION_NAMES:
        value=globals()[name]
        members=vars(value).items() if inspect.isclass(value) else [(name,value)]
        for member,fn in members:
            if isinstance(fn,(staticmethod,classmethod)):fn=fn.__func__
            if inspect.isfunction(fn):
                fn=inspect.unwrap(fn)
                signatures[name+"."+member]=[_code_signature(fn.__code__),repr(fn.__defaults__),repr(fn.__kwdefaults__)]
    versions=dict(python=platform.python_version(),numpy=np.__version__,torch=str(torch.__version__),
                  scipy=scipy.__version__,sklearn=sklearn.__version__)
    return stable_id(dict(schema=1,code=signatures,versions=versions)),versions


def atomic_write(path,writer,binary=True):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    fd,temporary=tempfile.mkstemp(prefix=path.name+".",suffix=".tmp",dir=path.parent)
    try:
        with os.fdopen(fd,"wb" if binary else "w") as handle:
            writer(handle);handle.flush();os.fsync(handle.fileno())
        os.replace(temporary,path)
    finally:
        if os.path.exists(temporary):os.unlink(temporary)


def save_json(path,value):
    atomic_write(path,lambda f:json.dump(json_value(value),f,indent=2,sort_keys=True,allow_nan=False),binary=False)
def load_json(path):
    with Path(path).open() as f:return json.load(f)
def save_pickle(path,value):atomic_write(path,lambda f:pickle.dump(value,f,protocol=pickle.HIGHEST_PROTOCOL))
def load_pickle(path):
    # These are locally generated scientific caches; never load files from an untrusted source.
    with Path(path).open("rb") as f:return pickle.load(f)
def _tensor_payload(value):
    if isinstance(value,OrderedDict):return OrderedDict((k,_tensor_payload(v)) for k,v in value.items())
    if isinstance(value,dict):return {k:_tensor_payload(v) for k,v in value.items()}
    if isinstance(value,tuple):return tuple(_tensor_payload(v) for v in value)
    if isinstance(value,list):return [_tensor_payload(v) for v in value]
    if isinstance(value,np.generic):return value.item()
    if isinstance(value,np.ndarray):return torch.from_numpy(value.copy())
    return value


def save_torch(path,value):atomic_write(path,lambda f:torch.save(_tensor_payload(value),f))
def load_torch(path):return torch.load(path,map_location="cpu",weights_only=True)
def save_csv(path,frame):atomic_write(path,lambda f:frame.to_csv(f,index=False),binary=False)


def scientific_config(c):
    return {k:v for k,v in c.items() if k not in {"results_dir","figures_dir","retry_failed","checkpoint_every"}}


def data_source_id(config):
    if config["dataset"] != "Give Me Some Credit":
        return "fixed_loader_snapshot_v1"
    path=Path(config["gmsc_path"]).expanduser().resolve()
    digest=hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda:handle.read(1024*1024),b""):digest.update(block)
    return digest.hexdigest()


def experiment_cases(c):
    if c['experiment'] in [9,10]:
        return [dict(seed=int(seed),alpha=float(c['alpha']),B=float(c['B']),lambda_=float(lam),
                     tau=float(tau),eta=float(eta),T=int(c['T']))
                for seed in c['seeds']
                for lam in (c['lambda_values'] if c['experiment']==10 else [c['lambda_']])
                for tau in c['tau_values'] for eta in c['eta_values']]
    axis={1:"eta",2:"lambda_",3:"B",4:"alpha",5:"tau"}.get(c["experiment"])
    grids={"eta":"eta_values","lambda_":"lambda_values","B":"B_values","alpha":"alpha_values","tau":"tau_values"}
    values=c[grids[axis]] if axis else [None]
    cases=[]
    for seed in c["seeds"]:
        for value in values:
            case=dict(seed=int(seed),alpha=float(c["alpha"]),B=float(c["B"]),lambda_=float(c["lambda_"]),
                      tau=float(c["tau"]),eta=float(c["eta"]),T=int(c["T"]))
            if axis:case[axis]=float(value)
            cases.append(case)
    return cases


def parameter_filename(config, config_id):
    """Readable key settings plus the ID of the complete scientific configuration."""
    axis = {1:'eta', 2:'lambda_', 3:'B', 4:'alpha', 5:'tau'}.get(config['experiment'])
    grids = {'eta':'eta_values', 'lambda_':'lambda_values', 'B':'B_values',
             'alpha':'alpha_values', 'tau':'tau_values'}

    def number(value):
        return f'{value:.6g}' if isinstance(value, (float, np.floating)) else str(value)

    def values_label(values):
        label = '_'.join(number(value) for value in values)
        return label if len(label) <= 20 else f'{number(values[0])}to{number(values[-1])}n{len(values)}'

    parameters = []
    for key in ['alpha', 'B', 'lambda_', 'tau', 'eta']:
        swept = key == axis or (config['experiment'] in [9,10] and key in ['tau','eta']) or (config['experiment']==10 and key=='lambda_')
        value = values_label(config[grids[key]]) if swept else number(config[key])
        parameters.append(f"{key.rstrip('_')}{value}")
    parameters += [f"T{config['T']}", f"N{config['cohort_size']}",
                   'seeds' + values_label(config['seeds'])]
    readable = (f"experiment-{config['experiment']}-dataset-{slug(config['dataset'])}"
                f"-model-{slug(config['model'])}-parameters-" + '-'.join(parameters))
    # Leave room for plot/table suffixes and extensions on filesystems with 255-byte names.
    return readable[:170].rstrip('-_') + '-' + config_id


def save_parameter_results(store):
    """Refresh this run's aggregate after every trial; other configurations are untouched."""
    frame = store.manifest_results()
    save_csv(store.directory / 'results.csv', frame)
    table_directory = store.root / 'tables'
    stem = parameter_filename(store.config, store.config_id)
    result_path = table_directory / f'{stem}-results.csv'
    config_path = table_directory / f'{stem}-config.json'
    save_csv(result_path, frame)
    metadata = load_json(store.directory / 'config.json')
    save_json(config_path, dict(metadata, run_directory=str(store.directory),
                                results_file=str(result_path)))
    return frame


def export_saved_runs(results_dir='./Results', *, figures_dir='./figures',
                      experiment=None, dataset=None, model=None, show=False):
    """Export completed old/new runs using their saved IDs; no fitting or dataset loading."""
    output_root = Path(results_dir).expanduser().resolve()
    exports = []
    for path in sorted((output_root / 'experiments').glob('experiment_*/*/*/*/config.json')):
        metadata = load_json(path)
        config = metadata['config']
        if experiment is not None and config['experiment'] != experiment:continue
        if dataset is not None and config['dataset'] != dataset:continue
        if model is not None and config['model'] != model:continue
        # Use the recorded identity, even if code, files, or defaults changed since training.
        store = object.__new__(RunStore)
        store.config = copy.deepcopy(config)
        store.config['figures_dir'] = str(figures_dir)
        store.config_id = metadata['config_id']
        store.code_id = metadata['implementation_id']
        store.versions = metadata.get('versions', {})
        store.source_id = metadata.get('source_id')
        store.root = output_root
        store.directory = path.parent
        if store.manifest_results().empty:continue
        frame = save_parameter_results(store)
        runner = types.SimpleNamespace(config=store.config, store=store)
        # Older tables remain exportable, but their figures must not imply that
        # the revised cutoff/baseline protocol was run on those historical policies.
        figure_paths = (save_experiment_figures(runner, frame, show=show)
                        if config.get('comparison_protocol')=='validation_cutoff_grid_v1' else [])
        stem = parameter_filename(config, store.config_id)
        exports.append(dict(experiment=config['experiment'], dataset=config['dataset'],
                            model=config['model'], config_id=store.config_id,
                            results_file=str(output_root / 'tables' / f'{stem}-results.csv'),
                            figures=figure_paths))
    return pd.DataFrame(exports)


class RunStore:
    def __init__(self,config):
        self.config=copy.deepcopy(config);validate_config(self.config)
        self.code_id,self.versions=implementation_id()
        self.source_id=data_source_id(config)
        self.config_id=stable_id(dict(config=scientific_config(config),code=self.code_id,source=self.source_id))
        self.root=Path(config["results_dir"]).expanduser().resolve()
        self.directory=self.root/"experiments"/f"experiment_{config['experiment']:02d}"/slug(config["dataset"])/config["model"]/self.config_id
        self.directory.mkdir(parents=True,exist_ok=True)
        metadata=dict(config=self.config,config_id=self.config_id,implementation_id=self.code_id,
                      source_id=self.source_id,versions=self.versions)
        existing=self.directory/"config.json"
        if existing.exists() and load_json(existing)["config_id"]!=self.config_id:
            raise RuntimeError("Configuration identity mismatch.")
        save_json(existing,metadata)

    def cache(self,kind,key):
        path=self.root/"cache"/self.code_id/kind/key;path.mkdir(parents=True,exist_ok=True);return path

    def trial(self,case,method):
        return self.directory/"trials"/f"seed_{case['seed']}"/f"{stable_id(dict(case=case,method=method))}_{slug(method)}"

    def status(self,path,status,**extra):
        save_json(Path(path)/"status.json",dict(status=status,updated_at=utc_now(),**extra))

    def manifest_results(self):
        rows=[]
        for path in sorted((self.directory/"trials").glob("seed_*/*/result.json")):
            value=load_json(path)
            if isinstance(value,list):rows.extend(value)
            else:rows.append(value)
        return pd.DataFrame(rows)


## Projected training and validation-based cutoff selection


In [ ]:
def encode_policy(policy):
    if isinstance(policy,NoRecourse):return dict(type="none")
    if isinstance(policy,CommonTarget):return dict(type="common",q_c=policy.q_c,cost_limit=policy.cost_limit)
    if isinstance(policy,DefinitePolicy):
        s=policy.surrogates
        return dict(type="learned",d=s.theta_1.numel()-1,state_dict=copy.deepcopy(s.state_dict()),
                    Theta_1_radius=s.Theta_1_radius,Theta_2_radius=s.Theta_2_radius,
                    select_recipients=s.select_recipients,rho_cutoff=policy.rho_cutoff)
    raise TypeError(type(policy).__name__)


def decode_policy(value):
    if value["type"]=="none":return NoRecourse()
    if value["type"]=="common":return CommonTarget(value["q_c"],value["cost_limit"])
    s=PolicySurrogates(value["d"],Theta_1_radius=value["Theta_1_radius"],
                       Theta_2_radius=value["Theta_2_radius"],select_recipients=value["select_recipients"])
    s.load_state_dict(value["state_dict"])
    return DefinitePolicy(s,rho_cutoff=value.get("rho_cutoff",0.5))


def learn_recourse(reference,validation,*,tau=0.05,eta=0.1,T=500,lambda_=3.0,
                   initial_surrogates=None,seed=42,mode="full",select_recipients=True,
                   validation_every=5,Theta_1_radius=3.0,Theta_2_radius=3.0,
                   rho_cutoffs=(0.05,0.1,0.25,0.5,0.75,0.9,0.95),
                   checkpoint_path=None,checkpoint_key=None,checkpoint_every=5,
                   progress_callback=None, diagnostic_every=0):
    """Projected updates; joint validation selection of checkpoint and binary cutoff."""
    cutoff_grid=validated_rho_cutoffs(rho_cutoffs) if select_recipients else [0.5]
    if T<1 or eta<=0 or not np.isfinite(eta) or checkpoint_every<1:raise ValueError("Invalid training settings.")
    if reference.f_w0 is not validation.f_w0 or reference.t_0!=validation.t_0:
        raise ValueError("Reference and validation must share the scorer and initial threshold.")
    if reference.B!=validation.B or reference.alpha!=validation.alpha:raise ValueError("Cohort setting mismatch.")
    if checkpoint_path is not None and checkpoint_key is None:raise ValueError("Checkpoint identity is required.")
    surrogates=(PolicySurrogates(reference.X.shape[1],seed=seed,select_recipients=select_recipients,
                                Theta_1_radius=Theta_1_radius,Theta_2_radius=Theta_2_radius)
                if initial_surrogates is None else copy.deepcopy(initial_surrogates))
    surrogates.requires_grad_(True).project_();surrogates.select_recipients=select_recipients
    initial_state=copy.deepcopy(surrogates.state_dict())
    w_0_before={key:value.clone() for key,value in reference.f_w0.state_dict().items()}
    cutoff_history=[]
    history=[];selected_policy=NoRecourse();selected_iteration=-1;start=0;elapsed=0.0
    best=scalar_metrics(evaluate_policy(validation,selected_policy,lambda_))
    if checkpoint_path is not None and Path(checkpoint_path).exists():
        saved=load_torch(checkpoint_path)
        if saved["checkpoint_key"]!=checkpoint_key:raise RuntimeError("Checkpoint settings do not match this run.")
        surrogates.load_state_dict(saved["state_dict"])
        initial_state=saved["initial_state"];history=saved["history"];best=saved["validation"]
        selected_policy=decode_policy(saved["selected_policy"]);selected_iteration=saved["selected_iteration"]
        start=saved["next_iteration"];elapsed=saved["elapsed_seconds"]
        cutoff_history=saved["cutoff_history"]
        if start>T+1:raise RuntimeError("Invalid checkpoint iteration.")
    resumed_from=start
    started=time.perf_counter()

    def persist(next_iteration):
        if checkpoint_path is None:return
        payload=dict(checkpoint_key=checkpoint_key,next_iteration=next_iteration,
                     state_dict=copy.deepcopy(surrogates.state_dict()),initial_state=initial_state,
                     selected_policy=encode_policy(selected_policy),selected_iteration=selected_iteration,
                     validation=best,history=history,cutoff_history=cutoff_history,elapsed_seconds=elapsed+time.perf_counter()-started)
        save_torch(checkpoint_path,payload)
        save_csv(Path(checkpoint_path).with_name("history.csv"),pd.DataFrame(history))
        save_csv(Path(checkpoint_path).with_name("validation_cutoffs.csv"),pd.DataFrame(cutoff_history))

    for j in range(start,T+1):
        tick=time.perf_counter()
        surrogates.zero_grad(set_to_none=True)
        objective,outputs=J_hat(reference,surrogates,tau,lambda_,mode)
        if not torch.isfinite(objective):raise FloatingPointError("Nonfinite training objective.")
        objective.backward()
        parameters=[surrogates.theta_1,surrogates.theta_2]
        gradients=[parameter.grad.clone() for parameter in parameters]
        if not all(torch.isfinite(g).all() for g in gradients):raise FloatingPointError("Nonfinite gradient.")
        proposed=[]
        for parameter,gradient,radius in zip(parameters,gradients,[surrogates.Theta_1_radius,surrogates.Theta_2_radius]):
            candidate=parameter.detach()-eta*gradient;norm=torch.linalg.vector_norm(candidate)
            proposed.append(candidate*min(1.0,radius/float(norm)) if norm>0 else candidate)
        G_eta_sq=sum(float(((parameter.detach()-candidate)/eta).square().sum())
                     for parameter,candidate in zip(parameters,proposed))
        validation_J=validation_J_at_0_5=validation_rho_cutoff=float("nan")
        if j%validation_every==0 or j==T:
            # One best-response computation per checkpoint, reused across cutoffs.
            candidates=validation_cutoff_metrics(validation,surrogates,lambda_,cutoff_grid)
            cutoff_history.extend(dict(iteration=j,**metrics) for metrics in candidates)
            validation_J_at_0_5=next(metrics["J"] for metrics in candidates if metrics["rho_cutoff"]==0.5)
            candidate_metrics=min(candidates,key=lambda metrics:metrics["J"])
            validation_J=candidate_metrics["J"]
            validation_rho_cutoff=candidate_metrics["rho_cutoff"]
            if validation_J<best["J"]:
                selected_policy=DefinitePolicy(surrogates,rho_cutoff=validation_rho_cutoff)
                best=scalar_metrics(evaluate_policy(validation,selected_policy,lambda_))
                selected_iteration=j
        row=dict(iteration=j,J_hat=float(objective.detach()),C_hat=float(outputs["C_hat"].detach()),
                 V_hat=float(outputs["V_hat"].detach()),t_hat=float(outputs["t_hat"].detach()),G_eta_sq=G_eta_sq,
                 capacity_residual=float(outputs["a_i"].detach().sum())-reference.k,validation_J=validation_J,
                 validation_J_at_0_5=validation_J_at_0_5,validation_rho_cutoff=validation_rho_cutoff,
                 selected_rho_cutoff=getattr(selected_policy,"rho_cutoff",float("nan")),
                 theta_1_norm=float(surrogates.theta_1.detach().norm()),theta_2_norm=float(surrogates.theta_2.detach().norm()),
                 step_seconds=time.perf_counter()-tick)
        row["sufficient_decrease"]=(not history or row["J_hat"]<=history[-1]["J_hat"]-eta*history[-1]["G_eta_sq"]/2+1e-10)
        if diagnostic_every and (j % diagnostic_every == 0 or j == T):
            diagnostic = allocation_diagnostics(reference, outputs['q_hat'], outputs['rho_hat'],
                                               tau, lambda_, br_X=outputs['br'])
            row.update({'diag_'+key:value for key,value in diagnostic.items()})
        history.append(row)
        if j<T:
            with torch.no_grad():
                for parameter,candidate in zip(parameters,proposed):parameter.copy_(candidate)
        if j==0 or (j+1)%checkpoint_every==0 or j==T:persist(j+1)
        # Used by the runner for logging; also permits controlled interruption tests.
        if progress_callback is not None:progress_callback(j,row)
    assert all(torch.equal(value,w_0_before[key]) for key,value in reference.f_w0.state_dict().items())
    return dict(policy=selected_policy,last_iterate=copy.deepcopy(surrogates).eval(),history=pd.DataFrame(history),
                validation=best,selected_iteration=selected_iteration,initial_state=initial_state,
                selected_rho_cutoff=getattr(selected_policy,"rho_cutoff",None),
                cutoff_history=pd.DataFrame(cutoff_history),
                resumed_from=resumed_from,runtime_seconds=elapsed+time.perf_counter()-started)


## Numerical checks and experiment runners


In [ ]:
def run_framework_checks():
    passed=[]
    # Closed-form curved example: nearest point to a circular superlevel set.
    f_w0=QuadraticScorer([1.,0.,0.,1.],[1.,1.,.2])
    X=as_tensor([[2.,0.,.5],[2.5,.1,.6],[.1,.2,.7]])
    cohort=make_cohort(X,f_w0,.65,2.,1/3,[True,True,False])
    target=float(f_w0(as_tensor([[1.,0.,.5]]))[0])
    q=torch.full((3,),target,requires_grad=True)
    response=br(cohort,q)
    assert torch.allclose(response[0],as_tensor([1.,0.,.5]),atol=1e-10)
    assert torch.equal(response[:,2],X[:,2])
    assert torch.equal(response[~cohort.X_minus],X[~cohort.X_minus])
    assert torch.autograd.gradcheck(lambda target:br(cohort,target),(q,),eps=1e-6,atol=1e-5,rtol=1e-4)
    assert torch.all(f_w0(cohort.witness)>=cohort.q_cap)
    passed.append("Global quadratic response, analytic curved example, mutable-subspace gradient")

    affine=FixedScorer([.1,1.,4.])
    sample=make_cohort([[-1.,0.],[1.,0.]],affine,.5,2.,.5,[True,False])
    q_a=torch.tensor([.6,.6])
    expected=(torch.logit(q_a[0])-.1)
    assert abs(float(br(sample,q_a)[0,0]-expected))<1e-12
    frozen=make_cohort([[-1.,0.],[1.,0.]],affine,.5,2.,.5,[False,False])
    assert not frozen.eligible.any() and torch.equal(br(frozen,q_a),frozen.X)
    zero=make_cohort([[-1.,0.],[1.,0.]],affine,.5,0.,.5,[True,False])
    assert not zero.eligible.any()
    # Immutable input is used in scoring even though response derivatives there are zero.
    assert affine(as_tensor([[0.,0.]])) != affine(as_tensor([[0.,1.]]))
    passed.append("Affine specialization; all-immutable/zero-budget cases; immutable inputs affect scores")

    class AnalyticNonlinear(nn.Module):
        model_type="mlp"
        def logits(self,X):return X[:,0]+.2*torch.sin(X[:,0])+.7*X[:,1]-.1*X[:,1].square()
        def forward(self,X):return torch.sigmoid(self.logits(X))
    nonlinear=AnalyticNonlinear().eval()
    local=make_cohort([[-1.5,.5],[-.5,-.7],[2.,0.]],nonlinear,.5,2.,1/3,[True,False])
    q_m=torch.tensor([.55,.55,.55],requires_grad=True)
    response=br(local,q_m)
    assert torch.allclose(nonlinear(response)[:2],q_m[:2],atol=2e-10)
    assert torch.equal(response[:,1],local.X[:,1])
    assert local.q_bar is None and len(local.solver_audit)>0
    assert torch.autograd.gradcheck(lambda target:br(local,target),(q_m,),eps=1e-5,atol=2e-5,rtol=3e-4)
    _,dq,_=solve_local_br(local,0,.55)
    expected_derivative=1/(.55*.45*(1+.2*torch.cos(response[0,0])))
    assert abs(float((dq[0]-expected_derivative).detach()))<1e-8
    assert all(a["kkt_residual"]<2e-7 for a in local.solver_audit)
    passed.append("Nonlinear local solver and KKT implicit derivative against an analytic inverse")

    for context in [cohort,local]:
        policy=PolicySurrogates(context.X.shape[1],seed=3)
        objective,_=J_hat(context,policy,.08,2.)
        objective.backward()
        for parameter in [policy.theta_1,policy.theta_2]:
            gradient=parameter.grad.clone()
            numerical=torch.empty_like(parameter)
            for j in range(len(parameter)):
                original=float(parameter[j].detach())
                with torch.no_grad():
                    parameter[j]=original+2e-5;high=float(J_hat(context,policy,.08,2.)[0])
                    parameter[j]=original-2e-5;low=float(J_hat(context,policy,.08,2.)[0])
                    parameter[j]=original
                numerical[j]=(high-low)/4e-5
            assert torch.allclose(gradient,numerical,atol=3e-5,rtol=1e-3)
    passed.append("Full theta_1/theta_2 gradients through nonlinear responses and competitive threshold")

    s_0=torch.tensor([.1,.3,.55,.8],requires_grad=True)
    s_br=torch.tensor([.4,.6,.7,.9],requires_grad=True)
    weights=torch.tensor([.2,.5,.7,.3],requires_grad=True)
    assert torch.autograd.gradcheck(lambda a,b,c:SmoothCapacity.apply(a,b,c,2,.07),
                                   (s_0,s_br,weights),eps=1e-6,atol=1e-6)
    t_hat=SmoothCapacity.apply(s_0,s_br,weights,2,.07)
    assert abs(float(a_i(t_hat,s_0,s_br,weights,.07).sum().detach())-2)<1e-12
    y_hat_1,_,beta=exact_allocation(torch.ones(5)*.5,2)
    assert beta==.4 and abs(float(y_hat_1.sum())-2)<1e-12
    passed.append("Integer capacity, cutoff ties, and all implicit threshold derivatives")

    class OneRecommendation:
        def __call__(self,c):return torch.full_like(c.s_0,.55),torch.tensor([True,False,False])
    result=evaluate_policy(local,OneRecommendation(),2.)
    assert abs(result["C"]-float(cost(local.X,result["r"])[0])/local.N_minus)<1e-12
    assert torch.equal(result["r"][1:],local.X[1:])
    assert torch.equal(result["r"][:,~local.actionable_mask],local.X[:,~local.actionable_mask])
    try:evaluate_policy(local,PolicySurrogates(2),2.)
    except ValueError as error:assert "binary" in str(error)
    else:raise AssertionError("Smooth recommendation weights were silently deployed.")
    fit=learn_recourse(local,local,T=2,eta=.01,lambda_=1e-8,validation_every=1)
    assert isinstance(fit["policy"],NoRecourse)
    for name in ["theta_1","theta_2"]:
        assert not torch.equal(fit["initial_state"][name],fit["last_iterate"].state_dict()[name])
    passed.append("All-rejection cost normalization, binary deployment, real no-action selection, both updates")

    rng=np.random.default_rng(7)
    raw=pd.DataFrame({"age":rng.normal(size=240),"age_bucket":rng.choice(["young","old"],240),
                      "sex":rng.choice(["F","M"],240),"income":rng.normal(size=240)})
    labels=(raw.income.to_numpy()+rng.normal(size=240)>.0).astype(int)
    pop=prepare_real_population(raw,labels,seed=7,immutable_features=["age","sex"])
    for column,mutable in zip(pop["encoded_source_features"],pop["actionable_mask"]):
        assert mutable == (column not in {"age","sex"})
    assert any(c=="age_bucket" and m for c,m in zip(pop["encoded_source_features"],pop["actionable_mask"]))
    duplicates=np.repeat(np.arange(80)[:,None],2,axis=0)
    groups=split_indices(duplicates)
    for name,ids in groups.items():
        for other,other_ids in groups.items():
            if name!=other:assert set(duplicates[ids,0]).isdisjoint(duplicates[other_ids,0])
    passed.append("Exact one-hot immutable mapping (including prefix collision) and leakage-free splits")
    return pd.DataFrame({"check":passed,"status":"passed"})


In [ ]:
def gradient_decomposition(cohort,surrogates,tau,lambda_):
    surrogates=copy.deepcopy(surrogates).requires_grad_(True)
    parameters=[surrogates.theta_1,surrogates.theta_2]
    objective,outputs=J_hat(cohort,surrogates,tau,lambda_)
    full=torch.autograd.grad(objective,parameters,retain_graph=True)
    dt=torch.autograd.grad(outputs["t_hat"],parameters)
    direct_objective,_=J_hat(cohort,surrogates,tau,lambda_,mode="direct_only")
    direct=torch.autograd.grad(direct_objective,parameters)
    with torch.no_grad():
        weights=outputs["rho_hat"]
        derivative_0=sigmoid_prime((cohort.s_0-outputs["t_hat"])/tau)
        derivative_br=sigmoid_prime((outputs["s_br"]-outputs["t_hat"])/tau)
        dJ_dt=lambda_/tau*((1-weights)*derivative_0+weights*derivative_br)[cohort.X_minus].mean()
        rows=[]
        for name,g_full,g_direct,dt_block in zip(["theta_1","theta_2"],full,direct,dt):
            assert torch.allclose(g_full-g_direct,dJ_dt*dt_block,atol=2e-8,rtol=1e-5)
            rows.append(dict(parameter=name,full_norm=float(g_full.norm()),
                             direct_norm=float(g_direct.norm()),threshold_norm=float((g_full-g_direct).norm())))
    return pd.DataFrame(rows)


@torch.no_grad()
def smoothing_diagnostic(cohort,surrogates,tau,lambda_):
    _,outputs=J_hat(cohort,surrogates,tau,lambda_)
    # Identical fractional mixture: separates acceptance smoothing from rho rounding.
    scores=torch.cat([cohort.s_0,outputs["s_br"]])
    weights=torch.cat([1-outputs["rho_hat"],outputs["rho_hat"]])
    allocation,_,_=exact_allocation(scores,cohort.k,mass=weights)
    hard=(1-outputs["rho_hat"])*allocation[:cohort.N]+outputs["rho_hat"]*allocation[cohort.N:]
    return dict(mean_acceptance_error=float((outputs["a_i"]-hard).abs().mean()),
                capacity_residual=abs(float(outputs["a_i"].sum())-cohort.k))


In [ ]:
def prepare_named_population(config, seed):
    dataset=config['dataset']
    if dataset.startswith('Synthetic'):
        X,f_star_scores,_=generate_correlated_nonlinear_population(config['synthetic_n'],correlation=0.5,seed=seed)
        if dataset=='Synthetic curved':
            f_star_scores=expit(0.6*X[:,0]+1.2*X[:,1]+0.3*X[:,2]-0.8*X[:,1]**2-0.2*X[:,2]**2)
        population=prepare_population(X,f_star_scores,seed,immutable_indices=[0])
        if config['fit_cap'] is not None and len(population['X_fit'])>config['fit_cap']:
            idx=np.random.default_rng(seed).choice(len(population['X_fit']),config['fit_cap'],replace=False)
            population['X_fit']=population['X_fit'][idx]
            population['f_star_fit']=population['f_star_fit'][idx]
            population['indices']['fit']=population['indices']['fit'][idx]
    else:
        loaders={'Adult':prepare_adult_data,'German Credit':prepare_credit_data,
                 'ACSIncome':prepare_folktables_data,'Give Me Some Credit':prepare_gmsc_data}
        options=dict(n_samples=config['fit_cap'],seed=seed,proxy_options=config['qualification_proxy'])
        if dataset=='Give Me Some Credit':options['path']=config['gmsc_path']
        population=loaders[dataset](**options)
    # Always permute a partition before taking prefixes: cohort sizes share nested applicants.
    rng=np.random.default_rng(seed+1000)
    for split in ['train','val','test']:
        indices=rng.permutation(len(population[f'X_{split}']))[:config['cohort_size']]
        population[f'X_{split}']=population[f'X_{split}'][indices]
        population['indices'][split]=population['indices'][split][indices]
        if split=='test' and 'y_test' in population:population['y_test']=population['y_test'][indices]
    return population


def scorer_payload(f_w0,config):
    return dict(model=config['model'],hidden=config['scorer']['hidden'],
                state_dict=copy.deepcopy(f_w0.state_dict()))


def restore_scorer(payload):
    state=payload['state_dict'];kind=payload['model']
    if kind=='affine':model=FixedScorer(state['w_0'])
    elif kind=='quadratic':model=QuadraticScorer(state['w_0'],state['h'])
    else:model=SmoothMLPScorer(state['w_0'],hidden=payload['hidden'])
    model.load_state_dict(state)
    return model.eval().requires_grad_(False)


def cohort_payload(cohort):
    return {key:value for key,value in vars(cohort).items() if key!='f_w0'}


def require_training_cohort(cohort):
    if cohort.N_minus==0 or not 1<=cohort.k<cohort.N:
        raise ValueError('A cohort has no initial rejections or invalid integer capacity; increase cohort_size.')


class ExperimentRunner:
    """One configuration; lazy cached data/scorer/cohorts and resumable per-method trials."""
    def __init__(self,config):
        self.store=RunStore(config);self.config=self.store.config
        self.populations={};self.scorers={};self.cohorts={};self.cohort_paths={}

    def population(self,seed):
        c=self.config
        if seed not in self.populations:
            key=stable_id(dict(dataset=c['dataset'],seed=seed,source=self.store.source_id,
                               fit_cap=c['fit_cap'],cohort_size=c['cohort_size'],synthetic_n=c['synthetic_n'],
                               qualification_proxy=c['qualification_proxy']))
            directory=self.store.cache('populations',key);path=directory/'population.pkl'
            if path.exists():population=load_pickle(path)
            else:
                print(f"Preparing {c['dataset']}, seed={seed}",flush=True)
                population=prepare_named_population(c,seed);save_pickle(path,population)
            signature=stable_id({name:array_id(value) for name,value in population.items()
                                 if name.startswith('X_') or name in ['f_star_fit','actionable_mask','y_test']})
            self.populations[seed]=(population,signature)
            save_json(directory/'metadata.json',dict(dataset=c['dataset'],seed=seed,data_id=signature,
                                                     feature_names=population['feature_names'],
                                                     immutable_features=population.get('immutable_features',['x_0'])))
        return self.populations[seed]

    def scorer(self,seed):
        if seed not in self.scorers:
            population,data_id=self.population(seed);c=self.config
            key=stable_id(dict(X_fit=array_id(population['X_fit']),f_star_fit=array_id(population['f_star_fit']),
                               seed=seed,model=c['model'],scorer=c['scorer']))
            path=self.store.cache('scorers',key)/'scorer.pt'
            if path.exists():f_w0=restore_scorer(load_torch(path))
            else:
                print(f"Fitting frozen {c['model']} scorer, seed={seed}",flush=True)
                f_w0=fit_f_w0(population['X_fit'],population['f_star_fit'],c['model'],seed,**c['scorer'])
                save_torch(path,scorer_payload(f_w0,c))
            self.scorers[seed]=(f_w0,key)
        return self.scorers[seed]

    def contexts(self,case,*,pilot=False,include_test=True):
        seed=case['seed'];population,data_id=self.population(seed);f_w0,model_id=self.scorer(seed)
        # Pilot uses the same deployed scorer and full-reference t_0, on smaller cohorts.
        t_0=initial_threshold(f_w0(as_tensor(population['X_train'])),case['alpha'])
        result={};keys={}
        for split in (['train','val','test'] if include_test else ['train','val']):
            X=population[f'X_{split}']
            if pilot:X=X[:min(len(X),self.config['preflight']['cohort_size'])]
            key=stable_id(dict(X=array_id(X),model=model_id,mask=array_id(population['actionable_mask']),
                               t_0=t_0,B=case['B'],alpha=case['alpha'],seed=seed,solver=self.config['solver']))
            if key not in self.cohorts:
                path=self.store.cache('cohorts',key)/'cohort.pt'
                if path.exists():cohort=Cohort(f_w0=f_w0,**load_torch(path))
                else:
                    cohort=make_cohort(X,f_w0,t_0,case['B'],case['alpha'],population['actionable_mask'],
                                       seed=seed,solver_options=self.config['solver'])
                    save_torch(path,cohort_payload(cohort))
                require_training_cohort(cohort)
                self.cohorts[key]=cohort;self.cohort_paths[key]=path
            result[split]=self.cohorts[key];keys[split]=key
        return result,keys

    def flush_cohorts(self,keys):
        for key in set(keys.values()):save_torch(self.cohort_paths[key],cohort_payload(self.cohorts[key]))

    def fit(self,case,contexts,keys,method,*,pilot=False,progress_callback=None,trial_directory=None):
        mode,select=learned_method(method)
        options=dict(tau=case['tau'],eta=case['eta'],T=case['T'],lambda_=case['lambda_'],seed=case['seed'],
                     mode=mode,select_recipients=select,**self.config['policy'])
        diagnostic=diagnostic_options(self.config)
        options['diagnostic_every']=diagnostic['every'] if diagnostic['enabled'] else 0
        key=stable_id(dict(train=keys['train'],val=keys['val'],options=options,pilot=pilot))
        path=self.store.cache('pilot_fits' if pilot else 'fits',key)
        if trial_directory is not None:
            save_json(Path(trial_directory)/'fit.json',dict(fit_id=key,fit_path=str(path),
                        checkpoint_path=str(path/'checkpoint.pt'),history_path=str(path/'history.csv')))
        fit=learn_recourse(contexts['train'],contexts['val'],**options,
                           checkpoint_path=path/'checkpoint.pt',checkpoint_key=key,
                           checkpoint_every=self.config['checkpoint_every'],progress_callback=progress_callback)
        save_torch(path/'policy.pt',encode_policy(fit['policy']))
        save_json(path/'summary.json',dict(selected_iteration=fit['selected_iteration'],
                                           validation=fit['validation'],runtime_seconds=fit['runtime_seconds'],
                                           selected_rho_cutoff=fit['selected_rho_cutoff']))
        fit['fit_path']=str(path);fit['fit_id']=key
        return fit

    def baseline(self,case,contexts,keys,method):
        reference,validation=contexts['train'],contexts['val']
        if method=='No recourse':return NoRecourse()
        if method=='Original threshold':return CommonTarget(reference.t_0)
        if method!='Common target':raise ValueError(f'Unknown baseline: {method}')
        settings=self.config['baselines']
        key=stable_id(dict(train=keys['train'],val=keys['val'],method=method,settings=settings))
        directory=self.store.cache('baseline_searches',key);state_path=directory/'search.pt'
        state=load_torch(state_path) if state_path.exists() else dict(next_target=0,entries=[])
        # Thirty targets by default, including t_0 and 1. Cache C,V once per
        # cohort/target grid and reuse them for every lambda; selection is on validation.
        grid=np.linspace(reference.t_0,1.0,settings['target_points'])
        for index in range(state['next_target'],len(grid)):
            policy=CommonTarget(float(grid[index]))
            metrics=scalar_metrics(evaluate_policy(validation,policy,1.0))
            state['entries'].append(dict(policy=encode_policy(policy),C=metrics['C'],V=metrics['V']))
            state['next_target']=index+1
            save_torch(state_path,state)
        selected=NoRecourse();best=evaluate_policy(validation,selected,case['lambda_'])['J']
        for entry in state['entries']:
            value=entry['C']-case['lambda_']*entry['V']
            if value<best:selected,best=decode_policy(entry['policy']),value
        return selected

    def tags(self,case,method):
        return dict(experiment=self.config['experiment'],dataset=self.config['dataset'],model=self.config['model'],
                    config_id=self.store.config_id,implementation_id=self.store.code_id,
                    comparison_protocol=self.config["comparison_protocol"],method=method,**case)

    def run_trial(self,case,method):
        directory=self.store.trial(case,method);result_path=directory/'result.json'
        if result_path.exists():
            status_path=directory/'status.json'
            if not status_path.exists() or load_json(status_path)['status']!='completed':
                self.store.status(directory,'completed',case=case,method=method)
            if diagnostic_options(self.config)['enabled']:
                diagnose_one_trial(self.store,directory)
            print(f"  Saved: {method}, seed={case['seed']}",flush=True);return
        status_path=directory/'status.json'
        if status_path.exists() and load_json(status_path)['status']=='failed' and not self.config['retry_failed']:
            print(f"  Prior failure: {method}; set retry_failed=True after checking error.txt.",flush=True);return
        self.store.status(directory,'running',case=case,method=method)
        keys={};started=time.perf_counter()
        try:
            contexts,keys=self.contexts(case);fit=None
            model_key=self.scorers[case['seed']][1]
            save_json(directory/'context.json',dict(schema=1,
                scorer=str((self.store.cache('scorers',model_key)/'scorer.pt').relative_to(self.store.root)),
                cohorts={split:str(self.cohort_paths[key].relative_to(self.store.root)) for split,key in keys.items()}))
            print(f"  Running {method}: seed={case['seed']}, lambda={case['lambda_']:g}, "
                  f"B={case['B']:g}, alpha={case['alpha']:g}, tau={case['tau']:g}, eta={case['eta']:g}",flush=True)
            if learned_method(method) is not None:
                def progress(j,row):
                    if j%max(100,self.config['checkpoint_every'])==0 or j==case['T']:
                        print(f"    {j}/{case['T']}: J_hat={row['J_hat']:.5g}, G_eta_sq={row['G_eta_sq']:.3g}",flush=True)
                fit=self.fit(case,contexts,keys,method,progress_callback=progress,trial_directory=directory)
                policy=fit['policy'];save_csv(directory/'history.csv',fit['history'])
                save_csv(directory/'validation_cutoffs.csv',fit['cutoff_history'])
                save_json(directory/'fit.json',dict(fit_id=fit['fit_id'],fit_path=fit['fit_path'],
                                                    checkpoint_path=str(Path(fit['fit_path'])/'checkpoint.pt'),
                                                    history_path=str(Path(fit['fit_path'])/'history.csv'),
                                                    selected_iteration=fit['selected_iteration'],
                                                    runtime_seconds=fit['runtime_seconds']))
            else:policy=self.baseline(case,contexts,keys,method)
            save_torch(directory/'policy.pt',encode_policy(policy))
            validation=evaluate_policy(contexts['val'],policy,case['lambda_'])
            metrics=evaluate_policy(contexts['test'],policy,case['lambda_'])
            row=dict(**self.tags(case,method),status='ok',expected_methods=len(self.config['methods']),
                     validation_J=validation['J'],trial_seconds=time.perf_counter()-started,**scalar_metrics(metrics))
            no_action=scalar_metrics(evaluate_policy(contexts['test'],NoRecourse(),case['lambda_']))
            row.update(no_action_V=no_action['V'],no_action_J=no_action['J'],
                       incremental_validity=metrics['V']-no_action['V'],
                       delta_J_no_action=metrics['J']-no_action['J'],
                       wasted_cost_fraction=metrics['wasted_cost']/metrics['C'] if metrics['C']>0 else None,
                       rho_cutoff=getattr(policy,'rho_cutoff',None))
            if fit is not None:
                row.update(selected_iteration=fit['selected_iteration'],training_seconds=fit['runtime_seconds'],
                           selected_rho_cutoff=fit['selected_rho_cutoff'],
                           best_validation_J_at_0_5=min(
                               evaluate_policy(contexts['val'],NoRecourse(),case['lambda_'])['J'],
                               float(fit['history'].validation_J_at_0_5.min())))
                if self.config['experiment']==5:
                    row.update(smoothing_diagnostic(contexts['train'],fit['last_iterate'],case['tau'],case['lambda_']))
                if self.config['experiment']==6 and method=='Personalized targets + selection':
                    save_csv(directory/'gradient_decomposition.csv',gradient_decomposition(
                        contexts['train'],fit['last_iterate'],case['tau'],case['lambda_']))
            if self.config['experiment']==8:
                self.adoption(case,method,contexts['test'],policy,directory,row)
            else:save_json(result_path,row)
            self.store.status(directory,'completed',case=case,method=method)
            if diagnostic_options(self.config)['enabled']:
                diagnose_one_trial(self.store,directory,contexts=contexts)
        except KeyboardInterrupt:
            self.store.status(directory,'interrupted',case=case,method=method)
            print('Interrupted. Completed trials and the last training checkpoint are saved.',flush=True)
            raise
        except Exception as error:
            atomic_write(directory/'error.txt',lambda f:f.write(traceback.format_exc()),binary=False)
            self.store.status(directory,'failed',case=case,method=method,error=str(error))
            print(f"  Failed (saved): {type(error).__name__}: {error}",flush=True)
        finally:
            self.flush_cohorts(keys)
            save_parameter_results(self.store)

    def adoption(self,case,method,cohort,policy,directory,base_row):
        rng=np.random.default_rng(case['seed'])
        uniforms=rng.uniform(size=(self.config['adoption_draws'],cohort.N))
        rows=[]
        for p_adopt in self.config['adoption_probabilities']:
            path=directory/'adoption'/f'{stable_id(p_adopt)}.json'
            if path.exists():part=load_json(path)
            else:
                part=[]
                for draw,uniform in enumerate(uniforms):
                    metrics=scalar_metrics(evaluate_policy(cohort,policy,case['lambda_'],adoption_mask=uniform<p_adopt))
                    part.append(dict(base_row,**dict(metrics,p_adopt=p_adopt,draw=draw)))
                save_json(path,part)
            rows.extend(part)
        save_json(directory/'result.json',rows)

    def audit(self,case):
        method='Solver and model audit';directory=self.store.trial(case,method)
        if (directory/'result.json').exists():return
        old=directory/'status.json'
        if old.exists() and load_json(old)['status']=='failed' and not self.config['retry_failed']:return
        keys={};self.store.status(directory,'running',case=case,method=method)
        try:
            contexts,keys=self.contexts(case);cohort=contexts['train'];records=[]
            selected=torch.where(cohort.eligible)[0][:self.config['audit_limit']]
            for fraction in self.config['audit_fractions']:
                q=cohort.s_0.clone();q[selected]=cohort.t_0+fraction*(cohort.q_cap[selected]-cohort.t_0)
                # Only audit selected rejected applicants; other q entries are deliberately infeasible/no-action.
                with torch.no_grad():responses=br(cohort,q)
                for i in selected.tolist():
                    records.append(dict(applicant=i,target_fraction=fraction,
                                        target_error=abs(float(cohort.f_w0(responses[i:i+1])[0])-float(q[i])),
                                        cost=float(cost(cohort.X[i:i+1],responses[i:i+1])[0]),
                                        immutable_error=float((responses[i,~cohort.actionable_mask]-cohort.X[i,~cohort.actionable_mask]).abs().max())
                                            if (~cohort.actionable_mask).any() else 0.))
            save_csv(directory/'response_probes.csv',pd.DataFrame(records))
            save_csv(directory/'solver_audit.csv',pd.DataFrame(cohort.solver_audit))
            population,_=self.population(case['seed']);audit=cohort.solver_audit
            row=dict(**self.tags(case,method),status='ok',expected_methods=1,n_probes=len(records),
                     eligible_fraction=float(cohort.eligible[cohort.X_minus].double().mean()),
                     reachability_exact=cohort.q_bar is not None,globally_optimal_br=cohort.q_bar is not None,
                     max_target_error=max([r['target_error'] for r in records],default=0.),
                     max_immutable_error=max([r['immutable_error'] for r in records],default=0.),
                     max_kkt_residual=max([r['kkt_residual'] for r in audit],default=0.),
                     max_local_cost_spread=max([r['local_cost_spread'] for r in audit],default=0.),
                     immutable_features=int((~cohort.actionable_mask).sum()))
            if 'y_test' in population:
                scores=contexts['test'].s_0.numpy();y=population['y_test']
                row.update(predictive_log_loss=log_loss(y,scores,labels=[0,1]),
                           predictive_auc=roc_auc_score(y,scores) if len(np.unique(y))==2 else None)
            save_json(directory/'result.json',row);self.store.status(directory,'completed')
        except KeyboardInterrupt:
            self.store.status(directory,'interrupted');raise
        except Exception as error:
            atomic_write(directory/'error.txt',lambda f:f.write(traceback.format_exc()),binary=False)
            self.store.status(directory,'failed',error=str(error));print(f'Audit failed (saved): {error}')
        finally:
            self.flush_cohorts(keys);save_parameter_results(self.store)


In [ ]:
def directional_gradient_check(cohort,surrogates,tau,lambda_):
    surrogates=copy.deepcopy(surrogates).requires_grad_(True)
    objective,_=J_hat(cohort,surrogates,tau,lambda_)
    parameters=[surrogates.theta_1,surrogates.theta_2]
    gradients=torch.autograd.grad(objective,parameters)
    generator=torch.Generator().manual_seed(91)
    directions=[torch.randn(p.shape,generator=generator) for p in parameters]
    scale=sum(float(d.square().sum()) for d in directions)**0.5
    directions=[d/scale for d in directions];epsilon=1e-5
    analytic=sum(float((g*d).sum()) for g,d in zip(gradients,directions))
    values=[]
    for sign in [-1,1]:
        candidate=copy.deepcopy(surrogates)
        with torch.no_grad():
            for p,d in zip([candidate.theta_1,candidate.theta_2],directions):p.add_(sign*epsilon*d)
            values.append(float(J_hat(cohort,candidate,tau,lambda_)[0]))
    numerical=(values[1]-values[0])/(2*epsilon)
    error=abs(analytic-numerical)
    if not np.isfinite(error) or error>2e-5+2e-3*max(abs(analytic),abs(numerical)):
        raise ArithmeticError(f'Directional gradient check failed: analytic={analytic}, finite difference={numerical}.')
    return dict(analytic_derivative=analytic,finite_difference_derivative=numerical,absolute_error=error)


def preflight(runner):
    c=runner.config;directory=runner.store.directory/'preflight';path=directory/'report.json'
    if path.exists() and load_json(path).get('status')=='passed':
        report=load_json(path);print('Loaded saved preflight.',flush=True);return report
    started=time.perf_counter();rows=[];warnings_list=[];keys_seen={}
    runner.store.status(directory,'running')
    try:
        cases=experiment_cases(c);first_contexts=None;first_keys=None
        for index,case in enumerate(cases):
            contexts,keys=runner.contexts(case,pilot=True,include_test=False);keys_seen.update({v:v for v in keys.values()})
            if first_contexts is None:first_contexts,first_keys=contexts,keys
            surrogates=PolicySurrogates(contexts['train'].X.shape[1],seed=case['seed'],
                                       Theta_1_radius=c['policy']['Theta_1_radius'],Theta_2_radius=c['policy']['Theta_2_radius'])
            objective,outputs=J_hat(contexts['train'],surrogates,case['tau'],case['lambda_'])
            gradients=torch.autograd.grad(objective,[surrogates.theta_1,surrogates.theta_2])
            if not torch.isfinite(objective) or not all(torch.isfinite(g).all() for g in gradients):
                raise FloatingPointError('Preflight found a nonfinite objective/gradient.')
            for split,cohort in contexts.items():
                result=evaluate_policy(cohort,DefinitePolicy(surrogates),case['lambda_'])
                quantiles=torch.quantile(cohort.s_0,as_tensor([0.,0.5,1.]))
                rows.append(dict(case,split=split,N=cohort.N,N_minus=cohort.N_minus,k=cohort.k,t_0=cohort.t_0,
                                 eligible=int(cohort.eligible.sum()),eligible_fraction=result['eligible_fraction'],
                                 min_score=float(quantiles[0]),median_score=float(quantiles[1]),max_score=float(quantiles[2]),
                                 max_reachable_improvement=float((cohort.q_cap-cohort.s_0).max()),
                                 immutable_features=int((~cohort.actionable_mask).sum()),
                                 recommendation_rate=result['recommendation_rate'],max_target_error=result['max_target_error'],
                                 gradient_norm=float(sum(g.square().sum() for g in gradients).sqrt()) if split=='train' else None))
                if result['eligible_fraction']<0.05:
                    warnings_list.append(f"seed={case['seed']}, B={case['B']}, alpha={case['alpha']}, {split}: fewer than 5% eligible.")
            save_csv(directory/'diagnostics.csv',pd.DataFrame(rows))
        gradient=None
        if c['preflight']['finite_difference']:
            gradient=directional_gradient_check(first_contexts['train'],PolicySurrogates(
                first_contexts['train'].X.shape[1],seed=cases[0]['seed'],
                Theta_1_radius=c['policy']['Theta_1_radius'],Theta_2_radius=c['policy']['Theta_2_radius']),
                cases[0]['tau'],cases[0]['lambda_'])
        fit=None;training_count=0;estimate=0.;step_seconds=0.
        if c['experiment']!=7:
            pilot_case=dict(cases[0],T=min(c['T'],c['preflight']['T']))
            method=next((m for m in c['methods'] if learned_method(m)),None)
            if method is not None:
                fit=runner.fit(pilot_case,first_contexts,first_keys,method,pilot=True)
                save_csv(directory/'pilot_history.csv',fit['history'])
                step_seconds=float(fit['history'].step_seconds.median())
                training_count=len(cases)*sum(learned_method(m) is not None for m in c['methods'])
                population,_=runner.population(cases[0]['seed'])
                ratio=len(population['X_train'])/first_contexts['train'].N
                estimate=training_count*(c['T']+1)*step_seconds*ratio
                if float(fit['history'].G_eta_sq.max())<1e-14:
                    warnings_list.append('Pilot gradients are near zero; inspect eligibility, tau, and the objective before a full run.')
                if float(fit['history'].sufficient_decrease.mean())<0.8:
                    warnings_list.append('Frequent pilot descent-check failures; inspect the history and consider a smaller eta.')
        report=dict(status='passed',config_id=runner.store.config_id,created_at=utc_now(),
                    planned_cases=len(cases),planned_trials=len(cases)*(1 if c['experiment']==7 else len(c['methods'])),
                    planned_training_runs=training_count,pilot_seconds_per_iteration=step_seconds,
                    rough_uncached_training_hours=estimate/3600,
                    rough_training_hours_range=[0.5*estimate/3600,2*estimate/3600],
                    timing_note='A rough extrapolation from one small pilot, not a runtime bound. Excludes data/scorer setup, '
                                'reachability, baseline searches, test evaluation and plotting; nonlinear solver difficulty can vary.',
                    gradient_check=gradient,warnings=sorted(set(warnings_list)),elapsed_seconds=time.perf_counter()-started)
        save_json(path,report);runner.store.status(directory,'passed');return report
    except KeyboardInterrupt:
        runner.store.status(directory,'interrupted');raise
    except Exception as error:
        atomic_write(directory/'error.txt',lambda f:f.write(traceback.format_exc()),binary=False)
        save_json(path,dict(status='failed',error=str(error),config_id=runner.store.config_id))
        runner.store.status(directory,'failed',error=str(error));raise
    finally:runner.flush_cohorts(keys_seen)


def load_histories(store):
    frames=[]
    for path in sorted((store.directory/'trials').glob('seed_*/*/history.csv')):
        status_path=path.parent/'status.json'
        if not status_path.exists():continue
        metadata=load_json(status_path)
        if 'case' not in metadata:continue
        history=pd.read_csv(path)
        for key,value in metadata['case'].items():history[key]=value
        history['method']=metadata['method'];history['trial_id']=path.parent.name
        frames.append(history)
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()


def complete_comparisons(frame):
    if frame.empty or 'expected_methods' not in frame:return frame
    keys=['config_id','seed','alpha','B','lambda_','tau','eta','T']
    count=frame.groupby(keys)['method'].transform('nunique')
    keep=count==frame.expected_methods
    if not keep.all():print('Plots omit incomplete method comparisons. All raw successes and failure records remain saved.')
    return frame[keep].copy()


def nondominated_mask(C, V, tolerance=1e-12):
    """Keep observed points with no other point of lower cost and higher validity."""
    C,V=np.asarray(C,dtype=float),np.asarray(V,dtype=float)
    if C.shape!=V.shape or C.ndim!=1 or not np.isfinite(C).all() or not np.isfinite(V).all():
        raise ValueError('Frontier coordinates must be matching, finite vectors.')
    no_worse=(C[:,None]<=C[None,:]+tolerance)&(V[:,None]>=V[None,:]-tolerance)
    better=(C[:,None]<C[None,:]-tolerance)|(V[:,None]>V[None,:]+tolerance)
    return ~(no_worse&better).any(axis=0)


def achieved_tradeoff_points(frame,config):
    """Descriptive means; require the same complete seeds across the whole sweep."""
    lambdas=config['lambda_values'];methods=config['methods'];retained=[]
    for seed in config['seeds']:
        part=frame[frame.seed==seed]
        present=set(zip(part.method,part.lambda_))
        expected={(method,lam) for method in methods for lam in lambdas}
        if expected.issubset(present):retained.append(seed)
    missing=sorted(set(config['seeds'])-set(retained))
    if missing:print(f'Trade-off plots exclude incomplete seeds: {missing}; raw results are retained.')
    frame=frame[frame.seed.isin(retained)]
    rows=[]
    for method in ordered_methods(frame.method):
        part=frame[frame.method==method]
        # One independent record per seed and lambda; never count repeated exports as seeds.
        if part.duplicated(['seed','lambda_']).any():
            raise ValueError('Duplicate seed/lambda records in trade-off data.')
        grouped=part.groupby('lambda_')[['C','V','J']]
        mean=grouped.mean();se=grouped.sem().fillna(0.)
        keep=nondominated_mask(mean.C,mean.V)
        for index,(lam,values) in enumerate(mean.iterrows()):
            rows.append(dict(method=method,label=METHOD_LABELS[method],lambda_=lam,
                             C=values.C,V=values.V,J=values.J,C_se=se.loc[lam,'C'],
                             V_se=se.loc[lam,'V'],J_se=se.loc[lam,'J'],
                             n_seeds=len(retained),nondominated=bool(keep[index])))
    return pd.DataFrame(rows)


def save_experiment_figures(runner,results,*,show=True):
    c=runner.config;e=c['experiment']
    if e in [9,10]:
        return save_focused_figures(runner,results,show=show)
    directory=Path(c.get('figures_dir','./figures')).expanduser().resolve()
    directory.mkdir(parents=True,exist_ok=True)
    stem=parameter_filename(c,runner.store.config_id)
    figures=[];frame=complete_comparisons(results)
    if e==1:
        histories=load_histories(runner.store)
        if not histories.empty:
            fig,axes=plt.subplots(1,2,figsize=(9,3.6))
            etas=sorted(histories.eta.unique());styles=['-','--',':','-.']
            for method in ordered_methods(histories.method):
                for eta in etas:
                    group=histories[(histories.method==method)&(histories.eta==eta)]
                    if group.empty:continue
                    group=group.sort_values(['seed','iteration']).copy()
                    group['best_G']=group.groupby('seed').G_eta_sq.cummin()
                    mean=group.groupby('iteration')[['J_hat','best_G']].mean()
                    for ax,key in zip(axes,['J_hat','best_G']):
                        ax.plot(mean.index,mean[key],color=METHOD_COLORS[method],
                                linestyle=styles[etas.index(eta)%len(styles)],
                                label=f'{METHOD_LABELS[method]}; eta={eta:g}')
            for ax,label in zip(axes,['Mean training objective','Mean best-so-far squared gradient mapping']):
                ax.set(xlabel='Iterations',ylabel=label);ax.grid(alpha=.25)
            axes[1].set_yscale('symlog',linthresh=1e-12)
            positive=histories.loc[histories.G_eta_sq>0,'G_eta_sq']
            if len(positive):
                lower=10.**math.floor(math.log10(float(positive.min())))
                upper=10.**math.ceil(math.log10(float(positive.max())))
                axes[1].set_ylim(0 if (histories.G_eta_sq==0).any() else lower,
                                 upper if upper>lower else 10*upper)
            axes[0].ticklabel_format(axis='y',style='plain',useOffset=False)
            axes[0].legend(fontsize=6);figures.append(('convergence',fig))
            # Keep solver accuracy as a saved/text diagnostic, not a convergence panel.
            capacity_summary=dict(
                max_abs_capacity_residual=float(histories.capacity_residual.abs().max()),
                recorded_iterates=len(histories),
                description='Maximum absolute capacity residual across recorded seeds, step sizes, and iterations.')
            save_json(runner.store.directory/'capacity_residual_summary.json',capacity_summary)
            print('Capacity residual (diagnostic only): maximum |sum_i a_i - k| = '
                  f"{capacity_summary['max_abs_capacity_residual']:.3e} across "
                  f"{capacity_summary['recorded_iterates']} recorded iterates.")
    elif e==2 and not frame.empty:
        points=achieved_tradeoff_points(frame,c)
        if not points.empty:
            # This only summarizes already-frozen test outcomes; it never selects a policy.
            save_csv(runner.store.directory/'cost_validity_points.csv',points)
            save_csv(runner.store.root/'tables'/f'{stem}-cost_validity_points.csv',points)
            fig,ax=plt.subplots(figsize=(8,5.4))
            objective_fig,objective_ax=plt.subplots(figsize=(8,4.8))
            for method in ordered_methods(points.method):
                group=points[points.method==method].sort_values('lambda_')
                color=METHOD_COLORS[method];marker=METHOD_MARKERS[method]
                # All sampled means stay visible; only nondominated ones are connected.
                ax.scatter(group.V,group.C,color=color,marker=marker,s=22,alpha=.25)
                frontier=group[group.nondominated].sort_values(['V','C']).drop_duplicates(['C','V'])
                ax.errorbar(frontier.V,frontier.C,xerr=frontier.V_se,yerr=frontier.C_se,
                            color=color,marker=marker,capsize=2,label=METHOD_LABELS[method])
                objective_ax.errorbar(group.lambda_,group.J,yerr=group.J_se,color=color,
                                      marker=marker,capsize=2,label=METHOD_LABELS[method])
            ax.set(xlabel='Post-shift validity V',ylabel='Mean recourse cost C')
            ax.grid(alpha=.25);ax.legend(fontsize=8)
            objective_ax.set(xlabel='Validity weight lambda',ylabel='Test objective J = C - lambda V',xscale='log')
            objective_ax.grid(alpha=.25);objective_ax.legend(fontsize=8)
            figures.extend([('cost_validity',fig),('objective_lambda',objective_fig)])
    elif e in [3,4,5] and not frame.empty:
        axis={3:'B',4:'alpha',5:'tau'}[e];fig,axes=plt.subplots(1,3,figsize=(13,4))
        for method in ordered_methods(frame.method):
            group=frame[frame.method==method]
            aggregate=group.groupby(axis)[['C','V','J']];mean=aggregate.mean();se=aggregate.sem().fillna(0)
            for ax,key in zip(axes,['C','V','J']):
                ax.errorbar(mean.index,mean[key],yerr=se[key],color=METHOD_COLORS[method],
                            marker=METHOD_MARKERS[method],capsize=2,label=METHOD_LABELS[method])
                ax.set(xlabel=axis,ylabel=key);ax.grid(alpha=.25)
        axes[0].legend(fontsize=6);figures.append((f'{axis}_sensitivity',fig))
    elif e==6 and not frame.empty:
        groups=[('baseline_comparison',MAIN_METHODS),
                ('threshold_ablations',MAIN_METHODS[3:]+ABLATION_METHODS)]
        for name,requested in groups:
            methods=[m for m in ordered_methods(frame.method) if m in requested]
            if not methods:continue
            aggregate=frame[frame.method.isin(methods)].groupby('method')[['C','V','J']]
            mean=aggregate.mean().reindex(methods);se=aggregate.sem().fillna(0).reindex(methods)
            fig,axes=plt.subplots(1,3,figsize=(14,4.4),sharey=True)
            for index,(ax,key) in enumerate(zip(axes,['C','V','J'])):
                ax.barh(np.arange(len(methods)),mean[key],xerr=se[key],
                        color=[METHOD_COLORS[method] for method in methods])
                ax.set_yticks(np.arange(len(methods)),[METHOD_LABELS[m] for m in methods])
                ax.tick_params(axis='y',labelleft=index==0);ax.set_xlabel(key);ax.grid(axis='x',alpha=.2)
            axes[0].invert_yaxis();figures.append((name,fig))
    elif e==8 and not frame.empty:
        # Average matched Monte Carlo draws within seeds, then SE across seeds.
        seed_means=frame.groupby(['method','seed','p_adopt'])[['C','V']].mean().reset_index()
        fig,axes=plt.subplots(1,2,figsize=(11,4.2))
        for method in ordered_methods(seed_means.method):
            group=seed_means[seed_means.method==method]
            aggregate=group.groupby('p_adopt')[['C','V']];mean=aggregate.mean();se=aggregate.sem().fillna(0)
            for ax,key in zip(axes,['C','V']):
                ax.errorbar(mean.index,mean[key],yerr=se[key],color=METHOD_COLORS[method],
                            marker=METHOD_MARKERS[method],capsize=2,label=METHOD_LABELS[method])
                ax.set(xlabel='Adoption probability',ylabel=key);ax.grid(alpha=.25)
        axes[0].legend(fontsize=6);figures.append(('adoption',fig))
    for name,fig in figures:
        fig.suptitle(f"{c['dataset']} | {c['model']}");standardize_figure(fig);fig.tight_layout()
        path=directory/f'{stem}-{name}.png'
        atomic_write(path,lambda handle:fig.savefig(handle,format='png',dpi=160,bbox_inches='tight'))
        save_json(directory/f'{stem}-{name}.json',dict(
            config=c,config_id=runner.store.config_id,plot=name,method_order=list(METHOD_ORDER),
            method_labels=METHOD_LABELS,method_colors=METHOD_COLORS,
            run_directory=str(runner.store.directory),figure_file=str(path),
            frontier_note=('Within-method nondominated seed means; descriptive, no global-optimality guarantee. '
                           'Lines are visual guides; all sampled means and standard errors are saved.') if e==2 else None))
        if show:plt.show()
        plt.close(fig)
    return [str(directory/f'{stem}-{name}.png') for name,_ in figures]


def select_experiment_configs(configs_by_dataset, selected_datasets, selected_models=None):
    """Dataset order first, then model order; returned configurations are independent copies."""
    if isinstance(selected_datasets, str):
        raise ValueError('Use a list of dataset names, for example ["German Credit"].')
    names = list(selected_datasets)
    if len(names) != len(set(names)):
        raise ValueError('Select each dataset only once.')
    unknown = [name for name in names if name not in configs_by_dataset]
    if unknown:
        raise ValueError(f'Unknown selected datasets: {unknown}. Available: {list(configs_by_dataset)}')
    if isinstance(selected_models, str):
        raise ValueError('Use a list of models, for example ["affine", "quadratic"].')
    models = None if selected_models is None else list(selected_models)
    if models is not None and len(models) != len(set(models)):
        raise ValueError('Select each model only once.')
    selected = []
    for dataset in names:
        entry = configs_by_dataset[dataset]
        # Older flat configuration dictionaries remain accepted for programmatic callers.
        by_model = {entry['model']:entry} if 'model' in entry else entry
        requested = list(by_model) if models is None else models
        missing = [name for name in requested if name not in by_model]
        if missing:
            raise ValueError(f'{dataset}: models {missing} are not configured. Available: {list(by_model)}')
        for model in requested:
            config = copy.deepcopy(by_model[model])
            if config['dataset'] != dataset or config['model'] != model:
                raise ValueError('The configuration dataset/model must match its dictionary keys.')
            selected.append(config)
    return selected


def execute_experiment(configs,*,mode='run',show=True):
    """Run selected configurations sequentially; each keeps its own saved results."""
    if mode not in {'preflight','run','load','skip'}:raise ValueError('Unknown execution mode.')
    if mode=='skip':return []
    configs=list(configs)
    if not configs:
        print('No datasets selected; nothing to run.')
        return []
    for config in configs:validate_config(config)
    reports=[]
    for position,config in enumerate(configs,1):
        print(f"Selected configuration {position}/{len(configs)}: {config['dataset']}",flush=True)
        runner=ExperimentRunner(config);c=runner.config
        print(f"Experiment {c['experiment']} | {c['dataset']} | {c['model']} | {runner.store.config_id}")
        print(f'Results: {runner.store.directory}',flush=True)
        print(f"Named tables: {runner.store.root / 'tables'} | Figures: {Path(c.get('figures_dir','./figures')).resolve()}",flush=True)
        if mode!='load':
            report=preflight(runner);reports.append(report)
            display(pd.DataFrame([{key:value for key,value in report.items()
                                   if key in ['status','planned_trials','planned_training_runs','pilot_seconds_per_iteration','rough_uncached_training_hours']}]))
            for message in report['warnings']:print('Preflight note:',message)
            print(report['timing_note'])
            if mode=='preflight':
                display(pd.read_csv(runner.store.directory/'preflight'/'diagnostics.csv'))
                print('Review these diagnostics; set this experiment mode to "run" to start its full configuration.')
                continue
            for case in experiment_cases(c):
                if c['experiment']==7:runner.audit(case)
                else:
                    for method in c['methods']:runner.run_trial(case,method)
        frame=save_parameter_results(runner.store)
        failures=[]
        for path in sorted((runner.store.directory/'trials').glob('seed_*/*/status.json')):
            record=load_json(path)
            if record['status']!='completed':failures.append(dict(trial_directory=str(path.parent),**record))
        save_json(runner.store.directory/'incomplete_trials.json',failures)
        if failures:print(f'{len(failures)} incomplete/failed trials; see incomplete_trials.json. No fabricated zero results were added.')
        if not frame.empty:
            display(frame);save_experiment_figures(runner,frame,show=show)
            if c['experiment'] in [9,10]:save_focused_tables(runner,frame)
        else:print('No saved successful trials for this configuration.')
        reports.append(dict(directory=str(runner.store.directory),results=frame,incomplete=failures))
    return reports


def collect_saved_results(results_dir='./Results/',*,experiment=None,dataset=None):
    """Collect all saved datasets/configurations without fitting models or rerunning trials."""
    root=Path(results_dir).expanduser().resolve();rows=[]
    for path in sorted((root/'experiments').glob('experiment_*/*/*/*/trials/seed_*/*/result.json')):
        value=load_json(path);entries=value if isinstance(value,list) else [value]
        for row in entries:
            if experiment is not None and row['experiment']!=experiment:continue
            if dataset is not None and row['dataset']!=dataset:continue
            rows.append(row)
    frame=pd.DataFrame(rows)
    suffix=slug(f"experiment_{experiment or 'all'}_{dataset or 'all_datasets'}")
    save_csv(root/'summaries'/f'{suffix}.csv',frame)
    return frame


## Saved diagnostics and focused-study summaries


In [ ]:
DIAGNOSTIC_VERSION = 'consolidated-v1'
def diagnostic_options(config):
    options = dict(enabled=True, every=25, figures=True,
                   rho_cutoffs=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
    options.update(config.get('diagnostics', {}))
    return options

@torch.no_grad()
def allocation_diagnostics(cohort, q_hat, rho_hat, tau, lambda_, *, br_X=None, rho_cutoff=0.5):
    """Same cohort/policy, three allocations. All C,V,J average over X_minus.

    Exact mixture means weighted population mass, not a sampled adoption experiment.
    Canonical target scores preserve ties exactly as evaluate_policy does.
    """
    if cohort.N_minus == 0 or not 1 <= cohort.k < cohort.N:
        raise ValueError('Diagnostics require rejections and a valid integer capacity.')
    q_hat = as_tensor(q_hat).expand(cohort.N).detach()
    rho_hat = as_tensor(rho_hat).expand(cohort.N).detach()
    if not torch.isfinite(rho_hat).all() or ((rho_hat < 0) | (rho_hat > 1)).any():
        raise ValueError('rho_hat must lie in [0,1].')
    feasible = target_feasible(cohort, q_hat)
    rho_hat = torch.where(feasible, rho_hat, torch.zeros_like(rho_hat))
    if br_X is None: br_X = br(cohort, q_hat)
    br_X = br_X.detach()
    costs = cost(cohort.X, br_X)
    s_br_actual = cohort.f_w0(br_X)
    s_br = torch.where(feasible, q_hat, cohort.s_0)
    if not torch.allclose(s_br_actual, s_br, atol=2e-9, rtol=2e-9):
        raise ArithmeticError('Diagnostic response failed target attainment.')
    minus = cohort.X_minus
    t_hat = SmoothCapacity.apply(cohort.s_0, s_br_actual, rho_hat, cohort.k, float(tau))
    smooth = a_i(t_hat, cohort.s_0, s_br_actual, rho_hat, tau)
    allocation, t_mixture, _ = exact_allocation(
        torch.cat([cohort.s_0, s_br]), cohort.k,
        mass=torch.cat([1-rho_hat, rho_hat]))
    mixture = (1-rho_hat)*allocation[:cohort.N] + rho_hat*allocation[cohort.N:]
    rho = feasible & (rho_hat >= rho_cutoff)
    hard, t_hard, _ = exact_allocation(torch.where(rho, s_br, cohort.s_0), cohort.k)
    C_hat = float((rho_hat*costs)[minus].mean())
    V_hat = float(smooth[minus].mean())
    V_mixture = float(mixture[minus].mean())
    C_hard = float((rho*costs)[minus].mean())
    V_hard = float(hard[minus].mean())
    wasted_cost = float((rho*costs*(1-hard))[minus].mean())
    zero = torch.zeros_like(rho_hat)
    t_no_action = SmoothCapacity.apply(cohort.s_0, cohort.s_0, zero, cohort.k, float(tau))
    no_action_soft = a_i(t_no_action, cohort.s_0, cohort.s_0, zero, tau)
    no_action_hard, _, _ = exact_allocation(cohort.s_0, cohort.k)
    eligible = cohort.eligible
    weights = rho_hat[eligible]
    gap = (q_hat-cohort.t_0)[eligible]
    def mean(values):return float(values.mean()) if values.numel() else float('nan')
    def quantile(values, p):return float(torch.quantile(values, p)) if values.numel() else float('nan')
    J_hat_value = C_hat-lambda_*V_hat
    J_mixture = C_hat-lambda_*V_mixture
    J_hard = C_hard-lambda_*V_hard
    return dict(
        C_hat=C_hat, V_hat=V_hat, J_hat=J_hat_value,
        C_mixture=C_hat, V_mixture=V_mixture, J_mixture=J_mixture,
        C_hard=C_hard, V_hard=V_hard, J_hard=J_hard,
        smoothing_gap_V=V_hat-V_mixture, rounding_gap_V=V_mixture-V_hard,
        smoothing_gap_J=J_mixture-J_hat_value, rounding_gap_J=J_hard-J_mixture,
        mean_acceptance_error=float((smooth-mixture).abs().mean()),
        capacity_residual=abs(float(smooth.sum())-cohort.k),
        mixture_capacity_residual=abs(float(mixture.sum())-cohort.k),
        hard_capacity_residual=abs(float(hard.sum())-cohort.k),
        t_hat=float(t_hat), t_mixture=t_mixture, t_hard=t_hard,
        no_action_V_hat=mean(no_action_soft[minus]), no_action_V_hard=mean(no_action_hard[minus]),
        no_action_J_hat=-lambda_*mean(no_action_soft[minus]),
        no_action_J_hard=-lambda_*mean(no_action_hard[minus]),
        recommendation_rate=mean(rho[minus].double()), eligible_fraction=mean(eligible[minus].double()),
        rho_hat_mean=mean(weights), rho_hat_q10=quantile(weights, .1),
        rho_hat_median=quantile(weights, .5), rho_hat_q90=quantile(weights, .9),
        rho_hat_above_half=mean((weights>=.5).double()),
        rho_hat_near_half=mean((torch.abs(weights-.5)<=.05).double()),
        recommendations_collapsed=bool(eligible.any() and not rho.any()),
        q_hat_gap_mean=mean(gap), q_hat_gap_median=quantile(gap, .5),
        tau_over_q_hat_gap=tau/mean(gap) if gap.numel() and mean(gap)>0 else float('nan'),
        score_std=float(cohort.s_0.std(unbiased=False)),
        score_iqr=quantile(cohort.s_0,.75)-quantile(cohort.s_0,.25),
        wasted_cost=wasted_cost, wasted_cost_fraction=wasted_cost/C_hard if C_hard>0 else float('nan'),
        N=cohort.N, N_minus=cohort.N_minus, k=cohort.k,
        max_target_error=float((s_br_actual-s_br).abs().max()))

def surrogates_from_state(state, config, method):
    s = PolicySurrogates(state['theta_1'].numel()-1,
                        Theta_1_radius=config['policy']['Theta_1_radius'],
                        Theta_2_radius=config['policy']['Theta_2_radius'],
                        select_recipients=learned_method(method)[1])
    s.load_state_dict(state)
    return s.eval().requires_grad_(False)

def history_diagnostics(history, selected_iteration, T, eta):
    if history.empty:return dict(history_available=False)
    h=history.sort_values('iteration')
    differences=h.J_hat.diff()
    full=bool(np.array_equal(h.iteration.to_numpy(),np.arange(T+1)))
    descent=(h.J_hat.to_numpy()[1:] <= h.J_hat.to_numpy()[:-1]
             -eta*h.G_eta_sq.to_numpy()[:-1]/2+1e-10)
    return dict(history_available=True, history_complete=full, history_rows=len(h),
                initial_J_hat=float(h.J_hat.iloc[0]), final_J_hat=float(h.J_hat.iloc[-1]),
                final_G_eta_sq=float(h.G_eta_sq.iloc[-1]), best_G_eta_sq=float(h.G_eta_sq.min()),
                objective_increases=int((differences>1e-10).sum()),
                sufficient_decrease_failures=int((~descent).sum()) if full else None,
                max_capacity_residual=float(h.capacity_residual.abs().max()),
                initial_validation_J=float(h.validation_J.iloc[0]),
                final_validation_J=float(h.validation_J.iloc[-1]),
                selected_iteration=selected_iteration,
                selected_no_action=selected_iteration==-1,
                selected_early=bool(selected_iteration is not None and 0<=selected_iteration<=10),
                diagnostic_history_points=int(h.get('diag_J_hard',pd.Series(dtype=float)).notna().sum()))

@torch.no_grad()
def trial_diagnostics(contexts, config, case, method, policy, *, checkpoint=None, history=None):
    """Initial/final on train/validation; only the selected policy is examined on test."""
    snapshots=[];cutoffs=[]
    candidates=[]
    if checkpoint is not None:
        for stage,key,iteration in [('initial','initial_state',0),('final','state_dict',case['T'])]:
            s=surrogates_from_state(checkpoint[key],config,method)
            candidates.append((stage,iteration,s,False))
    selected_iteration=checkpoint.get('selected_iteration') if checkpoint is not None else None
    if method=='Personalized targets + validated cutoff' and checkpoint is not None:
        selected_iteration=checkpoint['cutoff_iteration']
        if history is not None and 'validation_cutoff_J' in history:
            history=history.copy();history['validation_J']=history['validation_cutoff_J']
    selected_source = ('no_action' if isinstance(policy,NoRecourse) else
                       'learned' if isinstance(policy,DefinitePolicy) else 'common_target')
    selected_model=policy.surrogates if isinstance(policy,DefinitePolicy) else policy
    candidates.append(('selected',selected_iteration,selected_model,not isinstance(policy,DefinitePolicy)))
    for stage,iteration,model,binary in candidates:
        for split in (['train','val','test'] if stage=='selected' else ['train','val']):
            cohort=contexts[split];q_hat,rho_hat=model(cohort)
            cutoff=policy.rho_cutoff if stage=='selected' and isinstance(policy,DefinitePolicy) else .5
            values=allocation_diagnostics(cohort,q_hat,rho_hat.double(),case['tau'],case['lambda_'],rho_cutoff=cutoff)
            snapshots.append(dict(stage=stage,split=split,iteration=iteration,
                                  selected_source=selected_source,
                                  recommendation_type='binary' if binary else 'fractional',rho_cutoff=cutoff,**values))
        # Cutoff comparisons are validation diagnostics, never test-based selection.
        if stage=='final':
            cohort=contexts['val'];q_hat,rho_hat=model(cohort)
            for cutoff in sorted(set(diagnostic_options(config)['rho_cutoffs']+[.5])):
                def candidate(pop, cutoff=cutoff):
                    q,weights=model(pop)
                    return q,pop.eligible & (weights>=cutoff)
                metrics=scalar_metrics(evaluate_policy(cohort,candidate,case['lambda_']))
                cutoffs.append(dict(stage=stage,split='val',rho_cutoff=cutoff,rule='threshold',**metrics))
            metrics=scalar_metrics(evaluate_policy(cohort,NoRecourse(),case['lambda_']))
            cutoffs.append(dict(stage=stage,split='val',rho_cutoff=None,rule='no_action',**metrics))
            best=min(range(len(cutoffs)),key=lambda i:cutoffs[i]['J'])
            for i,row in enumerate(cutoffs):row['best_on_validation']=i==best
    summary=history_diagnostics(history if history is not None else pd.DataFrame(),
                                selected_iteration,case['T'],case['eta'])
    summary['selected_source']=selected_source
    summary['diagnostic_version']=DIAGNOSTIC_VERSION
    return dict(snapshots=pd.DataFrame(snapshots),cutoffs=pd.DataFrame(cutoffs),summary=summary)

def diagnostic_id(config):
    return stable_id(dict(version=DIAGNOSTIC_VERSION,engine=implementation_id()[0],
                          rho_cutoffs=diagnostic_options(config)['rho_cutoffs']))

def save_trial_diagnostics(store, case, method, bundle, *, identity=None):
    identity=identity or diagnostic_id(store.config)
    trial=store.trial(case,method).relative_to(store.directory)
    directory=store.directory/'diagnostics'/identity/trial
    tags=dict(experiment=store.config['experiment'],dataset=store.config['dataset'],model=store.config['model'],
              config_id=store.config_id,implementation_id=store.code_id,diagnostic_id=identity,method=method,**case)
    for name in ['snapshots','cutoffs']:
        frame=bundle[name].copy()
        for key,value in tags.items():frame[key]=value
        save_csv(directory/f'{name}.csv',frame)
    save_json(directory/'summary.json',dict(tags,**bundle['summary']))
    save_json(directory/'status.json',dict(status='completed',diagnostic_version=DIAGNOSTIC_VERSION))
    return directory

def collect_diagnostics(store, identity):
    directory=store.directory/'diagnostics'/identity
    tables={}
    for name in ['snapshots','cutoffs']:
        frames=[]
        for path in sorted(directory.glob(f'trials/seed_*/*/{name}.csv')):
            status=path.parent/'status.json'
            if not status.exists() or load_json(status)['status']!='completed':continue
            try:frame=pd.read_csv(path)
            except pd.errors.EmptyDataError:continue
            if not frame.empty:frames.append(frame)
        tables[name]=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
    summaries=[load_json(p) for p in sorted(directory.glob('trials/seed_*/*/summary.json'))
               if (p.parent/'status.json').exists() and load_json(p.parent/'status.json')['status']=='completed']
    tables['history_summary']=pd.DataFrame(summaries)
    # Pair within a seed and exact experimental setting; never treat reused runs as extra seeds.
    results=store.manifest_results()
    keys=['config_id','seed','alpha','B','lambda_','tau','eta','T']
    if (not results.empty and 'J' in results and
        {'Personalized targets + selection','Common target'}.issubset(set(results.method))):
        main=results[results.method=='Personalized targets + selection']
        baseline=results[results.method=='Common target']
        paired=main.merge(baseline,on=keys,suffixes=('_personalized','_common'),validate='one_to_one')
        for metric in ['C','V','J']:
            paired[f'delta_{metric}']=paired[f'{metric}_personalized']-paired[f'{metric}_common']
        tables['paired_comparisons']=paired
    else:tables['paired_comparisons']=pd.DataFrame()
    stem=parameter_filename(store.config,store.config_id)[:185]+'-diag'+identity[:12]
    for name,frame in tables.items():
        save_csv(directory/f'{name}.csv',frame)
        save_csv(store.root/'tables'/f'{stem}-{name}.csv',frame)
    save_json(directory/'metadata.json',dict(config=store.config,config_id=store.config_id,diagnostic_id=identity,
              diagnostic_version=DIAGNOSTIC_VERSION,notes=[
                  'All C,V,J are conditional on initial rejection.',
                  'Smoothing and rounding gaps compare the same cohort and policy.',
                  'Cutoff comparisons use validation only and do not change the saved policy.',
                  'Old histories without diagnostic columns are not retrospectively fabricated.',
                  'wasted_cost_fraction is undefined (blank) when no costs are incurred.']))
    return tables


In [ ]:
class SavedDiagnosticReader:
    """Read existing tensor caches only. Missing data never triggers fitting or downloading."""
    def __init__(self, root, code_id):
        self.root=Path(root);self.cache=self.root/'cache'/code_id
        self.index=None;self.models=None;self.resolved={}

    def payload(self, relative):
        path=(self.root/relative).resolve()
        if self.root.resolve() not in path.parents:
            raise ValueError('Saved diagnostic paths must be inside results_dir.')
        if not path.exists():raise FileNotFoundError(f'Missing saved cache: {path}')
        return load_torch(path)

    def contexts(self, directory, config, case, record, policy):
        manifest=directory/'context.json'
        if manifest.exists():
            metadata=load_json(manifest);model=restore_scorer(self.payload(metadata['scorer']))
            return {split:Cohort(f_w0=model,**self.payload(path)) for split,path in metadata['cohorts'].items()}
        # Older notebooks did not save context manifests. Identify train/validation by
        # their exact saved fit hash, then verify the test cohort against saved metrics.
        cache_key=(config['dataset'],config['model'],case['seed'],case['alpha'],case['B'],record['t_0'])
        if cache_key in self.resolved:return self.resolved[cache_key]
        if self.index is None:
            self.index=[]
            for path in sorted((self.cache/'cohorts').glob('*/cohort.pt')):
                c=load_torch(path)
                self.index.append(dict(path=path,key=path.parent.name,
                                       **{k:c[k] for k in ['t_0','B','alpha','N','N_minus','k']}))
        possible=[item for item in self.index if item['B']==case['B'] and item['alpha']==case['alpha']
                  and abs(item['t_0']-record['t_0'])<1e-12]
        fit_path=directory/'fit.json'
        fit_method=record['method']
        if not fit_path.exists():
            for status_path in sorted(directory.parent.glob('*/status.json')):
                status=load_json(status_path)
                candidate=status_path.parent/'fit.json'
                if status.get('case')==case and candidate.exists():
                    fit_path=candidate;fit_method=status['method'];break
        if not fit_path.exists():
            raise FileNotFoundError('Legacy run needs one learned fit for the same case to identify its cohorts.')
        fit=load_json(fit_path);mode,select=learned_method(fit_method)
        options=dict(tau=case['tau'],eta=case['eta'],T=case['T'],lambda_=case['lambda_'],seed=case['seed'],
                     mode=mode,select_recipients=select,**config['policy'])
        if 'diagnostics' in config:
            opts=diagnostic_options(config);options['diagnostic_every']=opts['every'] if opts['enabled'] else 0
        matches=[(a,b) for a in possible for b in possible
                 if stable_id(dict(train=a['key'],val=b['key'],options=options,pilot=False))==fit['fit_id']]
        if len(matches)!=1:
            raise ValueError('Could not uniquely identify saved train/validation cohorts; original caches are required.')
        a,b=matches[0];train=load_torch(a['path']);val=load_torch(b['path'])
        if self.models is None:
            self.models=[restore_scorer(load_torch(p)) for p in sorted((self.cache/'scorers').glob('*/scorer.pt'))]
        matching=[]
        with torch.no_grad():
            for model in self.models:
                if model.model_type!=config['model']:continue
                try:
                    if torch.allclose(model(train['X']),train['s_0'],atol=1e-12,rtol=1e-12):matching.append(model)
                except (RuntimeError,ValueError):continue
        if len(matching)!=1:raise ValueError('Could not uniquely identify the original frozen scorer.')
        model=matching[0]
        tests=[]
        for item in possible:
            if item['key'] in {a['key'],b['key']} or item['N_minus']!=record['N_minus'] or item['k']!=record['k']:continue
            test=Cohort(f_w0=model,**load_torch(item['path']))
            with torch.no_grad():
                if not torch.allclose(model(test.X),test.s_0,atol=1e-12,rtol=1e-12):continue
            metrics=evaluate_policy(test,policy,case['lambda_'])
            if all(abs(metrics[k]-record[k])<=1e-8 for k in ['C','V','J']):tests.append(test)
        if len(tests)!=1:raise ValueError('Could not uniquely verify the original test cohort.')
        contexts=dict(train=Cohort(f_w0=model,**train),val=Cohort(f_w0=model,**val),test=tests[0])
        self.resolved[cache_key]=contexts
        return contexts

def diagnose_one_trial(store, directory, *, reader=None, contexts=None, overwrite=False, identity=None):
    identity=identity or diagnostic_id(store.config)
    output=store.directory/'diagnostics'/identity/directory.relative_to(store.directory)
    status_path=output/'status.json'
    if not overwrite and status_path.exists() and load_json(status_path).get('status')=='completed':
        return dict(status='cached',trial=str(directory),diagnostics=str(output))
    try:
        status=load_json(directory/'status.json')
        if status.get('status')!='completed':return dict(status='skipped',trial=str(directory),reason='Trial incomplete.')
        case=status['case'];method=status['method'];record=load_json(directory/'result.json')
        if isinstance(record,list):
            # Adoption stress tests: reconstruct the fully implemented base policy only.
            rows=[r for r in record if r.get('p_adopt')==1]
            if not rows:raise ValueError('Saved adoption run has no full-adoption reference result.')
            record=rows[0]
        policy=decode_policy(load_torch(directory/'policy.pt'))
        if contexts is None:
            reader=reader or SavedDiagnosticReader(store.root,store.code_id)
            contexts=reader.contexts(directory,store.config,case,record,policy)
        checkpoint=None;history=None
        if learned_method(method) is not None:
            fit=load_json(directory/'fit.json')
            path=store.root/'cache'/store.code_id/'fits'/fit['fit_id']/'checkpoint.pt'
            checkpoint=load_torch(path)
            history=pd.read_csv(directory/'history.csv')
            if checkpoint['next_iteration']!=case['T']+1:
                raise ValueError('The saved training checkpoint is incomplete.')
        bundle=trial_diagnostics(contexts,store.config,case,method,policy,checkpoint=checkpoint,history=history)
        # Verify reconstruction before labeling the diagnostic bundle complete.
        selected=bundle['snapshots'].query("stage == 'selected' and split == 'test'").iloc[0]
        if not isinstance(load_json(directory/'result.json'),list):
            for key in ['C','V','J']:
                if abs(selected[key+'_hard']-record[key])>1e-8:
                    raise ArithmeticError(f'Reconstructed {key} differs from the saved result.')
        save_trial_diagnostics(store,case,method,bundle,identity=identity)
        return dict(status='completed',trial=str(directory),diagnostics=str(output))
    except Exception as error:
        save_json(status_path,dict(status='failed',error=f'{type(error).__name__}: {error}'))
        return dict(status='failed',trial=str(directory),error=f'{type(error).__name__}: {error}')

def diagnose_saved_runs(results_dir='./Results', *, figures_dir='./figures', experiment=None,
                        dataset=None, model=None, config_id=None, seeds=None, lambda_=None,
                        B=None, alpha=None, rho_cutoffs=None, overwrite=False, show=False,
                        make_figures=True):
    """Add diagnostic sidecars to matching saved runs; never refit or change saved policies.

    Original result/config/history files are retained. Historical per-iteration hard
    metrics cannot be recovered when those intermediate parameter states were not saved.
    """
    root=Path(results_dir).expanduser().resolve();reports=[];readers={}
    for path in sorted((root/'experiments').glob('experiment_*/*/*/*/config.json')):
        metadata=load_json(path);config=metadata['config']
        if any(value is not None and config[key]!=value for key,value in
               [('experiment',experiment),('dataset',dataset),('model',model)]):continue
        if config_id is not None and metadata['config_id']!=config_id:continue
        store=object.__new__(RunStore);store.root=root;store.directory=path.parent
        store.config=copy.deepcopy(config);store.config['figures_dir']=str(figures_dir)
        if rho_cutoffs is not None:
            if not rho_cutoffs or any(not np.isfinite(v) or not 0<=v<=1 for v in rho_cutoffs):
                raise ValueError('rho_cutoffs must be a nonempty list in [0,1].')
            store.config.setdefault('diagnostics',{})['rho_cutoffs']=list(rho_cutoffs)
        store.config_id=metadata['config_id'];store.code_id=metadata['implementation_id']
        identity=diagnostic_id(store.config)
        reader=readers.setdefault(store.code_id,SavedDiagnosticReader(root,store.code_id))
        processed=[]
        for status_path in sorted(store.directory.glob('trials/seed_*/*/status.json')):
            status=load_json(status_path);case=status.get('case')
            if case is None or status.get('status')!='completed':continue
            if seeds is not None and case['seed'] not in seeds:continue
            if any(value is not None and case[key]!=value for key,value in
                   [('lambda_',lambda_),('B',B),('alpha',alpha)]):continue
            if not (status_path.parent/'policy.pt').exists():continue
            report=diagnose_one_trial(store,status_path.parent,reader=reader,overwrite=overwrite,identity=identity)
            processed.append(report)
            if report['status']=='failed':print('Diagnostic unavailable:',report['error'])
        if not processed:continue
        tables=collect_diagnostics(store,identity)
        figures=save_diagnostic_figures(store,tables,identity,show=show) if make_figures else []
        save_json(store.directory/'diagnostics'/identity/'last_request.json',dict(
            filters=dict(experiment=experiment,dataset=dataset,model=model,seeds=seeds,lambda_=lambda_,B=B,alpha=alpha),
            trials=processed))
        report=dict(dataset=config['dataset'],model=config['model'],experiment=config['experiment'],
                    config_id=store.config_id,diagnostic_id=identity,
                    completed=sum(p['status']=='completed' for p in processed),
                    cached=sum(p['status']=='cached' for p in processed),failed=sum(p['status']=='failed' for p in processed),
                    directory=str(store.directory/'diagnostics'/identity),figures=figures)
        reports.append(report)
        print(f"{config['dataset']} | {config['model']}: {report['completed']} diagnosed, "
              f"{report['cached']} already saved, {report['failed']} unavailable.",flush=True)
    return pd.DataFrame(reports)

def save_diagnostic_figures(store, tables, identity, *, show=False):
    """One diagnostic figure per saved configuration; same-case seed means with SEM."""
    frame=tables['snapshots']
    if frame.empty:return []
    # Focused figures explicitly separate eta and tau; do not pool settings here.
    if store.config['experiment'] in [9,10]:return []
    methods=list(frame.method.unique())
    method='Personalized targets + selection' if 'Personalized targets + selection' in methods else methods[0]
    final=frame[(frame.method==method)&(frame.stage=='final')&(frame.split=='train')]
    selected=frame[(frame.method==method)&(frame.stage=='selected')&(frame.split=='test')]
    if final.empty:return []
    axis={1:'eta',2:'lambda_',3:'B',4:'alpha',5:'tau'}.get(store.config['experiment'])
    if axis is None:
        final=final.assign(setting=0.);selected=selected.assign(setting=0.);axis='setting'
    labels={'lambda_':'lambda','alpha':'Admission rate','B':'Budget','eta':'Step size','tau':'Temperature','setting':'Configured setting'}
    fig,axes=plt.subplots(2,3,figsize=(14,7))
    def curves(ax,data,columns,ylabel):
        for key,label,color in columns:
            grouped=data.groupby(axis)[key];mean=grouped.mean();sem=grouped.sem()
            ax.errorbar(mean.index,mean.values,yerr=None if sem.isna().all() else sem.to_numpy(),
                        marker='o',capsize=3,label=label,color=color)
        ax.set(xlabel=labels[axis],ylabel=ylabel);ax.grid(alpha=.25);ax.legend(fontsize=7)
        if axis in ['lambda_','tau','eta'] and (data[axis]>0).all():ax.set_xscale('log')
    curves(axes[0,0],final,[('V_hat','Smooth / fractional','#1f77b4'),
        ('V_mixture','Exact / fractional','#ff7f0e'),('V_hard','Exact / binary','#2ca02c')],'Final training validity')
    curves(axes[0,1],final,[('J_hat','Smooth / fractional','#1f77b4'),
        ('J_mixture','Exact / fractional','#ff7f0e'),('J_hard','Exact / binary','#2ca02c')],'Final training objective')
    curves(axes[0,2],final,[('rho_hat_mean','Mean rho_hat','#1f77b4'),
        ('rho_hat_above_half','Fraction with rho_hat >= 0.5','#2ca02c')],'Among eligible training applicants')
    axes[0,2].set_ylim(-.03,1.03)
    curves(axes[1,0],selected,[('C_hard','Total cost','#1f77b4'),
        ('wasted_cost','Unsuccessful cost','#d62728')],'Selected policy: test cost')
    histories=tables['history_summary']
    histories=histories[histories.method==method].copy()
    if axis=='setting':histories[axis]=0.
    curves(axes[1,1],histories,[('selected_iteration','Selected iteration (-1 = no action)','#9467bd')],'Validation-selected iteration')
    curves(axes[1,2],final,[('no_action_V_hat','Smooth no action','#1f77b4'),
        ('no_action_V_hard','Exact no action','#2ca02c')],'No-action training validity')
    fig.suptitle(f"{store.config['dataset']} | {store.config['model']}\n{method}")
    standardize_figure(fig);fig.tight_layout()
    directory=Path(store.config.get('figures_dir','./figures')).expanduser().resolve()
    stem=parameter_filename(store.config,store.config_id)[:185]+'-diag'+identity[:12]
    path=directory/f'{stem}-diagnostics.png'
    atomic_write(path,lambda f:fig.savefig(f,format='png',dpi=160,bbox_inches='tight'))
    save_json(path.with_suffix('.json'),dict(config=store.config,config_id=store.config_id,
              diagnostic_id=identity,method=method,notes='Error bars: SEM over available seeds; inspect CSV counts.'))
    if show:plt.show()
    plt.close(fig)
    paths=[str(path)]
    if store.config['experiment']==1:
        histories=load_histories(store)
        if 'diag_J_hard' in histories and histories.diag_J_hard.notna().any():
            histories=histories[histories.method==method]
            values=sorted(histories.eta.unique())
            fig,axes=plt.subplots(1,len(values),figsize=(6*len(values),3.8),squeeze=False)
            for ax,eta in zip(axes[0],values):
                group=histories[histories.eta==eta]
                for key,label,color in [('J_hat','Training: optimized objective','#1f77b4'),
                    ('diag_J_mixture','Training: exact fractional','#ff7f0e'),
                    ('diag_J_hard','Training: exact binary','#2ca02c'),
                    ('validation_J','Validation: exact binary','#9467bd')]:
                    observed=group.dropna(subset=[key]).groupby('iteration')[key]
                    mean=observed.mean();sem=observed.sem()
                    ax.plot(mean.index,mean.values,label=label,color=color)
                    if sem.notna().any():ax.fill_between(mean.index,mean-sem,mean+sem,color=color,alpha=.12)
                ax.set(title=f'eta = {eta:g}',xlabel='Projected updates',ylabel='Objective')
                ax.grid(alpha=.25);ax.legend(fontsize=7)
            fig.suptitle(f"{store.config['dataset']} | {store.config['model']}")
            standardize_figure(fig);fig.tight_layout()
            path=directory/f'{stem}-training-diagnostics.png'
            atomic_write(path,lambda f:fig.savefig(f,format='png',dpi=160,bbox_inches='tight'))
            if show:plt.show()
            plt.close(fig);paths.append(str(path))
    return paths

def select_config_pairs(configs_by_dataset, selected_pairs):
    """Run explicitly selected (dataset, model) pairs in the requested order."""
    pairs=[tuple(pair) for pair in selected_pairs]
    if any(len(pair)!=2 for pair in pairs) or len(pairs)!=len(set(pairs)):
        raise ValueError('Select distinct (dataset, model) pairs.')
    return [select_experiment_configs(configs_by_dataset,[dataset],[model])[0]
            for dataset,model in pairs]

def focused_stability_pilots(runner, contexts, keys, first_case):
    """Check every requested temperature/step-size pair on the same small pilot."""
    rows=[]
    c=runner.config
    method=next((m for m in c['methods'] if learned_method(m)),None)
    if method is None:return rows
    for tau in c['tau_values']:
        check={}
        if c['preflight']['finite_difference']:
            s=PolicySurrogates(contexts['train'].X.shape[1],seed=first_case['seed'],
                Theta_1_radius=c['policy']['Theta_1_radius'],Theta_2_radius=c['policy']['Theta_2_radius'])
            check=directional_gradient_check(contexts['train'],s,tau,first_case['lambda_'])
        for eta in c['eta_values']:
            case=dict(first_case,tau=float(tau),eta=float(eta),T=min(c['T'],c['preflight']['T']))
            fit=runner.fit(case,contexts,keys,method,pilot=True)
            h=fit['history']
            rows.append(dict(tau=tau,eta=eta,initial_J_hat=float(h.J_hat.iloc[0]),
                final_J_hat=float(h.J_hat.iloc[-1]),objective_increases=int((h.J_hat.diff()>1e-10).sum()),
                descent_failures=int((~h.sufficient_decrease).sum()),
                final_G_eta_sq=float(h.G_eta_sq.iloc[-1]),
                finite_difference_error=check.get('absolute_error')))
    save_csv(runner.store.directory/'preflight'/'stability.csv',pd.DataFrame(rows))
    return rows

def focused_comparisons(runner, frame):
    """Select only by validation J; never use test columns to choose a grid point."""
    c=runner.config
    complete=[];status=[]
    lambdas=c['lambda_values'] if c['experiment']==10 else [c['lambda_']]
    expected={(float(tau),float(eta),method) for tau in c['tau_values']
              for eta in c['eta_values'] for method in c['methods']}
    for seed in c['seeds']:
        for lam in lambdas:
            group=frame[(frame.seed==seed)&(frame.lambda_==lam)].copy() if not frame.empty else frame.copy()
            present=set(zip(group.tau,group.eta,group.method)) if not group.empty else set()
            missing=sorted(expected-present)
            valid=not missing and len(group)==len(expected) and group.validation_J.notna().all()
            status.append(dict(seed=seed,lambda_=lam,complete=bool(valid),expected_rows=len(expected),
                               successful_rows=len(group),missing=[dict(tau=t,eta=e,method=m) for t,e,m in missing]))
            if valid:complete.append(group)
    full=pd.concat(complete,ignore_index=True) if complete else frame.iloc[:0].copy()
    if full.empty:return full,full.copy(),pd.DataFrame(),status
    keys=['seed','lambda_','method']
    # Test C,V,J are carried along only after validation has selected the row.
    order=full.sort_values(keys+['validation_J','tau','eta'],kind='stable')
    selected=order.drop_duplicates(keys,keep='first').copy()
    selected['selection_rule']='minimum validation J over tau and eta; test never used'
    pairs=[]
    for (seed,lam),group in selected.groupby(['seed','lambda_'],sort=False):
        by_method=group.set_index('method')
        for baseline in ['No recourse','Common target']:
            if baseline not in by_method.index:continue
            base=by_method.loc[baseline]
            for _,row in group.iterrows():
                if row['method']==baseline:continue
                pairs.append(dict(seed=seed,lambda_=lam,method=row['method'],baseline=baseline,
                    delta_C=row.C-base.C,delta_V=row.V-base.V,delta_J=row.J-base.J,
                    tau=row.tau,eta=row.eta,baseline_tau=base.tau,baseline_eta=base.eta,
                    better_J=bool(row.J<base.J-1e-10)))
    return full,selected,pd.DataFrame(pairs),status

def save_focused_tables(runner, frame):
    full,selected,paired,status=focused_comparisons(runner,frame)
    store=runner.store;stem=parameter_filename(store.config,store.config_id)
    directory=store.directory/'focused'
    save_json(directory/'comparison_status.json',status)
    for name,table in [('complete_grid',full),('validation_selected',selected),('paired_selected',paired)]:
        save_csv(directory/f'{name}.csv',table)
        save_csv(store.root/'tables'/f'{stem}-{name}.csv',table)
    missing=sum(not row['complete'] for row in status)
    if missing:print(f'{missing} seed/lambda grids are incomplete; inspect focused/comparison_status.json.')
    save_json(directory/'interpretation.json',dict(
        selection='Checkpoint and rho cutoff use validation; selected summaries also choose tau/eta by validation.',
        cost='C and wasted_cost average over all initially rejected applicants.',
        incremental_validity='Test V minus test V of no action on the same cohort.',
        uncertainty='Plot bars show standard errors over available complete seeds, not over grid points.',
        pairing='All comparisons use identical scorer, cohorts, seed, lambda, alpha and B.',
        limitation='No surrogate-to-binary or population-optimality guarantee is claimed.'))
    return selected

def save_focused_figures(runner, results, *, show=True):
    c=runner.config;frame,selected,_,_=focused_comparisons(runner,results)
    if frame.empty:return []
    labels={'No recourse':'No action','Original threshold':'Original threshold',
        'Common target':'Common target',
        'Personalized targets + selection':'Personalized target with selection (ours)',
        'Personalized targets + validated cutoff':'Personalized, validated cutoff',
        'Personalized targets, all eligible':'Personalized, all eligible'}
    figures=[]
    def curve(ax,group,axis,value,label,**kwargs):
        if group.empty or value not in group:return
        stat=group.groupby(axis)[value].agg(['mean','sem'])
        ax.errorbar(stat.index,stat['mean'],yerr=None if stat['sem'].isna().all() else stat['sem'],
                    marker='o',capsize=2,label=label,**kwargs)
        ax.grid(alpha=.25)
    if c['experiment']==9:
        etas=sorted(frame.eta.unique());fig,axes=plt.subplots(3,len(etas),figsize=(6*len(etas),9),squeeze=False)
        for col,eta in enumerate(etas):
            data=frame[frame.eta==eta]
            for method in ordered_methods(c['methods']):
                group=data[data.method==method]
                for row,(key,ylabel) in enumerate([('J','Test objective C - lambda V'),
                         ('incremental_validity','Test validity gain over no action'),('wasted_cost','Unsuccessful cost')]):
                    curve(axes[row,col],group,'tau',key,labels.get(method,method))
                    axes[row,col].set(xscale='log',xlabel='Temperature tau',ylabel=ylabel)
                    if key=='incremental_validity' and data[key].abs().max()<1e-8:
                        axes[row,col].set_ylim(-1e-4,1e-4)
            axes[0,col].set_title(f'eta = {eta:g}')
        handles,legend=axes[0,0].get_legend_handles_labels()
        fig.legend(handles,legend,loc='lower center',ncol=3,fontsize=8)
        figures.append(('temperature_cutoff',fig,.09))
        histories=load_histories(runner.store)
        histories=histories[histories.method=='Personalized targets + selection']
        if not histories.empty:
            # Keep matched complete grids and avoid counting the shared tuned-cutoff fit twice.
            histories=histories.merge(frame[frame.method=='Personalized targets + selection'][
                ['seed','tau','eta','lambda_']].drop_duplicates(),on=['seed','tau','eta','lambda_'],how='inner')
            final=histories.sort_values('iteration').groupby(['seed','tau','eta'],as_index=False).tail(1)
            fig,axes=plt.subplots(len(etas),3,figsize=(15,3.6*len(etas)),squeeze=False)
            for row,eta in enumerate(etas):
                data=final[final.eta==eta]
                if 'diag_V_hat' not in data:
                    for col in [0,1]:axes[row,col].text(.5,.5,'Training diagnostics disabled',ha='center',transform=axes[row,col].transAxes)
                for key,label in [('diag_V_hat','Smooth / fractional'),('diag_V_mixture','Exact / fractional'),
                                   ('diag_V_hard','Exact / binary'),('diag_no_action_V_hat','Smooth no action')]:
                    curve(axes[row,0],data,'tau',key,label)
                for key,label in [('diag_rho_hat_mean','Mean rho_hat'),('diag_rho_hat_above_half','Fraction >= 0.5')]:
                    curve(axes[row,1],data,'tau',key,label)
                for tau,group in histories[histories.eta==eta].groupby('tau'):
                    curve(axes[row,2],group,'iteration','G_eta_sq',f'tau = {tau:g}')
                for col,ylabel in enumerate(['Final training validity','Among eligible training applicants','Current squared gradient mapping']):
                    axes[row,col].set(xlabel='Temperature tau' if col<2 else 'Projected updates',ylabel=ylabel,
                                       title=f'eta = {eta:g}')
                    axes[row,col].legend(fontsize=7)
                    if col<2:axes[row,col].set_xscale('log')
                axes[row,1].set_ylim(-.03,1.03)
                axes[row,2].set_yscale('symlog',linthresh=1e-10)
            figures.append(('smoothing_rounding_convergence',fig,0.))
    else:
        fig,axes=plt.subplots(1,3,figsize=(16,4.8))
        for method in ordered_methods(c['methods']):
            group=selected[selected.method==method]
            stat=group.groupby('lambda_')[['C','V']]
            mean,se=stat.mean(),stat.sem();label=labels.get(method,method)
            axes[0].errorbar(mean.V,mean.C,xerr=None if se.V.isna().all() else se.V,
                             yerr=None if se.C.isna().all() else se.C,marker='o',capsize=2,label=label)
            curve(axes[1],group,'lambda_','delta_J_no_action',label)
            curve(axes[2],group,'lambda_','wasted_cost',label)
        axes[0].set(xlabel='Test validity V',ylabel='Mean recourse cost C');axes[0].grid(alpha=.25)
        from matplotlib.ticker import MaxNLocator
        axes[0].xaxis.set_major_locator(MaxNLocator(nbins=5))
        axes[1].set(xlabel='lambda',ylabel='Test objective minus no-action objective',xscale='log')
        axes[2].set(xlabel='lambda',ylabel='Unsuccessful cost',xscale='log')
        axes[1].axhline(0,color='grey',linewidth=.8)
        handles,legend=axes[0].get_legend_handles_labels()
        fig.legend(handles,legend,loc='lower center',ncol=3,fontsize=8)
        figures.append(('validated_cost_validity',fig,.16))
    paths=[];directory=Path(c.get('figures_dir','./figures')).expanduser().resolve()
    stem=parameter_filename(c,runner.store.config_id)
    for name,fig,space in figures:
        fig.suptitle(f"{c['dataset']} | {c['model']}")
        standardize_figure(fig);fig.tight_layout(rect=(0,space,1,.97))
        path=directory/f'{stem}-{name}.png'
        atomic_write(path,lambda handle:fig.savefig(handle,format='png',dpi=160,bbox_inches='tight'))
        save_json(path.with_suffix('.json'),dict(config=c,config_id=runner.store.config_id,plot=name,
            uncertainty='SEM across complete seeds',selection='validation J only',figure_file=str(path)))
        paths.append(str(path))
        if show:plt.show()
        plt.close(fig)
    return paths


In [ ]:
def _plot_method(label):
    """Map display/diagnostic labels onto a stable paper-method style."""
    if not isinstance(label,str):return None
    for key in METHOD_ORDER:
        if label in (key,METHOD_LABELS[key]):return key
    aliases={'Full':'Personalized targets + selection','All eligible':'Personalized targets, all eligible',
             'Personalized, all eligible':'Personalized targets, all eligible',
             'Personalized, validated cutoff':'Personalized targets + selection'}
    if label in aliases:return aliases[label]
    if label.startswith('Common target'):return 'Common target'
    if label.startswith('Personalized target with selection'):return 'Personalized targets + selection'
    if label.startswith('Personalized target without selection'):return 'Personalized targets, all eligible'
    return None


def standardize_figure(fig):
    """Use the same method colors/order, including in older follow-up plots."""
    from matplotlib.container import ErrorbarContainer
    def apply(handle,color,marker):
        if isinstance(handle,ErrorbarContainer):
            line,caps,bars=handle.lines
            if line is not None:
                line.set_color(color)
                if line.get_marker() not in (None,'None','',' '):line.set_marker(marker)
            for artist in list(caps)+list(bars):artist.set_color(color)
        else:
            if hasattr(handle,'set_color'):handle.set_color(color)
            if hasattr(handle,'get_marker') and handle.get_marker() not in (None,'None','',' '):handle.set_marker(marker)
    for ax in fig.axes:
        handles,labels=ax.get_legend_handles_labels()
        for handle,label in zip(handles,labels):
            method=_plot_method(label)
            if method:apply(handle,METHOD_COLORS[method],METHOD_MARKERS[method])
        # Method comparison bars also share the palette.
        ticks=[t.get_text() for t in ax.get_xticklabels()]
        if len(ax.patches)==len(ticks) and ticks and all(_plot_method(t) for t in ticks):
            for patch,tick in zip(ax.patches,ticks):patch.set_facecolor(METHOD_COLORS[_plot_method(tick)])
        if ax.get_legend() is not None and handles:
            order=sorted(range(len(labels)),key=lambda i: METHOD_ORDER.index(_plot_method(labels[i])) if _plot_method(labels[i]) else len(METHOD_ORDER)+i)
            ax.legend([handles[i] for i in order],[labels[i] for i in order],fontsize=7)
    # Figure-level legends are reconstructed after coloring the underlying artists.
    if fig.legends:
        unique={}
        for ax in fig.axes:
            h,l=ax.get_legend_handles_labels()
            for a,b in zip(h,l):unique.setdefault(b,a)
        for legend in list(fig.legends):legend.remove()
        labels=sorted(unique,key=lambda l:METHOD_ORDER.index(_plot_method(l)) if _plot_method(l) else len(METHOD_ORDER))
        if labels:fig.legend([unique[l] for l in labels],labels,loc='lower center',ncol=min(3,len(labels)),fontsize=8)
    return fig


def inventory_saved_studies(results_dir='./Results'):
    """Read-only inventory; historical outputs retain their original method labels."""
    root=Path(results_dir).expanduser().resolve();rows=[]
    if not root.exists():return pd.DataFrame(columns=['directory','dataset','model','id','complete'])
    for path in sorted(root.rglob('config.json')):
        if 'cache' in path.relative_to(root).parts:continue
        try:
            meta=load_json(path);c=meta.get('config',{});base=c.get('base_config',c)
            rows.append(dict(directory=str(path.parent),dataset=c.get('dataset',base.get('dataset')),
                model=c.get('model',base.get('model')),id=meta.get('config_id',meta.get('run_id')),
                methods=base.get('methods'),complete=(path.parent/'complete.json').exists()))
        except (ValueError,OSError):continue
    return pd.DataFrame(rows)


In [ ]:
IMPLEMENTATION_NAMES = tuple(sorted(name for name,value in list(globals().items())
    if (inspect.isfunction(value) or inspect.isclass(value)) and getattr(value,"__module__",None)==__name__
    and name not in {"_code_signature","implementation_id"}))
print("Numerical setup ready; no datasets or experiments have been run.")


In [ ]:
def refinement_config(source_config, *, seeds=None, lambda_values=None, grid_levels=((30, 1), (100, 1), (300, 1)), source_config_id=None):
    """Independent settings for a read-only comparison with a saved experiment 10."""
    return dict(dataset=source_config['dataset'], model=source_config['model'], source_config=copy.deepcopy(source_config), source_config_id=source_config_id, seeds=list(source_config['seeds'] if seeds is None else seeds), lambda_values=list([3.0, 10.0, 30.0] if lambda_values is None else lambda_values), grid_levels=[list(level) for level in grid_levels], results_dir=source_config['results_dir'], figures_dir=source_config['figures_dir'], audit_descent=True)

def refinement_source(config):
    """Resolve one compatible run; never silently choose among different saved runs."""
    root = Path(config['results_dir']).expanduser().resolve()
    wanted = config['source_config']
    fields = ['dataset', 'model', 'alpha', 'B', 'T', 'cohort_size', 'fit_cap', 'synthetic_n', 'scorer', 'qualification_proxy', 'solver', 'policy', 'tau_values', 'eta_values', 'diagnostics', 'methods', 'comparison_protocol']
    matches = []
    for path in sorted((root / 'experiments' / 'experiment_10' / slug(config['dataset']) / config['model']).glob('*/config.json')):
        metadata = load_json(path)
        old = metadata['config']
        if config['source_config_id'] and metadata['config_id'] != config['source_config_id']:
            continue
        if all((old.get(key) == wanted.get(key) for key in fields)) and set(config['seeds']) <= set(old['seeds']) and (set(config['lambda_values']) <= set(old['lambda_values'])):
            matches.append((path.parent, metadata))
    if len(matches) != 1:
        ids = [metadata['config_id'] for _, metadata in matches]
        raise ValueError(f"Expected one compatible saved experiment 10 for {config['dataset']} / {config['model']}; found {ids}. Check results_dir and source_config settings, or set source_config_id. This experiment never starts training to replace missing results.")
    directory, metadata = matches[0]
    records, paths = ([], {})
    for path in sorted((directory / 'trials').glob('seed_*/*/result.json')):
        row = load_json(path)
        if row['seed'] in config['seeds'] and row['lambda_'] in config['lambda_values']:
            if row.get('status') != 'ok':
                continue
            key = (row['seed'], row['lambda_'], row['tau'], row['eta'], row['method'])
            if key in paths:
                raise ValueError(f'Duplicate source trial: {key}')
            paths[key] = path.parent
            records.append(row)
    frame = pd.DataFrame(records)
    source = copy.deepcopy(metadata['config'])
    source.update(seeds=config['seeds'], lambda_values=config['lambda_values'])
    _, selected, _, status = focused_comparisons(types.SimpleNamespace(config=source), frame)
    if not status or any((not row['complete'] for row in status)):
        raise ValueError('The requested source grids are incomplete. Finish experiment 10 first; partial grids are not used for paired comparisons.')
    manifests, fingerprints = ({}, {})
    for seed in config['seeds']:
        values = []
        for key, trial in paths.items():
            if key[0] != seed:
                continue
            manifest_path = trial / 'context.json'
            if not manifest_path.exists():
                raise FileNotFoundError(f'Missing {manifest_path}; saved cohort references are required.')
            values.append(load_json(manifest_path))
        if any((value != values[0] for value in values[1:])):
            raise ValueError(f'Source trials do not share identical cohorts for seed {seed}.')
        manifest = values[0]
        if set(manifest['cohorts']) != {'train', 'val', 'test'}:
            raise ValueError('Source must include separate train, validation, and test cohorts.')
        for relative in [manifest['scorer'], *manifest['cohorts'].values()]:
            path = (root / relative).resolve()
            if root not in path.parents or not path.is_file():
                raise FileNotFoundError(f'Missing original cache: {path}. Restore Results/cache alongside Results/experiments. Exported tables alone are insufficient; no refitting is performed.')
            if relative not in fingerprints:
                fingerprints[relative] = hashlib.sha256(path.read_bytes()).hexdigest()
        manifests[seed] = manifest
    return dict(root=root, directory=directory, metadata=metadata, selected=selected, records=records, paths=paths, manifests=manifests, fingerprints=fingerprints)

def refinement_contexts(source, seed):
    manifest = source['manifests'][seed]
    reader = SavedDiagnosticReader(source['root'], source['metadata']['implementation_id'])
    model = restore_scorer(reader.payload(manifest['scorer']))
    contexts = {split: Cohort(f_w0=model, **reader.payload(path)) for split, path in manifest['cohorts'].items()}
    with torch.no_grad():
        for cohort in contexts.values():
            require_training_cohort(cohort)
            if not torch.allclose(model(cohort.X), cohort.s_0, atol=1e-12, rtol=1e-12):
                raise ValueError('Saved scorer and cohort scores disagree.')
            if (cohort.t_0, cohort.B, cohort.alpha) != (contexts['train'].t_0, contexts['train'].B, contexts['train'].alpha):
                raise ValueError('Saved cohorts have inconsistent settings.')
    return contexts

@torch.no_grad()
def refinement_target_data(cohort, q_c):
    """One exact best-response solve per target and cohort, shared by all cutoffs."""
    q = torch.full_like(cohort.s_0, q_c)
    feasible = target_feasible(cohort, q)
    br_X = br(cohort, q)
    costs = cost(cohort.X, br_X)
    scores = torch.where(feasible, q, cohort.s_0)
    if not torch.allclose(cohort.f_w0(br_X), scores, atol=2e-09, rtol=2e-09):
        raise ArithmeticError('Baseline target attainment failed.')
    if (costs > cohort.B + 1e-10 * max(1.0, cohort.B)).any():
        raise ArithmeticError('Baseline response exceeded the budget.')
    if not torch.equal(br_X[:, ~cohort.actionable_mask], cohort.X[:, ~cohort.actionable_mask]):
        raise ArithmeticError('Baseline changed an immutable feature.')
    return (feasible, costs)

@torch.no_grad()
def refinement_target_entries(reference, validation, q_c, quantile_levels):
    """Plain common target: recommend to EVERY feasible rejected applicant."""
    feasible,costs=refinement_target_data(validation,q_c)
    scores=torch.where(feasible,torch.full_like(validation.s_0,q_c),validation.s_0)
    allocation,_,_=exact_allocation(scores,validation.k)
    return [dict(q_c=float(q_c),cost_limit=math.inf,
                 C=float((costs*feasible)[validation.X_minus].mean()),
                 V=float(allocation[validation.X_minus].mean()))]

def refinement_search(reference, validation, config, seed_directory):
    """Nested candidate sets; select all lambdas from the same validation evaluations."""
    path = seed_directory / 'search.pt'
    state = load_torch(path) if path.exists() else dict(blocks={}, stages=[])
    targets, quantiles, cumulative = (set(), set(), {})
    no_action = scalar_metrics(evaluate_policy(validation, NoRecourse(), 1.0))
    for stage, (n_targets, n_quantiles) in enumerate(config['grid_levels']):
        targets.update(map(float, np.linspace(reference.t_0, 1.0, n_targets)))
        quantiles.update(map(float, torch.linspace(0.0, 1.0, n_quantiles).tolist()))
        levels = sorted(quantiles)
        for index, q_c in enumerate(sorted(targets)):
            block_key = stable_id(dict(q_c=q_c, quantiles=levels))
            if block_key not in state['blocks']:
                state['blocks'][block_key] = refinement_target_entries(reference, validation, q_c, levels)
                save_torch(path, state)
            for entry in state['blocks'][block_key]:
                cumulative.setdefault((entry['q_c'], entry['cost_limit']), entry)
            if index % 50 == 0 or index + 1 == len(targets):
                print(f'    grid {n_targets}x{n_quantiles}: {index + 1}/{len(targets)} targets', flush=True)
        selected = []
        for lam in config['lambda_values']:
            best = dict(policy=encode_policy(NoRecourse()), C=no_action['C'], V=no_action['V'])
            best_J = best['C'] - lam * best['V']
            for entry in cumulative.values():
                value = entry['C'] - lam * entry['V']
                if value < best_J:
                    best = dict(policy=encode_policy(CommonTarget(entry['q_c'], entry['cost_limit'])), C=entry['C'], V=entry['V'])
                    best_J = value
            selected.append(dict(lambda_=lam, validation_J=best_J, **best))
        item = dict(stage=stage, target_points=n_targets, cost_quantiles=n_quantiles, effective_targets=len(targets), candidates=len(cumulative) + 1, selected=selected)
        if stage < len(state['stages']):
            state['stages'][stage] = item
        else:
            state['stages'].append(item)
        save_torch(path, state)
    return state['stages']

def refinement_descent_audit(source):
    """Reconstruct each one-step bound using the PREVIOUS iterate's gradient mapping."""
    summaries, failures, seen = ([], [], set())
    for row in source['records']:
        if learned_method(row['method']) is None:
            continue
        key = (row['seed'], row['lambda_'], row['tau'], row['eta'], row['method'])
        trial = source['paths'][key]
        fit = load_json(trial / 'fit.json') if (trial / 'fit.json').exists() else {}
        fit_id = fit.get('fit_id', stable_id(dict(seed=row['seed'], lambda_=row['lambda_'], tau=row['tau'], eta=row['eta'], mode=learned_method(row['method']))))
        if fit_id in seen:
            continue
        seen.add(fit_id)
        tags = {k: row[k] for k in ['dataset', 'model', 'seed', 'lambda_', 'tau', 'eta', 'method']}
        tags['fit_id'] = fit_id
        path = trial / 'history.csv'
        if not path.exists():
            summaries.append(dict(**tags, status='missing_history'))
            continue
        h = pd.read_csv(path).sort_values('iteration')
        if len(h) < 2 or h.iteration.duplicated().any() or (not np.isfinite(h[['J_hat', 'G_eta_sq']].to_numpy()).all()):
            raise ValueError(f'Malformed training history: {path}')
        previous = h.J_hat.shift(1)
        consecutive = h.iteration.diff().eq(1)
        actual = previous - h.J_hat
        required = row['eta'] * h.G_eta_sq.shift(1) / 2
        gap = required - actual
        tolerance = 1e-10
        failed = consecutive & (gap > tolerance)
        objective_increased = consecutive & (actual < -tolerance)
        summaries.append(dict(**tags, status='ok', steps=int(consecutive.sum()), expected_steps=row['T'], complete=bool(consecutive.sum() == row['T']), sufficient_decrease_failures=int(failed.sum()), objective_increases=int(objective_increased.sum()), max_bound_gap=float(gap[consecutive].max()), final_G_eta_sq=float(h.G_eta_sq.iloc[-1]), max_capacity_residual=float(h.capacity_residual.abs().max()), recorded_flag_disagreements=int((~h.sufficient_decrease.astype(bool) != failed)[consecutive].sum()) if 'sufficient_decrease' in h else None))
        for index in h.index[failed]:
            failures.append(dict(**tags, iteration=int(h.loc[index, 'iteration']), J_previous=float(previous.loc[index]), J_current=float(h.loc[index, 'J_hat']), actual_decrease=float(actual.loc[index]), required_decrease=float(required.loc[index]), bound_gap=float(gap.loc[index]), tolerance=tolerance, gap_over_tolerance=float(gap.loc[index] / tolerance), achieved_fraction=float(actual.loc[index] / required.loc[index]) if required.loc[index] > 0 else None, objective_increased=bool(objective_increased.loc[index])))
    columns = ['dataset', 'model', 'seed', 'lambda_', 'tau', 'eta', 'method', 'fit_id', 'iteration', 'J_previous', 'J_current', 'actual_decrease', 'required_decrease', 'bound_gap', 'tolerance', 'gap_over_tolerance', 'achieved_fraction', 'objective_increased']
    return (pd.DataFrame(summaries), pd.DataFrame(failures, columns=columns))

def refinement_tables(frame):
    summary = frame.groupby(['method', 'lambda_'], sort=False).agg(seeds=('seed', 'nunique'), C_mean=('C', 'mean'), C_sem=('C', 'sem'), V_mean=('V', 'mean'), V_sem=('V', 'sem'), J_mean=('J', 'mean'), J_sem=('J', 'sem'), validation_J_mean=('validation_J', 'mean'), validation_J_sem=('validation_J', 'sem'), incremental_validity_mean=('incremental_validity', 'mean'), wasted_cost_mean=('wasted_cost', 'mean')).reset_index()
    pairs = []
    for (seed, lam), group in frame.groupby(['seed', 'lambda_']):
        for _, baseline in group[group.origin == 'refined'].iterrows():
            for _, row in group[group.method.str.startswith('Personalized')].iterrows():
                pairs.append(dict(dataset=row.dataset, model=row.model, seed=seed, lambda_=lam, method=row.method, baseline=baseline.method, stage=int(baseline.stage), delta_C=row.C - baseline.C, delta_V=row.V - baseline.V, delta_J=row.J - baseline.J))
    paired = pd.DataFrame(pairs)
    if paired.empty:
        return (summary, paired, pd.DataFrame())
    paired['win'] = paired.delta_J < -1e-10
    paired['tie'] = paired.delta_J.abs() <= 1e-10
    paired_summary = paired.groupby(['method', 'baseline', 'lambda_', 'stage'], sort=False).agg(seeds=('seed', 'nunique'), delta_J_mean=('delta_J', 'mean'), delta_J_sem=('delta_J', 'sem'), delta_C_mean=('delta_C', 'mean'), delta_V_mean=('delta_V', 'mean'), wins=('win', 'sum'), ties=('tie', 'sum')).reset_index()
    paired_summary['losses'] = paired_summary.seeds - paired_summary.wins - paired_summary.ties
    return (summary, paired, paired_summary)

def refinement_figures(frame, config, config_id, stem, *, show=True):
    summary, paired, paired_summary = refinement_tables(frame)
    figures = []
    levels = config['grid_levels']
    refined = frame[frame.origin == 'refined']
    fig, axes = plt.subplots(1, len(config['lambda_values']), figsize=(5 * len(config['lambda_values']), 4), squeeze=False)
    for ax, lam in zip(axes[0], config['lambda_values']):
        stat = refined[refined.lambda_ == lam].groupby('stage').validation_J.agg(['mean', 'sem'])
        ax.errorbar(stat.index, stat['mean'], yerr=stat['sem'].fillna(0), marker='o', capsize=3)
        ax.set(xticks=range(len(levels)), xticklabels=[str(a) for a, b in levels], xlabel='Added target points (nested)', ylabel='Validation C - lambda V', title=f'lambda = {lam:g}')
        ax.grid(alpha=0.25)
    figures.append(('baseline_search_convergence', fig))
    labels = {'Personalized targets + selection': 'Personalized target with selection (ours)', 'Personalized targets + validated cutoff': 'Personalized, validated cutoff', 'Personalized targets, all eligible': 'Personalized, all eligible'}
    fig, ax = plt.subplots(figsize=(8, 5))
    for method, stat in summary.groupby('method', sort=False):
        if method in ['Original threshold', 'Common target']:
            continue
        stat = stat.sort_values('lambda_')
        ax.errorbar(stat.V_mean, stat.C_mean, xerr=stat.V_sem.fillna(0), yerr=stat.C_sem.fillna(0), marker='o', capsize=2, label=labels.get(method, method))
    ax.set(xlabel='Test validity V', ylabel='Mean recourse cost C')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    figures.append(('cost_validity', fig))
    fig, ax = plt.subplots(figsize=(8, 5))
    if not paired_summary.empty:
        for method, stat in paired_summary[paired_summary.stage == len(levels) - 1].groupby('method'):
            stat = stat.sort_values('lambda_')
            ax.errorbar(stat.lambda_, stat.delta_J_mean, yerr=stat.delta_J_sem.fillna(0), marker='o', capsize=3, label=labels.get(method, method))
        ax.legend(fontsize=8)
    ax.axhline(0, color='grey', linewidth=1)
    ax.set(xscale='log', xlabel='lambda', ylabel='Test J(personalized) - J(finest common target)')
    ax.grid(alpha=0.25)
    figures.append(('paired_objective', fig))
    paths = []
    for name, fig in figures:
        fig.suptitle(f"{config['dataset']} | {config['model']}")
        standardize_figure(fig);fig.tight_layout()
        path = Path(config['figures_dir']).expanduser().resolve() / f'{stem}-{name}.png'
        atomic_write(path, lambda handle: fig.savefig(handle, format='png', dpi=160, bbox_inches='tight'))
        save_json(path.with_suffix('.json'), dict(config=config, config_id=config_id, plot=name, uncertainty='SEM across paired seeds; candidates selected using validation only'))
        paths.append(str(path))
        if show:
            plt.show()
        plt.close(fig)
    return paths

def refinement_identity(config, source):
    helpers = {name: _code_signature(inspect.unwrap(value).__code__) for name, value in globals().items() if name.startswith('refinement_') and inspect.isfunction(value)}
    scientific = {key: value for key, value in config.items() if key not in ['results_dir', 'figures_dir']}
    scientific['source_config'] = scientific_config(scientific['source_config'])
    return stable_id(dict(schema=1, config=scientific, helpers=helpers, core_implementation=implementation_id()[0], source_config_id=source['metadata']['config_id'], source_records=source['records'], source_files=source['fingerprints']))

def refinement_run(config, *, mode='run', show=True):
    if any(len(v)!=2 or v[0]<2 or v[1]!=1 for v in config['grid_levels']):
        raise ValueError('Use (target_points,1): common target does not search recipient cost cutoffs.')
    """No calls to fit(), learn_recourse(), dataset loaders, or the old experiment runner."""
    if mode not in ['run', 'preflight', 'load', 'skip']:
        raise ValueError('Choose run, preflight, load, or skip.')
    if mode == 'skip':
        return None
    if not config['seeds'] or len(set(config['seeds'])) != len(config['seeds']) or (not config['lambda_values']) or (len(set(config['lambda_values'])) != len(config['lambda_values'])) or any((not np.isfinite(v) or v <= 0 for v in config['lambda_values'])) or (not config['grid_levels']) or any((len(pair) != 2 or any((isinstance(n, bool) or not isinstance(n, (int, np.integer)) or n < 1 for n in pair)) for pair in config['grid_levels'])):
        raise ValueError('Provide distinct seeds/lambdas and positive integer grid sizes >= 2.')
    source = refinement_source(config)
    config_id = refinement_identity(config, source)
    directory = source['root'] / 'refinements' / 'experiment_11' / slug(config['dataset']) / config['model'] / config_id
    grids = '_'.join((f'{a}x{b}' for a, b in config['grid_levels']))
    lambdas = '_'.join((f'{v:g}' for v in config['lambda_values']))
    base = config['source_config']
    stem = f"experiment-11-dataset-{slug(config['dataset'])}-model-{config['model']}-parameters-alpha{base['alpha']:g}-B{base['B']:g}-lambda{lambdas}-grids{grids}"[:165] + '-' + config_id
    print(f"{config['dataset']} / {config['model']}: source {source['metadata']['config_id']}; {len(config['seeds'])} seeds, {len(config['grid_levels'])} nested grids; 0 new training fits.", flush=True)
    if mode == 'preflight':
        for seed in config['seeds']:
            contexts = refinement_contexts(source, seed)
            entries = refinement_target_entries(contexts['train'], contexts['val'], contexts['train'].t_0, [0.0, 1.0])
            for entry in entries:
                exact = evaluate_policy(contexts['val'], CommonTarget(entry['q_c'], entry['cost_limit']), 1.0)
                if not np.allclose([entry['C'], entry['V']], [exact['C'], exact['V']], atol=1e-12, rtol=1e-12):
                    raise ArithmeticError('Cached candidate evaluation disagrees with exact evaluation.')
        print('Saved inputs and baseline evaluation checks passed. No grid search or training was run.')
        return dict(status='preflight_passed', directory=str(directory))
    result_path = directory / 'results.csv'
    if mode == 'load' and (not result_path.exists()):
        raise FileNotFoundError(f'No saved refinement for these settings: {directory}')
    if mode == 'run':
        save_json(directory / 'config.json', dict(config=config, config_id=config_id, source_config_id=source['metadata']['config_id'], source_directory=str(source['directory']), source_fingerprints=source['fingerprints'], new_training_fits=0))
        for seed in config['seeds']:
            seed_directory = directory / f'seed_{seed}'
            if (seed_directory / 'complete.json').exists():
                continue
            print(f'  Baseline refinement, seed {seed}', flush=True)
            contexts = refinement_contexts(source, seed)
            stages = refinement_search(contexts['train'], contexts['val'], config, seed_directory)
            rows = []
            for stage in stages:
                for selected in stage['selected']:
                    lam = selected['lambda_']
                    policy = decode_policy(selected['policy'])
                    validation = scalar_metrics(evaluate_policy(contexts['val'], policy, lam))
                    if not np.allclose([validation['C'], validation['V']], [selected['C'], selected['V']], atol=1e-10, rtol=1e-10):
                        raise ArithmeticError('Selected baseline failed exact validation recheck.')
                    metrics = scalar_metrics(evaluate_policy(contexts['test'], policy, lam))
                    no_action = scalar_metrics(evaluate_policy(contexts['test'], NoRecourse(), lam))
                    rows.append(dict(dataset=config['dataset'], model=config['model'], seed=seed, lambda_=lam, method=f"Common target [{stage['target_points']} targets]", origin='refined', stage=stage['stage'], target_points=stage['target_points'], cost_quantiles=stage['cost_quantiles'], effective_targets=stage['effective_targets'], candidates=stage['candidates'], validation_J=validation['J'], validation_C=validation['C'], validation_V=validation['V'], incremental_validity=metrics['V'] - no_action['V'], no_action_V=no_action['V'], delta_J_no_action=metrics['J'] - no_action['J'], **metrics))
                    save_torch(seed_directory / f"stage_{stage['stage']}_lambda_{lam:g}_policy.pt", selected['policy'])
            imported = source['selected'][source['selected'].seed == seed].copy()
            imported['origin'] = 'saved'
            imported['stage'] = -1
            result = pd.concat([pd.DataFrame(rows), imported], ignore_index=True)
            save_csv(seed_directory / 'results.csv', result)
            save_json(seed_directory / 'complete.json', dict(seed=seed, rows=len(result)))
            partial = pd.concat([pd.read_csv(p) for p in sorted(directory.glob('seed_*/results.csv'))], ignore_index=True)
            save_csv(result_path, partial)
        frame = pd.concat([pd.read_csv(directory / f'seed_{seed}' / 'results.csv') for seed in config['seeds']], ignore_index=True)
        frame['experiment'] = 11
        frame['config_id'] = config_id
        frame['source_config_id'] = source['metadata']['config_id']
        save_csv(result_path, frame)
    else:
        frame = pd.read_csv(result_path)
    expected = len(config['seeds']) * len(config['lambda_values']) * (len(config['grid_levels']) + len(source['selected'].method.unique()))
    if len(frame) != expected or frame.duplicated(['seed', 'lambda_', 'method']).any():
        raise ValueError('Incomplete refinement results; run again to resume before comparing methods.')
    summary, paired, paired_summary = refinement_tables(frame)
    tables = dict(results=frame, summary=summary, paired=paired, paired_summary=paired_summary)
    if config['audit_descent']:
        tables['descent_summary'], tables['descent_failures'] = refinement_descent_audit(source)
    for name, table in tables.items():
        save_csv(directory / f'{name}.csv', table)
        save_csv(source['root'] / 'tables' / f'{stem}-{name}.csv', table)
    save_json(source['root'] / 'tables' / f'{stem}-config.json', dict(config=config, config_id=config_id, source_config_id=source['metadata']['config_id'], directory=str(directory)))
    figures = refinement_figures(frame, config, config_id, stem, show=show)
    display(summary[['method', 'lambda_', 'seeds', 'C_mean', 'V_mean', 'J_mean']])
    if not paired_summary.empty:
        display(paired_summary[paired_summary.stage == len(config['grid_levels']) - 1])
    if config['audit_descent']:
        audit = tables['descent_summary']
        print(f"Descent audit: {len(audit)} distinct fits; {len(tables['descent_failures'])} flagged steps. Inspect actual_decrease, required_decrease and bound_gap; a failed bound need not mean an objective increase.")
    return dict(config_id=config_id, directory=str(directory), results=frame, tables=tables, figures=figures)


In [ ]:
def transfer_config(refinement_settings, *, seeds=None, lambda_values=None, validation_counts=(1, 5, 20), test_cohorts=50, cohort_size=256, fresh_seed=20260925, refinement_config_id=None):
    """A synthetic transfer diagnostic; no scorer or recourse-policy training."""
    return dict(dataset=refinement_settings['dataset'], model=refinement_settings['model'], source_refinement=copy.deepcopy(refinement_settings), refinement_config_id=refinement_config_id, seeds=list(refinement_settings['seeds'] if seeds is None else seeds), lambda_values=list(refinement_settings['lambda_values'] if lambda_values is None else lambda_values), validation_counts=list(validation_counts), test_cohorts=test_cohorts, cohort_size=cohort_size, fresh_seed=fresh_seed, results_dir=refinement_settings['results_dir'], figures_dir=refinement_settings['figures_dir'])

def transfer_payload_id(value):

    def convert(item):
        if torch.is_tensor(item) or isinstance(item, np.ndarray):
            return dict(array_id=array_id(item))
        if isinstance(item, dict):
            return {key: convert(val) for key, val in item.items()}
        if isinstance(item, (list, tuple)):
            return [convert(val) for val in item]
        return item
    return stable_id(convert(value))

def transfer_source(config):
    settings = copy.deepcopy(config['source_refinement'])
    settings.update(seeds=config['seeds'], lambda_values=config['lambda_values'], results_dir=config['results_dir'], figures_dir=config['figures_dir'])
    source = refinement_source(settings)
    matches = []
    parent = source['root'] / 'refinements' / 'experiment_11' / slug(config['dataset']) / config['model']
    for path in sorted(parent.glob('*/config.json')):
        metadata = load_json(path)
        old = metadata['config']
        if config['refinement_config_id'] and metadata['config_id'] != config['refinement_config_id']:
            continue
        if metadata['source_config_id'] == source['metadata']['config_id'] and old['grid_levels'] == settings['grid_levels'] and (set(config['seeds']) <= set(old['seeds'])) and (set(config['lambda_values']) <= set(old['lambda_values'])):
            matches.append((path.parent, metadata))
    if len(matches) != 1:
        raise ValueError(f"Expected one compatible completed experiment 11; found {[m['config_id'] for _, m in matches]}. Check results_dir/source_refinement, or set refinement_config_id. Missing inputs never trigger training.")
    directory, metadata = matches[0]
    if any((metadata['source_fingerprints'].get(key) != value for key, value in source['fingerprints'].items())):
        raise ValueError('The original scorer/cohort caches changed after experiment 11. Restore its original inputs before using this comparison.')
    files = {}
    for seed in config['seeds']:
        seed_dir = directory / f'seed_{seed}'
        needed = [seed_dir / 'search.pt', seed_dir / 'complete.json', seed_dir / 'results.csv']
        stage = len(settings['grid_levels']) - 1
        needed += [seed_dir / f'stage_{stage}_lambda_{lam:g}_policy.pt' for lam in config['lambda_values']]
        for row in source['selected'][source['selected'].seed == seed].to_dict('records'):
            key = (seed, row['lambda_'], row['tau'], row['eta'], row['method'])
            needed.append(source['paths'][key] / 'policy.pt')
        for path in needed:
            if not path.is_file():
                raise FileNotFoundError(f'Missing saved input: {path}. Keep the full Results folder, including policies, search.pt, and cache files.')
            files[str(path.relative_to(source['root']))] = hashlib.sha256(path.read_bytes()).hexdigest()
    return dict(source=source, directory=directory, metadata=metadata, fingerprints=files)

def transfer_candidates(bundle, seed):
    """Reuse the exact candidate family from experiment 11; ignore its validation scores."""
    state = load_torch(bundle['directory'] / f'seed_{seed}' / 'search.pt')
    expected = len(bundle['metadata']['config']['grid_levels'])
    if len(state['stages']) != expected or state['stages'][-1]['stage'] != expected - 1:
        raise ValueError('The saved common-target search is incomplete.')
    candidates = [encode_policy(NoRecourse())]
    seen = set()
    for block in state['blocks'].values():
        for entry in block:
            key = (entry['q_c'], entry['cost_limit'])
            if key not in seen:
                candidates.append(encode_policy(CommonTarget(*key)))
                seen.add(key)
    if len(candidates) != state['stages'][-1]['candidates']:
        raise ValueError('Saved candidate count does not match the completed search.')
    groups = {}
    for index, candidate in enumerate(candidates[1:], start=1):
        groups.setdefault(candidate['q_c'], []).append((index, candidate['cost_limit']))
    return (candidates, groups)

def transfer_saved_policies(bundle, config, seed):
    source = bundle['source']
    records = []
    for row in source['selected'][source['selected'].seed == seed].to_dict('records'):
        if row['method'] == 'Common target':
            continue
        key = (seed, row['lambda_'], row['tau'], row['eta'], row['method'])
        payload = load_torch(source['paths'][key] / 'policy.pt')
        records.append(dict(method=row['method'], lambda_=row['lambda_'], validation_count=0, payload=payload, policy_id=transfer_payload_id(payload), selection_J=row['validation_J'], selection_source='original validation cohort; saved policy unchanged'))
    seed_dir = bundle['directory'] / f'seed_{seed}'
    frame = pd.read_csv(seed_dir / 'results.csv')
    stage = len(bundle['metadata']['config']['grid_levels']) - 1
    for lam in config['lambda_values']:
        row = frame[(frame.origin == 'refined') & (frame.stage == stage) & (frame.lambda_ == lam)]
        if len(row) != 1:
            raise ValueError('Missing or duplicate fine-grid baseline result.')
        payload = load_torch(seed_dir / f'stage_{stage}_lambda_{lam:g}_policy.pt')
        records.append(dict(method='Common target, original validation', lambda_=lam, validation_count=0, payload=payload, policy_id=transfer_payload_id(payload), selection_J=float(row.validation_J.iloc[0]), selection_source='original validation cohort; saved fine-grid policy unchanged'))
    return records

def transfer_fresh_cohort(reference, config, seed, split, index):
    """Disjoint RNG streams, paired across policies/lambdas; t_0 is NEVER recalibrated."""
    if split not in ['validation', 'test']:
        raise ValueError('Choose validation or test.')
    entropy = [int(config['fresh_seed']), 1201, int(seed), 101 if split == 'validation' else 202, int(index)]
    X, _, _ = generate_correlated_nonlinear_population(config['cohort_size'], correlation=0.5, d=reference.X.shape[1], noise_std=0.0, seed=np.random.SeedSequence(entropy))
    cohort = make_cohort(X, reference.f_w0, reference.t_0, reference.B, reference.alpha, reference.actionable_mask, solver_options=reference.options)
    require_training_cohort(cohort)
    return (cohort, dict(split=split, cohort_id=index, rng_entropy=entropy, X_id=array_id(cohort.X), N=cohort.N, N_minus=cohort.N_minus, k=cohort.k))

@torch.no_grad()
def transfer_target_metrics(cohort, q_c, limits):
    """Vectorized exact top-k allocation for all cost cutoffs at one common target."""
    feasible, costs = refinement_target_data(cohort, q_c)
    costs = costs.cpu().numpy()
    feasible = feasible.cpu().numpy()
    rejected = cohort.X_minus.cpu().numpy()
    rho = feasible[None, :] & (costs[None, :] <= np.asarray(limits)[:, None])
    scores = np.where(rho, q_c, cohort.s_0.cpu().numpy()[None, :])
    cutoff = np.partition(scores, cohort.N - cohort.k, axis=1)[:, cohort.N - cohort.k]
    above = scores > cutoff[:, None]
    tied = scores == cutoff[:, None]
    beta = (cohort.k - above.sum(axis=1)) / tied.sum(axis=1)
    if np.any((beta < -1e-12) | (beta > 1 + 1e-12)):
        raise ArithmeticError('Invalid tie probability in batched baseline evaluation.')
    allocation = above + beta[:, None] * tied
    if np.max(np.abs(allocation.sum(axis=1) - cohort.k)) > 1e-10 * cohort.N:
        raise ArithmeticError('Batched allocation violated capacity.')
    return ((rho[:, rejected] * costs[None, rejected]).mean(axis=1), allocation[:, rejected].mean(axis=1))

@torch.no_grad()
def transfer_validation(cohort, candidates, groups, saved):
    C, V = (np.empty(len(candidates)), np.empty(len(candidates)))
    no_action = scalar_metrics(evaluate_policy(cohort, NoRecourse(), 1.0))
    C[0], V[0] = (no_action['C'], no_action['V'])
    for q_c, members in groups.items():
        indices, limits = zip(*members)
        C[list(indices)], V[list(indices)] = transfer_target_metrics(cohort, q_c, limits)
    indices = {transfer_payload_id(payload): i for i, payload in enumerate(candidates)}
    metrics = {}
    for record in saved:
        key = record['policy_id']
        if key in metrics:
            continue
        if key in indices:
            i = indices[key]
            metrics[key] = dict(C=float(C[i]), V=float(V[i]))
        else:
            result = evaluate_policy(cohort, decode_policy(record['payload']), 1.0)
            metrics[key] = dict(C=result['C'], V=result['V'])
    return dict(C=as_tensor(C), V=as_tensor(V), saved=metrics)

def transfer_select(candidates, validation, counts, lambdas):
    """Only validation metrics enter selection; equal weight per independent cohort."""
    records = []
    for count in counts:
        C = torch.stack([row['C'] for row in validation[:count]]).mean(0)
        V = torch.stack([row['V'] for row in validation[:count]]).mean(0)
        for lam in lambdas:
            index = int(torch.argmin(C - lam * V))
            payload = candidates[index]
            records.append(dict(method=f'Common target, {count} fresh validation cohorts', lambda_=lam, validation_count=count, candidate_index=index, payload=payload, policy_id=transfer_payload_id(payload), selection_C=float(C[index]), selection_V=float(V[index]), selection_J=float(C[index] - lam * V[index]), selection_source=f'mean C-lambda*V on first {count} fresh validation cohorts'))
    return records

def transfer_scientific(value):
    if isinstance(value, dict):
        return {k: transfer_scientific(v) for k, v in value.items() if k not in ['results_dir', 'figures_dir']}
    if isinstance(value, (list, tuple)):
        return [transfer_scientific(v) for v in value]
    return value

def transfer_identity(config, bundle):
    helpers = {name: [_code_signature(inspect.unwrap(value).__code__), repr(value.__defaults__), repr(value.__kwdefaults__)] for name, value in globals().items() if name.startswith(('transfer_', 'refinement_')) and inspect.isfunction(value)}
    return stable_id(dict(schema=1, config=transfer_scientific(config), helpers=helpers, core_implementation=implementation_id()[0], source_id=bundle['source']['metadata']['config_id'], refinement_id=bundle['metadata']['config_id'], policies_and_search=bundle['fingerprints'], original_cohorts=bundle['source']['fingerprints']))

def transfer_tables(frame, config):
    keys = ['dataset', 'model', 'seed', 'lambda_', 'method', 'validation_count']
    seed_means = frame.groupby(keys, sort=False).agg(cohorts=('cohort_id', 'nunique'), C=('C', 'mean'), V=('V', 'mean'), J=('J', 'mean'), cohort_J_sd=('J', 'std'), cohort_V_sd=('V', 'std'), wasted_cost=('wasted_cost', 'mean'), incremental_validity=('incremental_validity', 'mean'), selection_J=('selection_J', 'first'), fresh_validation_J=('fresh_validation_J', 'first')).reset_index()
    summary = seed_means.groupby(['method', 'lambda_', 'validation_count'], sort=False).agg(seeds=('seed', 'nunique'), C_mean=('C', 'mean'), C_sem=('C', 'sem'), V_mean=('V', 'mean'), V_sem=('V', 'sem'), J_mean=('J', 'mean'), J_sem=('J', 'sem'), cohort_J_sd_mean=('cohort_J_sd', 'mean'), cohort_V_sd_mean=('cohort_V_sd', 'mean'), incremental_validity_mean=('incremental_validity', 'mean'), wasted_cost_mean=('wasted_cost', 'mean'), selection_J_mean=('selection_J', 'mean'), fresh_validation_J_mean=('fresh_validation_J', 'mean')).reset_index()
    baselines = ['Common target, original validation', f"Common target, {max(config['validation_counts'])} fresh validation cohorts"]
    pairs = []
    for (seed, lam, index), group in frame.groupby(['seed', 'lambda_', 'cohort_id']):
        for baseline in baselines:
            base = group[group.method == baseline]
            if len(base) != 1:
                raise ValueError('A paired cohort is missing its baseline.')
            base = base.iloc[0]
            for _, row in group.iterrows():
                if row['method'] == baseline:
                    continue
                pairs.append(dict(seed=seed, lambda_=lam, cohort_id=index, method=row['method'], baseline=baseline, delta_C=row.C - base.C, delta_V=row.V - base.V, delta_J=row.J - base.J))
    paired = pd.DataFrame(pairs)
    paired_seed = paired.groupby(['seed', 'lambda_', 'method', 'baseline'], sort=False).agg(delta_C=('delta_C', 'mean'), delta_V=('delta_V', 'mean'), delta_J=('delta_J', 'mean'), cohort_delta_J_sd=('delta_J', 'std')).reset_index()
    paired_seed['win'] = paired_seed.delta_J < -1e-10
    paired_seed['tie'] = paired_seed.delta_J.abs() <= 1e-10
    paired_summary = paired_seed.groupby(['method', 'baseline', 'lambda_'], sort=False).agg(seeds=('seed', 'nunique'), delta_C_mean=('delta_C', 'mean'), delta_V_mean=('delta_V', 'mean'), delta_J_mean=('delta_J', 'mean'), delta_J_sem=('delta_J', 'sem'), wins=('win', 'sum'), ties=('tie', 'sum')).reset_index()
    paired_summary['losses'] = paired_summary.seeds - paired_summary.wins - paired_summary.ties
    return dict(results=frame, seed_means=seed_means, summary=summary, paired_cohorts=paired, paired_seeds=paired_seed, paired_summary=paired_summary)

def transfer_figures(tables, config, config_id, stem, *, show=True):
    summary = tables['summary']
    pairs = tables['paired_summary']
    original = 'Common target, original validation'
    final = f"Common target, {max(config['validation_counts'])} fresh validation cohorts"
    primary = 'Personalized targets + selection'
    tuned = 'Personalized targets + validated cutoff'
    labels = {original: 'Common target, original selection', final: 'Common target, multiple validation cohorts', primary: 'Personalized target with selection (ours)', tuned: 'Personalized, validated cutoff', 'No recourse': 'No action'}
    figures = []
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for method in ['No recourse', original, primary, tuned, final]:
        stat = summary[summary.method == method].sort_values('lambda_')
        if stat.empty:
            continue
        axes[0].errorbar(stat.V_mean, stat.C_mean, xerr=stat.V_sem.fillna(0), yerr=stat.C_sem.fillna(0), marker='o', capsize=2, label=labels[method])
    for method, baseline, style in [(primary, original, '--'), (primary, final, '-'), (tuned, final, ':')]:
        stat = pairs[(pairs.method == method) & (pairs.baseline == baseline)].sort_values('lambda_')
        axes[1].errorbar(stat.lambda_, stat.delta_J_mean, yerr=stat.delta_J_sem.fillna(0), marker='o', linestyle=style, capsize=3, label=f"{labels[method]} vs {('original' if baseline == original else 'multiple-cohort')} common target")
    axes[0].set(xlabel='Fresh-test validity V', ylabel='Mean recourse cost C')
    axes[1].set(xlabel='lambda', ylabel='Paired test objective difference', xscale='log')
    axes[1].axhline(0, color='grey', linewidth=1)
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7)
    figures.append(('fresh_test_comparison', fig))
    fig, axes = plt.subplots(1, len(config['lambda_values']), figsize=(5 * len(config['lambda_values']), 4), squeeze=False)
    for ax, lam in zip(axes[0], config['lambda_values']):
        stat = summary[(summary.lambda_ == lam) & (summary.validation_count > 0)].sort_values('validation_count')
        ax.errorbar(stat.validation_count, stat.J_mean, yerr=stat.J_sem.fillna(0), marker='o', capsize=3, label='Fresh-test objective')
        ax.plot(stat.validation_count, stat.selection_J_mean, marker='s', linestyle='--', label='Selection objective')
        old = summary[(summary.lambda_ == lam) & (summary.method == original)].J_mean.iloc[0]
        ax.axhline(old, color='grey', linestyle=':', label='Original selection, fresh test')
        ax.set(xlabel='Fresh validation cohorts used', ylabel='C - lambda V', title=f'lambda = {lam:g}', xticks=config['validation_counts'])
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7)
    figures.append(('validation_cohort_count', fig))
    paths = []
    for name, fig in figures:
        fig.suptitle(f"{config['dataset']} | {config['model']}")
        standardize_figure(fig);fig.tight_layout()
        path = Path(config['figures_dir']).expanduser().resolve() / f'{stem}-{name}.png'
        atomic_write(path, lambda handle: fig.savefig(handle, format='png', dpi=160, bbox_inches='tight'))
        save_json(path.with_suffix('.json'), dict(config=config, config_id=config_id, plot=name, uncertainty='SEM across training-seed means; repeated cohorts are averaged within each seed.'))
        paths.append(str(path))
        if show:
            plt.show()
        plt.close(fig)
    return paths

def transfer_validate_config(config):
    if config['dataset'] not in ['Synthetic nonlinear', 'Synthetic curved'] or config['model'] not in ['affine', 'quadratic']:
        raise ValueError('Fresh independent populations are implemented only for the synthetic datasets and affine/quadratic models.')
    for key in ['cohort_size', 'test_cohorts', 'fresh_seed']:
        value = config[key]
        minimum = 2 if key == 'cohort_size' else 0 if key == 'fresh_seed' else 1
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < minimum:
            raise ValueError(f'{key} must be an integer >= {minimum}.')
    for key in ['seeds', 'validation_counts']:
        values = config[key]
        if not values or len(set(values)) != len(values) or any((isinstance(v, bool) or not isinstance(v, (int, np.integer)) or v < (0 if key == 'seeds' else 1) for v in values)):
            raise ValueError(f'Invalid {key}.')
    if config['validation_counts'] != sorted(config['validation_counts']):
        raise ValueError('validation_counts must be in increasing order.')
    if not config['lambda_values'] or len(set(config['lambda_values'])) != len(config['lambda_values']) or any((not np.isfinite(v) or v <= 0 for v in config['lambda_values'])):
        raise ValueError('Provide distinct positive lambda_values.')
    if (config['source_refinement']['dataset'], config['source_refinement']['model']) != (config['dataset'], config['model']):
        raise ValueError('Dataset/model must match the saved source.')

def transfer_run(config, *, mode='run', show=True):
    if mode not in ['run', 'preflight', 'load', 'skip']:
        raise ValueError('Choose run, preflight, load, or skip.')
    if mode == 'skip':
        return None
    transfer_validate_config(config)
    bundle = transfer_source(config)
    config_id = transfer_identity(config, bundle)
    root = bundle['source']['root']
    directory = root / 'cohort_transfer' / 'experiment_12' / slug(config['dataset']) / config['model'] / config_id
    counts = '_'.join(map(str, config['validation_counts']))
    lambdas = '_'.join((f'{v:g}' for v in config['lambda_values']))
    base = config['source_refinement']['source_config']
    stem = f"experiment-12-dataset-{slug(config['dataset'])}-model-{config['model']}-parameters-alpha{base['alpha']:g}-B{base['B']:g}-lambda{lambdas}-N{config['cohort_size']}-val{counts}-test{config['test_cohorts']}"[:165] + '-' + config_id
    print(f"{config['dataset']} / {config['model']}: {len(config['seeds'])} fitted seeds; {max(config['validation_counts'])} fresh validation and {config['test_cohorts']} fresh test cohorts per seed; 0 new training fits.", flush=True)
    if mode == 'preflight':
        for seed in config['seeds']:
            reference = refinement_contexts(bundle['source'], seed)['train']
            candidates, groups = transfer_candidates(bundle, seed)
            transfer_saved_policies(bundle, config, seed)
            val, meta = transfer_fresh_cohort(reference, config, seed, 'validation', 0)
            other, other_meta = transfer_fresh_cohort(reference, config, seed, 'test', 0)
            if meta['X_id'] == other_meta['X_id']:
                raise AssertionError('Validation and test streams overlap.')
            q = reference.t_0
            limits = [0.0, reference.B / 2, math.inf]
            C, V = transfer_target_metrics(val, q, limits)
            for index, limit in enumerate(limits):
                exact = evaluate_policy(val, CommonTarget(q, limit), 1.0)
                if not np.allclose([C[index], V[index]], [exact['C'], exact['V']], atol=1e-12, rtol=1e-12):
                    raise AssertionError('Batched baseline evaluator failed its exact check.')
            print(f'  seed {seed}: {len(candidates)} saved candidates; source and evaluation checks passed.')
        return dict(status='preflight_passed', config_id=config_id, directory=str(directory))
    if mode == 'load' and (not (directory / 'complete.json').exists()):
        raise FileNotFoundError(f'No complete transfer experiment with these settings: {directory}')
    if mode == 'run':
        save_json(directory / 'config.json', dict(config=config, config_id=config_id, source_config_id=bundle['source']['metadata']['config_id'], refinement_config_id=bundle['metadata']['config_id'], source_files=bundle['fingerprints'], source_cohorts=bundle['source']['fingerprints'], new_training_fits=0, estimand='Mean cohort objective with fixed original t_0 and separate capacity allocation per cohort.', scope='Selection robustness diagnostic: only the common-target family receives new validation-based selection.'))
        for seed in config['seeds']:
            seed_dir = directory / f'seed_{seed}'
            if (seed_dir / 'complete.json').exists():
                print(f'  seed {seed}: saved results reused.', flush=True)
                continue
            reference = refinement_contexts(bundle['source'], seed)['train']
            scorer_before = transfer_payload_id(reference.f_w0.state_dict())
            candidates, groups = transfer_candidates(bundle, seed)
            saved = transfer_saved_policies(bundle, config, seed)
            print(f'  seed {seed}: {len(candidates)} common-target candidates.', flush=True)
            validation = []
            for index in range(max(config['validation_counts'])):
                path = seed_dir / 'validation' / f'cohort_{index:04d}.pt'
                if path.exists():
                    payload = load_torch(path)
                else:
                    cohort, meta = transfer_fresh_cohort(reference, config, seed, 'validation', index)
                    payload = dict(metadata=meta, **transfer_validation(cohort, candidates, groups, saved))
                    save_torch(path, payload)
                validation.append(payload)
                if (index + 1) % 5 == 0 or index + 1 == max(config['validation_counts']):
                    print(f"    validation {index + 1}/{max(config['validation_counts'])}", flush=True)
            selected = transfer_select(candidates, validation, config['validation_counts'], config['lambda_values'])
            policies = saved + selected
            for record in policies:
                key = record['policy_id']
                if record['validation_count']:
                    index = record['candidate_index']
                    C = np.mean([float(row['C'][index]) for row in validation])
                    V = np.mean([float(row['V'][index]) for row in validation])
                else:
                    C = np.mean([row['saved'][key]['C'] for row in validation])
                    V = np.mean([row['saved'][key]['V'] for row in validation])
                record.update(fresh_validation_C=float(C), fresh_validation_V=float(V), fresh_validation_J=float(C - record['lambda_'] * V))
            save_torch(seed_dir / 'selected_policies.pt', policies)
            selection_table = pd.DataFrame([{k: v for k, v in row.items() if k != 'payload'} for row in policies])
            save_csv(seed_dir / 'selection.csv', selection_table)
            for index in range(config['test_cohorts']):
                path = seed_dir / 'test' / f'cohort_{index:04d}.csv'
                if path.exists():
                    continue
                cohort, meta = transfer_fresh_cohort(reference, config, seed, 'test', index)
                no_action = scalar_metrics(evaluate_policy(cohort, NoRecourse(), 1.0))
                computed, rows = ({}, [])
                for record in policies:
                    key = record['policy_id']
                    lam = record['lambda_']
                    if key not in computed:
                        computed[key] = scalar_metrics(evaluate_policy(cohort, decode_policy(record['payload']), 1.0))
                    metrics = dict(computed[key])
                    metrics['J'] = metrics['C'] - lam * metrics['V']
                    rows.append(dict(dataset=config['dataset'], model=config['model'], seed=seed, cohort_id=index, X_id=meta['X_id'], lambda_=lam, method=record['method'], policy_id=key, validation_count=record['validation_count'], selection_J=record['selection_J'], fresh_validation_J=record['fresh_validation_J'], no_action_V=no_action['V'], incremental_validity=metrics['V'] - no_action['V'], **metrics))
                save_csv(path, pd.DataFrame(rows))
                save_json(path.with_suffix('.json'), meta)
                if (index + 1) % 10 == 0 or index + 1 == config['test_cohorts']:
                    print(f"    test {index + 1}/{config['test_cohorts']}", flush=True)
            if transfer_payload_id(reference.f_w0.state_dict()) != scorer_before:
                raise AssertionError('The frozen scoring model changed.')
            frame = pd.concat([pd.read_csv(seed_dir / 'test' / f'cohort_{index:04d}.csv') for index in range(config['test_cohorts'])], ignore_index=True)
            if len(frame) != len(policies) * config['test_cohorts']:
                raise ValueError('A saved test cohort is incomplete; inspect its CSV before resuming.')
            save_csv(seed_dir / 'results.csv', frame)
            save_json(seed_dir / 'complete.json', dict(seed=seed, rows=len(frame), test_cohorts=config['test_cohorts']))
        frame = pd.concat([pd.read_csv(directory / f'seed_{seed}' / 'results.csv') for seed in config['seeds']], ignore_index=True)
        frame['experiment'] = 12
        frame['config_id'] = config_id
        save_csv(directory / 'results.csv', frame)
        selections = pd.concat([pd.read_csv(directory / f'seed_{seed}' / 'selection.csv').assign(seed=seed) for seed in config['seeds']], ignore_index=True)
        save_csv(directory / 'selection.csv', selections)
        save_json(directory / 'complete.json', dict(rows=len(frame), seeds=config['seeds']))
    else:
        frame = pd.read_csv(directory / 'results.csv')
    if frame.duplicated(['seed', 'lambda_', 'cohort_id', 'method']).any():
        raise ValueError('Duplicate policy/cohort result.')
    tables = transfer_tables(frame, config)
    tables['selection'] = pd.read_csv(directory / 'selection.csv')
    for name, table in tables.items():
        save_csv(directory / f'{name}.csv', table)
        save_csv(root / 'tables' / f'{stem}-{name}.csv', table)
    save_json(root / 'tables' / f'{stem}-config.json', dict(config=config, config_id=config_id, directory=str(directory)))
    figures = transfer_figures(tables, config, config_id, stem, show=show)
    display(tables['summary'][['method', 'lambda_', 'seeds', 'C_mean', 'V_mean', 'J_mean', 'J_sem', 'cohort_J_sd_mean']])
    print('Error bars use training-seed means, not the number of repeated test cohorts. Only common-target policies were reselected using fresh validation data.')
    return dict(config_id=config_id, directory=str(directory), tables=tables, figures=figures)


In [ ]:
def selection_config(transfer_settings, *, validation_count=20, test_cohorts=50, fresh_seed=20260926, source_transfer_id=None, learned_methods=None):
    """Choose among saved policies using common validation data; never train."""
    return dict(dataset=transfer_settings['dataset'], model=transfer_settings['model'], source_transfer=copy.deepcopy(transfer_settings), source_transfer_id=source_transfer_id, seeds=list(transfer_settings['seeds']), lambda_values=list(transfer_settings['lambda_values']), validation_count=validation_count, test_cohorts=test_cohorts, cohort_size=transfer_settings['cohort_size'], fresh_seed=fresh_seed, learned_methods=list(learned_methods if learned_methods is not None else ['Personalized targets + selection', 'Personalized targets, all eligible']), results_dir=transfer_settings['results_dir'], figures_dir=transfer_settings['figures_dir'])

def selection_validate_config(config):
    settings = copy.deepcopy(config['source_transfer'])
    settings.update(seeds=config['seeds'], lambda_values=config['lambda_values'], validation_counts=[config['validation_count']], test_cohorts=config['test_cohorts'], cohort_size=config['cohort_size'], fresh_seed=config['fresh_seed'])
    transfer_validate_config(settings)
    if (config['dataset'], config['model'], config['cohort_size']) != (settings['dataset'], settings['model'], config['source_transfer']['cohort_size']):
        raise ValueError('Dataset, model and cohort_size must match the saved validation study.')
    allowed = ['Personalized targets + selection', 'Personalized targets, all eligible']
    methods = config['learned_methods']
    if not methods or len(set(methods)) != len(methods) or (not set(methods) <= set(allowed)) or (allowed[0] not in methods):
        raise ValueError('Include the primary learned policy and choose distinct supported learned_methods.')

def selection_source(config):
    settings = copy.deepcopy(config['source_transfer'])
    settings.update(seeds=config['seeds'], lambda_values=config['lambda_values'], results_dir=config['results_dir'], figures_dir=config['figures_dir'])
    bundle = transfer_source(settings)
    root = bundle['source']['root']
    matches = []
    parent = root / 'cohort_transfer' / 'experiment_12' / slug(config['dataset']) / config['model']
    for path in sorted(parent.glob('*/config.json')):
        metadata = load_json(path)
        old = metadata['config']
        if config['source_transfer_id'] and metadata['config_id'] != config['source_transfer_id']:
            continue
        if metadata['source_config_id'] == bundle['source']['metadata']['config_id'] and metadata['refinement_config_id'] == bundle['metadata']['config_id'] and (old['fresh_seed'] == settings['fresh_seed']) and (old['cohort_size'] == config['cohort_size']) and (config['validation_count'] <= max(old['validation_counts'])) and (set(config['seeds']) <= set(old['seeds'])) and (set(config['lambda_values']) <= set(old['lambda_values'])) and (path.parent / 'complete.json').exists():
            matches.append((path.parent, metadata))
    if len(matches) != 1:
        raise ValueError(f"Expected one completed experiment 12; found {[m['config_id'] for _, m in matches]}. Check source_transfer_id and Results/cohort_transfer. Missing inputs never trigger training or a new search.")
    directory, metadata = matches[0]
    for field, current in [('source_files', bundle['fingerprints']), ('source_cohorts', bundle['source']['fingerprints'])]:
        if any((metadata[field].get(key) != value for key, value in current.items())):
            raise ValueError('Saved validation metrics no longer match the original policies/caches.')
    files, excluded = ({}, {})
    for seed in config['seeds']:
        seed_dir = directory / f'seed_{seed}'
        needed = [seed_dir / 'complete.json'] + [seed_dir / 'validation' / f'cohort_{i:04d}.pt' for i in range(config['validation_count'])]
        old_tests = [seed_dir / 'test' / f'cohort_{i:04d}.json' for i in range(metadata['config']['test_cohorts'])]
        for path in needed + old_tests:
            if not path.is_file():
                raise FileNotFoundError(f'Missing saved input: {path}. Keep the full original Results folder.')
            files[str(path.relative_to(root))] = hashlib.sha256(path.read_bytes()).hexdigest()
        excluded[seed] = {load_json(path)['X_id'] for path in old_tests}
        candidates, _ = transfer_candidates(bundle, seed)
        saved = transfer_saved_policies(bundle, settings, seed)
        ids = {row['policy_id'] for row in saved if row['method'] in config['learned_methods']}
        validation_ids = set()
        for index in range(config['validation_count']):
            payload = load_torch(seed_dir / 'validation' / f'cohort_{index:04d}.pt')
            meta = payload['metadata']
            entropy = [int(settings['fresh_seed']), 1201, int(seed), 101, index]
            if meta['split'] != 'validation' or meta['rng_entropy'] != entropy or meta['N'] != config['cohort_size'] or (meta['X_id'] in validation_ids):
                raise ValueError('Saved validation cohort identity is inconsistent.')
            validation_ids.add(meta['X_id'])
            if payload['C'].shape != (len(candidates),) or payload['V'].shape != (len(candidates),) or (not ids <= set(payload['saved'])) or (not torch.isfinite(payload['C']).all()) or (not torch.isfinite(payload['V']).all()):
                raise ValueError('Saved validation metrics are incomplete or inconsistent.')
            for key in ids:
                if not np.isfinite([payload['saved'][key]['C'], payload['saved'][key]['V']]).all():
                    raise ValueError('Nonfinite learned-policy validation metrics.')
        if validation_ids & excluded[seed]:
            raise ValueError('The saved validation and test cohorts overlap.')
        excluded[seed].update(validation_ids)
    return dict(bundle=bundle, directory=directory, metadata=metadata, fingerprints=files, excluded_X_ids=excluded)

def selection_choose(candidates, validation, saved, config):
    """Pure validation-only selection. No test data are accepted by this function."""
    count = config['validation_count']
    if len(validation) != count:
        raise ValueError('Wrong number of validation cohorts.')
    C = torch.stack([row['C'] for row in validation]).mean(0)
    V = torch.stack([row['V'] for row in validation]).mean(0)
    selected_common = transfer_select(candidates, validation, [count], config['lambda_values'])
    records = []
    for lam in config['lambda_values']:
        common = next((row for row in selected_common if row['lambda_'] == lam))
        options = [dict(method='No recourse', payload=candidates[0], C=float(C[0]), V=float(V[0])), dict(method='Common target, shared validation', payload=common['payload'], C=common['selection_C'], V=common['selection_V'])]
        for method in config['learned_methods']:
            matches = [row for row in saved if row['method'] == method and row['lambda_'] == lam]
            if len(matches) != 1:
                raise ValueError(f'Missing or duplicate saved policy for {method}, lambda={lam}.')
            row = matches[0]
            key = row['policy_id']
            options.append(dict(method=method, payload=row['payload'], C=float(np.mean([v['saved'][key]['C'] for v in validation])), V=float(np.mean([v['saved'][key]['V'] for v in validation]))))
        evaluated = []
        for row in options:
            evaluated.append(dict(method=row['method'], chosen_method=row['method'], selected_family=row['payload']['type'], payload=row['payload'], policy_id=transfer_payload_id(row['payload']), lambda_=lam, validation_count=count, selection_C=row['C'], selection_V=row['V'], selection_J=row['C'] - lam * row['V'], selection_source=f'mean exact objective over {count} shared validation cohorts'))
        best = min(evaluated, key=lambda row: row['selection_J'])
        records.extend(evaluated)
        records.append(dict(best, method='Validation-selected policy'))
    return records

def selection_fresh_cohort(reference, config, seed, index):
    entropy = [int(config['fresh_seed']), 1301, int(seed), 202, int(index)]
    X, _, _ = generate_correlated_nonlinear_population(config['cohort_size'], correlation=0.5, d=reference.X.shape[1], noise_std=0.0, seed=np.random.SeedSequence(entropy))
    cohort = make_cohort(X, reference.f_w0, reference.t_0, reference.B, reference.alpha, reference.actionable_mask, solver_options=reference.options)
    require_training_cohort(cohort)
    return (cohort, dict(split='test', cohort_id=index, rng_entropy=entropy, X_id=array_id(cohort.X), N=cohort.N, N_minus=cohort.N_minus, k=cohort.k))

def selection_identity(config, source):
    helpers = {name: [_code_signature(inspect.unwrap(value).__code__), repr(value.__defaults__), repr(value.__kwdefaults__)] for name, value in globals().items() if name.startswith(('selection_', 'transfer_', 'refinement_')) and inspect.isfunction(value)}
    bundle = source['bundle']
    return stable_id(dict(schema=1, config=transfer_scientific(config), helpers=helpers, core_implementation=implementation_id()[0], source_transfer_id=source['metadata']['config_id'], policies_and_search=bundle['fingerprints'], original_cohorts=bundle['source']['fingerprints'], validation_and_cohort_identities=source['fingerprints']))

def selection_tables(frame, selections):
    keys = ['dataset', 'model', 'seed', 'lambda_', 'method']
    seed_means = frame.groupby(keys, sort=False).agg(cohorts=('cohort_id', 'nunique'), C=('C', 'mean'), V=('V', 'mean'), J=('J', 'mean'), cohort_J_sd=('J', 'std'), incremental_validity=('incremental_validity', 'mean'), wasted_cost=('wasted_cost', 'mean'), selection_J=('selection_J', 'first')).reset_index()
    summary = seed_means.groupby(['method', 'lambda_'], sort=False).agg(seeds=('seed', 'nunique'), C_mean=('C', 'mean'), C_sem=('C', 'sem'), V_mean=('V', 'mean'), V_sem=('V', 'sem'), J_mean=('J', 'mean'), J_sem=('J', 'sem'), selection_J_mean=('selection_J', 'mean'), cohort_J_sd_mean=('cohort_J_sd', 'mean'), incremental_validity_mean=('incremental_validity', 'mean'), wasted_cost_mean=('wasted_cost', 'mean')).reset_index()
    pairs = []
    for (seed, lam, index), group in frame.groupby(['seed', 'lambda_', 'cohort_id']):
        for baseline in ['Common target, shared validation', 'Personalized targets + selection']:
            matches = group[group.method == baseline]
            if len(matches) != 1:
                raise ValueError('A paired cohort is missing its baseline.')
            base = matches.iloc[0]
            for row in group.to_dict('records'):
                if row['method'] != baseline:
                    pairs.append(dict(seed=seed, lambda_=lam, cohort_id=index, method=row['method'], baseline=baseline, delta_C=row['C'] - base.C, delta_V=row['V'] - base.V, delta_J=row['J'] - base.J))
    paired = pd.DataFrame(pairs)
    paired_seed = paired.groupby(['seed', 'lambda_', 'method', 'baseline'], sort=False).agg(delta_C=('delta_C', 'mean'), delta_V=('delta_V', 'mean'), delta_J=('delta_J', 'mean')).reset_index()
    paired_seed['win'] = paired_seed.delta_J < -1e-10
    paired_seed['tie'] = paired_seed.delta_J.abs() <= 1e-10
    paired_summary = paired_seed.groupby(['method', 'baseline', 'lambda_'], sort=False).agg(seeds=('seed', 'nunique'), delta_C_mean=('delta_C', 'mean'), delta_V_mean=('delta_V', 'mean'), delta_J_mean=('delta_J', 'mean'), delta_J_sem=('delta_J', 'sem'), wins=('win', 'sum'), ties=('tie', 'sum')).reset_index()
    paired_summary['losses'] = paired_summary.seeds - paired_summary.wins - paired_summary.ties
    choices = selections[selections.method == 'Validation-selected policy']
    counts = choices.groupby(['lambda_', 'chosen_method', 'selected_family'], sort=False).agg(seeds=('seed', 'nunique')).reset_index()
    counts['fraction'] = counts.seeds / counts.groupby('lambda_').seeds.transform('sum')
    return dict(results=frame, selection=selections, selection_counts=counts, seed_means=seed_means, summary=summary, paired_cohorts=paired, paired_seeds=paired_seed, paired_summary=paired_summary)

def selection_figures(tables, config, config_id, stem, *, show=True):
    summary = tables['summary']
    pairs = tables['paired_summary']
    figures = []
    labels = {'Common target, shared validation': 'Common target', 'Personalized targets + selection': 'Personalized target with selection (ours)', 'Validation-selected policy': 'Validation-selected policy'}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for method, label in labels.items():
        stat = summary[summary.method == method].sort_values('lambda_')
        axes[0].errorbar(stat.V_mean, stat.C_mean, xerr=stat.V_sem.fillna(0), yerr=stat.C_sem.fillna(0), marker='o', capsize=3, label=label)
        for row in stat.itertuples():
            axes[0].annotate(f'{row.lambda_:g}', (row.V_mean, row.C_mean), xytext=(4, 4), textcoords='offset points', fontsize=8)
    for baseline, label in list(labels.items())[:2]:
        stat = pairs[(pairs.method == 'Validation-selected policy') & (pairs.baseline == baseline)].sort_values('lambda_')
        axes[1].errorbar(stat.lambda_, stat.delta_J_mean, yerr=stat.delta_J_sem.fillna(0), marker='o', capsize=3, label=f'Selected minus {label.lower()}')
    axes[0].set(xlabel='New-test validity V (labels show lambda)', ylabel='Mean recourse cost C')
    axes[1].set(xlabel='lambda', ylabel='Paired objective difference', xscale='log')
    axes[1].axhline(0, color='grey', linewidth=1)
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    figures.append(('cost_validity_and_objective', fig))
    fig, ax = plt.subplots(figsize=(7, 4))
    choices = tables['selection_counts']
    lambdas = config['lambda_values']
    bottom = np.zeros(len(lambdas))
    for method in ['No recourse', 'Common target, shared validation'] + config['learned_methods']:
        values = choices[choices.chosen_method == method].groupby('lambda_').fraction.sum().reindex(lambdas, fill_value=0).to_numpy()
        if values.any():
            ax.bar(np.arange(len(lambdas)), values, bottom=bottom, label=labels.get(method, method))
            bottom += values
    ax.set(xticks=np.arange(len(lambdas)), xticklabels=[f'{v:g}' for v in lambdas], xlabel='lambda', ylabel='Fraction of fitting seeds', ylim=(0, 1.02))
    ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
    figures.append(('selected_policy', fig))
    paths = []
    for name, fig in figures:
        fig.suptitle(f"{config['dataset']} | {config['model']}")
        standardize_figure(fig);fig.tight_layout()
        path = Path(config['figures_dir']).expanduser().resolve() / f'{stem}-{name}.png'
        atomic_write(path, lambda handle: fig.savefig(handle, format='png', dpi=160, bbox_inches='tight'))
        save_json(path.with_suffix('.json'), dict(config=config, config_id=config_id, plot=name, uncertainty='SEM across fitting-seed means; cohorts averaged within seed.'))
        paths.append(str(path))
        if show:
            plt.show()
        plt.close(fig)
    return paths

def selection_check_rows(frame, policies, config, seed, index=None):
    count = config['test_cohorts'] if index is None else 1
    required = {(row['lambda_'], row['method'], row['policy_id']) for row in policies}
    if len(frame) != count * len(required) or frame.duplicated(['seed', 'lambda_', 'method', 'cohort_id']).any():
        raise ValueError('Saved new-test results are incomplete or duplicated.')
    expected_indices = set(range(config['test_cohorts'])) if index is None else {index}
    if set(frame.seed) != {seed} or set(frame.cohort_id) != expected_indices:
        raise ValueError('Unexpected seed or cohort in saved results.')
    for _, group in frame.groupby('cohort_id'):
        if set(zip(group.lambda_, group.method, group.policy_id)) != required or group.X_id.nunique() != 1:
            raise ValueError('Saved policies or shared cohort identities do not match.')
    if not np.isfinite(frame[['C', 'V', 'J']].to_numpy()).all() or not np.allclose(frame.J, frame.C - frame.lambda_ * frame.V, atol=1e-11, rtol=1e-11) or (not np.allclose(frame.accepted_mass, frame.k, atol=1e-10, rtol=0)):
        raise ValueError('Saved objective or capacity checks failed.')

def selection_run(config, *, mode='run', show=True):
    if mode not in ['run', 'preflight', 'load', 'skip']:
        raise ValueError('Choose run, preflight, load, or skip.')
    if mode == 'skip':
        return None
    selection_validate_config(config)
    source = selection_source(config)
    config_id = selection_identity(config, source)
    bundle = source['bundle']
    root = bundle['source']['root']
    directory = root / 'policy_selection' / 'experiment_13' / slug(config['dataset']) / config['model'] / config_id
    base = config['source_transfer']['source_refinement']['source_config']
    lambdas = '_'.join((f'{v:g}' for v in config['lambda_values']))
    stem = f"experiment-13-dataset-{slug(config['dataset'])}-model-{config['model']}-parameters-alpha{base['alpha']:g}-B{base['B']:g}-lambda{lambdas}-N{config['cohort_size']}-val{config['validation_count']}-test{config['test_cohorts']}"[:165] + '-' + config_id
    print(f"{config['dataset']} / {config['model']}: {len(config['seeds'])} fitted seeds; {config['validation_count']} saved shared validation cohorts; {config['test_cohorts']} new test cohorts per seed; 0 training fits, 0 new target searches.", flush=True)
    if mode == 'load' and (not (directory / 'complete.json').exists()):
        raise FileNotFoundError(f'No completed selection experiment with these settings: {directory}')
    if mode in ['run', 'preflight']:
        if mode == 'run':
            save_json(directory / 'config.json', dict(config=config, config_id=config_id, source_transfer_id=source['metadata']['config_id'], source_files=source['fingerprints'], new_training_fits=0, new_validation_cohorts=0, scope='Select a fixed policy per seed/lambda using only shared saved validation cohorts; independent new test stream.'))
        for seed in config['seeds']:
            seed_dir = directory / f'seed_{seed}'
            candidates, _ = transfer_candidates(bundle, seed)
            saved = transfer_saved_policies(bundle, config, seed)
            validation = [load_torch(source['directory'] / f'seed_{seed}' / 'validation' / f'cohort_{i:04d}.pt') for i in range(config['validation_count'])]
            policies = selection_choose(candidates, validation, saved, config)
            selected = [row for row in policies if row['method'] == 'Validation-selected policy']
            print(f'  seed {seed}: ' + ', '.join((f"lambda {row['lambda_']:g}: {row['chosen_method']}" for row in selected)), flush=True)
            reference = refinement_contexts(bundle['source'], seed)['train']
            excluded = source['excluded_X_ids'][seed] | {array_id(reference.X)}
            if mode == 'preflight':
                _, meta = selection_fresh_cohort(reference, config, seed, 0)
                if meta['X_id'] in excluded:
                    raise ValueError('New test cohort overlaps earlier cohorts.')
                continue
            saved_path = seed_dir / 'selected_policies.pt'
            if saved_path.exists():
                if transfer_payload_id(load_torch(saved_path)) != transfer_payload_id(policies):
                    raise ValueError('Saved selections differ from deterministic validation selection.')
            else:
                save_torch(saved_path, policies)
            selection_table = pd.DataFrame([{k: v for k, v in row.items() if k != 'payload'} for row in policies]).assign(seed=seed)
            save_csv(seed_dir / 'selection.csv', selection_table)
            scorer_before = transfer_payload_id(reference.f_w0.state_dict())
            seen = set(excluded)
            for index in range(config['test_cohorts']):
                path = seed_dir / 'test' / f'cohort_{index:04d}.csv'
                meta_path = path.with_suffix('.json')
                if path.exists() and meta_path.exists():
                    meta = load_json(meta_path)
                    part = pd.read_csv(path)
                    selection_check_rows(part, policies, config, seed, index)
                    if set(part.X_id) != {meta['X_id']}:
                        raise ValueError('Saved cohort identity disagrees with results.')
                else:
                    cohort, meta = selection_fresh_cohort(reference, config, seed, index)
                    no_action = scalar_metrics(evaluate_policy(cohort, NoRecourse(), 1.0))
                    computed, rows = ({}, [])
                    for record in policies:
                        key = record['policy_id']
                        lam = record['lambda_']
                        if key not in computed:
                            computed[key] = scalar_metrics(evaluate_policy(cohort, decode_policy(record['payload']), 1.0))
                        metrics = dict(computed[key])
                        metrics['J'] = metrics['C'] - lam * metrics['V']
                        rows.append(dict(dataset=config['dataset'], model=config['model'], seed=seed, cohort_id=index, X_id=meta['X_id'], lambda_=lam, method=record['method'], policy_id=key, chosen_method=record['chosen_method'], selected_family=record['selected_family'], validation_count=config['validation_count'], selection_J=record['selection_J'], no_action_V=no_action['V'], incremental_validity=metrics['V'] - no_action['V'], **metrics))
                    part = pd.DataFrame(rows)
                    selection_check_rows(part, policies, config, seed, index)
                    save_csv(path, part)
                    save_json(meta_path, meta)
                expected = [int(config['fresh_seed']), 1301, int(seed), 202, index]
                if meta['rng_entropy'] != expected or meta['X_id'] in seen:
                    raise ValueError('New test streams overlap or their saved identities are inconsistent.')
                seen.add(meta['X_id'])
                if (index + 1) % 10 == 0 or index + 1 == config['test_cohorts']:
                    print(f"    test {index + 1}/{config['test_cohorts']}", flush=True)
            if transfer_payload_id(reference.f_w0.state_dict()) != scorer_before:
                raise AssertionError('The frozen scorer changed.')
            frame = pd.concat([pd.read_csv(seed_dir / 'test' / f'cohort_{i:04d}.csv') for i in range(config['test_cohorts'])], ignore_index=True)
            selection_check_rows(frame, policies, config, seed)
            if not np.allclose(frame.t_0, reference.t_0, atol=1e-15, rtol=0):
                raise ValueError('The original threshold changed.')
            save_csv(seed_dir / 'results.csv', frame)
            save_json(seed_dir / 'complete.json', dict(seed=seed, rows=len(frame), test_cohorts=config['test_cohorts']))
        if mode == 'preflight':
            return dict(status='preflight_passed', config_id=config_id, directory=str(directory))
        frame = pd.concat([pd.read_csv(directory / f'seed_{seed}' / 'results.csv') for seed in config['seeds']], ignore_index=True)
        frame['experiment'] = 13
        frame['config_id'] = config_id
        selections = pd.concat([pd.read_csv(directory / f'seed_{seed}' / 'selection.csv') for seed in config['seeds']], ignore_index=True)
        save_csv(directory / 'results.csv', frame)
        save_csv(directory / 'selection.csv', selections)
        save_json(directory / 'complete.json', dict(rows=len(frame), seeds=config['seeds']))
    else:
        frame = pd.read_csv(directory / 'results.csv')
        selections = pd.read_csv(directory / 'selection.csv')
        for seed in config['seeds']:
            policies = load_torch(directory / f'seed_{seed}' / 'selected_policies.pt')
            selection_check_rows(frame[frame.seed == seed], policies, config, seed)
    tables = selection_tables(frame, selections)
    for name, table in tables.items():
        save_csv(directory / f'{name}.csv', table)
        save_csv(root / 'tables' / f'{stem}-{name}.csv', table)
    save_json(root / 'tables' / f'{stem}-config.json', dict(config=config, config_id=config_id, directory=str(directory)))
    figures = selection_figures(tables, config, config_id, stem, show=show)
    display(tables['summary'][['method', 'lambda_', 'seeds', 'C_mean', 'V_mean', 'J_mean', 'J_sem']])
    display(tables['selection_counts'])
    print('Selection uses validation only. Validation dominance is guaranteed over the candidates; new-test dominance is not guaranteed. Error bars use fitting-seed means.')
    return dict(config_id=config_id, directory=str(directory), tables=tables, figures=figures)


In [ ]:
def sensitivity_config(source, *, axis, values, seeds=None, lambda_values=(3.0, 10.0, 30.0), tau_values=(0.01,), eta=0.2, validation_cohorts=None, test_cohorts=None, fresh_seed=20260927, grid_levels=((30, 1),)):
    """Independent one-factor study; scientific settings are fixed before testing."""
    if axis not in ['B', 'alpha']:
        raise ValueError('Choose B or alpha.')
    base = copy.deepcopy(source)
    base.update(experiment=10, seeds=list(source['seeds'] if seeds is None else seeds), lambda_values=list(lambda_values), tau_values=list(tau_values), eta_values=[eta], eta=eta, methods=list(MAIN_METHODS))
    base['policy'].setdefault('rho_cutoffs', [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
    synthetic = base['dataset'].startswith('Synthetic')
    return dict(experiment=14 if axis == 'B' else 15, axis=axis, values=list(values), base_config=base, dataset=base['dataset'], model=base['model'], evaluation='fresh_synthetic' if synthetic else 'heldout', validation_cohorts=(20 if synthetic else 1) if validation_cohorts is None else validation_cohorts, test_cohorts=(50 if synthetic else 1) if test_cohorts is None else test_cohorts, fresh_seed=fresh_seed, grid_levels=[list(v) for v in grid_levels], results_dir=base['results_dir'], figures_dir=base['figures_dir'])

def sensitivity_validate(config):
    base = config['base_config']
    validate_config(base)
    if config['axis'] not in ['B', 'alpha'] or config['experiment'] != (14 if config['axis'] == 'B' else 15):
        raise ValueError('Use experiment 14 for B and 15 for alpha.')
    if (config['dataset'], config['model']) != (base['dataset'], base['model']) or base['model'] not in ['affine', 'quadratic']:
        raise ValueError('Dataset/model must match base_config; use affine or quadratic.')
    values = config['values']
    if not values or len(set(values)) != len(values) or (not np.isfinite(values).all()) or (values != sorted(values)) or any((v < 0 if config['axis'] == 'B' else not 0 < v < 1 for v in values)):
        raise ValueError('Provide sorted distinct nonnegative budgets or admission rates in (0,1).')
    synthetic = base['dataset'].startswith('Synthetic')
    if config['evaluation'] != ('fresh_synthetic' if synthetic else 'heldout'):
        raise ValueError('Use fresh_synthetic for synthetic data and heldout for real data.')
    for key in ['validation_cohorts', 'test_cohorts', 'fresh_seed']:
        value = config[key]
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < (0 if key == 'fresh_seed' else 1):
            raise ValueError(f'Invalid {key}.')
    if not synthetic and (config['validation_cohorts'] != 1 or config['test_cohorts'] != 1):
        raise ValueError('Real-data studies use one held-out validation/test cohort; repeated resamples are not independent cohorts.')
    levels = config['grid_levels']
    if not levels or any((len(v) != 2 or any((isinstance(n, bool) or not isinstance(n, (int, np.integer)) or n < 1 for n in v)) for v in levels)):
        raise ValueError('grid_levels must contain target counts >=2 and cost-quantile count 1.')
    if any(v[0]<2 or v[1]!=1 for v in config['grid_levels']):
        raise ValueError('Use (target_points,1), with no cost-selection grid.')


def sensitivity_code_id():
    helpers = {name: [_code_signature(inspect.unwrap(value).__code__), repr(value.__defaults__), repr(value.__kwdefaults__)] for name, value in globals().items() if inspect.isfunction(value) and name.startswith(('sensitivity_', 'selection_', 'transfer_', 'refinement_'))}
    return stable_id(dict(schema=1, core=implementation_id()[0], helpers=helpers))

def sensitivity_study_id(config):
    return stable_id(dict(config=transfer_scientific(config), code=sensitivity_code_id(), data_source=data_source_id(config['base_config'])))

def sensitivity_case_id(config, case, contexts):
    base = config['base_config']
    ref = contexts['train']
    return stable_id(dict(schema=1, code=sensitivity_code_id(), dataset=base['dataset'], model=base['model'], seed=case['seed'], B=case['B'], alpha=case['alpha'], t_0=ref.t_0, train_X=array_id(ref.X), validation_X=array_id(contexts['val'].X), mask=array_id(ref.actionable_mask), scorer=transfer_payload_id(ref.f_w0.state_dict()), solver=ref.options, T=base['T'], tau_values=base['tau_values'], eta_values=base['eta_values'], lambda_values=base['lambda_values'], policy=base['policy'], diagnostics=base['diagnostics'], grid_levels=config['grid_levels'], evaluation=config['evaluation'], N=base['cohort_size'], validation_cohorts=config['validation_cohorts'], test_cohorts=config['test_cohorts'], fresh_seed=config['fresh_seed']))

def sensitivity_candidates(reference, config, directory):
    """Training-derived refined family; no validation outcomes used to form candidates."""
    path = directory / 'common_candidates.pt'
    targets = sorted(set((float(q) for n, _ in config['grid_levels'] for q in np.linspace(reference.t_0, 1.0, n))))
    quantiles = sorted(set((float(q) for _, n in config['grid_levels'] for q in torch.linspace(0.0, 1.0, n))))
    state = load_torch(path) if path.exists() else dict(next_target=0, candidates=[encode_policy(NoRecourse())])
    if reference.B == 0:
        state = dict(next_target=len(targets), candidates=[encode_policy(NoRecourse())])
        if not path.exists():
            save_torch(path, state)
    for index in range(state['next_target'], len(targets)):
        q = targets[index]
        feasible, costs = refinement_target_data(reference, q)
        limits = [math.inf]

        state['candidates'].extend((encode_policy(CommonTarget(q, float(limit))) for limit in limits))
        state['next_target'] = index + 1
        save_torch(path, state)
    groups = {}
    for index, payload in enumerate(state['candidates'][1:], start=1):
        groups.setdefault(payload['q_c'], []).append((index, payload['cost_limit']))
    return (state['candidates'], groups)

def sensitivity_learned(runner, config, case, contexts, keys, directory):
    path = directory / 'learned_policies.pt'
    if path.exists():
        return load_torch(path)
    base = config['base_config']
    records = []
    audits = []
    for lam in base['lambda_values']:
        best = {}
        for tau in base['tau_values']:
            for eta in base['eta_values']:
                for method in ['Personalized targets + selection', 'Personalized targets, all eligible']:
                    current = dict(case, lambda_=float(lam), tau=float(tau), eta=float(eta))
                    tag = f'lambda_{lam:g}_tau_{tau:g}_eta_{eta:g}_{slug(method)}'
                    if not contexts['train'].eligible.any():
                        metrics = scalar_metrics(evaluate_policy(contexts['val'], NoRecourse(), lam))
                        fit = dict(policy=NoRecourse(), cutoff_policy=NoRecourse(), validation=metrics, cutoff_validation=metrics, selected_iteration=-1, cutoff_iteration=-1)
                        audits.append(dict(seed=case['seed'], lambda_=lam, tau=tau, eta=eta, method=method, status='no eligible training applicants', steps=0, objective_increases=0, sufficient_decrease_failures=0, final_G_eta_sq=0.0))
                    else:
                        fit = runner.fit(current, contexts, keys, method, trial_directory=directory / 'training' / tag)
                        history = fit['history']
                        delta = np.diff(history.J_hat.to_numpy())
                        gap = delta + eta * history.G_eta_sq.to_numpy()[:-1] / 2
                        audits.append(dict(seed=case['seed'], lambda_=lam, tau=tau, eta=eta, method=method, status='ok', fit_id=fit['fit_id'], steps=len(history) - 1, objective_increases=int((delta > 1e-10).sum()), sufficient_decrease_failures=int((gap > 1e-10).sum()), final_G_eta_sq=float(history.G_eta_sq.iloc[-1]), max_capacity_residual=float(history.capacity_residual.abs().max())))
                        save_csv(directory / 'training' / tag / 'history.csv', history)
                    variants = [(method, fit['policy'], fit['validation'], fit['selected_iteration'])]
                    for name, policy, metrics, iteration in variants:
                        payload = encode_policy(policy)
                        row = dict(method=name, lambda_=lam, payload=payload, policy_id=transfer_payload_id(payload), selection_J=metrics['J'], tau=tau, eta=eta, selected_iteration=iteration)
                        if name not in best or row['selection_J'] < best[name]['selection_J']:
                            best[name] = row
                    for cohort in contexts.values():
                        cohort.response_cache.clear()
        records.extend(best.values())
    save_torch(path, records)
    save_csv(directory / 'training_summary.csv', pd.DataFrame(audits))
    return records

def sensitivity_cohort(reference, config, seed, split, index, contexts, runner, case):
    if config['evaluation'] == 'heldout':
        if index != 0:
            raise ValueError('Only one held-out cohort is available.')
        cohort = contexts['val'] if split == 'validation' else runner.contexts(case, include_test=True)[0]['test']
        entropy = None
    else:
        entropy = [int(config['fresh_seed']), 1401, int(seed), 101 if split == 'validation' else 202, int(index)]
        X, _, _ = generate_correlated_nonlinear_population(config['base_config']['cohort_size'], correlation=0.5, d=reference.X.shape[1], noise_std=0.0, seed=np.random.SeedSequence(entropy))
        cohort = make_cohort(X, reference.f_w0, reference.t_0, case['B'], case['alpha'], reference.actionable_mask, solver_options=reference.options)
    require_training_cohort(cohort)
    return (cohort, dict(split=split, cohort_id=index, rng_entropy=entropy, X_id=array_id(cohort.X), N=cohort.N, N_minus=cohort.N_minus, k=cohort.k, t_0=cohort.t_0))

def sensitivity_case(runner, config, case, contexts, keys, directory, case_id):
    ref = contexts['train']
    scorer_before = transfer_payload_id(ref.f_w0.state_dict())
    save_json(directory / 'config.json', dict(case_id=case_id, case=case, config=config, scorer_id=scorer_before, t_0=ref.t_0, train_X_id=array_id(ref.X), early_validation_X_id=array_id(contexts['val'].X)))
    policies_path = directory / 'selected_policies.pt'
    if policies_path.exists():
        policies = load_torch(policies_path)
    else:
        saved = sensitivity_learned(runner, config, case, contexts, keys, directory)
        candidates, groups = sensitivity_candidates(ref, config, directory)
        validation = []
        extra_payload = encode_policy(CommonTarget(ref.t_0))
        extra = dict(method='Original threshold', payload=extra_payload, policy_id=transfer_payload_id(extra_payload))
        for index in range(config['validation_cohorts']):
            path = directory / 'validation' / f'cohort_{index:04d}.pt'
            if path.exists():
                payload = load_torch(path)
            else:
                cohort, meta = sensitivity_cohort(ref, config, case['seed'], 'validation', index, contexts, runner, case)
                payload = dict(metadata=meta, **transfer_validation(cohort, candidates, groups, saved + [extra]))
                save_torch(path, payload)
            validation.append(payload)
            if (index + 1) % 5 == 0 or index + 1 == config['validation_cohorts']:
                print(f"    validation {index + 1}/{config['validation_cohorts']}", flush=True)
        selection_settings = dict(validation_count=config['validation_cohorts'], lambda_values=config['base_config']['lambda_values'], learned_methods=['Personalized targets + selection', 'Personalized targets, all eligible'])
        policies = selection_choose(candidates, validation, saved, selection_settings)
        C = np.mean([row['saved'][extra['policy_id']]['C'] for row in validation])
        V = np.mean([row['saved'][extra['policy_id']]['V'] for row in validation])
        for lam in config['base_config']['lambda_values']:
            policies.append(dict(extra, lambda_=lam, chosen_method='Original threshold', selected_family='common', validation_count=config['validation_cohorts'], selection_C=float(C), selection_V=float(V), selection_J=float(C - lam * V), selection_source='comparison only; excluded from selector'))
        save_torch(policies_path, policies)
    selections = pd.DataFrame([{k: v for k, v in row.items() if k != 'payload'} for row in policies]).assign(seed=case['seed'])
    save_csv(directory / 'selection.csv', selections)
    for row in policies:
        if row['method'] == 'Validation-selected policy':
            print(f"    lambda {row['lambda_']:g}: {row['chosen_method']}", flush=True)
    excluded = {array_id(ref.X), array_id(contexts['val'].X)}
    for index in range(config['validation_cohorts']):
        excluded.add(load_torch(directory / 'validation' / f'cohort_{index:04d}.pt')['metadata']['X_id'])
    seen = set(excluded)
    for index in range(config['test_cohorts']):
        path = directory / 'test' / f'cohort_{index:04d}.csv'
        meta_path = path.with_suffix('.json')
        if path.exists() and meta_path.exists():
            part = pd.read_csv(path)
            meta = load_json(meta_path)
        else:
            cohort, meta = sensitivity_cohort(ref, config, case['seed'], 'test', index, contexts, runner, case)
            no_action = scalar_metrics(evaluate_policy(cohort, NoRecourse(), 1.0))
            computed = {}
            rows = []
            for record in policies:
                key = record['policy_id']
                lam = record['lambda_']
                if key not in computed:
                    computed[key] = scalar_metrics(evaluate_policy(cohort, decode_policy(record['payload']), 1.0))
                metrics = dict(computed[key])
                metrics['J'] = metrics['C'] - lam * metrics['V']
                rows.append(dict(dataset=config['dataset'], model=config['model'], seed=case['seed'], B=case['B'], alpha=case['alpha'], lambda_=lam, cohort_id=index, X_id=meta['X_id'], N=cohort.N, method=record['method'], policy_id=key, chosen_method=record['chosen_method'], selected_family=record['selected_family'], selection_J=record['selection_J'], no_action_V=no_action['V'], incremental_validity=metrics['V'] - no_action['V'], rejected_fraction=cohort.N_minus / cohort.N, rejected_acceptance_mass=metrics['V'] * cohort.N_minus / cohort.N, share_of_places=metrics['V'] * cohort.N_minus / cohort.k, actual_admission_rate=cohort.k / cohort.N, **metrics))
            part = pd.DataFrame(rows)
            save_csv(path, part)
            save_json(meta_path, meta)
        selection_check_rows(part, policies, config, case['seed'], index)
        if meta['X_id'] in seen or set(part.X_id) != {meta['X_id']} or (not np.allclose(part.t_0, ref.t_0, atol=1e-15, rtol=0)):
            raise ValueError('Test pairing, independence, or the initial threshold is inconsistent.')
        seen.add(meta['X_id'])
        if (index + 1) % 10 == 0 or index + 1 == config['test_cohorts']:
            print(f"    test {index + 1}/{config['test_cohorts']}", flush=True)
    if transfer_payload_id(ref.f_w0.state_dict()) != scorer_before:
        raise AssertionError('The frozen scorer changed.')
    frame = pd.concat([pd.read_csv(directory / 'test' / f'cohort_{i:04d}.csv') for i in range(config['test_cohorts'])], ignore_index=True)
    selection_check_rows(frame, policies, config, case['seed'])
    save_csv(directory / 'results.csv', frame)
    save_json(directory / 'complete.json', dict(case_id=case_id, rows=len(frame)))
    return (frame, selections, pd.read_csv(directory / 'training_summary.csv'))

def sensitivity_tables(frame, selections, audits, config):
    collected = {}
    for value, group in frame.groupby('sweep_value', sort=True):
        tables = selection_tables(group, selections[selections.sweep_value == value])
        extra = ['share_of_places', 'rejected_acceptance_mass', 'rejected_fraction', 'actual_admission_rate', 'eligible_fraction']
        seed_extra = group.groupby(['dataset', 'model', 'seed', 'lambda_', 'method'])[extra].mean().reset_index()
        tables['seed_means'] = tables['seed_means'].merge(seed_extra, on=['dataset', 'model', 'seed', 'lambda_', 'method'], validate='one_to_one')
        means = seed_extra.groupby(['method', 'lambda_'])[extra].agg(['mean', 'sem'])
        means.columns = ['_'.join(c) for c in means.columns]
        tables['summary'] = tables['summary'].merge(means.reset_index(), on=['method', 'lambda_'], validate='one_to_one')
        for name, table in tables.items():
            collected.setdefault(name, []).append(table.assign(sweep_value=value, axis=config['axis']))
    tables = {name: pd.concat(values, ignore_index=True) for name, values in collected.items()}
    tables['training_summary'] = audits
    reference_value = config['base_config'][config['axis']]
    columns = ['C', 'V', 'J', 'share_of_places', 'rejected_acceptance_mass']
    means = tables['seed_means']
    reference = means[means.sweep_value == reference_value]
    if not reference.empty:
        joined = means.merge(reference[['seed', 'lambda_', 'method'] + columns], on=['seed', 'lambda_', 'method'], suffixes=('', '_reference'), validate='many_to_one')
        keys = ['seed', 'lambda_', 'method', 'sweep_value']
        contrasts = joined[keys].copy()
        contrasts['reference_value'] = reference_value
        for name in columns:
            contrasts['delta_' + name] = joined[name] - joined[name + '_reference']
        grouped = contrasts.groupby(['lambda_', 'method', 'sweep_value', 'reference_value'])
        stats = grouped[['delta_' + name for name in columns]].agg(['mean', 'sem'])
        stats.columns = ['_'.join(c) for c in stats.columns]
        tables['parameter_paired_seeds'] = contrasts
        tables['parameter_paired_summary'] = stats.reset_index().merge(grouped.seed.nunique().rename('seeds').reset_index(), on=['lambda_', 'method', 'sweep_value', 'reference_value'])
    return tables

def sensitivity_figures(tables, config, config_id, stem, *, show=True):
    summary = tables['summary']
    figures = []
    labels = {'Common target, shared validation': 'Common target', 'Personalized targets + selection': 'Personalized target with selection (ours)', 'Validation-selected policy': 'Validation-selected policy'}
    xlabel = 'Budget B' if config['axis'] == 'B' else 'Admission rate alpha'
    for lam in config['base_config']['lambda_values']:
        fig, axes = plt.subplots(2, 2, figsize=(11, 7))
        for method, label in labels.items():
            stat = summary[(summary.lambda_ == lam) & (summary.method == method)].sort_values('sweep_value')
            for ax, metric, ylabel in zip(axes.flat, ['V', 'C', 'J', 'share_of_places'], ['Post-shift validity V', 'Mean recourse cost C', 'C - lambda V', 'Share of places going to initially rejected']):
                ax.errorbar(stat.sweep_value, stat[metric + '_mean'], yerr=stat[metric + '_sem'].fillna(0), marker='o', capsize=3, label=label)
                ax.set(xlabel=xlabel, ylabel=ylabel, xticks=config['values'])
                ax.grid(alpha=0.25)
        axes[0, 0].legend(fontsize=8)
        fig.suptitle(f"{config['dataset']} | {config['model']} | lambda = {lam:g}")
        figures.append((f'sensitivity-lambda{lam:g}', fig))
    counts = tables['selection_counts']
    lambdas = config['base_config']['lambda_values']
    fig, axes = plt.subplots(1, len(lambdas), figsize=(5 * len(lambdas), 4), squeeze=False)
    methods = ['No recourse', 'Common target, shared validation', 'Personalized targets + selection', 'Personalized targets, all eligible']
    colors = dict(zip(methods, plt.get_cmap('tab10').colors))
    for ax, lam in zip(axes[0], lambdas):
        bottom = np.zeros(len(config['values']))
        for method in methods:
            s = counts[(counts.lambda_ == lam) & (counts.chosen_method == method)]
            values = s.groupby('sweep_value').fraction.sum().reindex(config['values'], fill_value=0).to_numpy()
            ax.bar(np.arange(len(values)), values, bottom=bottom, color=colors[method], label=labels.get(method, method))
            bottom += values
        ax.set(xticks=np.arange(len(config['values'])), xticklabels=[f'{v:g}' for v in config['values']], xlabel=xlabel, ylabel='Fraction of fitting seeds', ylim=(0, 1.02), title=f'lambda = {lam:g}')
    axes[0, -1].legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.01, 1))
    fig.suptitle(f"{config['dataset']} | {config['model']}")
    figures.append(('selected_policy', fig))
    paths = []
    for name, fig in figures:
        standardize_figure(fig);fig.tight_layout()
        path = Path(config['figures_dir']).expanduser().resolve() / f'{stem}-{name}.png'
        atomic_write(path, lambda f: fig.savefig(f, format='png', dpi=160, bbox_inches='tight'))
        save_json(path.with_suffix('.json'), dict(config=config, config_id=config_id, plot=name, uncertainty='SEM across fitting-seed means after averaging cohorts within seed.'))
        paths.append(str(path))
        if show:
            plt.show()
        plt.close(fig)
    return paths

def sensitivity_run(config, *, mode='run', show=True):
    if mode not in ['run', 'preflight', 'load', 'skip']:
        raise ValueError('Choose run, preflight, load or skip.')
    if mode == 'skip':
        return None
    sensitivity_validate(config)
    base = copy.deepcopy(config['base_config'])
    base.update(results_dir=config['results_dir'], figures_dir=config['figures_dir'])
    config_id = sensitivity_study_id(config)
    root = Path(config['results_dir']).expanduser().resolve()
    directory = root / 'sensitivity' / f"experiment_{config['experiment']}" / slug(config['dataset']) / config['model'] / config_id
    levels = '_'.join((f'{v:g}' for v in config['values']))
    lambdas = '_'.join((f'{v:g}' for v in base['lambda_values']))
    fixed = f"alpha{base['alpha']:g}" if config['axis'] == 'B' else f"B{base['B']:g}"
    stem = f"experiment-{config['experiment']}-dataset-{slug(config['dataset'])}-model-{config['model']}-parameters-{config['axis']}{levels}-{fixed}-lambda{lambdas}-N{base['cohort_size']}"[:165] + '-' + config_id
    fits = len(base['seeds']) * len(config['values']) * len(base['lambda_values']) * len(base['tau_values']) * len(base['eta_values']) * 2
    print(f"{config['dataset']} / {config['model']}: {config['axis']}={config['values']}; at most {fits} policy fits before cache reuse.", flush=True)
    if mode == 'load':
        if not (directory / 'complete.json').exists():
            raise FileNotFoundError(f'No completed matching study: {directory}')
        frame = pd.read_csv(directory / 'results.csv')
        selections = pd.read_csv(directory / 'selection.csv')
        audits = pd.read_csv(directory / 'training_summary.csv')
    else:
        runner = ExperimentRunner(base)
        frames = []
        choices = []
        audit_frames = []
        case_records = []
        for seed in base['seeds']:
            scorer_id = None
            train_X_id = None
            for value in config['values']:
                case = dict(seed=seed, B=base['B'], alpha=base['alpha'], lambda_=base['lambda_values'][0], tau=base['tau_values'][0], eta=base['eta_values'][0], T=base['T'])
                case[config['axis']] = float(value)
                contexts, keys = runner.contexts(case, include_test=False)
                ref = contexts['train']
                current = transfer_payload_id(ref.f_w0.state_dict())
                X_id = array_id(ref.X)
                if scorer_id is not None and (current != scorer_id or X_id != train_X_id):
                    raise AssertionError('Scorer or training applicants changed across the sweep.')
                scorer_id, train_X_id = (current, X_id)
                expected = initial_threshold(ref.s_0, case['alpha'])
                if ref.t_0 != expected:
                    raise AssertionError('Incorrect initial threshold for admission rate.')
                case_id = sensitivity_case_id(config, case, contexts)
                case_dir = root / 'sensitivity' / 'cases' / slug(config['dataset']) / config['model'] / case_id
                case_records.append(dict(seed=seed, sweep_value=value, case_id=case_id, directory=str(case_dir), B=case['B'], alpha=case['alpha'], t_0=ref.t_0, train_X_id=X_id, scorer_id=current, train_N=ref.N, train_N_minus=ref.N_minus, train_k=ref.k))
                print(f"  seed {seed}, B={case['B']:g}, alpha={case['alpha']:g}, t_0={ref.t_0:.6g}", flush=True)
                if mode == 'preflight':
                    val, vm = sensitivity_cohort(ref, config, seed, 'validation', 0, contexts, runner, case)
                    test, tm = sensitivity_cohort(ref, config, seed, 'test', 0, contexts, runner, case)
                    if vm['X_id'] == tm['X_id']:
                        raise ValueError('Validation and test cohorts overlap.')
                    evaluate_policy(val, CommonTarget(ref.t_0), 1.0)
                    if ref.eligible.any():
                        surrogates = PolicySurrogates(ref.X.shape[1], seed=seed)
                        loss, _ = J_hat(ref, surrogates, case['tau'], case['lambda_'], 'full')
                        loss.backward()
                        if not torch.isfinite(loss) or any((not torch.isfinite(p.grad).all() for p in surrogates.parameters())):
                            raise FloatingPointError('Nonfinite pilot objective or gradients.')
                    continue
                if (case_dir / 'complete.json').exists():
                    policies = load_torch(case_dir / 'selected_policies.pt')
                    part = pd.read_csv(case_dir / 'results.csv')
                    selection_check_rows(part, policies, config, seed)
                    selected = pd.read_csv(case_dir / 'selection.csv')
                    audit = pd.read_csv(case_dir / 'training_summary.csv')
                    print('    saved case reused', flush=True)
                else:
                    part, selected, audit = sensitivity_case(runner, config, case, contexts, keys, case_dir, case_id)
                frames.append(part.assign(sweep_value=value, axis=config['axis'], case_id=case_id))
                choices.append(selected.assign(sweep_value=value, axis=config['axis'], case_id=case_id))
                audit_frames.append(audit.assign(sweep_value=value, axis=config['axis'], case_id=case_id))
                runner.cohorts.clear()
        if mode == 'preflight':
            return dict(status='preflight_passed', config_id=config_id, cases=pd.DataFrame(case_records))
        frame = pd.concat(frames, ignore_index=True)
        selections = pd.concat(choices, ignore_index=True)
        audits = pd.concat(audit_frames, ignore_index=True)
        frame['experiment'] = config['experiment']
        frame['config_id'] = config_id
        if frame.groupby(['seed', 'cohort_id']).X_id.nunique().max() != 1:
            raise ValueError('Applicant pairing across sensitivity settings failed.')
        save_json(directory / 'config.json', dict(config=config, config_id=config_id, protocol='Recourse refitted per setting; fixed scorer per seed; shared-validation selection and held-out evaluation.'))
        save_csv(directory / 'cases.csv', pd.DataFrame(case_records))
        save_csv(directory / 'results.csv', frame)
        save_csv(directory / 'selection.csv', selections)
        save_csv(directory / 'training_summary.csv', audits)
        save_json(directory / 'complete.json', dict(rows=len(frame), seeds=base['seeds'], values=config['values']))
    if frame.duplicated(['sweep_value', 'seed', 'lambda_', 'cohort_id', 'method']).any():
        raise ValueError('Duplicate result rows.')
    tables = sensitivity_tables(frame, selections, audits, config)
    tables['cases'] = pd.read_csv(directory / 'cases.csv')
    for name, table in tables.items():
        save_csv(directory / f'{name}.csv', table)
        save_csv(root / 'tables' / f'{stem}-{name}.csv', table)
    save_json(root / 'tables' / f'{stem}-config.json', dict(config=config, config_id=config_id, directory=str(directory)))
    figures = sensitivity_figures(tables, config, config_id, stem, show=show)
    display(tables['summary'][['sweep_value', 'method', 'lambda_', 'seeds', 'C_mean', 'V_mean', 'J_mean', 'J_sem']])
    display(tables['selection_counts'])
    bad = audits[(audits.objective_increases > 0) | (audits.sufficient_decrease_failures > 0)]
    if not bad.empty:
        print(f'{len(bad)} fit records have descent flags; inspect training_summary.csv before claiming convergence.')
    return dict(config_id=config_id, directory=str(directory), tables=tables, figures=figures)


In [ ]:
def ablation_methods():
    return ['Personalized targets + selection', 'Personalized targets, all eligible', 'Detached threshold (ablation)']

def ablation_config(dataset, model, *, seeds=(42, 43, 44, 45, 46), lambda_values=(3.0, 30.0), B=0.75, alpha=0.4, tau=0.01, eta=0.2, T=500, cohort_size=256, validation_cohorts=None, test_cohorts=None, fresh_seed=20260928):
    synthetic = dataset.startswith('Synthetic')
    base = experiment_config(6, dataset, model=model, seeds=list(seeds), B=B, alpha=alpha, tau=tau, eta=eta, T=T, cohort_size=cohort_size, lambda_values=list(lambda_values), methods=ablation_methods(), policy=dict(validation_every=5), diagnostics=dict(every=25))
    return dict(experiment=16, base_config=base, dataset=dataset, model=model, evaluation='fresh_synthetic' if synthetic else 'heldout', validation_cohorts=(20 if synthetic else 1) if validation_cohorts is None else validation_cohorts, test_cohorts=(50 if synthetic else 1) if test_cohorts is None else test_cohorts, fresh_seed=fresh_seed, rho_cutoffs=[.05,.1,.25,.5,.75,.9,.95], results_dir='./Results', figures_dir='./figures')

def ablation_validate(config):
    b = config['base_config']
    validate_config(b)
    if config['experiment'] != 16 or (config['dataset'], config['model']) != (b['dataset'], b['model']):
        raise ValueError('Experiment 16 requires matching dataset/model settings.')
    if b['model'] not in ['affine', 'quadratic'] or b['methods'] != ablation_methods():
        raise ValueError('Use affine or quadratic and the three prescribed ablations.')
    synthetic = b['dataset'].startswith('Synthetic')
    if config['evaluation'] != ('fresh_synthetic' if synthetic else 'heldout'):
        raise ValueError('Use fresh synthetic cohorts or the real held-out partitions.')
    for key in ['validation_cohorts', 'test_cohorts', 'fresh_seed']:
        v = config[key]
        if isinstance(v, bool) or not isinstance(v, (int, np.integer)) or v < (0 if key == 'fresh_seed' else 1):
            raise ValueError(f'Invalid {key}.')
    if not synthetic and (config['validation_cohorts'] != 1 or config['test_cohorts'] != 1):
        raise ValueError('Real data have one held-out cohort per split; resampling does not create independent populations.')
    validated_rho_cutoffs(config['rho_cutoffs'])


def ablation_code_id():
    helpers = {name: [_code_signature(inspect.unwrap(fn).__code__), repr(fn.__defaults__), repr(fn.__kwdefaults__)] for name, fn in globals().items() if name.startswith('ablation_') and inspect.isfunction(fn)}
    return stable_id(dict(schema=1, core=implementation_id()[0], helpers=helpers))

def ablation_study_id(config):
    settings = copy.deepcopy(config)
    for d in [settings, settings['base_config']]:
        for key in ['results_dir', 'figures_dir', 'retry_failed']:
            d.pop(key, None)
    return stable_id(dict(settings=settings, code=ablation_code_id(), source=data_source_id(config['base_config'])))

def ablation_payload_id(payload):

    def convert(value):
        if torch.is_tensor(value):
            return dict(tensor_id=array_id(value))
        if isinstance(value, dict):
            return {k: convert(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [convert(v) for v in value]
        return value
    return stable_id(convert(payload))

def ablation_proposal(surrogates, gradients, eta):
    result = []
    for p, g, radius in zip([surrogates.theta_1, surrogates.theta_2], gradients, [surrogates.Theta_1_radius, surrogates.Theta_2_radius]):
        candidate = p.detach() - eta * g
        norm = float(candidate.norm())
        result.append(candidate * min(1.0, radius / norm) if norm else candidate)
    mapping_sq = sum((float(((p.detach() - v) / eta).square().sum()) for p, v in zip([surrogates.theta_1, surrogates.theta_2], result)))
    return (result, mapping_sq)

def ablation_train(reference, config, seed, lam, method, *, progress_callback=None):
    """Fit without validation; retain the same checkpoint schedule for all methods."""
    b = config['base_config']
    mode, select = learned_method(method)
    key = stable_id(dict(code=ablation_code_id(), X=array_id(reference.X), scorer=ablation_payload_id(reference.f_w0.state_dict()), mask=array_id(reference.actionable_mask), t_0=reference.t_0, B=reference.B, alpha=reference.alpha, solver=reference.options, seed=seed, lambda_=lam, method=method, tau=b['tau'], eta=b['eta'], T=b['T'], policy=b['policy'], diagnostics=b['diagnostics'], checkpoint_every=b['checkpoint_every']))
    directory = Path(config['results_dir']).expanduser().resolve() / 'multi_cohort_ablations' / 'fits' / key
    path = directory / 'checkpoint.pt'
    surrogates = PolicySurrogates(reference.X.shape[1], seed=seed, select_recipients=select, Theta_1_radius=b['policy']['Theta_1_radius'], Theta_2_radius=b['policy']['Theta_2_radius'])
    initial_id = ablation_payload_id(surrogates.state_dict())
    state = load_torch(path) if path.exists() else dict(fit_id=key, next_iteration=0, state_dict=copy.deepcopy(surrogates.state_dict()), initial_id=initial_id, history=[], snapshots=[])
    if state['fit_id'] != key or state['initial_id'] != initial_id:
        raise ValueError('Training cache mismatch.')
    surrogates.load_state_dict(state['state_dict'])
    start = state['next_iteration']
    if start > b['T'] + 1:
        raise ValueError('Invalid cached iteration.')
    every = b['policy']['validation_every']
    diagnostic_every = b['diagnostics']['every']
    scorer_before = ablation_payload_id(reference.f_w0.state_dict())
    for j in range(start, b['T'] + 1):
        tick = time.perf_counter()
        surrogates.zero_grad(set_to_none=True)
        objective, outputs = J_hat(reference, surrogates, b['tau'], lam, mode)
        objective.backward()
        parameters = [surrogates.theta_1, surrogates.theta_2]
        gradients = [p.grad.detach().clone() for p in parameters]
        if not torch.isfinite(objective) or any((not torch.isfinite(g).all() for g in gradients)):
            raise FloatingPointError('Nonfinite training objective or update.')
        proposed, mapping_sq = ablation_proposal(surrogates, gradients, b['eta'])
        true_mapping_sq = mapping_sq if mode == 'full' else float('nan')
        feedback_norm = float('nan')
        if mode == 'direct_only' and (j == 0 or j == b['T'] or (diagnostic_every and j % diagnostic_every == 0)):
            true_objective, _ = J_hat(reference, surrogates, b['tau'], lam, 'full')
            true_gradients = torch.autograd.grad(true_objective, parameters)
            _, true_mapping_sq = ablation_proposal(surrogates, true_gradients, b['eta'])
            feedback_norm = math.sqrt(sum((float((g - h).square().sum()) for g, h in zip(true_gradients, gradients))))
        row = dict(iteration=j, J_hat=float(objective.detach()), C_hat=float(outputs['C_hat'].detach()), V_hat=float(outputs['V_hat'].detach()), t_hat=float(outputs['t_hat'].detach()), update_mapping_sq=mapping_sq, true_G_eta_sq=true_mapping_sq, threshold_feedback_norm=feedback_norm, capacity_residual=float(outputs['a_i'].detach().sum()) - reference.k, step_seconds=time.perf_counter() - tick)
        previous = state['history'][-1] if state['history'] else None
        row['objective_change'] = row['J_hat'] - previous['J_hat'] if previous else 0.0
        row['sufficient_decrease_gap'] = row['objective_change'] + b['eta'] * previous['update_mapping_sq'] / 2 if previous else 0.0
        state['history'].append(row)
        if j % every == 0 or j == b['T']:
            state['snapshots'].append(dict(iteration=j, state_dict=copy.deepcopy(surrogates.state_dict())))
        if j < b['T']:
            with torch.no_grad():
                for p, v in zip(parameters, proposed):
                    p.copy_(v)
        state.update(next_iteration=j + 1, state_dict=copy.deepcopy(surrogates.state_dict()))
        if j == 0 or (j + 1) % b['checkpoint_every'] == 0 or j == b['T']:
            save_torch(path, state)
            save_csv(directory / 'history.csv', pd.DataFrame(state['history']))
        reference.response_cache.clear()
        if progress_callback is not None:
            progress_callback(j, row)
    if ablation_payload_id(reference.f_w0.state_dict()) != scorer_before:
        raise AssertionError('Scorer changed.')
    candidates = []
    for snapshot in state['snapshots']:
        surrogates.load_state_dict(snapshot['state_dict'])
        for cutoff in (validated_rho_cutoffs(config['rho_cutoffs']) if select else [.5]):
            payload=encode_policy(DefinitePolicy(surrogates,cutoff))
            candidates.append(dict(iteration=snapshot['iteration'],rho_cutoff=cutoff,payload=payload,policy_id=ablation_payload_id(payload)))
    history = pd.DataFrame(state['history'])
    audit = dict(seed=seed, lambda_=lam, method=method, fit_id=key, fit_directory=str(directory), initial_id=initial_id, steps=b['T'], candidates=len(candidates), resumed_from=start, objective_increases=int((history.objective_change > 1e-10).sum()), sufficient_decrease_failures=int((history.sufficient_decrease_gap > 1e-10).sum()) if mode == 'full' else np.nan, final_update_mapping_sq=float(history.update_mapping_sq.iloc[-1]), final_true_G_eta_sq=float(history.true_G_eta_sq.iloc[-1]), max_capacity_residual=float(history.capacity_residual.abs().max()), seconds=float(history.step_seconds.sum()), status='ok')
    return (candidates, audit, history)

def ablation_cohort(reference, config, seed, split, index, contexts, runner, case):
    if config['evaluation'] == 'heldout':
        if index != 0:
            raise ValueError('Only one real held-out cohort is available.')
        cohort = contexts['val'] if split == 'validation' else runner.contexts(case, include_test=True)[0]['test']
        entropy = None
    else:
        entropy = [int(config['fresh_seed']), 1601, int(seed), 101 if split == 'validation' else 202, int(index)]
        X, _, _ = generate_correlated_nonlinear_population(config['base_config']['cohort_size'], correlation=0.5, d=reference.X.shape[1], seed=np.random.SeedSequence(entropy))
        cohort = make_cohort(X, reference.f_w0, reference.t_0, case['B'], case['alpha'], reference.actionable_mask, solver_options=reference.options)
    require_training_cohort(cohort)
    return (cohort, dict(split=split, cohort_id=index, rng_entropy=entropy, X_id=array_id(cohort.X), N=cohort.N, N_minus=cohort.N_minus, k=cohort.k, t_0=cohort.t_0, evaluation=config['evaluation']))

def ablation_evaluate(cohort, payloads):
    rows = []
    for policy_id, payload in payloads.items():
        metrics = scalar_metrics(evaluate_policy(cohort, decode_policy(payload), 1.0))
        rows.append(dict(policy_id=policy_id, **metrics))
        cohort.response_cache.clear()
    return pd.DataFrame(rows)

def ablation_check_results(frame, config):
    b = config['base_config']
    methods = ablation_methods() + ['No recourse']
    keys = ['seed', 'lambda_', 'cohort_id', 'method']
    expected = {(s, float(l), i, m) for s in b['seeds'] for l in b['lambda_values'] for i in range(config['test_cohorts']) for m in methods}
    if frame.duplicated(keys).any() or set(frame[keys].itertuples(index=False, name=None)) != expected:
        raise ValueError('Incomplete or duplicated paired comparisons.')
    if frame.groupby(['seed', 'cohort_id']).X_id.nunique().max() != 1:
        raise ValueError('Methods or lambda values did not use the same applicants.')
    if frame.groupby(['seed', 'lambda_', 'method']).policy_id.nunique().max() != 1:
        raise ValueError('A policy changed between test cohorts.')
    if not np.allclose(frame.J, frame.C - frame.lambda_ * frame.V, atol=1e-12, rtol=0):
        raise ValueError('Incorrect objective.')
    if (frame.accepted_mass - frame.k).abs().max() > 1e-09 or (frame.V - frame.capacity_validity_bound).max() > 1e-12:
        raise ValueError('Hard allocation violated capacity.')

def ablation_tables(frame, selections, audits, histories):
    keys = ['dataset', 'model', 'seed', 'lambda_', 'method']
    metrics = ['C', 'V', 'J', 'incremental_validity', 'wasted_cost', 'recommendation_rate', 'eligible_fraction', 'share_of_places', 'selection_J']
    seed_means = frame.groupby(keys, sort=False)[metrics].mean().reset_index()
    seed_means = seed_means.merge(frame.groupby(keys).cohort_id.nunique().rename('cohorts').reset_index(), on=keys)
    summary = seed_means.groupby(['method', 'lambda_'], sort=False)[metrics].agg(['mean', 'sem'])
    summary.columns = ['_'.join(c) for c in summary.columns]
    summary = summary.reset_index()
    summary = summary.merge(seed_means.groupby(['method', 'lambda_']).seed.nunique().rename('seeds').reset_index())
    pairs = []
    full = ablation_methods()[0]
    for (seed, lam, index), group in frame.groupby(['seed', 'lambda_', 'cohort_id']):
        f = group.set_index('method').loc[full]
        for baseline in ablation_methods()[1:]:
            other = group.set_index('method').loc[baseline]
            pairs.append(dict(seed=seed, lambda_=lam, cohort_id=index, method=full, baseline=baseline, **{'delta_' + m: float(f[m] - other[m]) for m in ['C', 'V', 'J', 'wasted_cost', 'recommendation_rate']}))
    paired = pd.DataFrame(pairs)
    group_keys = ['seed', 'lambda_', 'method', 'baseline']
    delta = [c for c in paired if c.startswith('delta_')]
    paired_seed = paired.groupby(group_keys, sort=False)[delta].mean().reset_index()
    mc = paired.groupby(group_keys).delta_J.agg(['std', 'count']).reset_index()
    mc['within_seed_delta_J_mcse'] = mc['std'] / np.sqrt(mc['count'])
    paired_seed = paired_seed.merge(mc[group_keys + ['within_seed_delta_J_mcse']], on=group_keys)
    paired_seed['full_wins'] = paired_seed.delta_J < -1e-10
    paired_seed['tie'] = paired_seed.delta_J.abs() <= 1e-10
    ps = paired_seed.groupby(['method', 'baseline', 'lambda_'])[delta].agg(['mean', 'sem'])
    ps.columns = ['_'.join(c) for c in ps.columns]
    ps = ps.reset_index()
    counts = paired_seed.groupby(['method', 'baseline', 'lambda_']).agg(seeds=('seed', 'nunique'), full_wins=('full_wins', 'sum'), ties=('tie', 'sum'), within_seed_delta_J_mcse_mean=('within_seed_delta_J_mcse', 'mean')).reset_index()
    ps = ps.merge(counts)
    ps['full_losses'] = ps.seeds - ps.full_wins - ps.ties
    from scipy.stats import t as student_t
    for metric in delta:
        half = student_t.ppf(0.975, ps.seeds - 1) * ps[metric + '_sem']
        ps[metric + '_ci95_low'] = ps[metric + '_mean'] - half
        ps[metric + '_ci95_high'] = ps[metric + '_mean'] + half
    precision = []
    available = int(frame.cohort_id.max()) + 1
    for n in sorted(set([v for v in [1, 5, 10, 20, 50] if v <= available] + [available])):
        selected = paired[paired.cohort_id < n]
        for (lam, baseline), g in selected.groupby(['lambda_', 'baseline']):
            by_seed = g.groupby('seed').delta_J.agg(['mean', 'std', 'count'])
            precision.append(dict(lambda_=lam, baseline=baseline, test_cohorts=n, seeds=len(by_seed), delta_J_mean=by_seed['mean'].mean(), delta_J_seed_sem=by_seed['mean'].sem(), within_seed_delta_J_mcse_mean=(by_seed['std'] / np.sqrt(by_seed['count'])).mean()))
    return dict(results=frame, selection=selections, seed_means=seed_means, summary=summary, paired_cohorts=paired, paired_seeds=paired_seed, paired_summary=ps, cohort_precision=pd.DataFrame(precision), training_summary=audits, training_histories=histories)

def ablation_figures(tables, config, config_id, stem, *, show=True):
    paths = []
    summary = tables['summary']
    short = dict(zip(ablation_methods(), ['Full', 'All eligible', 'Detached threshold']))
    for lam in config['base_config']['lambda_values']:
        fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
        data = summary[summary.lambda_ == lam].set_index('method').loc[ordered_methods(ablation_methods())]
        for ax, metric, label in zip(axes, ['C', 'V', 'J'], ['Mean cost C', 'Post-shift validity V', 'C - lambda V']):
            ax.bar(range(3), data[metric + '_mean'], yerr=data[metric + '_sem'].fillna(0), capsize=3)
            ax.set(xticks=range(3), xticklabels=[short[m] for m in ordered_methods(ablation_methods())], ylabel=label)
            ax.tick_params(axis='x', rotation=15)
        fig.suptitle(f"{config['dataset']} | {config['model']} | lambda = {lam:g}")
        pairs = tables['paired_summary']
        pairs = pairs[pairs.lambda_ == lam]
        fig2, ax = plt.subplots(figsize=(7, 3))
        for row in pairs.itertuples():
            half = (row.delta_J_ci95_high - row.delta_J_ci95_low) / 2
            ax.errorbar(row.delta_J_mean, short[row.baseline], xerr=0 if pd.isna(half) else half, fmt='o', capsize=4)
        ax.axvline(0, color='black', lw=1)
        ax.set(xlabel='J(full) - J(comparator); negative favors full', ylabel='Comparator')
        ax.set_title(f"{config['dataset']} | {config['model']} | lambda = {lam:g}")
        for name, f in [(f'outcomes-lambda{lam:g}', fig), (f'paired-objective-lambda{lam:g}', fig2)]:
            standardize_figure(f);f.tight_layout()
            path = Path(config['figures_dir']).expanduser().resolve() / f'{stem}-{name}.png'
            atomic_write(path, lambda handle: f.savefig(handle, format='png', dpi=160, bbox_inches='tight'))
            save_json(path.with_suffix('.json'), dict(config=config, config_id=config_id, plot=name, uncertainty='Outcomes: SEM across seed means. Paired differences: 95% t intervals across seed means; descriptive with five seeds.'))
            paths.append(str(path))
            if show:
                plt.show()
            plt.close(f)
    return paths

def ablation_run(config, *, mode='run', show=True):
    if mode not in ['run', 'load', 'preflight', 'skip']:
        raise ValueError('Unknown execution mode.')
    if mode == 'skip':
        return None
    ablation_validate(config)
    b = copy.deepcopy(config['base_config'])
    b.update(results_dir=config['results_dir'], figures_dir=config['figures_dir'])
    config_id = ablation_study_id(config)
    root = Path(config['results_dir']).expanduser().resolve()
    directory = root / 'multi_cohort_ablations' / slug(config['dataset']) / config['model'] / config_id
    stem = f"experiment-16-dataset-{slug(config['dataset'])}-model-{config['model']}-parameters-B{b['B']:g}-alpha{b['alpha']:g}-lambda{'_'.join((f'{v:g}' for v in b['lambda_values']))}-tau{b['tau']:g}-eta{b['eta']:g}-N{b['cohort_size']}"[:165] + '-' + config_id
    fits = len(b['seeds']) * len(b['lambda_values']) * 3
    print(f"{config['dataset']} / {config['model']}: at most {fits} fits; {config['validation_cohorts']} validation and {config['test_cohorts']} test cohorts per seed.", flush=True)
    if mode == 'load' or (mode == 'run' and (directory / 'complete.json').exists()):
        if not (directory / 'complete.json').exists():
            raise FileNotFoundError(f'No completed matching study: {directory}')
        frame = pd.read_csv(directory / 'results.csv')
        selections = pd.read_csv(directory / 'selection.csv')
        audits = pd.read_csv(directory / 'training_summary.csv')
        histories = pd.read_csv(directory / 'training_histories.csv')
        print('Loaded completed study.', flush=True)
    else:
        runner = ExperimentRunner(b)
        frames = []
        choices = []
        audit_rows = []
        history_frames = []
        save_json(directory / 'config.json', dict(config=config, config_id=config_id, protocol='Matched initialization; checkpoint selection by shared mean validation J; frozen policies on paired test cohorts.'))
        for seed in b['seeds']:
            case = dict(seed=seed, B=b['B'], alpha=b['alpha'], tau=b['tau'], eta=b['eta'], T=b['T'], lambda_=b['lambda_values'][0])
            contexts, keys = runner.contexts(case, include_test=False)
            reference = contexts['train']
            seed_dir = directory / f'seed_{seed}'
            candidate_path = seed_dir / 'candidates.pt'
            print(f'  seed {seed}', flush=True)
            if mode == 'preflight':
                val, vm = ablation_cohort(reference, config, seed, 'validation', 0, contexts, runner, case)
                test, tm = ablation_cohort(reference, config, seed, 'test', 0, contexts, runner, case)
                if vm['X_id'] == tm['X_id']:
                    raise AssertionError('Validation/test overlap.')
                for method in ablation_methods():
                    gradient_mode, select = learned_method(method)
                    s = PolicySurrogates(reference.X.shape[1], seed=seed, select_recipients=select)
                    obj, _ = J_hat(reference, s, b['tau'], case['lambda_'], gradient_mode)
                    obj.backward()
                    if not torch.isfinite(obj) or any((not torch.isfinite(p.grad).all() for p in s.parameters())):
                        raise FloatingPointError('Nonfinite pilot gradient.')
                    evaluate_policy(val, DefinitePolicy(s, validated_rho_cutoffs(config['rho_cutoffs'])[0]), case['lambda_'])
                continue
            candidates = []
            seed_audits = []
            seed_history = []
            for lam in b['lambda_values']:
                for method in ablation_methods():
                    p = encode_policy(NoRecourse())
                    candidates.append(dict(lambda_=lam, method=method, iteration=-1, payload=p, policy_id=ablation_payload_id(p)))
                    if reference.eligible.any():
                        fitted, audit, history = ablation_train(reference, config, seed, lam, method)
                        candidates.extend((dict(c, lambda_=lam, method=method) for c in fitted))
                        seed_history.append(history.assign(seed=seed, lambda_=lam, method=method, fit_id=audit['fit_id']))
                    else:
                        audit = dict(seed=seed, lambda_=lam, method=method, status='no eligible training applicants', steps=0)
                    seed_audits.append(audit)
                    print(f'    {method}, lambda={lam:g}: trajectory ready', flush=True)
            ok = [a for a in seed_audits if a['status'] == 'ok']
            if ok and len(set((a['initial_id'] for a in ok))) != 1:
                raise AssertionError('Unmatched initial parameters.')
            save_torch(candidate_path, candidates)
            save_csv(seed_dir / 'training_summary.csv', pd.DataFrame(seed_audits))
            payloads = {c['policy_id']: c['payload'] for c in candidates}
            validation = []
            seen = {array_id(reference.X)}
            if config['evaluation'] == 'fresh_synthetic':
                seen.add(array_id(contexts['val'].X))
            for index in range(config['validation_cohorts']):
                path = seed_dir / 'validation' / f'cohort_{index:04d}.csv'
                meta_path = path.with_suffix('.json')
                if path.exists() and meta_path.exists():
                    v = pd.read_csv(path, float_precision='round_trip')
                    meta = load_json(meta_path)
                else:
                    cohort, meta = ablation_cohort(reference, config, seed, 'validation', index, contexts, runner, case)
                    v = ablation_evaluate(cohort, payloads)
                    save_csv(path, v)
                    save_json(meta_path, meta)
                if set(v.policy_id) != set(payloads) or v.policy_id.duplicated().any() or meta['X_id'] in seen:
                    raise AssertionError('Invalid validation candidates or repeated cohort.')
                seen.add(meta['X_id'])
                validation.append(v.assign(cohort_id=index))
                if (index + 1) % 5 == 0 or index + 1 == config['validation_cohorts']:
                    print(f"    validation {index + 1}/{config['validation_cohorts']}", flush=True)
            means = pd.concat(validation).groupby('policy_id')[['C', 'V']].mean()
            selected = []
            for lam in b['lambda_values']:
                for method in ablation_methods():
                    options = [c for c in candidates if c['method'] == method and c['lambda_'] == lam]
                    chosen = options[0]
                    best = float(means.loc[chosen['policy_id'], 'C'] - lam * means.loc[chosen['policy_id'], 'V'])
                    for candidate in options[1:]:
                        value = float(means.loc[candidate['policy_id'], 'C'] - lam * means.loc[candidate['policy_id'], 'V'])
                        if value < best - 1e-12:
                            chosen, best = (candidate, value)
                    m = means.loc[chosen['policy_id']]
                    selected.append(dict(chosen, selection_C=float(m.C), selection_V=float(m.V), selection_J=float(m.C - lam * m.V), candidate_count=len(options), validation_cohorts=config['validation_cohorts']))
                p = encode_policy(NoRecourse())
                pid = ablation_payload_id(p)
                m = means.loc[pid]
                selected.append(dict(lambda_=lam, method='No recourse', iteration=-1, payload=p, policy_id=pid, selection_C=float(m.C), selection_V=float(m.V), selection_J=float(m.C - lam * m.V), candidate_count=1, validation_cohorts=config['validation_cohorts']))
            save_torch(seed_dir / 'selected_policies.pt', selected)
            selection = pd.DataFrame([{k: v for k, v in c.items() if k != 'payload'} for c in selected]).assign(seed=seed)
            save_csv(seed_dir / 'selection.csv', selection)
            choices.append(selection)
            audit_rows.extend(seed_audits)
            history_frames.extend(seed_history)
            for index in range(config['test_cohorts']):
                path = seed_dir / 'test' / f'cohort_{index:04d}.csv'
                meta_path = path.with_suffix('.json')
                if path.exists() and meta_path.exists():
                    part = pd.read_csv(path, float_precision='round_trip')
                    meta = load_json(meta_path)
                else:
                    cohort, meta = ablation_cohort(reference, config, seed, 'test', index, contexts, runner, case)
                    metrics = ablation_evaluate(cohort, {c['policy_id']: c['payload'] for c in selected}).set_index('policy_id')
                    no_action = metrics.loc[ablation_payload_id(encode_policy(NoRecourse())), 'V']
                    rows = []
                    for c in selected:
                        m = metrics.loc[c['policy_id']].to_dict()
                        m['J'] = m['C'] - c['lambda_'] * m['V']
                        rows.append(dict(dataset=config['dataset'], model=config['model'], seed=seed, lambda_=c['lambda_'], method=c['method'], policy_id=c['policy_id'], cohort_id=index, X_id=meta['X_id'], N=cohort.N, selection_J=c['selection_J'], selected_iteration=c['iteration'], incremental_validity=m['V'] - no_action, share_of_places=m['V'] * cohort.N_minus / cohort.k, **m))
                    part = pd.DataFrame(rows)
                    save_csv(path, part)
                    save_json(meta_path, meta)
                if meta['X_id'] in seen or set(part.X_id) != {meta['X_id']}:
                    raise AssertionError('Repeated test cohort or pairing mismatch.')
                seen.add(meta['X_id'])
                for c in selected:
                    match = part[(part.method == c['method']) & (part.lambda_ == c['lambda_'])]
                    if len(match) != 1 or match.policy_id.iloc[0] != c['policy_id']:
                        raise AssertionError('Cached test policy differs from the validation choice.')
                frames.append(part)
                if (index + 1) % 10 == 0 or index + 1 == config['test_cohorts']:
                    print(f"    test {index + 1}/{config['test_cohorts']}", flush=True)
            runner.cohorts.clear()
        if mode == 'preflight':
            return dict(status='preflight_passed', config_id=config_id)
        frame = pd.concat(frames, ignore_index=True)
        selections = pd.concat(choices, ignore_index=True)
        audits = pd.DataFrame(audit_rows)
        histories = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame(columns=['seed', 'lambda_', 'method', 'iteration', 'J_hat', 'true_G_eta_sq'])
    ablation_check_results(frame, config)
    tables = ablation_tables(frame, selections, audits, histories)
    for name, table in tables.items():
        save_csv(directory / f'{name}.csv', table)
        save_csv(root / 'tables' / f'{stem}-{name}.csv', table)
    save_json(root / 'tables' / f'{stem}-config.json', dict(config=config, config_id=config_id, directory=str(directory)))
    save_json(directory / 'complete.json', dict(config_id=config_id, rows=len(frame), seeds=b['seeds']))
    figures = ablation_figures(tables, config, config_id, stem, show=show)
    display(tables['summary'][['method', 'lambda_', 'seeds', 'C_mean', 'V_mean', 'J_mean', 'J_sem']])
    display(tables['paired_summary'][['baseline', 'lambda_', 'delta_J_mean', 'delta_J_ci95_low', 'delta_J_ci95_high', 'full_wins', 'ties', 'full_losses']])
    return dict(config_id=config_id, directory=str(directory), tables=tables, figures=figures)


## Frozen-candidate transfer, pooling, cost decomposition, and joint calibration


In [ ]:
# Experiment 2 follow-up: reuse frozen policies; select on validation only.
MC_PROTOCOLS = ('single_matched', 'multi_matched', 'single_pooled', 'multi_pooled')
MC_PROTOCOL_LABELS = {
    'source_selected': 'Original saved selection (reference)',
    'single_matched': 'One validation cohort; matched lambda',
    'multi_matched': 'Multiple validation cohorts; matched lambda',
    'single_pooled': 'One validation cohort; pooled lambdas',
    'multi_pooled': 'Multiple validation cohorts; pooled lambdas',
}
MC_METRICS = ('C', 'V', 'J', 'wasted_cost', 'successful_cost',
              'successful_cutoff_cost', 'successful_excess_cost',
              'accepted_score_excess', 'recommendation_rate')


def experiment_2_diagnostic_config(dataset, model, **overrides):
    config = dict(dataset=dataset, model=model, source_results_dir='./Results',
                  source_run_dir=None, source_config_id=None, source_cache_root=None,
                  seeds=[42,43,44,45,46], lambda_values=None, training_lambda_values=None,
                  rho_cutoffs=[0.05,0.1,0.25,0.5,0.75,0.9,0.95],
                  validation_cohorts=20, test_cohorts=50, resampling_seed=20260927,
                  results_dir='./Results', figures_dir='./figures')
    unknown = set(overrides)-set(config)
    if unknown: raise ValueError(f'Unknown follow-up options: {sorted(unknown)}')
    config.update(copy.deepcopy(overrides))
    if model not in {'affine','quadratic'}:
        raise ValueError('This cost-decomposition follow-up supports affine and quadratic scorers.')
    for key in ['validation_cohorts','test_cohorts']:
        if not isinstance(config[key],int) or isinstance(config[key],bool) or config[key]<1:
            raise ValueError(f'{key} must be a positive integer.')
    if not isinstance(config['resampling_seed'],int) or config['resampling_seed']<0:
        raise ValueError('resampling_seed must be a nonnegative integer.')
    validated_rho_cutoffs(config['rho_cutoffs'])
    return config


def mc_file_hash(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''):digest.update(block)
    return digest.hexdigest()


def mc_payload_hash(value):
    def normalized(v):
        if torch.is_tensor(v):return {'tensor':array_id(v)}
        if isinstance(v,dict):return {k:normalized(x) for k,x in v.items()}
        if isinstance(v,(tuple,list)):return [normalized(x) for x in v]
        return v
    return stable_id(normalized(value))


def discover_experiment_2_sources(results_dir='./Results'):
    root=Path(results_dir).expanduser().resolve()
    bases=[root/'experiments'/'experiment_02',root/'experiment_02']
    if root.name=='experiment_02':bases.append(root)
    rows=[];seen=set()
    for base in bases:
        for path in sorted(base.glob('*/*/*/config.json')):
            if path in seen:continue
            seen.add(path);metadata=load_json(path);c=metadata['config']
            rows.append(dict(dataset=c['dataset'],model=c['model'],config_id=metadata['config_id'],
                             source_run_dir=str(path.parent),B=c['B'],alpha=c['alpha'],
                             tau=c['tau'],eta=c['eta'],T=c['T'],seeds=c['seeds']))
    return pd.DataFrame(rows)


def mc_resolve_source(config):
    if config['source_run_dir'] is not None:
        path=Path(config['source_run_dir']).expanduser().resolve()
    else:
        options=discover_experiment_2_sources(config['source_results_dir'])
        if options.empty:raise FileNotFoundError('No experiment-2 runs found. Set source_run_dir to the saved config folder.')
        options=options[(options.dataset==config['dataset'])&(options.model==config['model'])]
        if config['source_config_id'] is not None:options=options[options.config_id==config['source_config_id']]
        if len(options)!=1:
            raise ValueError('Select one source_config_id or source_run_dir. Matching sources: '+
                             str(options[['config_id','source_run_dir']].to_dict('records')))
        path=Path(options.iloc[0].source_run_dir)
    metadata=load_json(path/'config.json');c=metadata['config']
    if (c['experiment']!=2 or c['dataset']!=config['dataset'] or c['model']!=config['model']
            or c.get('comparison_protocol')!='validation_cutoff_grid_v1'):
        raise ValueError('Source must be the matching revised experiment-2 comparison.')
    if config['source_config_id'] is not None and metadata['config_id']!=config['source_config_id']:
        raise ValueError('source_run_dir and source_config_id disagree.')
    if c['methods']!=list(MAIN_METHODS):raise ValueError('Source must contain the five requested methods in their standard order.')
    resolved=copy.deepcopy(config)
    resolved['seeds']=list(c['seeds'] if config['seeds'] is None else config['seeds'])
    resolved['lambda_values']=list(c['lambda_values'] if config['lambda_values'] is None else config['lambda_values'])
    resolved['training_lambda_values']=list(c['lambda_values'] if config['training_lambda_values'] is None else config['training_lambda_values'])
    for key,available in [('seeds',c['seeds']),('lambda_values',resolved['training_lambda_values']),
                          ('training_lambda_values',c['lambda_values'])]:
        values=resolved[key]
        if not values or len(values)!=len(set(values)) or not set(values).issubset(available):
            raise ValueError(f'{key} must be a nonempty distinct subset of {available}.')
    rows=[]
    for record in sorted((path/'trials').glob('seed_*/*/result.json')):
        r=load_json(record)
        if r['seed'] not in resolved['seeds'] or r['lambda_'] not in resolved['training_lambda_values']:continue
        if r.get('status')!='ok' or r['method'] not in MAIN_METHODS:continue
        for key in ['B','alpha','tau','eta','T']:
            if r[key]!=c[key]:raise ValueError('Source mixes incompatible settings: '+key)
        if r['config_id']!=metadata['config_id']:raise ValueError('Source result/config identity mismatch.')
        policy_path=record.with_name('policy.pt')
        if not policy_path.exists():raise FileNotFoundError(policy_path)
        rows.append(dict(r,policy_path=str(policy_path),policy_hash=mc_file_hash(policy_path)))
    records=pd.DataFrame(rows)
    keys=['seed','lambda_','method']
    expected={(s,l,m) for s in resolved['seeds'] for l in resolved['training_lambda_values'] for m in MAIN_METHODS}
    actual=set(map(tuple,records[keys].to_numpy())) if not records.empty else set()
    if actual!=expected or records.duplicated(keys).any():
        raise ValueError(f'Incomplete or duplicate source comparisons; missing {len(expected-actual)} trials. Finish the source run first.')
    resolved['source_run_dir']=str(path);resolved['source_config_id']=metadata['config_id']
    return resolved,metadata,records


def mc_raw_features(c, cache):
    key=(c['dataset'],c.get('gmsc_path'))
    if key in cache:return cache[key]
    if c['dataset']=='Adult':X=fetch_openml(name='adult',version=2,as_frame=True).data
    elif c['dataset']=='German Credit':X=fetch_openml(data_id=31,as_frame=True).data
    elif c['dataset']=='Give Me Some Credit':
        X=pd.read_csv(c['gmsc_path']).drop(columns=['SeriousDlqin2yrs','Unnamed: 0'],errors='ignore')
    elif c['dataset']=='ACSIncome':
        from folktables import ACSDataSource,ACSIncome
        data=ACSDataSource(survey_year='2018',horizon='1-Year',survey='person').get_data(states=['CA'],download=True)
        features,_,_=ACSIncome.df_to_numpy(data);X=pd.DataFrame(features,columns=ACSIncome.features)
        for column in ['COW','SCHL','MAR','OCCP','POBP','RELP','SEX','RAC1P']:
            if column in X:X[column]=X[column].astype('category')
    else:raise ValueError('Unknown real dataset.')
    cache[key]=pd.DataFrame(X).reset_index(drop=True)
    return cache[key]


def mc_expand_holdout(c, population, seed, raw_cache):
    """Recover larger original held-out partitions with the SAVED preprocessing; fit nothing."""
    if c['dataset'].startswith('Synthetic'):
        X,scores,_=generate_correlated_nonlinear_population(c['synthetic_n'],correlation=.5,seed=seed)
        if c['dataset']=='Synthetic curved':
            scores=expit(.6*X[:,0]+1.2*X[:,1]+.3*X[:,2]-.8*X[:,1]**2-.2*X[:,2]**2)
        expanded=prepare_population(X,scores,seed,immutable_indices=[0])
        indices=expanded['indices']
        if c['fit_cap'] is not None and len(indices['fit'])>c['fit_cap']:
            choose=np.random.default_rng(seed).choice(len(indices['fit']),c['fit_cap'],replace=False)
            indices['fit']=indices['fit'][choose];expanded['X_fit']=expanded['X_fit'][choose]
    else:
        X=mc_raw_features(c,raw_cache);indices=split_indices(X,seed)
        if c['fit_cap'] is not None:
            rng=np.random.default_rng(seed)
            indices={name:rng.choice(idx,min(len(idx),c['fit_cap']),replace=False) for name,idx in indices.items()}
        if 'preprocessor' not in population:raise ValueError('The original cached preprocessor is required; no replacement is fitted.')
        expanded={f'X_{name}':np.asarray(population['preprocessor'].transform(X.iloc[idx]),dtype=float)
                  for name,idx in indices.items()}
    if not np.array_equal(indices['fit'],population['indices']['fit']):
        raise ValueError('The reconstructed fitting partition differs from the saved run.')
    if not np.allclose(expanded['X_fit'],population['X_fit'],atol=1e-12,rtol=1e-12):
        raise ValueError('Features/preprocessing changed; use the original source data and environment.')
    # Reproduce the old permutation/prefix exactly, including the training threshold cohort.
    rng=np.random.default_rng(seed+1000)
    original_positions={}
    for split in ['train','val','test']:
        positions=rng.permutation(len(indices[split]))[:c['cohort_size']]
        if not np.array_equal(indices[split][positions],population['indices'][split]):
            raise ValueError(f'{split} row identities differ from the original run.')
        if not np.allclose(expanded[f'X_{split}'][positions],population[f'X_{split}'],atol=1e-12,rtol=1e-12):
            raise ValueError(f'{split} features differ from the original run.')
        original_positions[split]=positions
    for left in indices:
        for right in indices:
            if left!=right and set(indices[left])&set(indices[right]):raise ValueError('Partitions overlap.')
    return dict(X_val=expanded['X_val'],X_test=expanded['X_test'],
                val_ids=indices['val'],test_ids=indices['test'],
                original_val_positions=original_positions['val'],original_test_positions=original_positions['test'])


def mc_frozen_context(config,metadata,seed,raw_cache):
    c=metadata['config'];root=Path(config['source_results_dir']).expanduser().resolve()
    cache=(Path(config['source_cache_root']).expanduser().resolve() if config['source_cache_root'] is not None else root/'cache')
    cache=cache/metadata['implementation_id']
    key=stable_id(dict(dataset=c['dataset'],seed=seed,source=metadata['source_id'],fit_cap=c['fit_cap'],
                       cohort_size=c['cohort_size'],synthetic_n=c['synthetic_n'],qualification_proxy=c['qualification_proxy']))
    population_path=cache/'populations'/key/'population.pkl'
    if not population_path.exists():
        raise FileNotFoundError(f'Original frozen population cache missing: {population_path}. '
                                'Set source_cache_root to the original Results/cache. The experiment-only ZIP is insufficient; nothing is refitted.')
    population=load_pickle(population_path)
    model_key=stable_id(dict(X_fit=array_id(population['X_fit']),f_star_fit=array_id(population['f_star_fit']),
                             seed=seed,model=c['model'],scorer=c['scorer']))
    scorer_path=cache/'scorers'/model_key/'scorer.pt'
    if not scorer_path.exists():raise FileNotFoundError(f'Original frozen scorer missing: {scorer_path}; no scorer is refitted.')
    f_w0=restore_scorer(load_torch(scorer_path))
    source_hash=stable_id(dict(population=mc_file_hash(population_path),scorer=mc_file_hash(scorer_path)))
    pool_key=stable_id(dict(population=mc_file_hash(population_path),seed=seed,pool_schema=1))
    snapshot=Path(config['results_dir']).expanduser().resolve()/'diagnostic_cache'/'heldout_pools'/pool_key/'pool.pkl'
    if snapshot.exists():pool=load_pickle(snapshot)
    else:
        if c['dataset']=='Give Me Some Credit' and data_source_id(c)!=metadata['source_id']:
            raise ValueError('Give Me Some Credit CSV differs from the original source.')
        pool=mc_expand_holdout(c,population,seed,raw_cache);save_pickle(snapshot,pool)
    pool_hash=stable_id({key:array_id(value) for key,value in pool.items()})
    t_0=initial_threshold(f_w0(as_tensor(population['X_train'])),c['alpha'])
    return dict(seed=seed,f_w0=f_w0,population=population,pool=pool,t_0=t_0,
                source_hash=source_hash,pool_hash=pool_hash,model_id=model_key,
                population_path=str(population_path),scorer_path=str(scorer_path))


def mc_catalog(config,source,records,context):
    candidates={};rows=[];seed=context['seed']
    def add(method,payload,training_lambda=None):
        identifier=mc_payload_hash(payload)
        candidates.setdefault(identifier,decode_policy(payload))
        rows.append(dict(method=method,candidate_id=identifier,training_lambda=training_lambda,
                         rho_cutoff=payload.get('rho_cutoff',np.nan),policy_type=payload['type']))
    none=encode_policy(NoRecourse())
    add(MAIN_METHODS[0],none)
    add(MAIN_METHODS[1],encode_policy(CommonTarget(context['t_0'])))
    add(MAIN_METHODS[2],none)
    for target in np.linspace(context['t_0'],1.,source['baselines']['target_points']):
        add(MAIN_METHODS[2],encode_policy(CommonTarget(float(target))))
    for method in MAIN_METHODS[3:]:
        add(method,none)
        for training_lambda in config['training_lambda_values']:
            row=records[(records.seed==seed)&(records.method==method)&(records.lambda_==training_lambda)].iloc[0]
            payload=load_torch(row.policy_path)
            if payload['type']=='none':continue
            if payload['type']!='learned' or payload['select_recipients']!=(method==MAIN_METHODS[4]):
                raise ValueError('Unexpected learned-policy payload in the source.')
            cutoffs=(validated_rho_cutoffs(config['rho_cutoffs']) if method==MAIN_METHODS[4] else [.5])
            for cutoff in cutoffs:
                candidate=copy.deepcopy(payload);candidate['rho_cutoff']=cutoff
                add(method,candidate,float(training_lambda))
    return candidates,pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)


def mc_make_pool_cohort(context,source,split):
    return make_cohort(context['pool'][f'X_{split}'],context['f_w0'],context['t_0'],source['B'],source['alpha'],
                       context['population']['actionable_mask'],seed=context['seed'],solver_options=source['solver'])


@torch.no_grad()
def mc_candidate_arrays(cohort,candidates,directory):
    """Best responses once per target model/pool; cutoff variants reuse these responses."""
    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
    arrays={};responses={}
    for identifier,policy in candidates.items():
        path=directory/f'{identifier}.pt'
        if path.exists():arrays[identifier]=load_torch(path);continue
        payload=encode_policy(policy);response_payload=copy.deepcopy(payload);response_payload.pop('rho_cutoff',None)
        response_id=mc_payload_hash(response_payload)
        q,rho=policy(cohort);q=as_tensor(q).expand(cohort.N)
        rho=binary_vector(rho,cohort.N)&target_feasible(cohort,q)
        if response_id not in responses:
            feasible=target_feasible(cohort,q)
            X_br=br(cohort,q)
            actual=cohort.f_w0(X_br);scores=torch.where(feasible,q,cohort.s_0)
            costs=cost(cohort.X,X_br)
            if not torch.allclose(actual,scores,atol=2e-9,rtol=2e-9):raise ArithmeticError('Target attainment failed.')
            if (costs>cohort.B+1e-9).any():raise ArithmeticError('Budget exceeded.')
            if not torch.equal(X_br[:,~cohort.actionable_mask],cohort.X[:,~cohort.actionable_mask]):
                raise ArithmeticError('Immutable feature changed.')
            responses[response_id]=(costs,scores)
        costs,scores=responses[response_id]
        value=dict(costs=torch.where(rho,costs,torch.zeros_like(costs)),
                   scores=torch.where(rho,scores,cohort.s_0),rho=rho)
        save_torch(path,value);arrays[identifier]=value
    return arrays


def mc_cohort_plan(context,source,config):
    pool=context['pool'];seed=context['seed'];rows=[]
    original=np.asarray(pool['original_val_positions'],dtype=int)
    rows.append(dict(split='val',role='original',cohort=0,positions=original.tolist(),
                     pool_size=len(pool['X_val']),N=len(original),unique_applicants=len(set(original)),
                     cohort_hash=array_id(pool['val_ids'][original])))
    for split,count,stream in [('val',config['validation_cohorts'],301),('test',config['test_cohorts'],701)]:
        rng=np.random.default_rng(np.random.SeedSequence([seed,config['resampling_seed'],stream]))
        size=len(pool[f'X_{split}']);N=min(source['cohort_size'],size)
        for index in range(count):
            positions=rng.choice(size,N,replace=True)
            item=dict(split=split,role='multi' if split=='val' else 'test',cohort=index,
                             positions=positions.tolist(),pool_size=size,N=N,unique_applicants=len(set(positions)),
                             cohort_hash=array_id(pool[f'{split}_ids'][positions]))
            rows.append(item)
            # Controlled one-vs-many comparison: the same first bootstrap cohort
            # is included in the multi-cohort average. Original selection is a separate reference.
            if split=='val' and index==0:rows.append(dict(item,role='single'))
    return rows


def mc_sample_cohort(pool,positions):
    cohort=copy.copy(pool);idx=torch.as_tensor(positions,dtype=torch.long)
    for name in ['X','X_aug','s_0','q_cap','witness','X_minus','eligible']:
        setattr(cohort,name,getattr(pool,name)[idx])
    if pool.q_bar is not None:cohort.q_bar=pool.q_bar[idx]
    cohort.N=len(idx);cohort.N_minus=int(cohort.X_minus.sum());cohort.k=math.floor(cohort.alpha*cohort.N)
    cohort.response_cache=OrderedDict();cohort.solver_audit=[]
    require_training_cohort(cohort)
    return cohort


@torch.no_grad()
def mc_measure(pool,arrays,positions,*,decompose=False):
    idx=torch.as_tensor(positions,dtype=torch.long);rejected=pool.X_minus[idx]
    N=len(idx);N_minus=int(rejected.sum());k=math.floor(pool.alpha*N)
    if N_minus==0 or not 1<=k<N:raise ValueError('A sampled cohort has no initial rejections or invalid capacity; change cohort settings.')
    costs=arrays['costs'][idx];scores=arrays['scores'][idx];rho=arrays['rho'][idx]
    acceptance,t_1,beta=exact_allocation(scores,k)
    C=float(costs[rejected].mean());V=float(acceptance[rejected].mean())
    wasted=float(((1-acceptance)*costs)[rejected].mean())
    result=dict(C=C,V=V,wasted_cost=wasted,successful_cost=C-wasted,
                recommendation_rate=float(rho[rejected].double().mean()),N=N,N_minus=N_minus,k=k,
                t_1=t_1,t_shift=t_1-pool.t_0,accepted_mass=float(acceptance.sum()),
                accepted_score_excess=float((rho*acceptance*(scores-t_1).clamp_min(0))[rejected].mean()))
    if decompose:
        # Retrospective diagnostic: hold this realized threshold fixed. This is
        # not a proposed policy; changing targets would change competition.
        cohort=mc_sample_cohort(pool,positions)
        active=rejected&rho&(acceptance>0)&(cohort.s_0<t_1)
        cutoff_cost=torch.zeros_like(costs)
        if active.any():
            diagnostic=copy.copy(cohort);diagnostic.t_0=float(t_1)
            diagnostic.X_minus=active;diagnostic.eligible=active
            targets=cohort.s_0.clone();targets[active]=t_1
            if not torch.equal(target_feasible(diagnostic,targets),active):raise ArithmeticError('Realized cutoff is not reachable by a successful recipient.')
            cutoff_cost=cost(cohort.X,br(diagnostic,targets))
            if (cutoff_cost[active]>costs[active]+1e-8).any():raise ArithmeticError('Cutoff cost exceeds implemented cost.')
        cutoff_cost=torch.minimum(cutoff_cost,costs)
        result['successful_cutoff_cost']=float((acceptance*cutoff_cost)[rejected].mean())
        result['successful_excess_cost']=float((acceptance*(costs-cutoff_cost))[rejected].mean())
        total=result['wasted_cost']+result['successful_cutoff_cost']+result['successful_excess_cost']
        if abs(total-C)>1e-10:raise ArithmeticError('Cost decomposition failed to sum to C.')
    return result


def mc_select(catalog,validation,lambda_values):
    means=validation.groupby(['role','candidate_id'])[['C','V']].mean()
    rows=[]
    for protocol in MC_PROTOCOLS:
        role='multi' if protocol.startswith('multi') else 'single'
        for lam in lambda_values:
            for method in MAIN_METHODS:
                choices=catalog[catalog.method==method]
                if protocol.endswith('matched') and method in MAIN_METHODS[3:]:
                    choices=choices[choices.training_lambda.isna()|(choices.training_lambda==lam)]
                # Catalog order gives no action first and cutoff .5 first on ties.
                selected=None;best=float('inf')
                for _,candidate in choices.iterrows():
                    metrics=means.loc[(role,candidate.candidate_id)]
                    J=float(metrics.C-lam*metrics.V)
                    if J<best-1e-12:
                        selected=candidate;best=J;C=float(metrics.C);V=float(metrics.V)
                if selected is None:raise ValueError('Empty candidate family.')
                rows.append(dict(protocol=protocol,method=method,lambda_=lam,candidate_id=selected.candidate_id,
                                 training_lambda=selected.training_lambda,rho_cutoff=selected.rho_cutoff,
                                 validation_C=C,validation_V=V,validation_J=best))
    return pd.DataFrame(rows)


def mc_stat_summary(frame,keys,metrics):
    from scipy.stats import t as student_t
    rows=[]
    for group,part in frame.groupby(keys,sort=False):
        group=group if isinstance(group,tuple) else (group,)
        base=dict(zip(keys,group))
        for metric in metrics:
            values=part[metric].dropna().to_numpy(dtype=float);n=len(values)
            mean=float(values.mean()) if n else np.nan
            se=float(values.std(ddof=1)/np.sqrt(n)) if n>1 else np.nan
            radius=float(student_t.ppf(.975,n-1)*se) if n>1 else np.nan
            rows.append(dict(base,metric=metric,mean=mean,se=se,ci_low=mean-radius,ci_high=mean+radius,n_seeds=n))
    return pd.DataFrame(rows)


def mc_summarize(results,selection):
    keys=['dataset','model','protocol','method','lambda_','seed']
    seed_means=results.groupby(keys)[list(MC_METRICS)].mean().reset_index()
    seed_means=seed_means.merge(selection[keys+['validation_C','validation_V','validation_J']],on=keys,validate='one_to_one')
    seed_means['test_minus_validation_J']=seed_means.J-seed_means.validation_J
    summary=mc_stat_summary(seed_means,keys[:-1],list(MC_METRICS)+['validation_J','test_minus_validation_J'])
    pairs=[]
    for new,old in [('multi_matched','single_matched'),('multi_pooled','multi_matched'),
                    ('single_pooled','single_matched'),('multi_pooled','single_pooled'),
                    ('multi_matched','source_selected'),('multi_pooled','source_selected')]:
        merge_keys=['dataset','model','method','lambda_','seed']
        p=seed_means[seed_means.protocol==new].merge(seed_means[seed_means.protocol==old],on=merge_keys,suffixes=('_new','_old'),validate='one_to_one')
        for _,r in p.iterrows():
            pairs.append(dict({key:r[key] for key in merge_keys},comparison=f'{new} minus {old}',
                              **{metric:r[metric+'_new']-r[metric+'_old'] for metric in MC_METRICS}))
    paired=pd.DataFrame(pairs)
    paired_summary=mc_stat_summary(paired,['dataset','model','comparison','method','lambda_'],MC_METRICS)
    method_pairs=[]
    merge_keys=['dataset','model','protocol','lambda_','seed']
    for baseline in MAIN_METHODS[:-1]:
        p=seed_means[seed_means.method==MAIN_METHODS[-1]].merge(seed_means[seed_means.method==baseline],on=merge_keys,suffixes=('_full','_baseline'),validate='one_to_one')
        for _,r in p.iterrows():
            method_pairs.append(dict({key:r[key] for key in merge_keys},baseline=baseline,
                                     **{metric:r[metric+'_full']-r[metric+'_baseline'] for metric in MC_METRICS}))
    method_pairs=pd.DataFrame(method_pairs)
    method_summary=mc_stat_summary(method_pairs,['dataset','model','protocol','baseline','lambda_'],MC_METRICS)
    return dict(seed_means=seed_means,summary=summary,paired_seeds=paired,paired_summary=paired_summary,
                method_paired_seeds=method_pairs,method_paired_summary=method_summary)


def mc_plot(config,run_id,tables,directory,show=True):
    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
    stem=f"experiment-2-followup-{slug(config['dataset'])}-{config['model']}-{run_id}"
    summary=tables['summary'];figures=[]
    for metric,suffix in [('J','objective'),('frontier','cost-validity')]:
        fig,axes=plt.subplots(2,2,figsize=(12,8))
        for ax,protocol in zip(axes.flat,MC_PROTOCOLS):
            for method in MAIN_METHODS:
                part=summary[(summary.protocol==protocol)&(summary.method==method)]
                if metric=='J':
                    values=part[part.metric=='J'].sort_values('lambda_')
                    ax.plot(values.lambda_,values['mean'],color=METHOD_COLORS[method],marker=METHOD_MARKERS[method],label=METHOD_LABELS[method])
                    ax.fill_between(values.lambda_,values.ci_low,values.ci_high,color=METHOD_COLORS[method],alpha=.10)
                    ax.set(xscale='log',xlabel='Validity weight lambda',ylabel='Test J')
                else:
                    values=part.pivot(index='lambda_',columns='metric',values='mean')
                    keep=nondominated_mask(values.C,values.V)
                    ax.scatter(values.V,values.C,color=METHOD_COLORS[method],alpha=.22,marker=METHOD_MARKERS[method])
                    front=values[keep].sort_values(['V','C']).drop_duplicates(['C','V'])
                    ax.plot(front.V,front.C,color=METHOD_COLORS[method],marker=METHOD_MARKERS[method],label=METHOD_LABELS[method])
                    ax.set(xlabel='Post-shift validity V',ylabel='Mean recourse cost C')
            ax.set_title(MC_PROTOCOL_LABELS[protocol],fontsize=10);ax.grid(alpha=.2)
        axes.flat[0].legend(fontsize=6);figures.append((suffix,fig))
    for protocol in ['multi_matched','multi_pooled']:
        fig,axes=plt.subplots(1,3,figsize=(13,4))
        for ax,metric,label in zip(axes,['wasted_cost','successful_cutoff_cost','successful_excess_cost'],
                                   ['Cost on unsuccessful recourse','Successful cost to realized cutoff','Successful cost above realized cutoff']):
            for method in MAIN_METHODS:
                values=summary[(summary.protocol==protocol)&(summary.method==method)&(summary.metric==metric)].sort_values('lambda_')
                ax.plot(values.lambda_,values['mean'],color=METHOD_COLORS[method],marker=METHOD_MARKERS[method],label=METHOD_LABELS[method])
            ax.set(xscale='log',xlabel='Validity weight lambda',ylabel=label);ax.grid(alpha=.2)
        axes[0].legend(fontsize=5);figures.append((f'cost-decomposition-{protocol}',fig))
    fig,axes=plt.subplots(1,2,figsize=(11,4));paired=tables['paired_summary']
    for ax,comparison in zip(axes,['multi_matched minus single_matched','multi_pooled minus multi_matched']):
        for method in MAIN_METHODS:
            values=paired[(paired.comparison==comparison)&(paired.method==method)&(paired.metric=='J')].sort_values('lambda_')
            ax.plot(values.lambda_,values['mean'],color=METHOD_COLORS[method],marker=METHOD_MARKERS[method],label=METHOD_LABELS[method])
            ax.fill_between(values.lambda_,values.ci_low,values.ci_high,color=METHOD_COLORS[method],alpha=.12)
        ax.axhline(0,color='black',linewidth=.7);ax.set(xscale='log',xlabel='Validity weight lambda',ylabel='Paired change in test J')
        ax.set_title(comparison.replace('_',' '),fontsize=9);ax.grid(alpha=.2)
    axes[0].legend(fontsize=5);figures.append(('paired-objective-changes',fig))
    paths=[]
    for suffix,fig in figures:
        fig.suptitle(f"{config['dataset']} | {config['model']}");standardize_figure(fig);fig.tight_layout()
        path=directory/f'{stem}-{suffix}.png'
        atomic_write(path,lambda handle:fig.savefig(handle,format='png',dpi=160,bbox_inches='tight'))
        save_json(path.with_suffix('.json'),dict(config=config,run_id=run_id,method_order=list(MAIN_METHODS),
                  method_colors=METHOD_COLORS,interval='95% t interval across seed means; cohorts are averaged within seed.'))
        paths.append(str(path))
        if show:plt.show()
        plt.close(fig)
    return paths


def run_experiment_2_diagnostics(config,*,mode='run',show=True):
    if mode not in {'run','preflight','load','skip'}:raise ValueError('Unknown mode.')
    if mode=='skip':return None
    config,metadata,records=mc_resolve_source(config);source=metadata['config']
    # Loading an existing follow-up requires no raw data, context recovery, or policy evaluation.
    code=stable_id(dict(core=implementation_id()[0],followup={name:_code_signature(inspect.unwrap(globals()[name]).__code__) for name in MC_IMPLEMENTATION_NAMES}))
    request_id=stable_id(dict(config={k:v for k,v in config.items() if k not in {'figures_dir','results_dir'}},code=code,
                             policies=records.policy_hash.tolist(),source=metadata['config_id']))
    root=Path(config['results_dir']).expanduser().resolve()
    request_path=root/'diagnostics'/'experiment_02'/slug(config['dataset'])/config['model']/request_id
    if mode=='load':
        locator=load_json(request_path/'run.json');directory=Path(locator['directory'])
        tables={name:pd.read_csv(directory/f'{name}.csv') for name in ['seed_means','summary','paired_seeds','paired_summary','method_paired_seeds','method_paired_summary']}
        mc_plot(config,locator['run_id'],tables,config['figures_dir'],show)
        return dict(directory=str(directory),**tables)
    contexts={};raw_cache={}
    for seed in config['seeds']:
        print(f"Recovering frozen context: {config['dataset']} | {config['model']} | seed={seed}",flush=True)
        contexts[seed]=mc_frozen_context(config,metadata,seed,raw_cache)
    inputs={seed:{key:context[key] for key in ['source_hash','pool_hash','model_id']} for seed,context in contexts.items()}
    run_id=stable_id(dict(request_id=request_id,inputs=inputs))
    directory=request_path/run_id;directory.mkdir(parents=True,exist_ok=True)
    save_json(request_path/'run.json',dict(run_id=run_id,directory=str(directory)))
    save_json(directory/'config.json',dict(config=config,source=metadata,input_fingerprints=inputs,run_id=run_id,
        notes=['No training or scorer/proxy fitting. Only saved final checkpoint candidates are available.',
               'Bootstrap cohorts sample the original held-out empirical partitions, with replacement; they are not new independent datasets.',
               'Single validation uses the first bootstrap cohort, which is also included in the multi-cohort mean. Original saved selection is a separate reference.',
               'Cost to the realized cutoff holds competition fixed retrospectively; it is a diagnostic, not an implementable cost-saving guarantee.']))
    for seed,context in contexts.items():
        seed_dir=directory/f'seed_{seed}';seed_dir.mkdir(exist_ok=True)
        completed_files=all((seed_dir/'test_cohorts'/f'cohort_{i:03d}.csv').exists() for i in range(config['test_cohorts']))
        if ((seed_dir/'status.json').exists() and load_json(seed_dir/'status.json')['status']=='completed'
                and completed_files and (seed_dir/'selection.csv').exists() and mode=='run'):
            print(f'  Seed {seed}: saved follow-up complete.',flush=True);continue
        try:
            save_json(seed_dir/'status.json',dict(status='running',seed=seed))
            candidates,catalog=mc_catalog(config,source,records,context)
            save_csv(seed_dir/'candidates.csv',catalog)
            plan=mc_cohort_plan(context,source,config);save_json(seed_dir/'cohorts.json',plan)
            val_pool=mc_make_pool_cohort(context,source,'val')
            arrays=mc_candidate_arrays(val_pool,candidates,seed_dir/'predictions'/'val')
            original=context['pool']['original_val_positions']
            # Check every saved source policy against its recorded hard validation J.
            reproduced={};source_selection=[]
            for _,row in records[records.seed==seed].iterrows():
                policy=decode_policy(load_torch(row.policy_path))
                identifier=mc_payload_hash(encode_policy(policy))
                if identifier not in reproduced:
                    data=(arrays[identifier] if identifier in arrays else mc_candidate_arrays(
                        val_pool,{identifier:policy},seed_dir/'predictions'/'reproduction')[identifier])
                    reproduced[identifier]=mc_measure(val_pool,data,original)
                measured=reproduced[identifier]['C']-float(row.lambda_)*reproduced[identifier]['V']
                if abs(measured-row.validation_J)>1e-7:
                    raise ArithmeticError('Frozen-context reproduction check failed; source scorer/data/code may differ.')
                if row.lambda_ in config['lambda_values']:
                    candidates.setdefault(identifier,policy)
                    source_selection.append(dict(protocol='source_selected',method=row.method,lambda_=row.lambda_,
                        candidate_id=identifier,training_lambda=row.lambda_ if row.method in MAIN_METHODS[3:] else np.nan,
                        rho_cutoff=getattr(policy,'rho_cutoff',np.nan),validation_C=reproduced[identifier]['C'],
                        validation_V=reproduced[identifier]['V'],validation_J=measured))
            save_json(seed_dir/'reproduction.json',dict(status='passed',source_policies=int((records.seed==seed).sum()),t_0=context['t_0']))
            if mode=='preflight':
                print(f'  Seed {seed}: source reproduction passed; {len(candidates)} unique candidates, '
                      f"validation pool={val_pool.N}, test pool={len(context['pool']['X_test'])}.",flush=True)
                save_json(seed_dir/'status.json',dict(status='preflight_passed',seed=seed));continue
            val_path=seed_dir/'validation_candidates.csv'
            if val_path.exists():validation=pd.read_csv(val_path)
            else:
                values=[]
                for item in [p for p in plan if p['split']=='val' and p['role']!='original']:
                    for identifier,data in arrays.items():
                        measured=mc_measure(val_pool,data,item['positions'])
                        values.append(dict(role=item['role'],cohort=item['cohort'],candidate_id=identifier,**measured))
                validation=pd.DataFrame(values);save_csv(val_path,validation)
            selection=mc_select(catalog,validation,config['lambda_values'])
            selection=pd.concat([selection,pd.DataFrame(source_selection)],ignore_index=True)
            selection=selection.assign(dataset=config['dataset'],model=config['model'],seed=seed)
            save_csv(seed_dir/'selection.csv',selection)
            # Seal all selection decisions before constructing or scoring test cohorts.
            chosen={identifier:candidates[identifier] for identifier in selection.candidate_id.unique()}
            for identifier,policy in chosen.items():save_torch(seed_dir/'selected_policies'/f'{identifier}.pt',encode_policy(policy))
            test_pool=mc_make_pool_cohort(context,source,'test')
            test_arrays=mc_candidate_arrays(test_pool,chosen,seed_dir/'predictions'/'test')
            for item in [p for p in plan if p['split']=='test']:
                path=seed_dir/'test_cohorts'/f"cohort_{item['cohort']:03d}.csv"
                if path.exists():continue
                metrics={identifier:mc_measure(test_pool,data,item['positions'],decompose=True) for identifier,data in test_arrays.items()}
                rows=[]
                for _,choice in selection.iterrows():
                    value=metrics[choice.candidate_id]
                    rows.append(dict(choice.to_dict(),cohort=item['cohort'],cohort_hash=item['cohort_hash'],
                                     J=value['C']-choice.lambda_*value['V'],**value))
                save_csv(path,pd.DataFrame(rows))
                if (item['cohort']+1)%10==0:print(f"  Seed {seed}: {item['cohort']+1}/{config['test_cohorts']} test cohorts saved.",flush=True)
            save_json(seed_dir/'status.json',dict(status='completed',seed=seed))
        except BaseException as error:
            save_json(seed_dir/'status.json',dict(status='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',seed=seed,error=str(error)))
            raise
    if mode=='preflight':return dict(directory=str(directory),status='preflight_passed')
    results=pd.concat([pd.read_csv(p) for seed in config['seeds'] for p in sorted((directory/f'seed_{seed}'/'test_cohorts').glob('cohort_*.csv'))],ignore_index=True)
    selection=pd.concat([pd.read_csv(directory/f'seed_{seed}'/'selection.csv') for seed in config['seeds']],ignore_index=True)
    expected=len(config['seeds'])*config['test_cohorts']*(len(MC_PROTOCOLS)+1)*len(MAIN_METHODS)*len(config['lambda_values'])
    if len(results)!=expected:raise ArithmeticError('Incomplete follow-up evaluation; raw completed cohorts are saved for resumption.')
    tables=mc_summarize(results,selection)
    stem=f"experiment-2-followup-{slug(config['dataset'])}-{config['model']}-{run_id}"
    for name,frame in dict(results=results,selection=selection,**tables).items():
        save_csv(directory/f'{name}.csv',frame);save_csv(root/'tables'/f'{stem}-{name}.csv',frame)
    save_json(root/'tables'/f'{stem}-config.json',load_json(directory/'config.json'))
    mc_plot(config,run_id,tables,config['figures_dir'],show)
    print(f'Follow-up saved: {directory}',flush=True)
    return dict(directory=str(directory),**tables)


MC_IMPLEMENTATION_NAMES=tuple(name for name in list(globals()) if name.startswith('mc_') and inspect.isfunction(globals()[name]))+('run_experiment_2_diagnostics',)


In [ ]:
# Joint target/cutoff calibration; requires the common setup and mc_* definitions.
JC_PROTOCOLS = ('cutoff_only', 'joint')
JC_LABELS = {'cutoff_only': 'Cutoff calibration only', 'joint': 'Joint target and cutoff calibration'}
JC_METRICS = MC_METRICS + ('V_no_action', 'V_gain', 'J_gain')


def experiment_2_calibration_config(dataset, model, **overrides):
    gamma_values = overrides.pop('gamma_values', [0.0, 0.25, 0.5, 0.75, 0.9, 1.0])
    values = [float(value) for value in gamma_values]
    if (not values or not np.isfinite(values).all() or len(values) != len(set(values))
            or any(value < 0 or value > 1 for value in values) or 1.0 not in values):
        raise ValueError('gamma_values must be distinct values in [0,1], including 1.')
    return dict(experiment_2_diagnostic_config(dataset, model, **overrides), gamma_values=values)


class TargetCalibratedPolicy:
    """Lower a frozen policy's targets; retain its binary recipient rule."""
    def __init__(self, policy, gamma):
        if not np.isfinite(gamma) or not 0 <= gamma <= 1:
            raise ValueError('gamma must lie in [0,1].')
        self.policy, self.gamma = policy, float(gamma)

    def __call__(self, cohort):
        q, rho = self.policy(cohort)
        # Preserve exact identity at gamma=1, including threshold ties.
        if self.gamma == 1.0: return q, rho
        return cohort.t_0 + self.gamma * (q - cohort.t_0), rho


def jc_encode_policy(policy):
    if isinstance(policy, TargetCalibratedPolicy):
        if policy.gamma == 1.0: return encode_policy(policy.policy)
        return dict(type='target_calibrated', gamma=policy.gamma, base=encode_policy(policy.policy))
    return encode_policy(policy)


def jc_decode_policy(payload):
    if payload['type'] == 'target_calibrated':
        return TargetCalibratedPolicy(decode_policy(payload['base']), payload['gamma'])
    return decode_policy(payload)


def jc_frozen_context(config, metadata, records, seed, raw_cache):
    context = mc_frozen_context(config, metadata, seed, raw_cache)
    # The original-threshold baseline serializes the original scalar exactly.
    # Re-evaluating a scorer on another BLAS/runtime may move it by one ULP.
    thresholds = []
    for _, row in records[(records.seed == seed) & (records.method == MAIN_METHODS[1])].iterrows():
        payload = load_torch(row.policy_path)
        if payload['type'] != 'common' or not math.isinf(payload['cost_limit']):
            raise ValueError('Unexpected original-threshold baseline payload.')
        thresholds.append(float(payload['q_c']))
    if not thresholds or len(set(thresholds)) != 1:
        raise ValueError('Source trials disagree on their original threshold.')
    original = thresholds[0]
    if abs(original - context['t_0']) > 1e-12:
        raise ArithmeticError('Saved initial threshold differs from the frozen scorer/reference sample.')
    context['recomputed_t_0'] = context['t_0']; context['t_0'] = original
    return context


def jc_catalog(config, source, records, context):
    candidates, catalog = mc_catalog(config, source, records, context)
    catalog = catalog.assign(gamma=1.0, base_candidate_id=catalog.candidate_id)
    additions = []
    # All identity candidates precede all adjustments: exact ties retain gamma=1.
    for gamma in sorted((v for v in config['gamma_values'] if v != 1.0), reverse=True):
        for row in catalog[catalog.policy_type == 'learned'].to_dict('records'):
            policy = TargetCalibratedPolicy(candidates[row['candidate_id']], gamma)
            identifier = mc_payload_hash(jc_encode_policy(policy))
            candidates.setdefault(identifier, policy)
            additions.append(dict(row, candidate_id=identifier, gamma=gamma))
    return candidates, pd.concat([catalog, pd.DataFrame(additions)], ignore_index=True)


@torch.no_grad()
def jc_candidate_arrays(cohort, candidates, directory):
    """Solve once per target model/gamma/pool; all rho cutoffs share that response."""
    directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)
    arrays, responses = {}, {}
    for identifier, policy in candidates.items():
        path = directory / f'{identifier}.pt'
        if path.exists(): arrays[identifier] = load_torch(path); continue
        payload = copy.deepcopy(jc_encode_policy(policy))
        base = payload.get('base', payload)
        base.pop('rho_cutoff', None)
        response_id = mc_payload_hash(payload)
        q, rho = policy(cohort); q = as_tensor(q).expand(cohort.N)
        feasible = target_feasible(cohort, q)
        rho = binary_vector(rho, cohort.N) & feasible
        if response_id not in responses:
            X_br = br(cohort, q)
            scores = torch.where(feasible, q, cohort.s_0)
            costs = cost(cohort.X, X_br)
            if not torch.allclose(cohort.f_w0(X_br), scores, atol=2e-9, rtol=2e-9):
                raise ArithmeticError('Calibrated target attainment failed.')
            if (costs > cohort.B + 1e-9).any(): raise ArithmeticError('Budget exceeded.')
            if not torch.equal(X_br[:, ~cohort.actionable_mask], cohort.X[:, ~cohort.actionable_mask]):
                raise ArithmeticError('Immutable feature changed.')
            responses[response_id] = costs, scores
        costs, scores = responses[response_id]
        value = dict(costs=torch.where(rho, costs, torch.zeros_like(costs)),
                     scores=torch.where(rho, scores, cohort.s_0), rho=rho)
        save_torch(path, value); arrays[identifier] = value
    return arrays


def jc_select(catalog, validation, lambda_values):
    means = validation.groupby('candidate_id')[['C', 'V']].mean()
    rows = []
    for protocol in JC_PROTOCOLS:
        for lam in lambda_values:
            for method in MAIN_METHODS:
                choices = catalog[catalog.method == method]
                if protocol == 'cutoff_only': choices = choices[choices.gamma == 1.0]
                selected, best = None, float('inf')
                for _, candidate in choices.iterrows():
                    metrics = means.loc[candidate.candidate_id]
                    J = float(metrics.C - lam * metrics.V)
                    if J < best - 1e-12: selected, best = candidate, J
                if selected is None: raise ValueError('Empty calibration candidate family.')
                metrics = means.loc[selected.candidate_id]
                rows.append(dict(protocol=protocol, method=method, lambda_=lam,
                    candidate_id=selected.candidate_id, base_candidate_id=selected.base_candidate_id,
                    training_lambda=selected.training_lambda, rho_cutoff=selected.rho_cutoff,
                    gamma=selected.gamma, validation_C=float(metrics.C),
                    validation_V=float(metrics.V), validation_J=best))
    selection = pd.DataFrame(rows)
    check = selection.pivot(index=['method', 'lambda_'], columns='protocol', values='validation_J')
    if (check.joint > check.cutoff_only + 1e-10).any():
        raise ArithmeticError('Joint validation search failed to include the cutoff-only candidates.')
    return selection


def jc_summarize(results, selection):
    keys = ['dataset', 'model', 'protocol', 'method', 'lambda_', 'seed']
    seed_means = results.groupby(keys)[list(JC_METRICS)].mean().reset_index()
    seed_means = seed_means.merge(selection[keys + ['validation_C', 'validation_V', 'validation_J',
        'gamma', 'rho_cutoff', 'training_lambda']], on=keys, validate='one_to_one')
    seed_means['test_minus_validation_J'] = seed_means.J - seed_means.validation_J
    summary = mc_stat_summary(seed_means, keys[:-1], list(JC_METRICS) + ['validation_J', 'test_minus_validation_J'])
    merge_keys = ['dataset', 'model', 'method', 'lambda_', 'seed']
    joined = seed_means[seed_means.protocol == 'joint'].merge(
        seed_means[seed_means.protocol == 'cutoff_only'], on=merge_keys,
        suffixes=('_joint', '_cutoff'), validate='one_to_one')
    paired = joined[merge_keys].copy()
    for metric in JC_METRICS: paired[metric] = joined[metric + '_joint'] - joined[metric + '_cutoff']
    paired['comparison'] = 'joint minus cutoff_only'
    paired_summary = mc_stat_summary(paired, merge_keys[:-1] + ['comparison'], JC_METRICS)
    method_pairs = []
    merge_keys = ['dataset', 'model', 'protocol', 'lambda_', 'seed']
    for baseline in MAIN_METHODS[:-1]:
        joined = seed_means[seed_means.method == MAIN_METHODS[-1]].merge(
            seed_means[seed_means.method == baseline], on=merge_keys,
            suffixes=('_full', '_baseline'), validate='one_to_one')
        part = joined[merge_keys].copy(); part['baseline'] = baseline
        for metric in JC_METRICS: part[metric] = joined[metric + '_full'] - joined[metric + '_baseline']
        method_pairs.append(part)
    method_pairs = pd.concat(method_pairs, ignore_index=True)
    method_summary = mc_stat_summary(method_pairs, merge_keys[:-1] + ['baseline'], JC_METRICS)
    return dict(seed_means=seed_means, summary=summary, paired_seeds=paired,
                paired_summary=paired_summary, method_paired_seeds=method_pairs,
                method_paired_summary=method_summary)


def jc_plot(config, run_id, tables, selection, show=True):
    summary = tables['summary']; figures = []
    for metric, suffix in [('J', 'objective'), ('frontier', 'cost-validity')]:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for ax, protocol in zip(axes, JC_PROTOCOLS):
            for method in MAIN_METHODS:
                part = summary[(summary.protocol == protocol) & (summary.method == method)]
                if metric == 'J':
                    data = part[part.metric == metric].sort_values('lambda_')
                    ax.plot(data.lambda_, data['mean'], color=METHOD_COLORS[method], marker=METHOD_MARKERS[method], label=METHOD_LABELS[method])
                    ax.fill_between(data.lambda_, data.ci_low, data.ci_high, color=METHOD_COLORS[method], alpha=.10)
                    ax.set(xscale='log', xlabel='Validity weight lambda', ylabel='Test J')
                else:
                    data = part.pivot(index='lambda_', columns='metric', values='mean')
                    front = data[nondominated_mask(data.C, data.V)].sort_values(['V','C']).drop_duplicates(['C','V'])
                    ax.scatter(data.V, data.C, color=METHOD_COLORS[method], marker=METHOD_MARKERS[method], alpha=.2)
                    ax.plot(front.V, front.C, color=METHOD_COLORS[method], marker=METHOD_MARKERS[method], label=METHOD_LABELS[method])
                    ax.set(xlabel='Post-shift validity V', ylabel='Mean recourse cost C')
            ax.set_title(JC_LABELS[protocol], fontsize=10); ax.grid(alpha=.2)
        figures.append((suffix, fig, axes[0]))
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, metric, label in zip(axes, ['J', 'wasted_cost', 'successful_excess_cost'],
            ['Change in test J', 'Change in unsuccessful cost', 'Change in cost above cutoff']):
        for method in MAIN_METHODS:
            data = tables['paired_summary']; data = data[(data.method == method) & (data.metric == metric)].sort_values('lambda_')
            ax.plot(data.lambda_, data['mean'], color=METHOD_COLORS[method], marker=METHOD_MARKERS[method], label=METHOD_LABELS[method])
            ax.fill_between(data.lambda_, data.ci_low, data.ci_high, color=METHOD_COLORS[method], alpha=.10)
        ax.axhline(0, color='black', linewidth=.7); ax.set(xscale='log', xlabel='Validity weight lambda', ylabel=label); ax.grid(alpha=.2)
    figures.append(('paired-changes', fig, axes[0]))
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    for row, protocol in zip(axes, JC_PROTOCOLS):
        for ax, metric in zip(row, ['wasted_cost', 'successful_cutoff_cost', 'successful_excess_cost']):
            for method in MAIN_METHODS:
                data = summary[(summary.protocol == protocol) & (summary.method == method) & (summary.metric == metric)].sort_values('lambda_')
                ax.plot(data.lambda_, data['mean'], color=METHOD_COLORS[method], marker=METHOD_MARKERS[method], label=METHOD_LABELS[method])
            ax.set(xscale='log', xlabel='Validity weight lambda', ylabel=metric.replace('_', ' '))
            ax.set_title(JC_LABELS[protocol], fontsize=9); ax.grid(alpha=.2)
    figures.append(('cost-decomposition', fig, axes.flat[0]))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, column, label in zip(axes, ['gamma', 'rho_cutoff'], ['Selected target scale gamma', 'Selected recommendation cutoff']):
        for method in MAIN_METHODS:
            if method not in MAIN_METHODS[3:] or (column == 'rho_cutoff' and method == MAIN_METHODS[3]): continue
            data = selection[(selection.protocol == 'joint') & (selection.method == method) & selection.training_lambda.notna()]
            for seed, part in data.groupby('seed'):
                part = part.sort_values('lambda_')
                ax.plot(part.lambda_, part[column], color=METHOD_COLORS[method], alpha=.35, linewidth=.7)
            ax.scatter(data.lambda_, data[column], color=METHOD_COLORS[method], marker=METHOD_MARKERS[method], label=METHOD_LABELS[method], alpha=.6)
        ax.set(xscale='log', xlabel='Validity weight lambda', ylabel=label, ylim=(-.03,1.03)); ax.grid(alpha=.2)
        if not np.isfinite(ax.dataLim.minposx):
            values=summary.lambda_.unique();ax.set_xlim(float(min(values))/1.2,float(max(values))*1.2)
            ax.text(.5,.5,'No learned policy selected',ha='center',transform=ax.transAxes)
    figures.append(('selected-calibration', fig, axes[0]))
    root = Path(config['figures_dir']); root.mkdir(parents=True, exist_ok=True)
    stem = f"experiment-2-joint-calibration-{slug(config['dataset'])}-{config['model']}-{run_id}"
    paths = []
    for suffix, fig, legend_ax in figures:
        handles, labels = legend_ax.get_legend_handles_labels()
        fig.suptitle(f"{config['dataset']} | {config['model']}")
        fig.legend(handles, labels, loc='lower center', ncol=min(3,len(handles)), fontsize=7)
        standardize_figure(fig);fig.tight_layout(rect=(0,.12,1,.95))
        path = root / f'{stem}-{suffix}.png'
        atomic_write(path, lambda handle: fig.savefig(handle, format='png', dpi=160, bbox_inches='tight'))
        save_json(path.with_suffix('.json'), dict(config=config, run_id=run_id,
            method_order=list(MAIN_METHODS), method_colors=METHOD_COLORS,
            interval='Pointwise 95% t intervals across seed means; cohorts averaged within each seed.'))
        paths.append(str(path))
        if show: plt.show()
        plt.close(fig)
    return paths


def jc_code_id():
    names = tuple(name for name in globals() if name.startswith('jc_') and inspect.isfunction(globals()[name]))
    names += ('run_experiment_2_calibration', 'experiment_2_calibration_config')
    return stable_id(dict(core=implementation_id()[0],
        followup={name: _code_signature(inspect.unwrap(globals()[name]).__code__) for name in MC_IMPLEMENTATION_NAMES},
        calibration={name: _code_signature(inspect.unwrap(globals()[name]).__code__) for name in names},
        wrapper={name: _code_signature(fn.__code__) for name,fn in vars(TargetCalibratedPolicy).items() if inspect.isfunction(fn)},
        protocols=JC_PROTOCOLS, metrics=JC_METRICS))


def run_experiment_2_calibration(config, *, mode='run', show=True):
    if mode not in {'run','preflight','load','skip'}: raise ValueError('Unknown mode.')
    if mode == 'skip': return None
    checked = experiment_2_calibration_config(**copy.deepcopy(config))
    gamma_values = checked.pop('gamma_values')
    config, metadata, records = mc_resolve_source(checked)
    config['gamma_values'] = gamma_values; source = metadata['config']
    request_id = stable_id(dict(config={k:v for k,v in config.items() if k not in {'results_dir','figures_dir'}},
        code=jc_code_id(), policies=records.policy_hash.tolist(), source=metadata['config_id']))
    root = Path(config['results_dir']).expanduser().resolve()
    request_path = root/'diagnostics'/'experiment_02_joint_calibration'/slug(config['dataset'])/config['model']/request_id
    if mode == 'load':
        locator = load_json(request_path/'run.json'); directory = Path(locator['directory'])
        tables = {name:pd.read_csv(directory/f'{name}.csv') for name in ['seed_means','summary','paired_seeds','paired_summary','method_paired_seeds','method_paired_summary']}
        selection = pd.read_csv(directory/'selection.csv')
        jc_plot(config, locator['run_id'], tables, selection, show)
        return dict(directory=str(directory), selection=selection, **tables)
    contexts, raw_cache = {}, {}
    for seed in config['seeds']:
        print(f"Recovering frozen context: {config['dataset']} | {config['model']} | seed={seed}", flush=True)
        contexts[seed] = jc_frozen_context(config, metadata, records, seed, raw_cache)
    inputs = {seed:{key:context[key] for key in ['source_hash','pool_hash','model_id']} for seed,context in contexts.items()}
    run_id = stable_id(dict(request_id=request_id, inputs=inputs))
    directory = request_path/run_id; directory.mkdir(parents=True, exist_ok=True)
    save_json(request_path/'run.json', dict(run_id=run_id, directory=str(directory)))
    save_json(directory/'config.json', dict(config=config, source=metadata, run_id=run_id, input_fingerprints=inputs,
        target_calibration='q_gamma = t_0 + gamma * (q_hat - t_0); gamma in [0,1].',
        notes=['Frozen final checkpoint pool; no training or scorer/proxy fitting.',
            'Both protocols pool all selected training lambdas and use the same multi-cohort validation and test plans.',
            'Both personalized families calibrate targets; only the selection method calibrates recipient cutoffs.',
            'Identity targets are included and preferred on exact validation ties. Baselines retain their original families.',
            'Calibration is selected only on validation; test outcomes never select parameters.',
            'Cohorts bootstrap existing held-out partitions; they are not independent new datasets.',
            'Costs to/above the realized cutoff are retrospective diagnostics, not guaranteed implementable savings.']))
    for seed, context in contexts.items():
        seed_dir = directory/f'seed_{seed}'; seed_dir.mkdir(exist_ok=True)
        complete = all((seed_dir/'test_cohorts'/f'cohort_{i:03d}.csv').exists() for i in range(config['test_cohorts']))
        if (mode == 'run' and complete and (seed_dir/'selection.csv').exists()
                and (seed_dir/'status.json').exists() and load_json(seed_dir/'status.json')['status']=='completed'):
            print(f'  Seed {seed}: saved joint calibration complete.', flush=True); continue
        try:
            save_json(seed_dir/'status.json', dict(status='running', seed=seed))
            candidates, catalog = jc_catalog(config, source, records, context)
            save_csv(seed_dir/'candidates.csv', catalog)
            plan = [item for item in mc_cohort_plan(context, source, config) if item['role'] in {'multi','test'}]
            save_json(seed_dir/'cohorts.json', plan)
            val_pool = mc_make_pool_cohort(context, source, 'val')
            arrays = jc_candidate_arrays(val_pool, candidates, seed_dir/'predictions'/'val')
            original = context['pool']['original_val_positions']; reproduced = {}
            for _, record in records[records.seed == seed].iterrows():
                policy = decode_policy(load_torch(record.policy_path)); identifier = mc_payload_hash(encode_policy(policy))
                if identifier not in reproduced:
                    data = arrays.get(identifier)
                    if data is None: data = jc_candidate_arrays(val_pool, {identifier:policy}, seed_dir/'predictions'/'reproduction')[identifier]
                    reproduced[identifier] = mc_measure(val_pool, data, original)
                value = reproduced[identifier]
                if abs(value['C']-record.lambda_*value['V']-record.validation_J)>1e-7:
                    raise ArithmeticError('Source validation reproduction failed; frozen context may differ.')
            save_json(seed_dir/'reproduction.json', dict(status='passed', source_policies=int((records.seed==seed).sum()),
                t_0=context['t_0'], recomputed_t_0=context['recomputed_t_0']))
            print(f'  Seed {seed}: {len(candidates)} candidates; {config["validation_cohorts"]} validation cohorts.', flush=True)
            if mode == 'preflight':
                save_json(seed_dir/'status.json', dict(status='preflight_passed', seed=seed)); continue
            paths = []
            for item in (p for p in plan if p['split']=='val'):
                path = seed_dir/'validation_cohorts'/f'cohort_{item["cohort"]:03d}.csv'; paths.append(path)
                if path.exists(): continue
                rows = [dict(cohort=item['cohort'], cohort_hash=item['cohort_hash'], candidate_id=identifier,
                    **mc_measure(val_pool, data, item['positions'])) for identifier,data in arrays.items()]
                save_csv(path, pd.DataFrame(rows))
            validation = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
            save_csv(seed_dir/'validation_candidates.csv', validation)
            selection = jc_select(catalog, validation, config['lambda_values'])
            selection = selection.assign(dataset=config['dataset'], model=config['model'], seed=seed)
            save_csv(seed_dir/'selection.csv', selection)
            chosen = {identifier:candidates[identifier] for identifier in selection.candidate_id.unique()}
            for identifier,policy in chosen.items(): save_torch(seed_dir/'selected_policies'/f'{identifier}.pt', jc_encode_policy(policy))
            # All choices are sealed above. Only now construct and evaluate test applicants.
            test_pool = mc_make_pool_cohort(context, source, 'test')
            test_arrays = jc_candidate_arrays(test_pool, chosen, seed_dir/'predictions'/'test')
            no_action_id = mc_payload_hash(encode_policy(NoRecourse()))
            for item in (p for p in plan if p['split']=='test'):
                path = seed_dir/'test_cohorts'/f'cohort_{item["cohort"]:03d}.csv'
                if path.exists(): continue
                metrics = {identifier:mc_measure(test_pool, data, item['positions'], decompose=True)
                           for identifier,data in test_arrays.items()}
                V_no_action = metrics[no_action_id]['V']; rows = []
                for _, choice in selection.iterrows():
                    value = metrics[choice.candidate_id]; V_gain = value['V'] - V_no_action
                    rows.append(dict(choice.to_dict(), cohort=item['cohort'], cohort_hash=item['cohort_hash'],
                        V_no_action=V_no_action, V_gain=V_gain, J_gain=value['C']-choice.lambda_*V_gain,
                        J=value['C']-choice.lambda_*value['V'], **value))
                save_csv(path, pd.DataFrame(rows))
                if (item['cohort']+1)%10 == 0: print(f'  Seed {seed}: {item["cohort"]+1}/{config["test_cohorts"]} test cohorts saved.', flush=True)
            save_json(seed_dir/'status.json', dict(status='completed', seed=seed))
        except BaseException as error:
            save_json(seed_dir/'status.json', dict(status='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed', seed=seed, error=str(error)))
            raise
    if mode == 'preflight': return dict(directory=str(directory), status='preflight_passed')
    results = pd.concat([pd.read_csv(directory/f'seed_{seed}'/'test_cohorts'/f'cohort_{i:03d}.csv')
        for seed in config['seeds'] for i in range(config['test_cohorts'])], ignore_index=True)
    selection = pd.concat([pd.read_csv(directory/f'seed_{seed}'/'selection.csv') for seed in config['seeds']], ignore_index=True)
    expected = len(config['seeds'])*config['test_cohorts']*len(JC_PROTOCOLS)*len(MAIN_METHODS)*len(config['lambda_values'])
    if len(results) != expected: raise ArithmeticError('Incomplete joint-calibration evaluation.')
    tables = jc_summarize(results, selection)
    stem = f"experiment-2-joint-calibration-{slug(config['dataset'])}-{config['model']}-{run_id}"
    for name, frame in dict(results=results, selection=selection, **tables).items():
        save_csv(directory/f'{name}.csv', frame); save_csv(root/'tables'/f'{stem}-{name}.csv', frame)
    save_json(root/'tables'/f'{stem}-config.json', load_json(directory/'config.json'))
    jc_plot(config, run_id, tables, selection, show)
    print(f'Joint calibration saved: {directory}', flush=True)
    return dict(directory=str(directory), selection=selection, **tables)


## Target and group-audit definitions


In [ ]:
def target_prepare_real(X, y, c, seed, immutable_features):
    """Group-disjoint records; preprocessing and qualification proxy fit on fit only.

    Fitting and held-out pool caps are independent; sampling is WITHOUT replacement.
    Recourse later treats all resulting encoded coordinates as continuous.
    """
    X = pd.DataFrame(X).reset_index(drop=True)
    y = np.asarray(y, dtype=int).reshape(-1)
    indices = split_indices(X, seed)
    rng = np.random.default_rng(seed)
    indices = {name: rng.choice(idx, min(len(idx), c['fit_cap'] if name=='fit' else c['pool_cap']), replace=False)
               for name,idx in indices.items()}
    proxy_options=c['qualification_proxy']
    numeric = X.select_dtypes(include="number").columns.tolist()
    categorical = [column for column in X if column not in numeric]
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])
    X_fit = preprocessor.fit_transform(X.iloc[indices["fit"]])
    if len(np.unique(y[indices["fit"]])) != 2:
        raise ValueError("The fitting partition must contain both outcome classes.")
    proxy_options = dict(proxy_options or {})
    qualification_proxy = MLPClassifier(
        hidden_layer_sizes=tuple(proxy_options.get("hidden", [32,16])), activation="tanh",
        max_iter=proxy_options.get("max_iter",1500), early_stopping=True,
        random_state=seed,
    ).fit(X_fit, y[indices["fit"]])
    population = {f"X_{name}": np.asarray(preprocessor.transform(X.iloc[idx]), dtype=float)
                  for name, idx in indices.items()}
    population["f_star_fit"] = qualification_proxy.predict_proba(X_fit)[:, 1]

    immutable_features = set(immutable_features or [])
    missing = immutable_features - set(X.columns)
    if missing:
        raise ValueError(f"Immutable columns were not found: {sorted(missing)}")
    # Exact source-column mapping, not string-prefix matching on one-hot names.
    source_features = list(numeric)
    if categorical:
        encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
        source_features += [column for column, categories in zip(categorical, encoder.categories_)
                            for _ in categories]
    feature_names = preprocessor.get_feature_names_out()
    assert len(source_features) == len(feature_names) == X_fit.shape[1]
    population["actionable_mask"] = np.array([name not in immutable_features for name in source_features])
    population["feature_names"] = feature_names.tolist()
    population["encoded_source_features"] = source_features
    population["immutable_features"] = sorted(immutable_features)
    population["y_test"] = y[indices["test"]]
    population.update(indices=indices, preprocessor=preprocessor,
                      qualification_proxy=qualification_proxy)
    return population


In [ ]:
# Standalone target experiment: configuration, persistent data, and cohort plans.
from scipy.stats import t as student_t
from threadpoolctl import threadpool_limits
import gzip

TARGET_DATASETS = ('German Credit', 'Adult', 'ACSIncome', 'Give Me Some Credit',
                   'Synthetic nonlinear', 'Synthetic curved')
TARGET_MODELS = ('affine', 'quadratic')
TARGET_FORCED = 'Selection off, same learned targets'
TARGET_METHODS = MAIN_METHODS + (TARGET_FORCED,)
TARGET_COLORS = dict(METHOD_COLORS, **{TARGET_FORCED:'#D55E00'})
TARGET_LABELS = dict(METHOD_LABELS, **{TARGET_FORCED:TARGET_FORCED})


def target_config(dataset, model, **overrides):
    """Independent configuration; grids do not mutate another dataset/model."""
    c = dict(dataset=dataset, model=model, seeds=[42,43,44,45,46],
        alpha=.4, B=.75, tau=.01, eta=.2, T=500,
        train_lambda_values=[1.,3.,10.], lambda_values=[1.,3.,10.],
        cohort_size=256, fit_cap=5000, pool_cap=5000, synthetic_n=6000,
        validation_cohorts=20, test_cohorts=50, snapshot_every=50,
        checkpoint_every=25, target_points=30,
        rho_cutoffs=[.05,.1,.25,.5,.75,.9,.95],
        Theta_1_radius=3., Theta_2_radius=3.,
        scorer=dict(l2_reg=.01, epochs=1000, lr=.01, hidden=12),
        qualification_proxy=dict(hidden=[32,16],max_iter=1500),
        solver=dict(exact_margin=1e-4),
        age_bins=[25,45,65], min_group_rejected=30,
        matched_cost_tolerance=.01, matched_validity_tolerance=.01,
        gmsc_path='./GiveMeSomeCredit.csv', results_dir='./Results', figures_dir='./figures',
        response_batch_size=512, group_distribution_plots=True, retry_failed=False)
    for key,value in overrides.items():
        if key not in c:raise ValueError(f'Unknown setting: {key}')
        if isinstance(c[key],dict):
            if not isinstance(value,dict) or set(value)-set(c[key]):raise ValueError(key)
            c[key].update(copy.deepcopy(value))
        else:c[key]=copy.deepcopy(value)
    validate_target_config(c)
    return c


def validate_target_config(c):
    if c['dataset'] not in TARGET_DATASETS or c['model'] not in TARGET_MODELS:
        raise ValueError('Choose a supported dataset and affine or quadratic scorer.')
    for key in ['cohort_size','fit_cap','pool_cap','synthetic_n','validation_cohorts',
                'test_cohorts','snapshot_every','checkpoint_every','T','response_batch_size','min_group_rejected']:
        if isinstance(c[key],bool) or not isinstance(c[key],(int,np.integer)) or c[key]<1:raise ValueError(key)
    if c['cohort_size']<2 or c['target_points']<2:raise ValueError('cohort_size/target_points')
    if not 0<c['alpha']<1 or c['B']<0 or not np.isfinite(c['B']):raise ValueError('alpha/B')
    for key in ['tau','eta','Theta_1_radius','Theta_2_radius']:
        if not np.isfinite(c[key]) or c[key]<=0:raise ValueError(key)
    for key in ['train_lambda_values','lambda_values']:
        a=np.asarray(c[key],float)
        if not len(a) or not np.isfinite(a).all() or (a<=0).any() or len(set(a))!=len(a):raise ValueError(key)
    if not c['seeds'] or len(set(c['seeds']))!=len(c['seeds']):raise ValueError('distinct seeds required')
    if any(isinstance(s,bool) or not isinstance(s,(int,np.integer)) or s<0 for s in c['seeds']):raise ValueError('seeds')
    if c['age_bins']!=sorted(set(c['age_bins'])) or not np.isfinite(c['age_bins']).all():raise ValueError('age_bins')
    if any(c[k]<0 or not np.isfinite(c[k]) for k in ['matched_cost_tolerance','matched_validity_tolerance']):raise ValueError('matching tolerance')
    validated_rho_cutoffs(c['rho_cutoffs'])
    if not 0<c['solver']['exact_margin']<1:raise ValueError('exact_margin')


def target_code_id():
    """Bytecode identity is stable across notebook cell filenames and restarts."""
    signatures={}
    for name in TARGET_CODE_NAMES:
        value=globals()[name]
        for member,fn in (vars(value).items() if inspect.isclass(value) else [(name,value)]):
            if isinstance(fn,(staticmethod,classmethod)):fn=fn.__func__
            if inspect.isfunction(fn):
                fn=inspect.unwrap(fn)
                signatures[name+'.'+member]=[_code_signature(fn.__code__),repr(fn.__defaults__),repr(fn.__kwdefaults__)]
    return stable_id(dict(code=signatures,versions=dict(python=platform.python_version(),
        numpy=np.__version__,pandas=pd.__version__,torch=str(torch.__version__),scipy=scipy.__version__,sklearn=sklearn.__version__)))


def target_file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


def target_run_directory(c):
    # File CONTENT, not only path, determines the local dataset identity.
    source=target_file_hash(c['gmsc_path']) if c['dataset']=='Give Me Some Credit' else 'pinned-loader-v1'
    science={k:v for k,v in c.items() if k not in ['results_dir','figures_dir','retry_failed','checkpoint_every','response_batch_size','gmsc_path']}
    run_id=stable_id(dict(config=science,code=target_code_id(),source=source))
    root=Path(c['results_dir']).expanduser().resolve()
    return root/'experiments'/'targets'/slug(c['dataset'])/c['model']/run_id,run_id,source


def target_raw_data(c):
    """Download/read once. Only split-specific fitting code sees outcome labels."""
    name=c['dataset']
    if name=='Adult':
        data=fetch_openml(name='adult',version=2,as_frame=True)
        X=data.data; y=data.target.astype(str).str.rstrip('.').eq('>50K').to_numpy(int)
        immutable=['age','sex','race']
    elif name=='German Credit':
        data=fetch_openml(data_id=31,as_frame=True)
        X=data.data; y=data.target.astype(str).eq('good').to_numpy(int)
        immutable=['age','personal_status','foreign_worker','num_dependents']
    elif name=='ACSIncome':
        from folktables import ACSDataSource,ACSIncome
        data=ACSDataSource(survey_year='2018',horizon='1-Year',survey='person').get_data(states=['CA'],download=True)
        features,labels,_=ACSIncome.df_to_numpy(data)
        X=pd.DataFrame(features,columns=ACSIncome.features);y=labels.astype(int)
        for col in ['COW','SCHL','MAR','OCCP','POBP','RELP','SEX','RAC1P']:X[col]=X[col].astype('category')
        immutable=['AGEP','SEX','RAC1P','MAR','RELP','POBP']
    elif name=='Give Me Some Credit':
        data=pd.read_csv(c['gmsc_path'])
        y=data['SeriousDlqin2yrs'].eq(0).to_numpy(int)
        X=data.drop(columns=['SeriousDlqin2yrs','Unnamed: 0'],errors='ignore')
        immutable=['age','NumberOfDependents']
    else:raise ValueError('Synthetic data are generated per seed.')
    return pd.DataFrame(X).reset_index(drop=True),np.asarray(y),immutable


def target_group_labels(X,c):
    """Raw attributes; never infer groups from standardized or one-hot test values."""
    labels=pd.DataFrame(index=np.arange(len(X))); notes={}
    name=c['dataset']
    def add(key,values,note):
        labels[key]=pd.Series(values).astype('string').fillna('Unknown').str.strip().replace({'?':'Unknown','':'Unknown'})
        notes[key]=note
    if name=='Adult':
        for col in ['sex','race']:add(col,X[col],f'Recorded {col}; descriptive group audit, not a causal fairness guarantee.')
    elif name=='ACSIncome':
        add('sex',pd.to_numeric(X['SEX'],errors='coerce').map({1:'Male',2:'Female'}),'Recorded SEX codes 1=male, 2=female.')
        race={1:'White',2:'Black',3:'American Indian',4:'Alaska Native',5:'American Indian and Alaska Native',6:'Asian',7:'Native Hawaiian/Pacific Islander',8:'Some other race',9:'Two or more races'}
        add('race',pd.to_numeric(X['RAC1P'],errors='coerce').map(race),'Recorded RAC1P categories; no binary race aggregation.')
    elif name=='German Credit':
        def sex(value):
            s=str(value).strip().lower()
            if s in ['a92','a95'] or 'female' in s:return 'Female'
            if s in ['a91','a93','a94'] or 'male' in s:return 'Male'
            return 'Unknown'
        add('sex',X['personal_status'].map(sex),'Sex derived from the documented personal_status codes; unknown codes remain Unknown.')
    if name.startswith('Synthetic'):
        add('synthetic_x0_sign',np.where(np.asarray(X.iloc[:,0])<0,'x0 < 0','x0 >= 0'),
            'Artificial diagnostic groups based on immutable x0; NOT protected demographic groups.')
    else:
        age_col='AGEP' if name=='ACSIncome' else 'age'
        age=pd.to_numeric(X[age_col],errors='coerce');bins=[-np.inf]+list(c['age_bins'])+[np.inf]
        tags=[f'< {bins[1]:g}']+[f'{a:g} to < {b:g}' for a,b in zip(bins[1:-2],bins[2:-1])]+[f'>= {bins[-2]:g}']
        add('age_band',pd.cut(age,bins=bins,labels=tags,right=False).astype('string'),
            'Predeclared descriptive age bands; not a legal definition of protected age groups.')
    return labels,notes


def target_population(c,seed,source):
    root=Path(c['results_dir']).expanduser().resolve()/'cache'/'targets'/target_code_id()
    if c['dataset'].startswith('Synthetic'):
        X,scores,_=generate_correlated_nonlinear_population(c['synthetic_n'],correlation=.5,seed=seed)
        if c['dataset']=='Synthetic curved':scores=expit(.6*X[:,0]+1.2*X[:,1]+.3*X[:,2]-.8*X[:,1]**2-.2*X[:,2]**2)
        raw=pd.DataFrame(X);y=None;immutable=None
        data_hash=stable_id(dict(X=array_id(X),scores=array_id(scores)))
    else:
        raw_key=stable_id(dict(dataset=c['dataset'],source=source))
        raw_path=root/'raw'/f'{raw_key}.pkl'
        if raw_path.exists():raw,y,immutable=load_pickle(raw_path)
        else:
            raw,y,immutable=target_raw_data(c);save_pickle(raw_path,(raw,y,immutable))
        data_hash=stable_id(dict(rows=array_id(pd.util.hash_pandas_object(raw,index=True).to_numpy()),
                                columns=list(raw.columns),labels=array_id(y)))
    key=stable_id(dict(data=data_hash,seed=seed,fit_cap=c['fit_cap'],pool_cap=c['pool_cap'],
                       proxy=c['qualification_proxy'],age_bins=c['age_bins']))
    path=root/'populations'/f'{key}.pkl'
    if path.exists():return load_pickle(path),key
    labels,notes=target_group_labels(raw,c)
    if c['dataset'].startswith('Synthetic'):
        pop=prepare_population(raw.to_numpy(),scores,seed,immutable_indices=[0])
        rng=np.random.default_rng(seed)
        for split in ['fit','train','val','test']:
            cap=c['fit_cap'] if split=='fit' else c['pool_cap']
            p=rng.choice(len(pop[f'X_{split}']),min(cap,len(pop[f'X_{split}'])),replace=False)
            pop[f'X_{split}']=pop[f'X_{split}'][p];pop['indices'][split]=pop['indices'][split][p]
            if split=='fit':pop['f_star_fit']=pop['f_star_fit'][p]
    else:
        # Existing fitter, with independent fit and held-out caps and raw group alignment.
        pop=target_prepare_real(raw,y,c,seed,immutable)
    pop['groups']={s:labels.iloc[ids].reset_index(drop=True) for s,ids in pop['indices'].items()}
    pop['group_notes']=notes;pop['data_hash']=data_hash
    for s in ['fit','train','val','test']:
        assert len(pop[f'X_{s}'])==len(pop['indices'][s])==len(pop['groups'][s])
    save_pickle(path,pop)
    return pop,key


def target_context(c,seed,source):
    pop,pop_id=target_population(c,seed,source)
    root=Path(c['results_dir']).expanduser().resolve()/'cache'/'targets'/target_code_id()
    score_id=stable_id(dict(pop=pop_id,model=c['model'],scorer=c['scorer']))
    path=root/'scorers'/f'{score_id}.pt'
    if path.exists():
        v=load_torch(path);f_w0=FixedScorer(v['w_0']) if c['model']=='affine' else QuadraticScorer(v['w_0'],v['h'])
    else:
        with threadpool_limits(limits=1):f_w0=fit_f_w0(pop['X_fit'],pop['f_star_fit'],c['model'],seed,**c['scorer'])
        save_torch(path,f_w0.state_dict())
    f_w0.eval().requires_grad_(False)
    positions=np.random.default_rng(seed+1000).permutation(len(pop['X_train']))[:c['cohort_size']]
    X=pop['X_train'][positions];t_0=initial_threshold(f_w0(as_tensor(X)),c['alpha'])
    train=make_cohort(X,f_w0,t_0,c['B'],c['alpha'],pop['actionable_mask'],seed,c['solver'])
    if train.N_minus==0:raise ValueError('No initially rejected training applicants.')
    return dict(pop=pop,pop_id=pop_id,score_id=score_id,f_w0=f_w0,t_0=t_0,train=train,
                train_positions=positions,context_id=stable_id(dict(score_id=score_id,B=c['B'],alpha=c['alpha'],
                            train_ids=pop['indices']['train'][positions].tolist(),solver=c['solver'])))


def target_plan(c,context,seed,split):
    size=len(context['pop'][f'X_{split}']);N=min(size,c['cohort_size'])
    if not 1<=math.floor(c['alpha']*N)<N:raise ValueError(f'{split}: too few observations for integer capacity.')
    count=c['validation_cohorts'] if split=='val' else c['test_cohorts']
    rng=np.random.default_rng(seed+(21000 if split=='val' else 41000))
    # If all pool rows are used, there is only ONE distinct competition, not 50 repetitions.
    plans=[np.arange(size)] if N==size else [rng.choice(size,N,replace=False) for _ in range(count)]
    return [np.asarray(p,dtype=int) for p in plans]


def target_subset_pool(context,c,seed,split,plans):
    used=np.unique(np.concatenate(plans));lookup={p:i for i,p in enumerate(used)}
    local=[np.asarray([lookup[p] for p in positions],int) for positions in plans]
    pop=context['pop'];cohort=make_cohort(pop[f'X_{split}'][used],context['f_w0'],context['t_0'],
        c['B'],c['alpha'],pop['actionable_mask'],seed,c['solver'])
    for p in local:
        if not cohort.X_minus[p].any():raise ValueError(f'{split} cohort has no initial rejections; increase cohort_size.')
    return dict(cohort=cohort,plans=local,positions=used,ids=pop['indices'][split][used],
                groups=pop['groups'][split].iloc[used].reset_index(drop=True))


def target_save_gzip(path,frame):
    def writer(f):
        with gzip.GzipFile(fileobj=f,mode='wb',mtime=0) as z:z.write(frame.to_csv(index=False).encode())
    atomic_write(path,writer)


In [ ]:
# Policy training, multi-cohort validation, and exact allocation.
def target_fit(c,context,seed,training_lambda,select_recipients):
    fit_settings={k:c[k] for k in ['T','tau','eta','Theta_1_radius','Theta_2_radius','snapshot_every']}
    key=stable_id(dict(context=context['context_id'],settings=fit_settings,seed=seed,
                       training_lambda=training_lambda,select_recipients=select_recipients))
    directory=Path(c['results_dir']).resolve()/'cache'/'targets'/target_code_id()/'fits'/key
    directory.mkdir(parents=True,exist_ok=True);checkpoint=directory/'checkpoint.pt'
    surrogates=PolicySurrogates(context['train'].X.shape[1],seed=seed,select_recipients=select_recipients,
        Theta_1_radius=c['Theta_1_radius'],Theta_2_radius=c['Theta_2_radius'])
    start=0;history=[]
    if checkpoint.exists():
        state=load_torch(checkpoint)
        if state['fit_key']!=key:raise ValueError('Training checkpoint mismatch.')
        surrogates.load_state_dict(state['state_dict']);start=state['next_iteration'];history=state['history']
    for j in range(start,c['T']+1):
        surrogates.zero_grad(set_to_none=True)
        objective,outputs=J_hat(context['train'],surrogates,c['tau'],training_lambda,'full')
        if not torch.isfinite(objective):raise FloatingPointError('Nonfinite objective.')
        objective.backward()
        parameters=[surrogates.theta_1,surrogates.theta_2];proposed=[]
        for p,radius in zip(parameters,[c['Theta_1_radius'],c['Theta_2_radius']]):
            if p.grad is None or not torch.isfinite(p.grad).all():raise FloatingPointError('Nonfinite policy gradient.')
            candidate=p.detach()-c['eta']*p.grad;norm=float(candidate.norm())
            proposed.append(candidate*min(1.,radius/norm) if norm else candidate)
        history.append(dict(iteration=j,J_hat=float(objective.detach()),C_hat=float(outputs['C_hat'].detach()),
            V_hat=float(outputs['V_hat'].detach()),G_eta_sq=sum(float(((p.detach()-v)/c['eta']).square().sum()) for p,v in zip(parameters,proposed)),
            capacity_residual=float(outputs['a_i'].detach().sum())-context['train'].k))
        if j%c['snapshot_every']==0 or j==c['T']:
            save_torch(directory/f'iterate_{j:06d}.pt',encode_policy(DefinitePolicy(surrogates)))
        if j<c['T']:
            with torch.no_grad():
                for p,v in zip(parameters,proposed):p.copy_(v)
        if (j+1)%c['checkpoint_every']==0 or j==c['T']:
            save_torch(checkpoint,dict(fit_key=key,next_iteration=j+1,state_dict=surrogates.state_dict(),history=history))
            save_csv(directory/'history.csv',pd.DataFrame(history))
    rows=[]
    for path in sorted(directory.glob('iterate_*.pt')):
        iteration=int(path.stem.split('_')[1])
        rows.append(dict(base_id=stable_id(dict(fit=key,iteration=iteration)),path=str(path.resolve()),
            training_lambda=float(training_lambda),iteration=iteration,select_recipients=select_recipients))
    return rows


def target_slice_cohort(cohort,indices):
    out=copy.copy(cohort)
    for field in ['X','X_aug','s_0','q_bar','q_cap','witness','X_minus','eligible']:
        value=getattr(cohort,field)
        setattr(out,field,None if value is None else value[indices])
    out.N=len(indices);out.N_minus=int(out.X_minus.sum());out.k=math.floor(out.alpha*out.N)
    out.response_cache=OrderedDict();out.solver_audit=[]
    return out


@torch.no_grad()
def target_arrays(cohort,policy,c):
    q,rho=policy(cohort);q=as_tensor(q).expand(cohort.N)
    feasible=target_feasible(cohort,q)
    costs=torch.zeros(cohort.N)
    # Compute the complete response once; reuse it for all recipient cutoffs/cohorts.
    for start in range(0,cohort.N,c['response_batch_size']):
        ids=torch.arange(start,min(start+c['response_batch_size'],cohort.N))
        small=target_slice_cohort(cohort,ids);response=br(small,q[ids])
        actual=small.f_w0(response);expected=torch.where(feasible[ids],q[ids],small.s_0)
        if not torch.allclose(actual,expected,atol=2e-9,rtol=2e-9):raise ArithmeticError('Target-attainment failure.')
        if not torch.equal(response[:,~small.actionable_mask],small.X[:,~small.actionable_mask]):raise AssertionError('Immutable features changed.')
        costs[ids]=cost(small.X,response)
    if (costs>cohort.B+1e-9).any():raise ArithmeticError('Budget exceeded.')
    if isinstance(policy,DefinitePolicy):_,weight=policy.surrogates(cohort)
    else:weight=as_tensor(rho)
    if isinstance(policy,NoRecourse):costs.zero_();feasible=torch.zeros_like(feasible);weight=torch.zeros_like(weight)
    return dict(q=q.numpy(),cost=costs.numpy(),rho_hat=weight.numpy(),feasible=feasible.numpy(),
                is_no_action=isinstance(policy,NoRecourse))


def target_allocation(cohort,arrays,positions,cutoff=.5,force=False):
    p=np.asarray(positions,int);s0=cohort.s_0[p]
    rho=arrays['feasible'][p] & ((arrays['rho_hat'][p]>=cutoff) | force)
    if arrays['is_no_action']:rho[:]=False
    scores=torch.where(torch.as_tensor(rho),as_tensor(arrays['q'][p]),s0)
    y,t1,beta=exact_allocation(scores,math.floor(cohort.alpha*len(p)))
    spent=arrays['cost'][p]*rho;minus=cohort.X_minus[p].numpy()
    if not minus.any():raise ValueError('Cohort without initially rejected applicants.')
    acceptance=y.numpy();C=float(spent[minus].mean());V=float(acceptance[minus].mean())
    return dict(C=C,V=V,rho=rho,cost=spent,p_accept=acceptance,t_1=t1,beta=beta,
                score=scores.numpy(),minus=minus,eligible=cohort.eligible[p].numpy())


def target_validate(c,context,seed,directory,pool,fit_catalog):
    """No test inputs. Rank each family using cohort-mean validation C-lambda*V."""
    directory=Path(directory);catalog=[]
    records=[dict(base_id='no_action',kind='none',path=None,training_lambda=np.nan,iteration=-1,select_recipients=False),
             dict(base_id='original',kind='original',q=context['t_0'],path=None,training_lambda=np.nan,iteration=-1,select_recipients=False)]
    for j,q in enumerate(np.linspace(context['t_0'],1,c['target_points'])):
        records.append(dict(base_id=f'common_{j:03d}',kind='common',q=float(q),path=None,training_lambda=np.nan,iteration=-1,select_recipients=False))
    records += [dict(row,kind='learned') for row in fit_catalog]
    validation_dir=directory/'validation_candidates';validation_dir.mkdir(parents=True,exist_ok=True)
    for record in records:
        base=record['base_id'];path=validation_dir/f'{base}.csv'
        cutoffs=validated_rho_cutoffs(c['rho_cutoffs']) if record['select_recipients'] else [.5]
        for cutoff in cutoffs:
            catalog.append(dict(record,candidate_id=base+f'_cutoff_{cutoff:g}',rho_cutoff=cutoff))
        if path.exists():continue
        policy=NoRecourse() if record['kind']=='none' else (CommonTarget(record['q']) if record['kind']!='learned' else decode_policy(load_torch(record['path'])))
        arrays=target_arrays(pool['cohort'],policy,c);rows=[]
        for cutoff in cutoffs:
            for ci,positions in enumerate(pool['plans']):
                result=target_allocation(pool['cohort'],arrays,positions,cutoff)
                rows.append(dict(candidate_id=base+f'_cutoff_{cutoff:g}',cohort=ci,C=result['C'],V=result['V']))
        save_csv(path,pd.DataFrame(rows))
    catalog=pd.DataFrame(catalog);save_csv(directory/'candidate_catalog.csv',catalog)
    validation=pd.concat([pd.read_csv(validation_dir/f'{record["base_id"]}.csv') for record in records],ignore_index=True)
    means=validation.groupby('candidate_id',sort=False)[['C','V']].mean().reset_index()
    candidates=catalog.merge(means,on='candidate_id',validate='one_to_one')
    save_csv(directory/'validation_candidates.csv',candidates)
    rows=[];forced_validation_cache={}
    for lam in c['lambda_values']:
        for method in MAIN_METHODS:
            none=candidates.kind.eq('none')
            if method=='No recourse':take=none
            elif method=='Original threshold':take=candidates.kind.eq('original')
            elif method=='Common target':take=none|candidates.kind.eq('common')
            else:
                selection=(method=='Personalized targets + selection')
                take=none | (candidates.kind.eq('learned') & candidates.select_recipients.eq(selection))
            choices=candidates.loc[take].copy();choices['J']=choices.C-lam*choices.V
            # Catalog order: no action first, then initial/earlier checkpoint, cutoff 0.5 first.
            chosen=choices.loc[choices.J.idxmin()].to_dict()
            rows.append(dict(chosen,method=method,lambda_=float(lam),validation_C=chosen['C'],
                             validation_V=chosen['V'],validation_J=chosen['J'],force_selection_off=False))
        full=rows[-1].copy();full.update(method=TARGET_FORCED,force_selection_off=True)
        # Evaluate the forced diagnostic but do not optimize its checkpoint or targets again.
        base=full['base_id']
        if base not in forced_validation_cache:
            policy=NoRecourse() if full['kind']=='none' else decode_policy(load_torch(full['path']))
            arrays=target_arrays(pool['cohort'],policy,c)
            forced=[target_allocation(pool['cohort'],arrays,p,force=True) for p in pool['plans']]
            forced_validation_cache[base]=(np.mean([r['C'] for r in forced]),np.mean([r['V'] for r in forced]))
        cost_off,validity_off=forced_validation_cache[base]
        full.update(C=cost_off,V=validity_off,J=cost_off-lam*validity_off,validation_C=cost_off,
                    validation_V=validity_off,validation_J=cost_off-lam*validity_off)
        rows.append(full)
    selections=pd.DataFrame(rows)
    save_csv(directory/'selection.csv',selections)
    for i,row in selections.iterrows():
        if row.kind=='none':payload=encode_policy(NoRecourse())
        elif row.kind in ['common','original']:payload=encode_policy(CommonTarget(float(row.q)))
        else:
            payload=load_torch(row.path);payload['rho_cutoff']=float(row.rho_cutoff)
        save_torch(directory/'selected_policies'/f'{i:03d}.pt',dict(policy=payload,force_selection_off=bool(row.force_selection_off),
            method=row.method,lambda_=float(row.lambda_),candidate_id=row.candidate_id))
    # Choices are sealed before any test responses or allocations are evaluated.
    save_json(directory/'selection_complete.json',dict(selection_hash=target_file_hash(directory/'selection.csv'),n_choices=len(rows)))
    return selections


def target_applicants(c,pool,arrays,row,positions,seed,ci):
    cohort=pool['cohort'];p=np.asarray(positions,int)
    out=target_allocation(cohort,arrays,p,float(row.rho_cutoff),bool(row.force_selection_off))
    minus=out['minus'];rho=out['rho'];q=arrays['q'][p]
    candidate_q=np.where(minus & (not arrays['is_no_action']),q,np.nan)
    # Targets are NOT set to zero for nonrecipients. Recommended q is missing there.
    df=pd.DataFrame(dict(seed=seed,cohort=ci,lambda_=float(row.lambda_),method=row.method,
        applicant_id=pool['ids'][p],initial_score=cohort.s_0[p].numpy(),t_0=cohort.t_0,
        q_bar=cohort.q_bar[p].numpy(),q_cap=cohort.q_cap[p].numpy(),eligible=out['eligible'],
        initially_rejected=minus,candidate_q=candidate_q,q=np.where(rho,q,np.nan),rho=rho,
        rho_hat=arrays['rho_hat'][p],target_feasible=arrays['feasible'][p],
        target_infeasible=minus & (q>cohort.q_cap[p].numpy()) & (not arrays['is_no_action']),
        candidate_cost=np.where(arrays['feasible'][p],arrays['cost'][p],np.nan),
        cost=out['cost'],p_accept=out['p_accept'],final_score=out['score'],t_1=out['t_1'],
        target_minus_cutoff=np.where(rho,q-out['t_1'],np.nan),
        required_score_gain=np.where(rho,q-cohort.s_0[p].numpy(),np.nan),
        min_cost_t0=pool['min_cost_t0'][p]))
    y0,_,_=exact_allocation(cohort.s_0[p],math.floor(c['alpha']*len(p)))
    df['p_accept_no_action']=y0.numpy()
    df['wasted_cost']=df.cost*(1-df.p_accept)
    for name in pool['groups']:df['group_'+name]=pool['groups'].iloc[p][name].to_numpy()
    return df


def target_cohort_metrics(frame):
    r=frame[frame.initially_rejected];e=r[r.eligible];a=frame[~frame.initially_rejected]
    lam=float(frame.lambda_.iloc[0]);C=r.cost.mean();V=r.p_accept.mean();v0=r.p_accept_no_action.mean()
    return dict(C=C,V=V,J=C-lam*V,V_no_action=v0,V_gain=V-v0,J_gain=C-lam*(V-v0),
        wasted_cost=r.wasted_cost.mean(),recommendation_rate=r.rho.mean(),
        selection_rate_eligible=e.rho.mean() if len(e) else np.nan,
        eligible_fraction=r.eligible.mean(),displacement=(1-a.p_accept).mean() if len(a) else np.nan,
        N=len(frame),N_minus=len(r),k=int(round(frame.p_accept.sum())),
        mean_recipient_q=r.loc[r.rho,'q'].mean(),mean_recipient_cost=r.loc[r.rho,'cost'].mean())


def target_paired_targets(frames):
    common=frames['Common target'];rows=[]
    for method in ['Personalized targets, all eligible','Personalized targets + selection',TARGET_FORCED]:
        other=frames[method]
        if not np.array_equal(common.applicant_id,other.applicant_id):raise AssertionError('Applicant pairing failed.')
        both=common.rho & other.rho;minus=common.initially_rejected
        delta=common.loc[both,'q'].to_numpy()-other.loc[both,'q'].to_numpy()
        rows.append(dict(method=method,n_rejected=int(minus.sum()),n_both=int(both.sum()),
            n_common_only=int((common.rho & ~other.rho).sum()),n_personalized_only=int((other.rho & ~common.rho).sum()),
            n_neither=int((minus & ~common.rho & ~other.rho).sum()),
            n_common_higher=int((delta>1e-9).sum()),n_common_lower=int((delta< -1e-9).sum()),
            sum_target_difference=float(delta.sum()),
            sum_cost_difference=float((common.loc[both,'cost']-other.loc[both,'cost']).sum())))
    return rows


In [ ]:
# Aggregation and plotting: cohorts are repeated competitions; seeds are the uncertainty units.
def target_group_metrics(frame,c):
    rows=[]
    for attribute in [s for s in frame if s.startswith('group_')]:
        for group,d in frame.groupby(attribute,dropna=False,sort=True):
            rejected=d[d.initially_rejected];eligible=rejected[rejected.eligible]
            recipients=rejected[rejected.rho];accepted=d[~d.initially_rejected]
            n_unique=rejected.applicant_id.nunique()
            C=rejected.cost.mean();V=rejected.p_accept.mean();V0=rejected.p_accept_no_action.mean()
            rows.append(dict(attribute=attribute.removeprefix('group_'),group=str(group),
                n_occurrences=len(d),n_rejected_occurrences=len(rejected),n_unique_rejected=n_unique,
                n_unique_eligible=eligible.applicant_id.nunique(),n_unique_recipients=recipients.applicant_id.nunique(),
                eligible_fraction=rejected.eligible.mean(),recommendation_rate=rejected.rho.mean(),
                selection_rate_eligible=eligible.rho.mean() if len(eligible) else np.nan,
                C=C,V=V,V_no_action=V0,V_gain=V-V0,J=C-float(frame.lambda_.iloc[0])*V,
                recipient_cost=recipients.cost.mean(),recipient_q=recipients.q.mean(),
                wasted_cost=rejected.wasted_cost.mean(),
                displacement=(1-accepted.p_accept).mean() if len(accepted) else np.nan,
                overall_acceptance=d.p_accept.mean(),
                reliable=n_unique>=c['min_group_rejected'] and str(group)!='Unknown'))
    return rows


def target_summary(frame,keys,metrics):
    rows=[]
    for key,d in frame.groupby(keys,sort=False,dropna=False):
        if not isinstance(key,tuple):key=(key,)
        for metric in metrics:
            x=d[metric].dropna().to_numpy(float);n=len(x)
            mean=float(x.mean()) if n else np.nan
            half=float(student_t.ppf(.975,n-1)*x.std(ddof=1)/np.sqrt(n)) if n>1 else np.nan
            rows.append(dict(zip(keys,key),metric=metric,mean=mean,n_seeds=n,ci_low=mean-half,ci_high=mean+half))
    return pd.DataFrame(rows)


def target_finish(c,directory,seed_directories):
    cohort_tables=[];group_tables=[];pair_tables=[];selections=[];matched_tables=[]
    # Load one seed/lambda at a time; do not retain millions of applicant rows in memory.
    for seed,sd in seed_directories:
        selections.append(pd.read_csv(sd/'selection.csv').assign(seed=seed))
        for lam in c['lambda_values']:
            files=sorted((sd/'test'/f'lambda_{lam:g}').glob('cohort_*.csv.gz'))
            if not files:continue
            all_frames=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
            for (ci,method),d in all_frames.groupby(['cohort','method'],sort=False):
                cohort_tables.append(dict(seed=seed,lambda_=lam,cohort=ci,method=method,**target_cohort_metrics(d)))
            for method,d in all_frames.groupby('method',sort=False):
                group_tables.extend(dict(seed=seed,lambda_=lam,method=method,**v) for v in target_group_metrics(d,c))
            for ci,d in all_frames.groupby('cohort',sort=False):
                frames={m:v.reset_index(drop=True) for m,v in d.groupby('method',sort=False)}
                pair_tables.extend(dict(seed=seed,lambda_=lam,cohort=ci,**v) for v in target_paired_targets(frames))
        matchpath=sd/'matched_selection.csv'
        if matchpath.exists():matched_tables.append(pd.read_csv(matchpath).assign(seed=seed))
    results=pd.DataFrame(cohort_tables);groups=pd.DataFrame(group_tables);paired=pd.DataFrame(pair_tables)
    metrics=['C','V','J','V_gain','J_gain','wasted_cost','recommendation_rate','selection_rate_eligible',
             'eligible_fraction','displacement','mean_recipient_q','mean_recipient_cost']
    seed_means=results.groupby(['seed','lambda_','method'],sort=False)[metrics].mean().reset_index()
    summary=target_summary(seed_means,['lambda_','method'],metrics)
    group_summary=target_summary(groups,['lambda_','method','attribute','group'],
        ['C','V','V_gain','wasted_cost','selection_rate_eligible','eligible_fraction','recipient_cost','recipient_q','displacement'])
    gaps=[]
    for key,d in groups.groupby(['seed','lambda_','method','attribute'],sort=False):
        reliable=d[d.reliable]
        for metric in ['V','V_gain','C','selection_rate_eligible','recipient_cost','wasted_cost']:
            x=reliable[metric].dropna()
            gaps.append(dict(zip(['seed','lambda_','method','attribute'],key),metric=metric,
                gap=float(x.max()-x.min()) if len(x)>=2 else np.nan,n_groups_used=len(x)))
    gaps=pd.DataFrame(gaps)
    # Avoid metric-name collision: target_summary's 'metric' is the summarized variable.
    if not gaps.empty:
        gap_summary=target_summary(gaps.rename(columns={'metric':'gap_metric'}),['lambda_','method','attribute','gap_metric'],['gap'])
    pair_counts=[col for col in paired if col.startswith('n_') or col.startswith('sum_')]
    pair_seed=paired.groupby(['seed','lambda_','method'],sort=False)[pair_counts].sum().reset_index()
    pair_seed['fraction_common_higher']=pair_seed.n_common_higher/pair_seed.n_both.replace(0,np.nan)
    pair_seed['mean_target_difference']=pair_seed.sum_target_difference/pair_seed.n_both.replace(0,np.nan)
    pair_seed['mean_cost_difference']=pair_seed.sum_cost_difference/pair_seed.n_both.replace(0,np.nan)
    pair_summary=target_summary(pair_seed,['lambda_','method'],['fraction_common_higher','mean_target_difference','mean_cost_difference'])
    method_pairs=[]
    for lam,d in seed_means.groupby('lambda_'):
        for comparator in ['Common target','Personalized targets, all eligible',TARGET_FORCED]:
            a=d[d.method=='Personalized targets + selection'].set_index('seed')
            b=d[d.method==comparator].set_index('seed')
            for seed in a.index.intersection(b.index):
                method_pairs.append(dict(seed=seed,lambda_=lam,comparison='Full minus '+comparator,
                    **{metric:float(a.loc[seed,metric]-b.loc[seed,metric]) for metric in ['C','V','J','V_gain','wasted_cost']}))
    method_pairs=pd.DataFrame(method_pairs)
    matched=pd.concat(matched_tables,ignore_index=True) if matched_tables else pd.DataFrame()
    matched_test=[]
    for _,pair in matched.iterrows():
        left=results[(results.seed==pair.seed)&(results.lambda_==pair.full_lambda)&(results.method=='Personalized targets + selection')]
        right=results[(results.seed==pair.seed)&(results.lambda_==pair.comparator_lambda)&(results.method==pair.comparator)]
        joined=left.merge(right,on='cohort',suffixes=('_full','_comparison'),validate='one_to_one')
        for _,r in joined.iterrows():
            matched_test.append(dict(pair.to_dict(),cohort=r.cohort,
                C_full=r.C_full,C_comparison=r.C_comparison,V_full=r.V_full,V_comparison=r.V_comparison,
                delta_C=r.C_full-r.C_comparison,delta_V=r.V_full-r.V_comparison,
                delta_J=(r.C_full-r.C_comparison)-pair.full_lambda*(r.V_full-r.V_comparison)))
    matched_test=pd.DataFrame(matched_test)
    matched_keys=['matching_metric','full_lambda','comparator','comparator_lambda','within_tolerance']
    matched_seed=matched_test.groupby(['seed']+matched_keys,dropna=False)[['delta_C','delta_V','delta_J']].mean().reset_index()
    # Comparator lambda can differ by seed; report that choice in matched_seed, aggregate by the matching rule.
    matched_summary=target_summary(matched_seed,['matching_metric','full_lambda','comparator','within_tolerance'],['delta_C','delta_V','delta_J'])
    tables=dict(results=results,seed_means=seed_means,summary=summary,group_seed_metrics=groups,
        group_summary=group_summary,group_gaps=gaps,group_gap_summary=gap_summary,paired_targets=paired,
        paired_target_seed_metrics=pair_seed,paired_target_summary=pair_summary,
        method_paired_seeds=method_pairs,method_paired_summary=target_summary(method_pairs,['lambda_','comparison'],['C','V','J','V_gain','wasted_cost']),
        selection=pd.concat(selections,ignore_index=True),matched_selection=matched,matched_test=matched_test,
        matched_seed_metrics=matched_seed,matched_summary=matched_summary)
    for name,frame in tables.items():save_csv(directory/f'{name}.csv',frame)
    return tables


def target_match_operating_points(c,selection,directory):
    """Use validation ONLY to pair AVAILABLE operating points; never interpolate a policy."""
    rows=[];full=selection[selection.method=='Personalized targets + selection']
    for _,a in full.iterrows():
        for metric,tolerance in [('C',c['matched_cost_tolerance']),('V',c['matched_validity_tolerance'])]:
            for method in ['Common target','Personalized targets, all eligible']:
                options=selection[selection.method==method].copy()
                options['distance']=abs(options['validation_'+metric]-a['validation_'+metric])
                b=options.loc[options.distance.idxmin()]
                rows.append(dict(matching_metric=metric,full_lambda=float(a.lambda_),comparator=method,
                    comparator_lambda=float(b.lambda_),validation_distance=float(b.distance),
                    within_tolerance=bool(b.distance<=tolerance),tolerance=tolerance,
                    full_candidate=a.candidate_id,comparator_candidate=b.candidate_id))
    save_csv(Path(directory)/'matched_selection.csv',pd.DataFrame(rows))


def target_plot(c,directory,run_id,tables,show=True):
    figure_dir=Path(c['figures_dir']).expanduser().resolve();figure_dir.mkdir(parents=True,exist_ok=True)
    prefix=f'targets-{slug(c["dataset"])}-{c["model"]}-B{c["B"]:g}-a{c["alpha"]:g}-{run_id[:12]}'
    paths=[];title=f'{c["dataset"]} | {c["model"]}'
    def finish(fig,suffix,extra=None):
        fig.suptitle(title);standardize_figure(fig);fig.tight_layout(rect=[0,0,1,.95])
        path=figure_dir/f'{prefix}-{suffix}.png';fig.savefig(path,dpi=180,bbox_inches='tight')
        save_json(path.with_suffix('.json'),dict(config=c,run_directory=str(directory),plot=suffix,**(extra or {})))
        paths.append(str(path))
        if show:display(fig)
        plt.close(fig)
    def legend_or_empty(ax,**kwargs):
        if ax.get_legend_handles_labels()[0]:ax.legend(**kwargs)
        else:ax.text(.5,.5,'No recommendations',ha='center',va='center',transform=ax.transAxes,fontsize=9)
    def ecdf(ax,values,color,label,weights=None,style='-'):
        values=np.asarray(values,float);weights=np.ones(len(values)) if weights is None else np.asarray(weights,float)
        good=np.isfinite(values)&np.isfinite(weights)&(weights>0);values=values[good];weights=weights[good]
        if not len(values):return
        order=np.argsort(values);v=values[order];w=weights[order]
        ax.step(v,np.cumsum(w)/w.sum(),where='post',color=color,label=label,linestyle=style)
    fig,ax=plt.subplots(figsize=(7,4))
    for method in MAIN_METHODS:
        d=tables['seed_means'].query('method == @method').groupby('lambda_')[['C','V']].mean()
        summarized=tables['summary'];sub=summarized[summarized.method==method]
        cstats=sub[sub.metric=='C'].set_index('lambda_').reindex(d.index)
        vstats=sub[sub.metric=='V'].set_index('lambda_').reindex(d.index)
        ax.errorbar(d.C,d.V,xerr=np.nan_to_num((cstats.ci_high-cstats.ci_low).to_numpy()/2),
            yerr=np.nan_to_num((vstats.ci_high-vstats.ci_low).to_numpy()/2),
            marker=METHOD_MARKERS[method],color=TARGET_COLORS[method],label=TARGET_LABELS[method],capsize=2)
    ax.set(xlabel='Mean cost',ylabel='Post-shift validity');legend_or_empty(ax,fontsize=8)
    finish(fig,'cost-validity')
    fig,axes=plt.subplots(1,3,figsize=(13,3.7))
    for ax,metric in zip(axes,['C','V','J']):
        for method in ['Personalized targets, all eligible','Personalized targets + selection',TARGET_FORCED]:
            d=tables['seed_means'].query('method == @method').groupby('lambda_')[metric].mean()
            ax.plot(d.index,d.values,'o-',color=TARGET_COLORS[method],label=TARGET_LABELS[method])
        ax.set(xlabel='lambda',ylabel=metric)
    legend_or_empty(axes[0],fontsize=7);finish(fig,'selection-off-diagnostic')
    # The first PREDECLARED seed shows a genuine single common target, not a mixture across fits.
    seed=c['seeds'][0];sd=directory/f'seed_{seed}'
    for lam in c['lambda_values']:
        files=sorted((sd/'test'/f'lambda_{lam:g}').glob('cohort_*.csv.gz'))
        if not files:continue
        frame=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
        fig,axes=plt.subplots(1,3,figsize=(14,3.7))
        for method in MAIN_METHODS[1:]:
            d=frame[(frame.method==method)&frame.rho]
            if d.empty:
                axes[0].plot([],[],color=TARGET_COLORS[method],label=TARGET_LABELS[method]+' (no recipients)')
                continue
            for ax,metric in zip(axes,['q','required_score_gain','cost']):
                if metric=='q' and method in ['Original threshold','Common target']:
                    ax.axvline(float(d.q.iloc[0]),color=TARGET_COLORS[method],linestyle='--',label=TARGET_LABELS[method])
                else:ecdf(ax,d[metric],TARGET_COLORS[method],TARGET_LABELS[method])
        for ax,label in zip(axes,['Recommended target','Required score increase','Recipient cost']):
            ax.set(xlabel=label,ylabel='Cumulative fraction of recipients',ylim=(0,1.02));ax.grid(alpha=.2)
        legend_or_empty(axes[0],fontsize=7)
        fig.text(.5,.005,f'Predeclared seed {seed}; lambda={lam:g}; repeated-cohort applicant occurrences',ha='center',fontsize=8)
        finish(fig,f'lambda{lam:g}-target-distributions',dict(representative_seed=seed))
        fig,axes=plt.subplots(1,3,figsize=(14,3.7))
        for ax,method in zip(axes,['Common target','Personalized targets, all eligible','Personalized targets + selection']):
            d=frame[(frame.method==method)&frame.rho]
            ecdf(ax,d.target_minus_cutoff,TARGET_COLORS[method],'Acceptance-weighted',d.p_accept)
            ecdf(ax,d.target_minus_cutoff,TARGET_COLORS[method],'Rejection-weighted',1-d.p_accept,style='--')
            ax.axvline(0,color='black',linewidth=.8);ax.set(title=TARGET_LABELS[method],xlabel='Target minus future cutoff',ylabel='Weighted cumulative fraction',ylim=(0,1.02));legend_or_empty(ax,fontsize=8)
        finish(fig,f'lambda{lam:g}-cutoff-margins',dict(representative_seed=seed))
        full=frame[(frame.method=='Personalized targets + selection')&frame.eligible]
        fig,axes=plt.subplots(1,3,figsize=(14,3.7))
        for selected,style in [(True,'-'),(False,'--')]:
            d=full[full.rho==selected]
            ecdf(axes[0],d.candidate_q,TARGET_COLORS['Personalized targets + selection'],
                 'Selected' if selected else 'Unselected',style=style)
        axes[0].set(xlabel='Candidate target',ylabel='Cumulative fraction');legend_or_empty(axes[0])
        for ax,metric,upper in [(axes[1],'initial_score',1.),(axes[2],'min_cost_t0',max(c['B'],1e-8))]:
            bins=np.linspace(0,upper,11);tmp=full.assign(bin=pd.cut(full[metric],bins,include_lowest=True))
            grouped=tmp.groupby('bin',observed=True).rho.agg(['mean','size'])
            ax.plot([v.mid for v in grouped.index],grouped['mean'],'o-',color=TARGET_COLORS['Personalized targets + selection'])
            ax.set(xlabel=metric.replace('_',' '),ylabel='Recommendation rate among eligible',ylim=(0,1.02))
        finish(fig,f'lambda{lam:g}-selection',dict(representative_seed=seed))
        gs=tables['group_summary'];gs=gs[gs.lambda_==lam]
        for attribute in gs.attribute.unique():
            sub=gs[gs.attribute==attribute];groups=list(sub.group.unique());fig,axes=plt.subplots(1,3,figsize=(15,4.6))
            for ax,metric,label in zip(axes,['selection_rate_eligible','V_gain','C'],
                ['Recommendation rate among eligible','Validity gain over no action','Cost per initially rejected applicant']):
                for j,method in enumerate(MAIN_METHODS):
                    d=sub[(sub.method==method)&(sub.metric==metric)].set_index('group').reindex(groups)
                    means=d['mean'].to_numpy(float);low=d.ci_low.to_numpy(float);high=d.ci_high.to_numpy(float)
                    ax.errorbar(np.arange(len(groups))+(j-2)*.13,means,yerr=np.vstack([np.nan_to_num(means-low),np.nan_to_num(high-means)]),
                        fmt=METHOD_MARKERS[method],color=TARGET_COLORS[method],label=TARGET_LABELS[method],capsize=2)
                ax.set(xticks=np.arange(len(groups)),xticklabels=groups,ylabel=label)
                ax.tick_params(axis='x',rotation=35)
            legend_or_empty(axes[0],fontsize=7);finish(fig,f'lambda{lam:g}-groups-{slug(attribute)}')
            if c['group_distribution_plots']:
                raw_attribute='group_'+attribute
                for page,start in enumerate(range(0,len(groups),3)):
                    shown=groups[start:start+3];fig,axes=plt.subplots(2,len(shown),figsize=(4.7*len(shown),6.7),squeeze=False)
                    for j,group in enumerate(shown):
                        subgroup=frame[frame[raw_attribute].astype(str)==str(group)]
                        for method in MAIN_METHODS[1:]:
                            d=subgroup[(subgroup.method==method)&subgroup.rho]
                            if d.empty:
                                axes[0,j].plot([],[],color=TARGET_COLORS[method],label=TARGET_LABELS[method]+' (no recipients)')
                                continue
                            if method in ['Original threshold','Common target']:
                                axes[0,j].axvline(float(d.q.iloc[0]),color=TARGET_COLORS[method],linestyle='--',label=TARGET_LABELS[method])
                            else:ecdf(axes[0,j],d.q,TARGET_COLORS[method],TARGET_LABELS[method])
                            ecdf(axes[1,j],d.cost,TARGET_COLORS[method],TARGET_LABELS[method])
                        n=subgroup[subgroup.initially_rejected].applicant_id.nunique()
                        axes[0,j].set(title=f'{group} (n={n} rejected)',xlabel='Recommended target',ylabel='Recipient cumulative fraction',ylim=(0,1.02))
                        axes[1,j].set(xlabel='Recipient cost',ylabel='Recipient cumulative fraction',ylim=(0,1.02))
                        legend_or_empty(axes[0,j],fontsize=6)
                    finish(fig,f'lambda{lam:g}-group-distributions-{slug(attribute)}-{page+1}',dict(representative_seed=seed))
    save_json(directory/'figures.json',paths)
    return paths


def run_target_experiment(config,mode='run',show=True):
    """run resumes training/evaluation; load regenerates reports; skip performs no work."""
    if mode=='skip':return None
    if mode not in ['run','load','preflight']:raise ValueError('mode must be run, load, preflight, or skip')
    c=copy.deepcopy(config);validate_target_config(c)
    directory,run_id,source=target_run_directory(c)
    if mode=='load':
        if not (directory/'complete.json').exists():raise FileNotFoundError(f'No completed run: {directory}')
        names=load_json(directory/'complete.json')['tables']
        tables={name:pd.read_csv(directory/f'{name}.csv') for name in names}
        figures=target_plot(c,directory,run_id,tables,show)
        return dict(directory=str(directory),tables=tables,figures=figures)
    directory.mkdir(parents=True,exist_ok=True)
    save_json(directory/'config.json',dict(config=c,run_id=run_id,code_id=target_code_id(),source=source))
    if mode=='preflight':
        seed=c['seeds'][0];context=target_context(c,seed,source)
        plans=target_plan(c,context,seed,'val');pool=target_subset_pool(context,c,seed,'val',plans[:1])
        policy=DefinitePolicy(PolicySurrogates(context['train'].X.shape[1],seed=seed))
        arrays=target_arrays(pool['cohort'],policy,c)
        result=target_allocation(pool['cohort'],arrays,pool['plans'][0])
        report=dict(dataset=c['dataset'],model=c['model'],processed_features=context['train'].X.shape[1],
            fitting_n=len(context['pop']['X_fit']),training_n=context['train'].N,
            val_pool_n=len(context['pop']['X_val']),test_pool_n=len(context['pop']['X_test']),
            validation_cohort_n=len(pool['plans'][0]),group_notes=context['pop']['group_notes'],
            total_training_fits=len(c['seeds'])*len(c['train_lambda_values'])*2,validation_C=result['C'],validation_V=result['V'])
        save_json(directory/'preflight.json',report);display(pd.Series(report));return dict(directory=str(directory),preflight=report)
    completed=[];failures=[]
    for seed in c['seeds']:
        sd=directory/f'seed_{seed}';sd.mkdir(parents=True,exist_ok=True)
        if (sd/'failed.json').exists() and not c['retry_failed']:
            failures.append(dict(seed=seed,error='Previous failure; inspect failed.json and set retry_failed=True.'));continue
        try:
            print(f'{c["dataset"]} | {c["model"]} | seed {seed}',flush=True)
            context=target_context(c,seed,source)
            val_plans=target_plan(c,context,seed,'val');test_plans=target_plan(c,context,seed,'test')
            save_json(sd/'cohort_plan.json',dict(pop_id=context['pop_id'],score_id=context['score_id'],t_0=context['t_0'],
                fitting_n=len(context['pop']['X_fit']),processed_features=context['train'].X.shape[1],
                train_ids=context['pop']['indices']['train'][context['train_positions']].tolist(),
                validation_ids=[context['pop']['indices']['val'][p].tolist() for p in val_plans],
                test_ids=[context['pop']['indices']['test'][p].tolist() for p in test_plans],
                group_notes=context['pop']['group_notes']))
            if (sd/'complete.json').exists():completed.append((seed,sd));continue
            if (sd/'selection_complete.json').exists():
                seal=load_json(sd/'selection_complete.json')
                if seal['selection_hash']!=target_file_hash(sd/'selection.csv'):raise RuntimeError('Saved selection was modified.')
                selection=pd.read_csv(sd/'selection.csv')
            else:
                catalog=[]
                for lam in c['train_lambda_values']:
                    for select in [False,True]:
                        print(f'  training/reusing lambda={lam:g}, selection={select}',flush=True)
                        catalog+=target_fit(c,context,seed,lam,select)
                val_pool=target_subset_pool(context,c,seed,'val',val_plans)
                print('  selecting checkpoints and cutoffs on validation cohorts',flush=True)
                selection=target_validate(c,context,seed,sd,val_pool,catalog)
                del val_pool
            target_match_operating_points(c,selection,sd)
            # No test predictions/recourse are computed until the validation choices are sealed.
            pool=target_subset_pool(context,c,seed,'test',test_plans)
            initial=target_arrays(pool['cohort'],CommonTarget(context['t_0']),c)
            pool['min_cost_t0']=np.where(initial['feasible'],initial['cost'],np.nan)
            array_cache={}
            for i,row in selection.iterrows():
                key=row.base_id
                if key in array_cache:continue
                payload=load_torch(sd/'selected_policies'/f'{i:03d}.pt')['policy']
                array_cache[key]=target_arrays(pool['cohort'],decode_policy(payload),c)
            print(f'  evaluating {len(test_plans)} distinct test cohorts',flush=True)
            for lam in c['lambda_values']:
                for ci,positions in enumerate(pool['plans']):
                    path=sd/'test'/f'lambda_{lam:g}'/f'cohort_{ci:04d}.csv.gz'
                    if path.exists():continue
                    rows=[target_applicants(c,pool,array_cache[row.base_id],row,positions,seed,ci)
                          for _,row in selection[selection.lambda_==lam].iterrows()]
                    target_save_gzip(path,pd.concat(rows,ignore_index=True))
            save_json(sd/'complete.json',dict(seed=seed,test_cohorts=len(test_plans),validation_cohorts=len(val_plans)))
            if (sd/'failed.json').exists():(sd/'failed.json').unlink()
            completed.append((seed,sd))
        except Exception as exc:
            error=dict(seed=seed,error=repr(exc),traceback=traceback.format_exc())
            save_json(sd/'failed.json',error);failures.append(error);print(f'  FAILED: {exc}',flush=True)
    save_json(directory/'failures.json',failures)
    if failures:
        raise RuntimeError(f'{len(failures)} seed(s) failed; successful work is saved in {directory}. Inspect failures.json; no complete aggregate is claimed.')
    tables=target_finish(c,directory,completed)
    figures=target_plot(c,directory,run_id,tables,show)
    save_json(directory/'complete.json',dict(tables=list(tables),seeds=c['seeds'],completed_at=utc_now()))
    print(f'Saved: {directory}',flush=True)
    return dict(directory=str(directory),tables=tables,figures=figures)


In [ ]:
TARGET_CODE_NAMES=tuple(sorted(name for name,value in list(globals().items())
    if (inspect.isfunction(value) or inspect.isclass(value)) and getattr(value,'__module__',None)==__name__
    and (name.startswith('target_') or name in IMPLEMENTATION_NAMES or name=='run_target_experiment')))


## Cost-matched personalized follow-up (saved target candidates only)


In [ ]:
def cost_matched_config(dataset,model,**overrides):
    c=dict(dataset=dataset,model=model,seeds=[42,43,44,45,46],lambda_values=[1.,3.,10.],
        rules=['budget','band'],tolerance=.01,source_run_dir=None,source_run_id=None,
        source_cache_root=None,source_results_dir='./Results',results_dir='./Results',figures_dir='./figures')
    for k,v in overrides.items():
        if k not in c:raise KeyError(k)
        c[k]=copy.deepcopy(v)
    return c


def cost_matched_select(candidates,common_cost,rule,tolerance):
    """Maximize validation validity subject to the common policy's validation cost."""
    if rule not in ['budget','band'] or tolerance<0:raise ValueError('Invalid cost-matching rule.')
    take=(candidates.C<=common_cost+1e-12) if rule=='budget' else ((candidates.C-common_cost).abs()<=tolerance+1e-12)
    feasible=candidates.loc[take].copy()
    if feasible.empty:return None
    feasible['_order']=np.arange(len(feasible))
    return feasible.sort_values(['V','C','_order'],ascending=[False,True,True],kind='stable').iloc[0].drop('_order').to_dict()


def cost_matched_source(c):
    root=Path(c['source_results_dir']).expanduser().resolve()
    paths=[Path(c['source_run_dir']).expanduser().resolve()/'config.json'] if c['source_run_dir'] else list(
        (root/'experiments'/'targets'/slug(c['dataset'])/c['model']).glob('*/config.json'))
    matches=[]
    for p in paths:
        if not p.exists():continue
        meta=load_json(p);old=meta.get('config',{})
        if ((old.get('dataset'),old.get('model'))!=(c['dataset'],c['model']) or
            (c['source_run_id'] and meta.get('run_id')!=c['source_run_id']) or
            not set(c['seeds'])<=set(old.get('seeds',[])) or
            not set(c['lambda_values'])<=set(old.get('lambda_values',[]))):continue
        if (p.parent/'complete.json').exists():matches.append((p.parent,meta))
    if len(matches)!=1:raise ValueError(f'Expected one completed target study, found {len(matches)}. Set source_run_dir or source_run_id; no training will be started.')
    return matches[0]


def cost_matched_cached_context(c,meta,plan,seed):
    """Restore exact population/scorer bytes; NEVER call a fitter or data loader."""
    old=meta['config'];root=Path(c['source_cache_root']).expanduser().resolve() if c['source_cache_root'] else Path(c['source_results_dir']).expanduser().resolve()/'cache'
    base=root/'targets'/meta['code_id']
    pop_path=base/'populations'/f"{plan['pop_id']}.pkl"
    score_path=base/'scorers'/f"{plan['score_id']}.pt"
    for p in [pop_path,score_path]:
        if not p.is_file():raise FileNotFoundError(f'Missing original cache: {p}. Restore Results/cache; no refitting is performed.')
    pop=load_pickle(pop_path);state=load_torch(score_path)
    f=FixedScorer(state['w_0']) if old['model']=='affine' else QuadraticScorer(state['w_0'],state['h'])
    f.eval().requires_grad_(False)
    positions=np.random.default_rng(seed+1000).permutation(len(pop['X_train']))[:old['cohort_size']]
    if pop['indices']['train'][positions].tolist()!=plan['train_ids']:raise ValueError('Training identities differ from the saved plan.')
    ref=make_cohort(pop['X_train'][positions],f,float(plan['t_0']),old['B'],old['alpha'],pop['actionable_mask'],seed,old['solver'])
    context=dict(pop=pop,f_w0=f,t_0=plan['t_0'],train=ref,train_positions=positions)
    lookup={int(v):i for i,v in enumerate(pop['indices']['test'])}
    plans=[np.asarray([lookup[int(v)] for v in ids],dtype=int) for ids in plan['test_ids']]
    # Split-disjointness check uses applicant identities, not test outcomes.
    forbidden=set(plan['train_ids'])|set(v for ids in plan['validation_ids'] for v in ids)
    if forbidden & set(v for ids in plan['test_ids'] for v in ids):raise AssertionError('Validation/test identity overlap.')
    return context,plans,base


def cost_matched_payload(row,seed_dir,cache_base):
    if row['kind']=='none':return encode_policy(NoRecourse())
    if row['kind'] in ['common','original']:return encode_policy(CommonTarget(float(row['q'])))
    original=Path(str(row['path']))
    path=original if original.is_file() else cache_base/'fits'/original.parent.name/original.name
    if not path.is_file():raise FileNotFoundError(f'Missing candidate checkpoint: {path}')
    payload=load_torch(path);payload=copy.deepcopy(payload);payload['rho_cutoff']=float(row['rho_cutoff'])
    return payload


def run_cost_matched(config,*,mode='run',show=True):
    if mode=='skip':return None
    if mode not in ['run','load','preflight']:raise ValueError('Unknown mode.')
    c=copy.deepcopy(config);source,meta=cost_matched_source(c)
    fingerprints={str(s):{name:target_file_hash(source/f'seed_{s}'/name) for name in
        ['selection.csv','validation_candidates.csv','cohort_plan.json']} for s in c['seeds']}
    settings={k:v for k,v in c.items() if k not in ['figures_dir','results_dir','source_cache_root','source_run_dir','source_results_dir']}
    run_id=stable_id(dict(protocol='validation-cost-matched-v1',settings=settings,source=meta['run_id'],fingerprints=fingerprints,
                          helper=_code_signature(cost_matched_select.__code__)))
    directory=Path(c['results_dir']).resolve()/'cost_matched_targets'/slug(c['dataset'])/c['model']/run_id
    if mode=='load':
        if not (directory/'complete.json').exists():raise FileNotFoundError(directory)
        return dict(directory=str(directory),summary=pd.read_csv(directory/'summary.csv'))
    choices=[]
    for seed in c['seeds']:
        sd=source/f'seed_{seed}';sel=pd.read_csv(sd/'selection.csv');catalog=pd.read_csv(sd/'validation_candidates.csv')
        # A learned target/selection policy and no action; all-eligible fits are excluded.
        eligible=catalog[(catalog.kind=='none')|((catalog.kind=='learned')&catalog.select_recipients.astype(bool))]
        for lam in c['lambda_values']:
            base=sel[(sel.method=='Common target')&np.isclose(sel.lambda_,lam)]
            if len(base)!=1:raise ValueError('Missing/duplicate common-target validation choice.')
            common=base.iloc[0];target_cost=float(common.validation_C)
            for rule in c['rules']:
                chosen=cost_matched_select(eligible,target_cost,rule,c['tolerance'])
                choices.append(dict(seed=seed,lambda_=lam,rule=rule,matched=chosen is not None,
                    common_candidate_id=common.candidate_id,common_validation_C=target_cost,
                    common_validation_V=float(common.validation_V),
                    candidate_id=None if chosen is None else chosen['candidate_id'],
                    validation_C=None if chosen is None else float(chosen['C']),
                    validation_V=None if chosen is None else float(chosen['V'])))
    choice_frame=pd.DataFrame(choices)
    # This file seals all choices before any test predictions are read or computed.
    save_csv(directory/'selection.csv',choice_frame)
    save_json(directory/'config.json',dict(config=c,run_id=run_id,source_run_id=meta['run_id'],fingerprints=fingerprints,
        protocol='Max validation V at common validation cost; deterministic candidates; no interpolation.'))
    if mode=='preflight':
        display(choice_frame);return dict(directory=str(directory),selection=choice_frame)
    frames=[]
    for seed in c['seeds']:
        sd=source/f'seed_{seed}';plan=load_json(sd/'cohort_plan.json');catalog=pd.read_csv(sd/'validation_candidates.csv').set_index('candidate_id')
        context,plans,cache=cost_matched_cached_context(c,meta,plan,seed)
        pool=target_subset_pool(context,meta['config'],seed,'test',plans)
        own=choice_frame[(choice_frame.seed==seed)&choice_frame.matched]
        array_cache={}
        for candidate_id in set(own.candidate_id)|set(own.common_candidate_id):
            row=catalog.loc[candidate_id].to_dict();payload=cost_matched_payload(row,sd,cache)
            array_cache[candidate_id]=target_arrays(pool['cohort'],decode_policy(payload),meta['config'])
        for choice in own.to_dict('records'):
            for ci,positions in enumerate(pool['plans']):
                metrics={}
                for label,key in [('common',choice['common_candidate_id']),('personalized',choice['candidate_id'])]:
                    result=target_allocation(pool['cohort'],array_cache[key],positions,float(catalog.loc[key,'rho_cutoff']))
                    metrics.update({label+'_C':result['C'],label+'_V':result['V']})
                frames.append(dict(seed=seed,lambda_=choice['lambda_'],rule=choice['rule'],cohort_id=ci,**metrics,
                    delta_C=metrics['personalized_C']-metrics['common_C'],delta_V=metrics['personalized_V']-metrics['common_V']))
        del array_cache,pool,context
    frame=pd.DataFrame(frames,columns=['seed','lambda_','rule','cohort_id','common_C','common_V','personalized_C','personalized_V','delta_C','delta_V'])
    save_csv(directory/'results.csv',frame)
    metrics=['common_C','common_V','personalized_C','personalized_V','delta_C','delta_V']
    seeds=frame.groupby(['seed','lambda_','rule'],as_index=False)[metrics].mean() if len(frame) else pd.DataFrame(columns=['seed','lambda_','rule']+metrics)
    summary=target_summary(seeds,['lambda_','rule'],metrics) if len(seeds) else pd.DataFrame(columns=['lambda_','rule','metric','mean','n_seeds','ci_low','ci_high'])
    save_csv(directory/'seed_means.csv',seeds);save_csv(directory/'summary.csv',summary)
    save_json(directory/'complete.json',dict(run_id=run_id,selected=len(choice_frame),matched=int(choice_frame.matched.sum()),test_rows=len(frame)))
    if len(summary):
        fig,axes=plt.subplots(1,2,figsize=(10,4))
        for rule in c['rules']:
            for ax,metric in zip(axes,['delta_C','delta_V']):
                d=summary[(summary.rule==rule)&(summary.metric==metric)]
                ax.errorbar(d.lambda_,d['mean'],yerr=np.nan_to_num((d.ci_high-d.ci_low)/2),marker='o',label=rule)
                ax.axhline(0,color='black',lw=.7);ax.set(xlabel='lambda',ylabel='Personalized minus common: '+metric[6:]);ax.legend()
        fig.suptitle(f"{c['dataset']} | {c['model']}");standardize_figure(fig);fig.tight_layout()
        path=Path(c['figures_dir']).resolve()/f"experiment-18-dataset-{slug(c['dataset'])}-model-{c['model']}-parameters-{run_id}.png"
        atomic_write(path,lambda h:fig.savefig(h,format='png',dpi=180,bbox_inches='tight'))
        if show:plt.show()
        plt.close(fig)
    display(choice_frame);display(summary)
    return dict(directory=str(directory),selection=choice_frame,results=frame,summary=summary)


## Configure the studies
All configurations below are independent copies. Change a pair's dictionary without changing the other experiments. Selection lists do not run anything until their execution cell is reached.


In [ ]:
def paper_config(experiment,dataset,model):
    """Independent fully resolved settings; experiment grids never modify another study."""
    c=experiment_config(experiment,dataset,model=model,seeds=PAPER_SEEDS.copy(),
        alpha=.4,B=.75,lambda_=30.,tau=.01,eta=.2,T=500,cohort_size=256,fit_cap=5000,synthetic_n=6000,
        results_dir=RESULTS_DIR,figures_dir=FIGURES_DIR,gmsc_path=GMSC_PATH,
        baselines=dict(target_points=30),policy=dict(validation_every=5,rho_cutoffs=RHO_CUTOFFS.copy()))
    grids={1:[3.,30.],2:PAPER_LAMBDAS,3:[3.,10.,30.],4:[3.,10.,30.],5:[30.],6:[3.,10.,30.],7:[30.],8:[3.,30.]}
    c['lambda_values']=list(grids.get(experiment,[.3,1.,3.,10.,30.]))
    if experiment==5:c['eta']=.05
    if experiment==9:
        c.update(lambda_=3.,lambda_values=[3.],tau_values=[.001,.005,.01,.02,.05],eta_values=[.05,.2])
    if experiment==10:
        c.update(tau_values=[.005,.01],eta_values=[.05,.2])
    return c


def independent_configs(factory,datasets=DATASETS):
    return {dataset:{model:factory(dataset,model) for model in MODELS} for dataset in datasets}


study_configs={n:independent_configs(lambda d,m,n=n:paper_config(n,d,m)) for n in range(1,11)}
# Aliases keep familiar configuration names available without copied settings drifting apart.
for _n in range(1,11):globals()[f'experiment_{_n}_configs']=study_configs[_n]
study_configs['2a']=independent_configs(lambda d,m:experiment_2_diagnostic_config(d,m,
    seeds=PAPER_SEEDS.copy(),validation_cohorts=20,test_cohorts=50,rho_cutoffs=RHO_CUTOFFS.copy(),
    source_results_dir=RESULTS_DIR,results_dir=RESULTS_DIR,figures_dir=FIGURES_DIR))
study_configs['2b']=independent_configs(lambda d,m:experiment_2_calibration_config(d,m,
    seeds=PAPER_SEEDS.copy(),validation_cohorts=20,test_cohorts=50,rho_cutoffs=RHO_CUTOFFS.copy(),
    gamma_values=[0.,.25,.5,.75,.9,1.],source_results_dir=RESULTS_DIR,results_dir=RESULTS_DIR,figures_dir=FIGURES_DIR))
study_configs[11]=independent_configs(lambda d,m:refinement_config(study_configs[10][d][m],
    seeds=PAPER_SEEDS.copy(),lambda_values=[3.,10.,30.],grid_levels=[(30,1),(100,1),(300,1)]))
study_configs[12]=independent_configs(lambda d,m:transfer_config(study_configs[11][d][m],
    seeds=PAPER_SEEDS.copy(),lambda_values=[3.,10.,30.],validation_counts=[1,5,20],test_cohorts=50,cohort_size=256),
    datasets=['Synthetic nonlinear','Synthetic curved'])
study_configs[13]=independent_configs(lambda d,m:selection_config(study_configs[12][d][m],
    validation_count=20,test_cohorts=50),datasets=['Synthetic nonlinear','Synthetic curved'])
study_configs[14]=independent_configs(lambda d,m:sensitivity_config(study_configs[10][d][m],
    axis='B',values=[.25,.75,1.5],seeds=PAPER_SEEDS.copy(),lambda_values=[3.,10.,30.],
    tau_values=[.01],eta=.2,grid_levels=[(30,1)]))
study_configs[15]=independent_configs(lambda d,m:sensitivity_config(study_configs[10][d][m],
    axis='alpha',values=[.2,.4,.6],seeds=PAPER_SEEDS.copy(),lambda_values=[3.,10.,30.],
    tau_values=[.01],eta=.2,grid_levels=[(30,1)]))
study_configs[16]=independent_configs(lambda d,m:ablation_config(d,m,seeds=PAPER_SEEDS.copy(),
    lambda_values=[3.,30.],B=.75,alpha=.4,tau=.01,eta=.2,T=500,cohort_size=256))
study_configs[17]=independent_configs(lambda d,m:target_config(d,m,seeds=PAPER_SEEDS.copy(),
    alpha=.4,B=.75,tau=.01,eta=.2,T=500,train_lambda_values=[1.,3.,10.],lambda_values=[1.,3.,10.],
    validation_cohorts=20,test_cohorts=50,snapshot_every=50,rho_cutoffs=RHO_CUTOFFS.copy(),
    results_dir=RESULTS_DIR,figures_dir=FIGURES_DIR,gmsc_path=GMSC_PATH))
study_configs[18]=independent_configs(lambda d,m:cost_matched_config(d,m,
    source_results_dir=RESULTS_DIR,results_dir=RESULTS_DIR,figures_dir=FIGURES_DIR))
for _n in range(11,19):globals()[f'experiment_{_n}_configs']=study_configs[_n]
target_configs=study_configs[17]
# Every section's list is independent. Unsupported real-data entries are not manufactured for 12/13.
study_pairs={key:[(d,m) for d,m in default_selected_pairs if d in configs and m in configs[d]]
             for key,configs in study_configs.items()}
for _n in [12,13]:
    if not study_pairs[_n]:study_pairs[_n]=[('Synthetic curved','quadratic')]
study_modes={key:RUN_MODE for key in study_configs}
reports={}
# Edit exact pairs locally, e.g.:
# study_pairs[2] = [('German Credit','affine'), ('German Credit','quadratic')]
# study_pairs[17] = ALL_PAIRS.copy()
# study_configs[3]['Adult']['quadratic']['B_values'] = [.25,.5,.75,1.,1.5]
# study_configs['2a']['Adult']['affine']['source_config_id'] = '<saved ID>'
# study_configs[18]['Adult']['affine']['source_run_dir'] = './Results/experiments/targets/Adult/affine/<ID>'
# study_modes[2] = 'load'


def validate_study_settings():
    for key,configs in study_configs.items():
        for d,models in configs.items():
            for m,c in models.items():
                if key in range(1,11):validate_config(c)
                elif key in [12]:transfer_validate_config(c)
                elif key==13:selection_validate_config(c)
                elif key in [14,15]:sensitivity_validate(c)
                elif key==16:ablation_validate(c)
                elif key==17:validate_target_config(c)
                if (c.get('dataset'),c.get('model'))!=(d,m):raise ValueError((key,d,m))
    return True


def run_study(study):
    if study not in experiments_to_run:
        print(f'Study {study}: disabled; add it to experiments_to_run to execute.');return []
    if study not in study_configs:raise KeyError(study)
    mode=study_modes[study];pairs=study_pairs[study]
    if mode=='skip':return []
    if not pairs:raise ValueError(f'Study {study}: select at least one supported dataset/model pair.')
    if len(pairs)!=len(set(pairs)):raise ValueError('Duplicate selected pairs.')
    configs=select_config_pairs(study_configs[study],pairs)
    if study in range(1,11):
        expanded=[]
        for source in configs:
            if study in [1,3,4,5,6,8]:
                for lam in source['lambda_values']:
                    c=copy.deepcopy(source);c['lambda_']=float(lam);c['lambda_values']=[float(lam)];expanded.append(c)
            else:expanded.append(copy.deepcopy(source))
        return execute_experiment(expanded,mode=mode,show=SHOW_FIGURES)
    runners={'2a':run_experiment_2_diagnostics,'2b':run_experiment_2_calibration,
        11:refinement_run,12:transfer_run,13:selection_run,14:sensitivity_run,15:sensitivity_run,
        16:ablation_run,17:run_target_experiment,18:run_cost_matched}
    rows=[];failures=[]
    for c in configs:
        try:
            rows.append(runners[study](c,mode=mode,show=SHOW_FIGURES))
        except Exception as exc:
            failure=dict(study=study,dataset=c['dataset'],model=c['model'],error=repr(exc),traceback=traceback.format_exc())
            failures.append(failure);print(f"FAILED {c['dataset']} / {c['model']}: {exc}",flush=True)
            failure_id=stable_id(dict(study=study,config=c))
            save_json(Path(c['results_dir'])/'failures'/f'study-{study}-{failure_id}.json',failure)
    if failures:display(pd.DataFrame(failures)[['study','dataset','model','error']])
    return rows


assert validate_study_settings()
print('Configurations validated. Active studies:',experiments_to_run)
# The focused ablation constructor has independent base settings; route its outputs too.
for _models in study_configs[16].values():
    for _c in _models.values():
        _c.update(results_dir=RESULTS_DIR,figures_dir=FIGURES_DIR)
        _c['base_config'].update(results_dir=RESULTS_DIR,figures_dir=FIGURES_DIR,gmsc_path=GMSC_PATH)


## 1. Convergence
Five-seed mean training objective and squared projected-gradient mapping against iterations; $\lambda\in\{3,30\}$ and $\eta\in\{0.05,0.2\}$. Capacity residuals remain in tables, not the convergence panels.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[1]; selected pairs: study_pairs[1].
reports[1] = run_study(1)


## 2. Cost–validity trade-offs
The five paper methods, 14 lambda values, and 30 common-target scores. Checkpoints and seven recommendation cutoffs use validation only. Plot achieved within-method nondominated mean points; this is not a certified global Pareto frontier.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[2]; selected pairs: study_pairs[2].
reports[2] = run_study(2)


## 2a. Frozen-candidate follow-up
Single versus multiple validation cohorts, pooled versus matching training-lambda candidates, and retrospective cost decomposition. Requires completed experiment 2 with its original caches; no model or policy fitting. Real-data bootstrap cohorts are repeated draws from empirical held-out pools. The retrospective cost-to-cutoff diagnostic is not an implementable policy guarantee.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs['2a']; selected pairs: study_pairs['2a'].
reports['2a'] = run_study('2a')


## 2b. Joint calibration
Jointly calibrate target contraction and recommendation cutoff using saved personalized candidates. Gamma values `[0,.25,.5,.75,.9,1]`; shared validation selects, paired held-out cohorts evaluate. The original frozen candidates remain available at gamma=1.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs['2b']; selected pairs: study_pairs['2b'].
reports['2b'] = run_study('2b')


## 3. Budget sensitivity
Budgets `[.25,.75,1.5]` at lambda `[3,10,30]`; other paper defaults fixed. Each lambda produces a separate, identified run.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[3]; selected pairs: study_pairs[3].
reports[3] = run_study(3)


## 4. Admission-rate sensitivity
Admission rates `[.2,.4,.6]` at lambda `[3,10,30]`. Recompute the reference initial cutoff for each admission rate; hold it fixed on that setting's validation/test cohorts. Report eligibility and the capacity validity bound alongside cost and validity.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[4]; selected pairs: study_pairs[4].
reports[4] = run_study(4)


## 5. Temperature sensitivity
Tau `[.005,.01,.02,.05]`, lambda=30, eta=.05. Compare hard-policy outcomes and the smooth/exact fractional-mixture discrepancy.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[5]; selected pairs: study_pairs[5].
reports[5] = run_study(5)


## 6. Selection and competition ablations
The five paper methods plus fixed-initial-threshold and detached-threshold diagnostics. Lambda `[3,10,30]`; the same validation cutoff grid is used for the learned selection methods.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[6]; selected pairs: study_pairs[6].
reports[6] = run_study(6)


## 7. Solver and predictive-model audit
Target attainment, budget/immutable constraints, KKT/gradient diagnostics, and scorer predictive quality. This is an audit, not another cost–validity sweep.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[7]; selected pairs: study_pairs[7].
reports[7] = run_study(7)


## 8. Partial-adoption stress test
Keep the validation-selected policies fixed, vary adoption probability `[0,.25,.5,.75,1]`, and recompute allocation. Lambda `[3,30]`; 200 adoption draws. No retraining for adoption levels.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[8]; selected pairs: study_pairs[8].
reports[8] = run_study(8)


## 9. Focused temperature and step-size audit
The five current methods at lambda=3. This follow-up additionally includes tau=.001 as an explicitly sharper stress setting; eta `[.05,.2]`. Diagnostics separate acceptance smoothing, fractional recommendation weights, and binary rounding. The primary learned method already selects its cutoff on validation.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[9]; selected pairs: study_pairs[9].
reports[9] = run_study(9)


## 10. Validation-selected focused trade-offs
Lambda `[.3,1,3,10,30]`, tau `[.005,.01]`, eta `[.05,.2]`. Choose settings by validation objective before comparing paired test outcomes. All five current methods are retained; source IDs for downstream studies are saved.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[10]; selected pairs: study_pairs[10].
reports[10] = run_study(10)


## 11. Common-target grid resolution
Read saved experiment 10; test nested grids adding 30, 100, and 300 target scores. Every feasible rejected applicant receives the common target. The second grid coordinate must remain 1: there is no recipient-cost grid. No new model or personalized-policy fits.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[11]; selected pairs: study_pairs[11].
reports[11] = run_study(11)


## 12. Independent synthetic cohort transfer
For synthetic populations, compare 1/5/20 fresh validation cohorts and 50 fresh test cohorts under a frozen scorer and reference cutoff. Requires completed studies 10–11. This protocol is deliberately unavailable for real datasets because it draws new observations from the known synthetic generator.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[12]; selected pairs: study_pairs[12].
reports[12] = run_study(12)


## 13. Shared-validation policy-family selection
Use the same saved validation cohorts to compare common targets, personalized policies, and no action; then evaluate the selected policy on 50 new synthetic cohorts. Requires 10–12. Family selection never uses test outcomes.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[13]; selected pairs: study_pairs[13].
reports[13] = run_study(13)


## 14. Multi-cohort budget sensitivity
Budgets `[.25,.75,1.5]`, lambda `[3,10,30]`, tau=.01, eta=.2. Use 20 fresh validation and 50 fresh test cohorts for synthetic datasets, one held-out cohort per split for real data. Common target uses 30 scores. All dataset/model configurations are available.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[14]; selected pairs: study_pairs[14].
reports[14] = run_study(14)


## 15. Multi-cohort admission sensitivity
Admission rates `[.2,.4,.6]` under the same protocol as study 14. The shared axis settings reuse compatible scientific caches; changed parameters get distinct result identities.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[15]; selected pairs: study_pairs[15].
reports[15] = run_study(15)


## 16. Focused multi-cohort ablations
Compare full, all-eligible, and detached-threshold training with matched initializations. Choose checkpoint and recommendation cutoff on shared validation cohorts. Lambda `[3,30]`, 20/50 fresh synthetic validation/test cohorts; one held-out cohort for real data. Detached update residuals are reported separately from stationarity of the full objective.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[16]; selected pairs: study_pairs[16].
reports[16] = run_study(16)


## 17. Targets, selection, and group outcomes
All six datasets and both scorers; training/evaluation lambda `[1,3,10]`; 20 validation and 50 test resampled cohorts where pool size permits (German Credit normally has one distinct cohort). Include recipient target/effort distributions, paired targets among shared recipients, selection-off with the same learned targets, and raw-feature group audits. Five seeds summarize uncertainty. Selection among checkpoints is every 50 iterations in this saved protocol.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[17]; selected pairs: study_pairs[17].
reports[17] = run_study(17)


## 18. Personalized validity at the common-target validation cost
Reuse all saved study-17 personalized candidates and their validation metrics. Choose maximum validity either under the common policy's validation cost (budget rule) or within ±.01 of it (band rule); no interpolation and no nearest-cost fallback. Seal choices before evaluating paired test cohorts. Missing original caches are reported instead of refitting. Test cost equality is not guaranteed by validation cost matching.


In [ ]:
# Edit this study's pair list or configuration before running this cell.
# Configuration: study_configs[18]; selected pairs: study_pairs[18].
reports[18] = run_study(18)


## Saved diagnostics without retraining
Enable `"diagnostics"` in `experiments_to_run`. Filter by dataset, model, experiment, or saved configuration ID. The diagnostic reads original caches and preserves historical labels.


In [ ]:
saved_diagnostic_selection = dict(experiment=None, dataset="German Credit", model="affine",
    config_id=None, seeds=None, lambda_=None, B=None, alpha=None)
if "diagnostics" in experiments_to_run:
    reports["diagnostics"] = diagnose_saved_runs(RESULTS_DIR,figures_dir=FIGURES_DIR,
        show=SHOW_FIGURES,**saved_diagnostic_selection)


## Inspect or export saved results
`inventory_saved_studies` lists saved studies, including historical configurations. A changed implementation receives a new identity; use the explicit source-ID fields for read-only follow-ups. No source experiment is trained implicitly.


In [ ]:
# Read-only examples (uncomment as needed).
# display(inventory_saved_studies(RESULTS_DIR))
# display(collect_saved_results(RESULTS_DIR))
# export_saved_runs(RESULTS_DIR, figures_dir=FIGURES_DIR)
